# 02 — Data Quality Assessment & Cleaning
## Objective

Assess the quality and consistency of the UCI Online Retail II dataset,
investigate potential data issues, and establish evidence-based cleaning rules.

## Scope

- Preserve the original transaction records.
- Identify duplicates and missing values.
- Investigate invalid and unusual values.
- Classify transaction types.
- Develop and validate cleaning rules.
- Prepare the dataset for downstream analytics and machine learning.

## Data Source

UCI Online Retail II — combined workbook worksheets.

In [26]:
import pandas as pd
import numpy as np
from pathlib import Path

In [4]:
import pandas as pd
from pathlib import Path

DATA_PATH = Path("../data/raw/online_retail_II.xlsx")

print("Dataset file exists:", DATA_PATH.exists())

df_2009_2010 = pd.read_excel(
    DATA_PATH,
    sheet_name="Year 2009-2010"
)

df_2010_2011 = pd.read_excel(
    DATA_PATH,
    sheet_name="Year 2010-2011"
)

df = pd.concat(
    [df_2009_2010, df_2010_2011],
    ignore_index=True
)

print("Combined dataset dimensions:", df.shape)

Dataset file exists: True
Combined dataset dimensions: (1067371, 8)


In [5]:
source_columns = [
    "Invoice",
    "StockCode",
    "Description",
    "Quantity",
    "InvoiceDate",
    "Price",
    "Customer ID",
    "Country"
]

df_combined_raw = df[source_columns].copy(deep=True)

df_work = df_combined_raw.copy(deep=True)

print("Raw dataset dimensions:", df_combined_raw.shape)
print("Working dataset dimensions:", df_work.shape)

print(
    "Both datasets identical:",
    df_combined_raw.equals(df_work)
)

Raw dataset dimensions: (1067371, 8)
Working dataset dimensions: (1067371, 8)
Both datasets identical: True


In [6]:
# Count exact duplicate rows
duplicate_count = df_work.duplicated().sum()

# Calculate the percentage of duplicate rows
duplicate_percentage = (
    duplicate_count / len(df_work)
) * 100

print("Total records:", len(df_work))
print("Exact duplicate rows:", duplicate_count)
print(f"Duplicate percentage: {duplicate_percentage:.4f}%")

Total records: 1067371
Exact duplicate rows: 34335
Duplicate percentage: 3.2168%


In [7]:
# Display a sample of exact duplicate records
duplicates = df_work[
    df_work.duplicated(keep=False)
].sort_values(by=source_columns)

display(duplicates.head(20))

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
379,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01 11:34:00,1.95,16329.0,United Kingdom
391,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01 11:34:00,1.95,16329.0,United Kingdom
365,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
386,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
363,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
371,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
394,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
362,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
385,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
368,489517,22130,PARTY CONE CHRISTMAS DECORATION,6,2009-12-01 11:34:00,0.85,16329.0,United Kingdom


In [9]:
# Group identical records and count their occurrences
duplicate_groups = (
    df_work
    .groupby(source_columns, dropna=False)
    .size()
    .reset_index(name="occurrences")
)

# Keep only groups that occur more than once
duplicate_groups = duplicate_groups[
    duplicate_groups["occurrences"] > 1
].sort_values("occurrences", ascending=False)

# Calculate duplicate statistics
duplicate_group_count = len(duplicate_groups)

rows_in_duplicate_groups = duplicate_groups["occurrences"].sum()

excess_duplicate_rows = (
    duplicate_groups["occurrences"] - 1
).sum()

print("Unique duplicated record groups:", duplicate_group_count)
print("Total rows belonging to duplicate groups:", rows_in_duplicate_groups)
print("Excess duplicate rows:", excess_duplicate_rows)

print("\nDuplicate occurrence distribution:")
display(duplicate_groups["occurrences"].value_counts().sort_index())

Unique duplicated record groups: 32907
Total rows belonging to duplicate groups: 67242
Excess duplicate rows: 34335

Duplicate occurrence distribution:


occurrences
2     31985
3       524
4       352
5        10
6        32
8         1
10        1
12        1
20        1
Name: count, dtype: int64

In [10]:
print("Top 20 most frequently duplicated records:")

display(
    duplicate_groups
    .head(20)
)

Top 20 most frequently duplicated records:


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,occurrences
694118,555524,22698,PINK REGENCY TEACUP AND SAUCER,1,2011-06-05 11:37:00,2.95,16923.0,United Kingdom,20
694117,555524,22697,GREEN REGENCY TEACUP AND SAUCER,1,2011-06-05 11:37:00,2.95,16923.0,United Kingdom,12
496282,537224,70007,HI TEC ALPINE HAND WARMER,1,2010-12-05 16:24:00,1.65,13174.0,United Kingdom,10
895184,572861,22775,PURPLE DRAWERKNOB ACRYLIC EDWARDIAN,12,2011-10-26 12:46:00,1.25,14102.0,United Kingdom,8
495167,537154,22727,ALARM CLOCK BAKELIKE RED,1,2010-12-05 13:05:00,3.75,15808.0,United Kingdom,6
512230,538514,21756,BATH BUILDING BLOCK WORD,1,2010-12-12 14:27:00,5.95,15044.0,United Kingdom,6
497951,537265,79000,MOROCCAN TEA GLASS,12,2010-12-06 11:26:00,0.85,15919.0,United Kingdom,6
495877,537208,90082B,DIAMANTE BOW BROOCH RED COLOUR,1,2010-12-05 15:12:00,6.35,15889.0,United Kingdom,6
357161,525065,20894,HANGING BAUBLE T-LIGHT HOLDER LARGE,1,2010-10-03 14:28:00,2.95,16799.0,United Kingdom,6
503982,537781,22898,CHILDRENS APRON APPLES DESIGN,1,2010-12-08 12:46:00,1.95,17341.0,United Kingdom,6


In [11]:
# Inspect complete invoice contents for selected duplicate examples

selected_invoices = [555524, 537224, 572861]

for invoice_id in selected_invoices:
    print(f"\n{'=' * 80}")
    print(f"Invoice: {invoice_id}")
    print(f"{'=' * 80}")

    invoice_data = (
        df_work[df_work["Invoice"] == invoice_id]
        .sort_values(["StockCode", "InvoiceDate"])
    )

    print("Total rows:", len(invoice_data))
    print("Unique products:", invoice_data["StockCode"].nunique())
    print("Exact duplicate rows:", invoice_data.duplicated().sum())

    display(invoice_data)


Invoice: 555524
Total rows: 76
Unique products: 39
Exact duplicate rows: 34


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
738683,555524,20717,STRAWBERRY SHOPPER BAG,3,2011-06-05 11:37:00,1.25,16923.0,United Kingdom
738628,555524,21155,RED RETROSPOT PEG BAG,1,2011-06-05 11:37:00,2.55,16923.0,United Kingdom
738700,555524,21155,RED RETROSPOT PEG BAG,1,2011-06-05 11:37:00,2.55,16923.0,United Kingdom
738680,555524,21166,COOK WITH WINE METAL SIGN,3,2011-06-05 11:37:00,2.08,16923.0,United Kingdom
738681,555524,21174,POTTERING IN THE SHED METAL SIGN,5,2011-06-05 11:37:00,2.08,16923.0,United Kingdom
...,...,...,...,...,...,...,...,...
738633,555524,23301,GARDENERS KNEELING PAD KEEP CALM,4,2011-06-05 11:37:00,1.65,16923.0,United Kingdom
738697,555524,82482,WOODEN PICTURE FRAME WHITE FINISH,3,2011-06-05 11:37:00,2.55,16923.0,United Kingdom
738629,555524,84978,HANGING HEART JAR T-LIGHT HOLDER,6,2011-06-05 11:37:00,1.25,16923.0,United Kingdom
738698,555524,82494L,WOODEN FRAME ANTIQUE WHITE,2,2011-06-05 11:37:00,2.95,16923.0,United Kingdom



Invoice: 537224
Total rows: 338
Unique products: 122
Exact duplicate rows: 186


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
512909,537224,20685,DOORMAT RED RETROSPOT,1,2010-12-05 16:24:00,7.95,13174.0,United Kingdom
535432,537224,20685,DOORMAT RED RETROSPOT,1,2010-12-05 16:24:00,7.95,13174.0,United Kingdom
512865,537224,20829,GLITTER HANGING BUTTERFLY STRING,3,2010-12-05 16:24:00,2.10,13174.0,United Kingdom
535388,537224,20829,GLITTER HANGING BUTTERFLY STRING,3,2010-12-05 16:24:00,2.10,13174.0,United Kingdom
512977,537224,20878,SET/9 CHRISTMAS T-LIGHTS SCENTED,5,2010-12-05 16:24:00,1.25,13174.0,United Kingdom
...,...,...,...,...,...,...,...,...
512961,537224,85025C,PINK HEART SHAPE PHOTO FRAME,2,2010-12-05 16:24:00,1.65,13174.0,United Kingdom
535377,537224,85025C,PINK HEART SHAPE PHOTO FRAME,1,2010-12-05 16:24:00,1.65,13174.0,United Kingdom
535484,537224,85025C,PINK HEART SHAPE PHOTO FRAME,2,2010-12-05 16:24:00,1.65,13174.0,United Kingdom
512978,537224,85123A,WHITE HANGING HEART T-LIGHT HOLDER,12,2010-12-05 16:24:00,2.95,13174.0,United Kingdom



Invoice: 572861
Total rows: 36
Unique products: 18
Exact duplicate rows: 18


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
945334,572861,21034,REX CASH+CARRY JUMBO SHOPPER,1,2011-10-26 12:46:00,0.95,14102.0,United Kingdom
945314,572861,21317,GLASS SPHERE CANDLE STAND MEDIUM,1,2011-10-26 12:46:00,5.45,14102.0,United Kingdom
945305,572861,21668,RED STRIPE CERAMIC DRAWER KNOB,6,2011-10-26 12:46:00,1.45,14102.0,United Kingdom
945306,572861,21668,RED STRIPE CERAMIC DRAWER KNOB,6,2011-10-26 12:46:00,1.45,14102.0,United Kingdom
945321,572861,21668,RED STRIPE CERAMIC DRAWER KNOB,6,2011-10-26 12:46:00,1.45,14102.0,United Kingdom
945322,572861,21668,RED STRIPE CERAMIC DRAWER KNOB,6,2011-10-26 12:46:00,1.45,14102.0,United Kingdom
945319,572861,21669,BLUE STRIPE CERAMIC DRAWER KNOB,6,2011-10-26 12:46:00,1.45,14102.0,United Kingdom
945320,572861,21669,BLUE STRIPE CERAMIC DRAWER KNOB,6,2011-10-26 12:46:00,1.45,14102.0,United Kingdom
945335,572861,21669,BLUE STRIPE CERAMIC DRAWER KNOB,6,2011-10-26 12:46:00,1.45,14102.0,United Kingdom
945336,572861,21669,BLUE STRIPE CERAMIC DRAWER KNOB,6,2011-10-26 12:46:00,1.45,14102.0,United Kingdom


In [12]:
# Summarize the size and product variety of each invoice

invoice_summary = (
    df_work
    .groupby("Invoice")
    .agg(
        total_rows=("Invoice", "size"),
        unique_products=("StockCode", "nunique")
    )
)

# Summarize duplicated record groups within each invoice

duplicate_invoice_summary = (
    duplicate_groups
    .groupby("Invoice")
    .agg(
        duplicate_groups=("occurrences", "size"),
        rows_in_duplicate_groups=("occurrences", "sum"),
        excess_duplicate_rows=(
            "occurrences",
            lambda x: (x - 1).sum()
        )
    )
)

# Combine both summaries

invoice_duplicate_analysis = (
    invoice_summary
    .join(duplicate_invoice_summary, how="left")
    .fillna(0)
)

# Calculate duplicate percentage per invoice

invoice_duplicate_analysis["duplicate_percentage"] = (
    invoice_duplicate_analysis["excess_duplicate_rows"]
    / invoice_duplicate_analysis["total_rows"]
    * 100
)

# Overall findings

affected_invoices = (
    invoice_duplicate_analysis["excess_duplicate_rows"] > 0
).sum()

total_invoices = len(invoice_duplicate_analysis)

print("Total invoices:", total_invoices)
print("Invoices containing exact duplicates:", affected_invoices)
print(
    "Percentage of invoices affected:",
    f"{affected_invoices / total_invoices * 100:.2f}%"
)

print(
    "Total excess duplicate rows:",
    int(invoice_duplicate_analysis["excess_duplicate_rows"].sum())
)

print("\nTop 20 invoices by excess duplicate rows:")

display(
    invoice_duplicate_analysis
    .sort_values("excess_duplicate_rows", ascending=False)
    .head(20)
)

Total invoices: 53628
Invoices containing exact duplicates: 5391
Percentage of invoices affected: 10.05%
Total excess duplicate rows: 34335

Top 20 invoices by excess duplicate rows:


,total_rows,unique_products,duplicate_groups,rows_in_duplicate_groups,excess_duplicate_rows,duplicate_percentage
Invoice,,,,,,
537434,1350,674,675.0,1350.0,675.0,50.000000
538071,1304,648,652.0,1304.0,652.0,50.000000
537638,1202,601,601.0,1202.0,601.0,50.000000
537237,1194,595,597.0,1194.0,597.0,50.000000
536876,1186,586,593.0,1186.0,593.0,50.000000
536592,1184,590,592.0,1184.0,592.0,50.000000
537823,1182,589,591.0,1182.0,591.0,50.000000
537240,1136,568,568.0,1136.0,568.0,50.000000
537666,1072,536,536.0,1072.0,536.0,50.000000


In [13]:
# Identify the date associated with each duplicated record group

duplicate_groups["InvoiceDate"] = pd.to_datetime(
    duplicate_groups["InvoiceDate"]
)

duplicate_groups["YearMonth"] = (
    duplicate_groups["InvoiceDate"].dt.to_period("M")
)

# Summarize duplicate records by month

monthly_duplicate_analysis = (
    duplicate_groups
    .groupby("YearMonth")
    .agg(
        duplicate_groups=("occurrences", "size"),
        rows_in_duplicate_groups=("occurrences", "sum"),
        excess_duplicate_rows=(
            "occurrences",
            lambda x: (x - 1).sum()
        ),
        affected_invoices=("Invoice", "nunique")
    )
)

print("Duplicate record analysis by month:")

display(monthly_duplicate_analysis)

Duplicate record analysis by month:


,duplicate_groups,rows_in_duplicate_groups,excess_duplicate_rows,affected_invoices
YearMonth,,,,
2009-12,471,977,506,208
2010-01,307,628,321,124
2010-02,301,631,330,137
2010-03,488,1014,526,209
2010-04,391,801,410,168
2010-05,399,814,415,171
2010-06,490,1004,514,197
2010-07,375,766,391,165
2010-08,341,708,367,162


In [14]:
# Inspect the characteristics of invoices with the most excess duplicates

top_invoice_ids = (
    invoice_duplicate_analysis
    .nlargest(10, "excess_duplicate_rows")
    .index
)

top_invoice_details = (
    df_work[df_work["Invoice"].isin(top_invoice_ids)]
    .groupby("Invoice")
    .agg(
        total_rows=("Invoice", "size"),
        unique_products=("StockCode", "nunique"),
        unique_dates=("InvoiceDate", "nunique"),
        unique_customers=("Customer ID", "nunique"),
        unique_countries=("Country", "nunique"),
        invoice_date=("InvoiceDate", "min"),
        customer_id=("Customer ID", "first"),
        country=("Country", "first")
    )
    .join(
        invoice_duplicate_analysis[
            ["excess_duplicate_rows", "duplicate_percentage"]
        ]
    )
    .sort_values("excess_duplicate_rows", ascending=False)
)

display(top_invoice_details)

,total_rows,unique_products,unique_dates,unique_customers,unique_countries,invoice_date,customer_id,country,excess_duplicate_rows,duplicate_percentage
Invoice,,,,,,,,,,
537434,1350,674,1,0,1,2010-12-06 16:57:00,NaN,United Kingdom,675.0,50.0
538071,1304,648,1,0,1,2010-12-09 14:09:00,NaN,United Kingdom,652.0,50.0
537638,1202,601,1,0,1,2010-12-07 15:28:00,NaN,United Kingdom,601.0,50.0
537237,1194,595,1,0,1,2010-12-06 09:58:00,NaN,United Kingdom,597.0,50.0
536876,1186,586,1,0,1,2010-12-03 11:36:00,NaN,United Kingdom,593.0,50.0
536592,1184,590,1,0,1,2010-12-01 17:06:00,NaN,United Kingdom,592.0,50.0
537823,1182,589,1,0,1,2010-12-08 14:25:00,NaN,United Kingdom,591.0,50.0
537240,1136,568,1,0,1,2010-12-06 10:08:00,NaN,United Kingdom,568.0,50.0
537666,1072,536,1,0,1,2010-12-07 18:36:00,NaN,United Kingdom,536.0,50.0


In [15]:
# Compare monthly duplicate rows against total monthly records

df_work["YearMonth"] = df_work["InvoiceDate"].dt.to_period("M")

monthly_total_rows = (
    df_work
    .groupby("YearMonth")
    .size()
    .rename("total_rows")
)

monthly_duplicate_comparison = (
    monthly_total_rows
    .to_frame()
    .join(monthly_duplicate_analysis)
    .fillna(0)
)

monthly_duplicate_comparison["excess_duplicate_percentage"] = (
    monthly_duplicate_comparison["excess_duplicate_rows"]
    / monthly_duplicate_comparison["total_rows"]
    * 100
)

display(
    monthly_duplicate_comparison.sort_values(
        "excess_duplicate_percentage",
        ascending=False
    )
)

,total_rows,duplicate_groups,rows_in_duplicate_groups,excess_duplicate_rows,affected_invoices,excess_duplicate_percentage
YearMonth,,,,,,
2010-12,65004,22358,45381,23023,1156,35.417820
2010-11,78015,1316,2747,1431,429,1.834263
2011-11,84711,1273,2641,1368,421,1.614902
2010-10,59098,787,1636,849,302,1.436597
2010-06,39983,490,1004,514,197,1.285546
2011-10,60742,709,1482,773,256,1.272596
2010-03,41511,488,1014,526,209,1.267134
2010-04,34057,391,801,410,168,1.203864
2010-05,35323,399,814,415,171,1.174872


In [16]:
# Re-examine the top 10 invoices and their customer information

top_invoice_ids = (
    invoice_duplicate_analysis
    .nlargest(10, "excess_duplicate_rows")
    .index
)

top_invoice_details = (
    df_work[df_work["Invoice"].isin(top_invoice_ids)]
    .groupby("Invoice")
    .agg(
        total_rows=("Invoice", "size"),
        unique_products=("StockCode", "nunique"),
        unique_dates=("InvoiceDate", "nunique"),
        unique_customer_ids=("Customer ID", "nunique"),
        missing_customer_id_rows=(
            "Customer ID",
            lambda s: s.isna().sum()
        ),
        customer_id=("Customer ID", lambda s: (
            s.dropna().iloc[0] if s.notna().any() else pd.NA
        )),
        country=("Country", "first"),
        invoice_date=("InvoiceDate", "min")
    )
    .join(
        invoice_duplicate_analysis[
            ["excess_duplicate_rows", "duplicate_percentage"]
        ]
    )
    .sort_values("excess_duplicate_rows", ascending=False)
)

display(top_invoice_details)

,total_rows,unique_products,unique_dates,unique_customer_ids,missing_customer_id_rows,customer_id,country,invoice_date,excess_duplicate_rows,duplicate_percentage
Invoice,,,,,,,,,,
537434,1350,674,1,0,1350,<NA>,United Kingdom,2010-12-06 16:57:00,675.0,50.0
538071,1304,648,1,0,1304,<NA>,United Kingdom,2010-12-09 14:09:00,652.0,50.0
537638,1202,601,1,0,1202,<NA>,United Kingdom,2010-12-07 15:28:00,601.0,50.0
537237,1194,595,1,0,1194,<NA>,United Kingdom,2010-12-06 09:58:00,597.0,50.0
536876,1186,586,1,0,1186,<NA>,United Kingdom,2010-12-03 11:36:00,593.0,50.0
536592,1184,590,1,0,1184,<NA>,United Kingdom,2010-12-01 17:06:00,592.0,50.0
537823,1182,589,1,0,1182,<NA>,United Kingdom,2010-12-08 14:25:00,591.0,50.0
537240,1136,568,1,0,1136,<NA>,United Kingdom,2010-12-06 10:08:00,568.0,50.0
537666,1072,536,1,0,1072,<NA>,United Kingdom,2010-12-07 18:36:00,536.0,50.0


In [17]:
# Preserve original row positions for invoice 537434

invoice_id = 537434

invoice_rows = (
    df_work[df_work["Invoice"] == invoice_id]
    .copy()
)

invoice_rows["source_row"] = invoice_rows.index

# Identify exact duplicate groups and their original row positions

invoice_duplicate_positions = (
    invoice_rows
    .groupby(source_columns, dropna=False)["source_row"]
    .apply(list)
    .reset_index(name="source_positions")
)

invoice_duplicate_positions["occurrences"] = (
    invoice_duplicate_positions["source_positions"].str.len()
)

invoice_duplicate_positions = invoice_duplicate_positions[
    invoice_duplicate_positions["occurrences"] > 1
].copy()

# Calculate the distance between repeated records

invoice_duplicate_positions["row_gaps"] = (
    invoice_duplicate_positions["source_positions"]
    .apply(lambda positions: [
        positions[i + 1] - positions[i]
        for i in range(len(positions) - 1)
    ])
)

print("Duplicate groups:", len(invoice_duplicate_positions))

print("\nMost common row gaps:")

display(
    invoice_duplicate_positions["row_gaps"]
    .explode()
    .value_counts()
    .head(15)
)

print("\nSample duplicate records with their original row positions:")

display(invoice_duplicate_positions.head(10))

Duplicate groups: 675

Most common row gaps:


row_gaps
22523    675
Name: count, dtype: int64


Sample duplicate records with their original row positions:


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,source_positions,occurrences,row_gaps
0,537434,11001,ASSTD DESIGN RACING CAR PEN,1,2010-12-06 16:57:00,3.36,NaN,United Kingdom,"[516863, 539386]",2,[22523]
1,537434,15036,ASSORTED COLOURS SILK FAN,2,2010-12-06 16:57:00,1.28,NaN,United Kingdom,"[516864, 539387]",2,[22523]
2,537434,16225,RATTLE SNAKE EGGS,3,2010-12-06 16:57:00,3.36,NaN,United Kingdom,"[516868, 539391]",2,[22523]
3,537434,16235,RECYCLED PENCIL WITH RABBIT ERASER,9,2010-12-06 16:57:00,0.43,NaN,United Kingdom,"[516869, 539392]",2,[22523]
4,537434,16236,KITTY PENCIL ERASERS,2,2010-12-06 16:57:00,0.43,NaN,United Kingdom,"[516870, 539393]",2,[22523]
5,537434,16237,SLEEPING CAT ERASERS,6,2010-12-06 16:57:00,0.43,NaN,United Kingdom,"[516871, 539394]",2,[22523]
6,537434,17003,BROCADE RING PURSE,1,2010-12-06 16:57:00,0.43,NaN,United Kingdom,"[516872, 539395]",2,[22523]
7,537434,18007,ESSENTIAL BALM 3.5g TIN IN ENVELOPE,1,2010-12-06 16:57:00,0.43,NaN,United Kingdom,"[516875, 539398]",2,[22523]
8,537434,20615,BLUE POLKADOT PASSPORT COVER,3,2010-12-06 16:57:00,4.21,NaN,United Kingdom,"[516877, 539400]",2,[22523]
9,537434,20652,BLUE POLKADOT LUGGAGE TAG,2,2010-12-06 16:57:00,2.51,NaN,United Kingdom,"[516878, 539401]",2,[22523]


In [18]:
# Test whether records are repeated at a constant offset
# throughout the entire dataset.

offset = 22523

comparison_data = df_work[source_columns].reset_index(drop=True)

original_rows = comparison_data.iloc[:-offset].reset_index(drop=True)
shifted_rows = comparison_data.iloc[offset:].reset_index(drop=True)

# Treat missing values in matching positions as equal
matching_positions = (
    original_rows.eq(shifted_rows)
    | (original_rows.isna() & shifted_rows.isna())
).all(axis=1)

matching_indices = matching_positions[matching_positions].index

print("Tested row offset:", offset)
print("Total row comparisons:", len(matching_positions))
print("Matching row pairs:", len(matching_indices))
print(
    "Percentage of matching pairs:",
    f"{matching_positions.mean() * 100:.4f}%"
)

# Identify consecutive matching positions
if len(matching_indices) > 0:
    matching_groups = (
        (matching_indices.to_series().diff() != 1)
        .cumsum()
    )

    matching_runs = (
        matching_indices.to_series()
        .groupby(matching_groups)
        .agg(["min", "max", "count"])
        .rename(columns={
            "min": "start_position",
            "max": "end_position",
            "count": "matching_rows"
        })
        .reset_index(drop=True)
    )

    print("\nLargest consecutive matching runs:")

    display(
        matching_runs
        .sort_values("matching_rows", ascending=False)
        .head(20)
    )

Tested row offset: 22523
Total row comparisons: 1044848
Matching row pairs: 18972
Percentage of matching pairs: 1.8158%

Largest consecutive matching runs:


,start_position,end_position,matching_rows
39,512584,513790,1207
57,517979,518939,961
8,505195,506069,875
51,516204,517050,847
31,510148,510981,834
67,521184,521991,808
74,523459,524229,771
4,503719,504447,729
22,508625,509328,704
59,519373,520003,631


In [19]:
# Examine the original and shifted records in the largest matching runs

offset = 22523

top_runs = (
    matching_runs
    .sort_values("matching_rows", ascending=False)
    .head(10)
    .copy()
)

run_details = []

for run_number, run in top_runs.iterrows():

    start = int(run["start_position"])
    end = int(run["end_position"])

    original_segment = df_work.iloc[start:end + 1]
    shifted_segment = df_work.iloc[start + offset:end + offset + 1]

    run_details.append({
        "run": run_number,
        "original_start": start,
        "original_end": end,
        "shifted_start": start + offset,
        "shifted_end": end + offset,
        "matching_rows": len(original_segment),
        "original_first_invoice": original_segment["Invoice"].iloc[0],
        "original_last_invoice": original_segment["Invoice"].iloc[-1],
        "shifted_first_invoice": shifted_segment["Invoice"].iloc[0],
        "shifted_last_invoice": shifted_segment["Invoice"].iloc[-1],
        "original_start_date": original_segment["InvoiceDate"].min(),
        "original_end_date": original_segment["InvoiceDate"].max(),
        "shifted_start_date": shifted_segment["InvoiceDate"].min(),
        "shifted_end_date": shifted_segment["InvoiceDate"].max(),
        "original_unique_invoices": original_segment["Invoice"].nunique(),
        "shifted_unique_invoices": shifted_segment["Invoice"].nunique()
    })

run_details_df = pd.DataFrame(run_details)

display(run_details_df)

,run,original_start,original_end,shifted_start,shifted_end,matching_rows,original_first_invoice,original_last_invoice,shifted_first_invoice,shifted_last_invoice,original_start_date,original_end_date,shifted_start_date,shifted_end_date,original_unique_invoices,shifted_unique_invoices
0,39,512584,513790,535107,536313,1207,537209,537240,537209,537240,2010-12-05 15:12:00,2010-12-06 10:08:00,2010-12-05 15:12:00,2010-12-06 10:08:00,32,32
1,57,517979,518939,540502,541462,961,537635,537642,537635,537642,2010-12-07 15:18:00,2010-12-07 15:33:00,2010-12-07 15:18:00,2010-12-07 15:33:00,8,8
2,8,505195,506069,527718,528592,875,536571,536601,536571,536601,2010-12-01 15:37:00,2010-12-02 08:33:00,2010-12-01 15:37:00,2010-12-02 08:33:00,31,31
3,51,516204,517050,538727,539573,847,537433,537460,537433,537460,2010-12-06 16:36:00,2010-12-07 10:04:00,2010-12-06 16:36:00,2010-12-07 10:04:00,27,27
4,31,510148,510981,532671,533504,834,537018,537117,537018,537117,2010-12-03 15:42:00,2010-12-05 12:09:00,2010-12-03 15:42:00,2010-12-05 12:09:00,48,48
5,67,521184,521991,543707,544514,808,C537812,537828,C537812,537828,2010-12-08 13:33:00,2010-12-08 14:43:00,2010-12-08 13:33:00,2010-12-08 14:43:00,16,16
6,74,523459,524229,545982,546752,771,538060,538071,538060,538071,2010-12-09 13:39:00,2010-12-09 14:09:00,2010-12-09 13:39:00,2010-12-09 14:09:00,12,12
7,4,503719,504447,526242,526970,729,536464,536544,536464,536544,2010-12-01 12:23:00,2010-12-01 14:32:00,2010-12-01 12:23:00,2010-12-01 14:32:00,34,34
8,22,508625,509328,531148,531851,704,536874,536938,536874,536938,2010-12-03 11:35:00,2010-12-03 12:05:00,2010-12-03 11:35:00,2010-12-03 12:05:00,8,8
9,59,519373,520003,541896,542526,631,537659,537674,537659,537674,2010-12-07 16:43:00,2010-12-08 09:51:00,2010-12-07 16:43:00,2010-12-08 09:51:00,12,12


In [20]:
# Inspect the beginning and end of the largest matching run

largest_run = top_runs.iloc[0]

start = int(largest_run["start_position"])
end = int(largest_run["end_position"])

print("Original sequence - beginning:")
display(df_work.iloc[start - 3:start + 5])

print("\nShifted sequence - beginning:")
display(df_work.iloc[start + offset - 3:start + offset + 5])

print("\nOriginal sequence - ending:")
display(df_work.iloc[end - 4:end + 4])

print("\nShifted sequence - ending:")
display(df_work.iloc[end + offset - 4:end + offset + 4])

Original sequence - beginning:


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,YearMonth
512581,537208,90082B,DIAMANTE BOW BROOCH RED COLOUR,1,2010-12-05 15:12:00,6.35,15889.0,United Kingdom,2010-12
512582,537208,90200E,GREEN SWEETHEART BRACELET,2,2010-12-05 15:12:00,4.25,15889.0,United Kingdom,2010-12
512583,537208,90062,CARNIVAL BRACELET,1,2010-12-05 15:12:00,12.75,15889.0,United Kingdom,2010-12
512584,537209,22577,WOODEN HEART CHRISTMAS SCANDINAVIAN,3,2010-12-05 15:12:00,0.85,14606.0,United Kingdom,2010-12
512585,537209,20972,PINK CREAM FELT CRAFT TRINKET BOX,2,2010-12-05 15:12:00,1.25,14606.0,United Kingdom,2010-12
512586,537209,85231G,ORANGE SCENTED SET/9 T-LIGHTS,1,2010-12-05 15:12:00,0.85,14606.0,United Kingdom,2010-12
512587,537209,17003,BROCADE RING PURSE,3,2010-12-05 15:12:00,0.21,14606.0,United Kingdom,2010-12
512588,537209,37461,FUNKY MONKEY MUG,1,2010-12-05 15:12:00,1.25,14606.0,United Kingdom,2010-12



Shifted sequence - beginning:


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,YearMonth
535104,537209,22568,FELTCRAFT CUSHION OWL,2,2010-12-05 15:12:00,3.75,14606.0,United Kingdom,2010-12
535105,537209,22570,FELTCRAFT CUSHION RABBIT,1,2010-12-05 15:12:00,3.75,14606.0,United Kingdom,2010-12
535106,537209,21894,POTTING SHED SEED ENVELOPES,1,2010-12-05 15:12:00,1.25,14606.0,United Kingdom,2010-12
535107,537209,22577,WOODEN HEART CHRISTMAS SCANDINAVIAN,3,2010-12-05 15:12:00,0.85,14606.0,United Kingdom,2010-12
535108,537209,20972,PINK CREAM FELT CRAFT TRINKET BOX,2,2010-12-05 15:12:00,1.25,14606.0,United Kingdom,2010-12
535109,537209,85231G,ORANGE SCENTED SET/9 T-LIGHTS,1,2010-12-05 15:12:00,0.85,14606.0,United Kingdom,2010-12
535110,537209,17003,BROCADE RING PURSE,3,2010-12-05 15:12:00,0.21,14606.0,United Kingdom,2010-12
535111,537209,37461,FUNKY MONKEY MUG,1,2010-12-05 15:12:00,1.25,14606.0,United Kingdom,2010-12



Original sequence - ending:


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,YearMonth
513786,537240,20963,APPLE BATH SPONGE,1,2010-12-06 10:08:00,2.51,NaN,United Kingdom,2010-12
513787,537240,20966,SANDWICH BATH SPONGE,3,2010-12-06 10:08:00,2.51,NaN,United Kingdom,2010-12
513788,537240,20967,GREY FLORAL FELTCRAFT SHOULDER BAG,1,2010-12-06 10:08:00,7.62,NaN,United Kingdom,2010-12
513789,537240,20969,RED FLORAL FELTCRAFT SHOULDER BAG,2,2010-12-06 10:08:00,7.62,NaN,United Kingdom,2010-12
513790,537240,20970,PINK FLORAL FELTCRAFT SHOULDER BAG,3,2010-12-06 10:08:00,7.62,NaN,United Kingdom,2010-12
513791,537240,20972,PINK CREAM FELT CRAFT TRINKET BOX,8,2010-12-06 10:08:00,2.51,NaN,United Kingdom,2010-12
513792,537240,20973,12 PENCIL SMALL TUBE WOODLAND,1,2010-12-06 10:08:00,1.28,NaN,United Kingdom,2010-12
513793,537240,20974,12 PENCILS SMALL TUBE SKULL,4,2010-12-06 10:08:00,1.28,NaN,United Kingdom,2010-12



Shifted sequence - ending:


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,YearMonth
536309,537240,20963,APPLE BATH SPONGE,1,2010-12-06 10:08:00,2.51,NaN,United Kingdom,2010-12
536310,537240,20966,SANDWICH BATH SPONGE,3,2010-12-06 10:08:00,2.51,NaN,United Kingdom,2010-12
536311,537240,20967,GREY FLORAL FELTCRAFT SHOULDER BAG,1,2010-12-06 10:08:00,7.62,NaN,United Kingdom,2010-12
536312,537240,20969,RED FLORAL FELTCRAFT SHOULDER BAG,2,2010-12-06 10:08:00,7.62,NaN,United Kingdom,2010-12
536313,537240,20970,PINK FLORAL FELTCRAFT SHOULDER BAG,3,2010-12-06 10:08:00,7.62,NaN,United Kingdom,2010-12
536314,537240,22358,KINGS CHOICE TEA CADDY,1,2010-12-06 10:08:00,5.91,NaN,United Kingdom,2010-12
536315,537240,22360,GLASS JAR ENGLISH CONFECTIONERY,1,2010-12-06 10:08:00,5.91,NaN,United Kingdom,2010-12
536316,537240,22372,AIRLINE BAG VINTAGE WORLD CHAMPION,1,2010-12-06 10:08:00,8.47,NaN,United Kingdom,2010-12


In [21]:
# Examine the original and shifted records in the largest matching runs

offset = 22523

top_runs = (
    matching_runs
    .sort_values("matching_rows", ascending=False)
    .head(10)
    .copy()
)

run_details = []

for run_number, run in top_runs.iterrows():

    start = int(run["start_position"])
    end = int(run["end_position"])

    original_segment = df_work.iloc[start:end + 1]
    shifted_segment = df_work.iloc[start + offset:end + offset + 1]

    run_details.append({
        "run": run_number,
        "original_start": start,
        "original_end": end,
        "shifted_start": start + offset,
        "shifted_end": end + offset,
        "matching_rows": len(original_segment),
        "original_first_invoice": original_segment["Invoice"].iloc[0],
        "original_last_invoice": original_segment["Invoice"].iloc[-1],
        "shifted_first_invoice": shifted_segment["Invoice"].iloc[0],
        "shifted_last_invoice": shifted_segment["Invoice"].iloc[-1],
        "original_start_date": original_segment["InvoiceDate"].min(),
        "original_end_date": original_segment["InvoiceDate"].max(),
        "shifted_start_date": shifted_segment["InvoiceDate"].min(),
        "shifted_end_date": shifted_segment["InvoiceDate"].max(),
        "original_unique_invoices": original_segment["Invoice"].nunique(),
        "shifted_unique_invoices": shifted_segment["Invoice"].nunique()
    })

run_details_df = pd.DataFrame(run_details)

display(run_details_df)

,run,original_start,original_end,shifted_start,shifted_end,matching_rows,original_first_invoice,original_last_invoice,shifted_first_invoice,shifted_last_invoice,original_start_date,original_end_date,shifted_start_date,shifted_end_date,original_unique_invoices,shifted_unique_invoices
0,39,512584,513790,535107,536313,1207,537209,537240,537209,537240,2010-12-05 15:12:00,2010-12-06 10:08:00,2010-12-05 15:12:00,2010-12-06 10:08:00,32,32
1,57,517979,518939,540502,541462,961,537635,537642,537635,537642,2010-12-07 15:18:00,2010-12-07 15:33:00,2010-12-07 15:18:00,2010-12-07 15:33:00,8,8
2,8,505195,506069,527718,528592,875,536571,536601,536571,536601,2010-12-01 15:37:00,2010-12-02 08:33:00,2010-12-01 15:37:00,2010-12-02 08:33:00,31,31
3,51,516204,517050,538727,539573,847,537433,537460,537433,537460,2010-12-06 16:36:00,2010-12-07 10:04:00,2010-12-06 16:36:00,2010-12-07 10:04:00,27,27
4,31,510148,510981,532671,533504,834,537018,537117,537018,537117,2010-12-03 15:42:00,2010-12-05 12:09:00,2010-12-03 15:42:00,2010-12-05 12:09:00,48,48
5,67,521184,521991,543707,544514,808,C537812,537828,C537812,537828,2010-12-08 13:33:00,2010-12-08 14:43:00,2010-12-08 13:33:00,2010-12-08 14:43:00,16,16
6,74,523459,524229,545982,546752,771,538060,538071,538060,538071,2010-12-09 13:39:00,2010-12-09 14:09:00,2010-12-09 13:39:00,2010-12-09 14:09:00,12,12
7,4,503719,504447,526242,526970,729,536464,536544,536464,536544,2010-12-01 12:23:00,2010-12-01 14:32:00,2010-12-01 12:23:00,2010-12-01 14:32:00,34,34
8,22,508625,509328,531148,531851,704,536874,536938,536874,536938,2010-12-03 11:35:00,2010-12-03 12:05:00,2010-12-03 11:35:00,2010-12-03 12:05:00,8,8
9,59,519373,520003,541896,542526,631,537659,537674,537659,537674,2010-12-07 16:43:00,2010-12-08 09:51:00,2010-12-07 16:43:00,2010-12-08 09:51:00,12,12


In [22]:
# Inspect the beginning and end of the largest matching run

largest_run = top_runs.iloc[0]

start = int(largest_run["start_position"])
end = int(largest_run["end_position"])

print("Original sequence - beginning:")
display(df_work.iloc[start - 3:start + 5])

print("\nShifted sequence - beginning:")
display(df_work.iloc[start + offset - 3:start + offset + 5])

print("\nOriginal sequence - ending:")
display(df_work.iloc[end - 4:end + 4])

print("\nShifted sequence - ending:")
display(df_work.iloc[end + offset - 4:end + offset + 4])

Original sequence - beginning:


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,YearMonth
512581,537208,90082B,DIAMANTE BOW BROOCH RED COLOUR,1,2010-12-05 15:12:00,6.35,15889.0,United Kingdom,2010-12
512582,537208,90200E,GREEN SWEETHEART BRACELET,2,2010-12-05 15:12:00,4.25,15889.0,United Kingdom,2010-12
512583,537208,90062,CARNIVAL BRACELET,1,2010-12-05 15:12:00,12.75,15889.0,United Kingdom,2010-12
512584,537209,22577,WOODEN HEART CHRISTMAS SCANDINAVIAN,3,2010-12-05 15:12:00,0.85,14606.0,United Kingdom,2010-12
512585,537209,20972,PINK CREAM FELT CRAFT TRINKET BOX,2,2010-12-05 15:12:00,1.25,14606.0,United Kingdom,2010-12
512586,537209,85231G,ORANGE SCENTED SET/9 T-LIGHTS,1,2010-12-05 15:12:00,0.85,14606.0,United Kingdom,2010-12
512587,537209,17003,BROCADE RING PURSE,3,2010-12-05 15:12:00,0.21,14606.0,United Kingdom,2010-12
512588,537209,37461,FUNKY MONKEY MUG,1,2010-12-05 15:12:00,1.25,14606.0,United Kingdom,2010-12



Shifted sequence - beginning:


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,YearMonth
535104,537209,22568,FELTCRAFT CUSHION OWL,2,2010-12-05 15:12:00,3.75,14606.0,United Kingdom,2010-12
535105,537209,22570,FELTCRAFT CUSHION RABBIT,1,2010-12-05 15:12:00,3.75,14606.0,United Kingdom,2010-12
535106,537209,21894,POTTING SHED SEED ENVELOPES,1,2010-12-05 15:12:00,1.25,14606.0,United Kingdom,2010-12
535107,537209,22577,WOODEN HEART CHRISTMAS SCANDINAVIAN,3,2010-12-05 15:12:00,0.85,14606.0,United Kingdom,2010-12
535108,537209,20972,PINK CREAM FELT CRAFT TRINKET BOX,2,2010-12-05 15:12:00,1.25,14606.0,United Kingdom,2010-12
535109,537209,85231G,ORANGE SCENTED SET/9 T-LIGHTS,1,2010-12-05 15:12:00,0.85,14606.0,United Kingdom,2010-12
535110,537209,17003,BROCADE RING PURSE,3,2010-12-05 15:12:00,0.21,14606.0,United Kingdom,2010-12
535111,537209,37461,FUNKY MONKEY MUG,1,2010-12-05 15:12:00,1.25,14606.0,United Kingdom,2010-12



Original sequence - ending:


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,YearMonth
513786,537240,20963,APPLE BATH SPONGE,1,2010-12-06 10:08:00,2.51,NaN,United Kingdom,2010-12
513787,537240,20966,SANDWICH BATH SPONGE,3,2010-12-06 10:08:00,2.51,NaN,United Kingdom,2010-12
513788,537240,20967,GREY FLORAL FELTCRAFT SHOULDER BAG,1,2010-12-06 10:08:00,7.62,NaN,United Kingdom,2010-12
513789,537240,20969,RED FLORAL FELTCRAFT SHOULDER BAG,2,2010-12-06 10:08:00,7.62,NaN,United Kingdom,2010-12
513790,537240,20970,PINK FLORAL FELTCRAFT SHOULDER BAG,3,2010-12-06 10:08:00,7.62,NaN,United Kingdom,2010-12
513791,537240,20972,PINK CREAM FELT CRAFT TRINKET BOX,8,2010-12-06 10:08:00,2.51,NaN,United Kingdom,2010-12
513792,537240,20973,12 PENCIL SMALL TUBE WOODLAND,1,2010-12-06 10:08:00,1.28,NaN,United Kingdom,2010-12
513793,537240,20974,12 PENCILS SMALL TUBE SKULL,4,2010-12-06 10:08:00,1.28,NaN,United Kingdom,2010-12



Shifted sequence - ending:


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,YearMonth
536309,537240,20963,APPLE BATH SPONGE,1,2010-12-06 10:08:00,2.51,NaN,United Kingdom,2010-12
536310,537240,20966,SANDWICH BATH SPONGE,3,2010-12-06 10:08:00,2.51,NaN,United Kingdom,2010-12
536311,537240,20967,GREY FLORAL FELTCRAFT SHOULDER BAG,1,2010-12-06 10:08:00,7.62,NaN,United Kingdom,2010-12
536312,537240,20969,RED FLORAL FELTCRAFT SHOULDER BAG,2,2010-12-06 10:08:00,7.62,NaN,United Kingdom,2010-12
536313,537240,20970,PINK FLORAL FELTCRAFT SHOULDER BAG,3,2010-12-06 10:08:00,7.62,NaN,United Kingdom,2010-12
536314,537240,22358,KINGS CHOICE TEA CADDY,1,2010-12-06 10:08:00,5.91,NaN,United Kingdom,2010-12
536315,537240,22360,GLASS JAR ENGLISH CONFECTIONERY,1,2010-12-06 10:08:00,5.91,NaN,United Kingdom,2010-12
536316,537240,22372,AIRLINE BAG VINTAGE WORLD CHAMPION,1,2010-12-06 10:08:00,8.47,NaN,United Kingdom,2010-12


In [23]:
# Identify invoices affected by the repeated sequences

offset = 22523

# Create a table of original and shifted row positions
matched_pairs = pd.DataFrame({
    "original_position": matching_indices,
    "shifted_position": matching_indices + offset
})

# Retrieve the original records corresponding to matching positions
matched_records = df_work.iloc[
    matched_pairs["original_position"]
].copy()

matched_records["source_position"] = (
    matched_pairs["original_position"].to_numpy()
)

# Summarize matched records by invoice
invoice_match_summary = (
    matched_records
    .groupby("Invoice")
    .agg(
        matching_rows=("Invoice", "size"),
        first_matching_position=("source_position", "min"),
        last_matching_position=("source_position", "max"),
        unique_matching_dates=("InvoiceDate", "nunique")
    )
)

# Count all original rows belonging to each invoice
invoice_total_rows = (
    df_work
    .groupby("Invoice")
    .size()
    .rename("total_invoice_rows")
)

# Combine the two summaries
invoice_match_summary = (
    invoice_total_rows.to_frame()
    .join(invoice_match_summary, how="left")
)

invoice_match_summary["matching_rows"] = (
    invoice_match_summary["matching_rows"].fillna(0)
)

invoice_match_summary["matching_percentage"] = (
    invoice_match_summary["matching_rows"]
    / invoice_match_summary["total_invoice_rows"]
    * 100
)

# Keep only affected invoices
invoice_match_summary = invoice_match_summary[
    invoice_match_summary["matching_rows"] > 0
].sort_values("matching_rows", ascending=False)

print("Total affected invoices:", len(invoice_match_summary))

print("\nInvoices with the most matching records:")
display(invoice_match_summary.head(20))

print("\nDistribution of invoice matching percentages:")
display(
    invoice_match_summary["matching_percentage"]
    .describe()
)

Total affected invoices: 914

Invoices with the most matching records:


,total_invoice_rows,matching_rows,first_matching_position,last_matching_position,unique_matching_dates,matching_percentage
Invoice,,,,,,
537434,1350,675.0,516208.0,516882.0,1.0,50.000000
538071,1304,606.0,523624.0,524229.0,1.0,46.472393
537638,1202,601.0,517988.0,518588.0,1.0,50.000000
537237,1194,597.0,513158.0,513754.0,1.0,50.000000
536876,1186,593.0,508694.0,509286.0,1.0,50.000000
536592,1184,592.0,505399.0,505990.0,1.0,50.000000
537823,1182,591.0,521321.0,521911.0,1.0,50.000000
537666,1072,536.0,519387.0,519922.0,1.0,50.000000
537224,338,169.0,512849.0,513017.0,1.0,50.000000



Distribution of invoice matching percentages:


count    914.000000
mean      49.448779
std        4.364047
min        0.704225
25%       50.000000
50%       50.000000
75%       50.000000
max       50.000000
Name: matching_percentage, dtype: float64

In [27]:
# Further summarize invoices affected by the repeated sequences

# Count invoices by matching percentage
exactly_half = invoice_match_summary[
    np.isclose(
        invoice_match_summary["matching_percentage"],
        50
    )
]

below_half = invoice_match_summary[
    invoice_match_summary["matching_percentage"] < 50
]

print("Total affected invoices:", len(invoice_match_summary))

print(
    "Invoices with exactly 50% matching rows:",
    len(exactly_half)
)

print(
    "Invoices with less than 50% matching rows:",
    len(below_half)
)

print(
    "\nPercentage of affected invoices with exactly 50% matching rows:",
    f"{len(exactly_half) / len(invoice_match_summary) * 100:.2f}%"
)

# Examine dates represented by the original matching records
matched_records["InvoiceDate"] = pd.to_datetime(
    matched_records["InvoiceDate"]
)

print("\nMatching record date range:")
print("First date:", matched_records["InvoiceDate"].min())
print("Last date:", matched_records["InvoiceDate"].max())

print("\nMatching records by month:")
display(
    matched_records["InvoiceDate"]
    .dt.to_period("M")
    .value_counts()
    .sort_index()
)

# Examine matching percentage distribution
print("\nMatching percentage distribution:")
display(
    invoice_match_summary["matching_percentage"]
    .round(2)
    .value_counts()
    .sort_index()
)

Total affected invoices: 914
Invoices with exactly 50% matching rows: 893
Invoices with less than 50% matching rows: 21

Percentage of affected invoices with exactly 50% matching rows: 97.70%

Matching record date range:
First date: 2010-12-01 08:26:00
Last date: 2010-12-09 20:01:00

Matching records by month:


InvoiceDate
2010-12    18972
Freq: M, Name: count, dtype: int64


Matching percentage distribution:


matching_percentage
0.70       1
0.89       1
0.91       1
6.36       1
8.33       1
9.09       1
12.50      1
20.00      1
23.68      1
27.78      1
31.34      1
31.82      1
32.14      1
34.38      1
38.89      1
39.29      1
41.43      1
44.29      1
46.47      1
47.56      1
48.33      1
50.00    893
Name: count, dtype: int64

In [28]:
# Compare offset-matched records with exact duplicate records

# Create a set of original positions involved in offset matches
offset_original_positions = set(
    matched_pairs["original_position"]
)

# Create a set of shifted positions involved in offset matches
offset_shifted_positions = set(
    matched_pairs["shifted_position"]
)

# Combine both sets of positions
all_offset_positions = (
    offset_original_positions | offset_shifted_positions
)

# Identify all rows marked as exact duplicates
exact_duplicate_mask = df_work.duplicated(
    subset=source_columns,
    keep=False
)

exact_duplicate_positions = set(
    df_work.index[exact_duplicate_mask]
)

# Compare the two groups
offset_exact_overlap = (
    all_offset_positions & exact_duplicate_positions
)

offset_not_exact = (
    all_offset_positions - exact_duplicate_positions
)

exact_not_offset = (
    exact_duplicate_positions - all_offset_positions
)

print("Unique rows involved in offset matches:", len(all_offset_positions))
print("Rows involved in offset matches also marked as exact duplicates:", len(offset_exact_overlap))
print("Offset-matched rows not marked as exact duplicates:", len(offset_not_exact))
print("Exact-duplicate rows not involved in offset matches:", len(exact_not_offset))

print("\nOffset-match coverage of exact-duplicate rows:")
print(
    f"{len(offset_exact_overlap) / len(exact_duplicate_positions) * 100:.2f}%"
)

Unique rows involved in offset matches: 37944
Rows involved in offset matches also marked as exact duplicates: 37944
Offset-matched rows not marked as exact duplicates: 0
Exact-duplicate rows not involved in offset matches: 29298

Offset-match coverage of exact-duplicate rows:
56.43%


In [29]:
# Compare offset-matched positions with actual excess duplicate rows

# Identify excess duplicate rows:
# keep="first" marks only subsequent occurrences
excess_duplicate_mask = df_work.duplicated(
    subset=source_columns,
    keep="first"
)

excess_duplicate_positions = set(
    df_work.index[excess_duplicate_mask]
)

# Compare shifted positions against excess duplicate positions
offset_shifted_excess = (
    offset_shifted_positions & excess_duplicate_positions
)

offset_shifted_not_excess = (
    offset_shifted_positions - excess_duplicate_positions
)

excess_not_offset_shifted = (
    excess_duplicate_positions - offset_shifted_positions
)

print("Total excess exact duplicate rows:", len(excess_duplicate_positions))

print(
    "Shifted positions that are excess duplicate rows:",
    len(offset_shifted_excess)
)

print(
    "Shifted positions not classified as excess duplicates:",
    len(offset_shifted_not_excess)
)

print(
    "Excess duplicate rows outside shifted positions:",
    len(excess_not_offset_shifted)
)

print("\nPercentage of excess duplicates explained by shifted positions:")

print(
    f"{len(offset_shifted_excess) / len(excess_duplicate_positions) * 100:.2f}%"
)

Total excess exact duplicate rows: 34335
Shifted positions that are excess duplicate rows: 18972
Shifted positions not classified as excess duplicates: 0
Excess duplicate rows outside shifted positions: 15363

Percentage of excess duplicates explained by shifted positions:
55.26%


In [30]:
# Identify excess duplicate rows outside the shifted positions

remaining_duplicate_positions = (
    excess_duplicate_positions - offset_shifted_positions
)

remaining_duplicates = df_work.loc[
    sorted(remaining_duplicate_positions)
].copy()

# Derive month for analysis
remaining_duplicates["YearMonth"] = (
    pd.to_datetime(remaining_duplicates["InvoiceDate"])
    .dt.to_period("M")
)

print("Remaining excess duplicate rows:", len(remaining_duplicates))

print("\nRemaining excess duplicates by month:")

display(
    remaining_duplicates
    .groupby("YearMonth")
    .size()
    .sort_values(ascending=False)
    .to_frame("excess_duplicate_rows")
)

print("\nRemaining excess duplicates by invoice:")

remaining_invoice_summary = (
    remaining_duplicates
    .groupby("Invoice")
    .agg(
        excess_duplicate_rows=("Invoice", "size"),
        unique_products=("StockCode", "nunique"),
        first_date=("InvoiceDate", "min"),
        last_date=("InvoiceDate", "max")
    )
    .sort_values("excess_duplicate_rows", ascending=False)
)

display(remaining_invoice_summary.head(20))

Remaining excess duplicate rows: 15363

Remaining excess duplicates by month:


,excess_duplicate_rows
YearMonth,
2010-12,4051
2010-11,1431
2011-11,1368
2010-10,849
2011-10,773
2010-03,526
2010-06,514
2009-12,506
2010-09,484



Remaining excess duplicates by invoice:


,excess_duplicate_rows,unique_products,first_date,last_date
Invoice,,,,
537240,560,560,2010-12-06 10:08:00,2010-12-06 10:08:00
536544,460,456,2010-12-01 14:32:00,2010-12-01 14:32:00
537643,123,123,2010-12-07 15:34:00,2010-12-07 15:34:00
537126,119,116,2010-12-05 12:13:00,2010-12-05 12:13:00
537645,107,107,2010-12-07 15:34:00,2010-12-07 15:34:00
537196,96,74,2010-12-05 13:55:00,2010-12-05 13:55:00
537374,92,74,2010-12-06 12:55:00,2010-12-06 12:55:00
536862,81,81,2010-12-03 11:13:00,2010-12-03 11:13:00
537154,73,54,2010-12-05 13:05:00,2010-12-05 13:05:00


In [31]:
# Investigate remaining duplicate groups within selected invoices

invoices_to_inspect = [537240, 536544]

for invoice_id in invoices_to_inspect:

    print(f"\n{'=' * 70}")
    print(f"Invoice: {invoice_id}")
    print(f"{'=' * 70}")

    invoice_rows = df_work[
        df_work["Invoice"] == invoice_id
    ].copy()

    invoice_rows["source_position"] = invoice_rows.index

    # Group identical records and preserve their original positions
    invoice_groups = (
        invoice_rows
        .groupby(source_columns, dropna=False)
        .agg(
            occurrences=("Invoice", "size"),
            source_positions=("source_position", list)
        )
        .reset_index()
    )

    # Keep only repeated record groups
    invoice_groups = invoice_groups[
        invoice_groups["occurrences"] > 1
    ].copy()

    # Calculate distances between repeated occurrences
    invoice_groups["row_gaps"] = (
        invoice_groups["source_positions"]
        .apply(
            lambda positions: [
                positions[i + 1] - positions[i]
                for i in range(len(positions) - 1)
            ]
        )
    )

    print("Total invoice rows:", len(invoice_rows))
    print("Unique duplicated record groups:", len(invoice_groups))
    print(
        "Rows belonging to duplicated groups:",
        invoice_groups["occurrences"].sum()
    )
    print(
        "Excess duplicate rows within invoice:",
        (invoice_groups["occurrences"] - 1).sum()
    )

    print("\nMost frequently repeated records:")

    display(
        invoice_groups
        .sort_values("occurrences", ascending=False)
        .head(15)
    )


Invoice: 537240
Total invoice rows: 1136
Unique duplicated record groups: 568
Rows belonging to duplicated groups: 1136
Excess duplicate rows within invoice: 568

Most frequently repeated records:


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,occurrences,source_positions,row_gaps
0,537240,10002,INFLATABLE POLITICAL GLOBE,1,2010-12-06 10:08:00,1.66,NaN,United Kingdom,2,"[514161, 536843]",[22682]
1,537240,10125,MINI FUNKY DESIGN TAPES,2,2010-12-06 10:08:00,1.66,NaN,United Kingdom,2,"[514162, 536844]",[22682]
2,537240,16235,RECYCLED PENCIL WITH RABBIT ERASER,1,2010-12-06 10:08:00,0.43,NaN,United Kingdom,2,"[514164, 536846]",[22682]
3,537240,16237,SLEEPING CAT ERASERS,1,2010-12-06 10:08:00,0.43,NaN,United Kingdom,2,"[514165, 536847]",[22682]
4,537240,17003,BROCADE RING PURSE,2,2010-12-06 10:08:00,0.43,NaN,United Kingdom,2,"[514166, 536848]",[22682]
5,537240,20665,RED RETROSPOT PURSE,2,2010-12-06 10:08:00,5.91,NaN,United Kingdom,2,"[514168, 536850]",[22682]
6,537240,20681,PINK POLKADOT CHILDRENS UMBRELLA,1,2010-12-06 10:08:00,6.77,NaN,United Kingdom,2,"[514169, 536851]",[22682]
7,537240,20682,RED RETROSPOT CHILDRENS UMBRELLA,3,2010-12-06 10:08:00,6.77,NaN,United Kingdom,2,"[514170, 536852]",[22682]
8,537240,20684,STRAWBERRY DREAM CHILDS UMBRELLA,1,2010-12-06 10:08:00,6.77,NaN,United Kingdom,2,"[514171, 536853]",[22682]
9,537240,20699,MOUSEY LONG LEGS SOFT TOY,1,2010-12-06 10:08:00,5.06,NaN,United Kingdom,2,"[514172, 536854]",[22682]



Invoice: 536544
Total invoice rows: 1054
Unique duplicated record groups: 527
Rows belonging to duplicated groups: 1054
Excess duplicate rows within invoice: 527

Most frequently repeated records:


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,occurrences,source_positions,row_gaps
0,536544,11001,ASSTD DESIGN RACING CAR PEN,3,2010-12-01 14:32:00,3.36,NaN,United Kingdom,2,"[504783, 527307]",[22524]
1,536544,16236,KITTY PENCIL ERASERS,1,2010-12-01 14:32:00,0.43,NaN,United Kingdom,2,"[504784, 527308]",[22524]
2,536544,16237,SLEEPING CAT ERASERS,4,2010-12-01 14:32:00,0.43,NaN,United Kingdom,2,"[504785, 527309]",[22524]
3,536544,16238,PARTY TIME PENCIL ERASERS,1,2010-12-01 14:32:00,0.43,NaN,United Kingdom,2,"[504786, 527310]",[22524]
4,536544,17003,BROCADE RING PURSE,3,2010-12-01 14:32:00,0.43,NaN,United Kingdom,2,"[504787, 527311]",[22524]
5,536544,20622,VIPPASSPORT COVER,1,2010-12-01 14:32:00,4.21,NaN,United Kingdom,2,"[504797, 527321]",[22524]
6,536544,20658,RED RETROSPOT LUGGAGE TAG,1,2010-12-01 14:32:00,2.51,NaN,United Kingdom,2,"[504798, 527322]",[22524]
7,536544,20662,FIRST CLASS HOLIDAY PURSE,1,2010-12-01 14:32:00,2.51,NaN,United Kingdom,2,"[504799, 527323]",[22524]
8,536544,20668,DISCO BALL CHRISTMAS DECORATION,2,2010-12-01 14:32:00,0.43,NaN,United Kingdom,2,"[504800, 527324]",[22524]
9,536544,20669,RED HEART LUGGAGE TAG,1,2010-12-01 14:32:00,2.51,NaN,United Kingdom,2,"[504801, 527325]",[22524]


In [32]:
# Analyze positional gaps across all remaining exact duplicate groups

remaining_duplicate_groups = (
    df_work
    .loc[sorted(remaining_duplicate_positions)]
    .copy()
)

remaining_duplicate_groups["source_position"] = (
    remaining_duplicate_groups.index
)

# Group all identical records and preserve their source positions
remaining_group_analysis = (
    df_work
    .reset_index(names="source_position")
    .groupby(source_columns, dropna=False)
    .agg(
        occurrences=("source_position", "size"),
        source_positions=("source_position", list)
    )
    .reset_index()
)

# Keep groups with more than one occurrence
remaining_group_analysis = remaining_group_analysis[
    remaining_group_analysis["occurrences"] > 1
].copy()

# Calculate gaps between consecutive occurrences
remaining_group_analysis["row_gaps"] = (
    remaining_group_analysis["source_positions"]
    .apply(
        lambda positions: [
            positions[i + 1] - positions[i]
            for i in range(len(positions) - 1)
        ]
    )
)

# Keep only duplicate groups that include a remaining excess position
remaining_group_analysis["has_remaining_excess"] = (
    remaining_group_analysis["source_positions"]
    .apply(
        lambda positions: any(
            position in remaining_duplicate_positions
            for position in positions[1:]
        )
    )
)

remaining_group_analysis = remaining_group_analysis[
    remaining_group_analysis["has_remaining_excess"]
].copy()

# Count the frequency of positional gaps
from collections import Counter

gap_counts = Counter(
    gap
    for gaps in remaining_group_analysis["row_gaps"]
    for gap in gaps
)

gap_frequency = pd.DataFrame(
    gap_counts.items(),
    columns=["row_gap", "frequency"]
).sort_values("frequency", ascending=False)

print("Remaining duplicated record groups:",
      len(remaining_group_analysis))

print("\nMost frequent positional gaps:")

display(gap_frequency.head(20))

Remaining duplicated record groups: 14433

Most frequent positional gaps:


,row_gap,frequency
16,1,1703
6,2,994
12,3,797
174,22524,777
13,4,652
23,5,498
32,6,491
30,7,412
236,22682,401
8,9,330


In [34]:
# Investigate additional candidate positional offsets

candidate_offsets = [22522, 22524, 22682]

comparison_data = df_work[source_columns].reset_index(drop=True)

offset_results = {}

for offset in candidate_offsets:

    original_rows = (
        comparison_data.iloc[:-offset]
        .reset_index(drop=True)
    )

    shifted_rows = (
        comparison_data.iloc[offset:]
        .reset_index(drop=True)
    )

    # Compare all source columns, treating corresponding missing values as equal
    matching_positions = (
        original_rows.eq(shifted_rows)
        | (original_rows.isna() & shifted_rows.isna())
    ).all(axis=1)

    matching_indices = np.flatnonzero(
        matching_positions.to_numpy()
    )

    # Identify consecutive matching sequences
    if len(matching_indices) > 0:

        breaks = np.where(
            np.diff(matching_indices) != 1
        )[0]

        run_starts = np.r_[0, breaks + 1]
        run_ends = np.r_[breaks, len(matching_indices) - 1]

        runs = pd.DataFrame({
            "start_position": matching_indices[run_starts],
            "end_position": matching_indices[run_ends],
            "matching_rows": (
                matching_indices[run_ends]
                - matching_indices[run_starts] + 1
            )
        }).sort_values(
            "matching_rows",
            ascending=False
        ).reset_index(drop=True)

    else:

        runs = pd.DataFrame(columns=[
            "start_position",
            "end_position",
            "matching_rows"
        ])

    # Store results for later investigation
    offset_results[offset] = {
        "matching_indices": matching_indices,
        "matching_positions": matching_positions,
        "runs": runs
    }

    # Display results
    print("\n" + "=" * 70)
    print(f"TESTED OFFSET: {offset:,}")
    print("=" * 70)

    print("Total row comparisons:", len(matching_positions))
    print("Matching row pairs:", len(matching_indices))

    print(
        "Matching percentage:",
        f"{matching_positions.mean() * 100:.4f}%"
    )

    print("Consecutive matching sequences:", len(runs))

    print("\nLargest consecutive matching sequences:")
    display(runs.head(10))


TESTED OFFSET: 22,522
Total row comparisons: 1044849
Matching row pairs: 209
Matching percentage: 0.0200%
Consecutive matching sequences: 41

Largest consecutive matching sequences:


,start_position,end_position,matching_rows
0,524459,524491,33
1,523407,523432,26
2,525372,525395,24
3,511956,511976,21
4,506666,506681,16
5,503705,503718,14
6,507274,507285,12
7,517052,517063,12
8,520005,520012,8
9,503074,503079,6



TESTED OFFSET: 22,524
Total row comparisons: 1044847
Matching row pairs: 791
Matching percentage: 0.0757%
Consecutive matching sequences: 39

Largest consecutive matching sequences:


,start_position,end_position,matching_rows
0,504448,504907,460
1,518940,519169,230
2,507206,507224,19
3,508608,508623,16
4,515030,515041,12
5,521059,521069,11
6,517697,517701,5
7,515976,515978,3
8,524725,524726,2
9,522563,522564,2



TESTED OFFSET: 22,682
Total row comparisons: 1044689
Matching row pairs: 401
Matching percentage: 0.0384%
Consecutive matching sequences: 1

Largest consecutive matching sequences:


,start_position,end_position,matching_rows
0,513791,514191,401


In [35]:
# Inspect the boundaries of the additional repeated sequences

offsets_to_inspect = [22522, 22524, 22682]

for offset in offsets_to_inspect:

    runs = offset_results[offset]["runs"]

    print("\n" + "=" * 75)
    print(f"OFFSET: {offset:,}")
    print("=" * 75)

    # Inspect the five longest sequences for each offset
    for _, run in runs.head(5).iterrows():

        start = int(run["start_position"])
        end = int(run["end_position"])

        original_block = df_work.iloc[start:end + 1]
        shifted_block = df_work.iloc[
            start + offset:end + offset + 1
        ]

        print("\n" + "-" * 65)
        print(
            f"Original positions: {start:,}–{end:,}"
        )
        print(
            f"Shifted positions: {start + offset:,}–{end + offset:,}"
        )
        print(
            "Matching rows:",
            int(run["matching_rows"])
        )

        print("\nOriginal block:")
        print(
            "Invoice range:",
            original_block["Invoice"].iloc[0],
            "to",
            original_block["Invoice"].iloc[-1]
        )
        print(
            "Unique invoices:",
            original_block["Invoice"].nunique()
        )
        print(
            "Date range:",
            original_block["InvoiceDate"].min(),
            "to",
            original_block["InvoiceDate"].max()
        )

        print("\nShifted block:")
        print(
            "Invoice range:",
            shifted_block["Invoice"].iloc[0],
            "to",
            shifted_block["Invoice"].iloc[-1]
        )
        print(
            "Unique invoices:",
            shifted_block["Invoice"].nunique()
        )
        print(
            "Date range:",
            shifted_block["InvoiceDate"].min(),
            "to",
            shifted_block["InvoiceDate"].max()
        )

        # Display the first and last matching records
        print("\nBoundary records:")

        boundary_positions = [
            start,
            end,
            start + offset,
            end + offset
        ]

        display(
            df_work.iloc[boundary_positions][
                source_columns
            ]
        )


OFFSET: 22,522

-----------------------------------------------------------------
Original positions: 524,459–524,491
Shifted positions: 546,981–547,013
Matching rows: 33

Original block:
Invoice range: 538093 to 538093
Unique invoices: 1
Date range: 2010-12-09 14:49:00 to 2010-12-09 14:49:00

Shifted block:
Invoice range: 538093 to 538093
Unique invoices: 1
Date range: 2010-12-09 14:49:00 to 2010-12-09 14:49:00

Boundary records:


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
524459,538093,22736,RIBBON REEL MAKING SNOWMEN,10,2010-12-09 14:49:00,1.65,12682.0,France
524491,538093,POST,POSTAGE,5,2010-12-09 14:49:00,18.00,12682.0,France
546981,538093,22736,RIBBON REEL MAKING SNOWMEN,10,2010-12-09 14:49:00,1.65,12682.0,France
547013,538093,POST,POSTAGE,5,2010-12-09 14:49:00,18.00,12682.0,France



-----------------------------------------------------------------
Original positions: 523,407–523,432
Shifted positions: 545,929–545,954
Matching rows: 26

Original block:
Invoice range: 538053 to 538055
Unique invoices: 2
Date range: 2010-12-09 13:34:00 to 2010-12-09 13:34:00

Shifted block:
Invoice range: 538053 to 538055
Unique invoices: 2
Date range: 2010-12-09 13:34:00 to 2010-12-09 13:34:00

Boundary records:


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
523407,538053,84029E,RED WOOLLY HOTTIE WHITE HEART.,24,2010-12-09 13:34:00,3.39,15856.0,United Kingdom
523432,538055,22454,NaN,-20,2010-12-09 13:34:00,0.00,NaN,United Kingdom
545929,538053,84029E,RED WOOLLY HOTTIE WHITE HEART.,24,2010-12-09 13:34:00,3.39,15856.0,United Kingdom
545954,538055,22454,NaN,-20,2010-12-09 13:34:00,0.00,NaN,United Kingdom



-----------------------------------------------------------------
Original positions: 525,372–525,395
Shifted positions: 547,894–547,917
Matching rows: 24

Original block:
Invoice range: 538170 to 538170
Unique invoices: 1
Date range: 2010-12-09 19:32:00 to 2010-12-09 19:32:00

Shifted block:
Invoice range: 538170 to 538170
Unique invoices: 1
Date range: 2010-12-09 19:32:00 to 2010-12-09 19:32:00

Boundary records:


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
525372,538170,22834,HAND WARMER BABUSHKA DESIGN,20,2010-12-09 19:32:00,2.10,13969.0,United Kingdom
525395,538170,22961,JAM MAKING SET PRINTED,4,2010-12-09 19:32:00,1.45,13969.0,United Kingdom
547894,538170,22834,HAND WARMER BABUSHKA DESIGN,20,2010-12-09 19:32:00,2.10,13969.0,United Kingdom
547917,538170,22961,JAM MAKING SET PRINTED,4,2010-12-09 19:32:00,1.45,13969.0,United Kingdom


... (12 outputs left out)


-----------------------------------------------------------------
Original positions: 515,030–515,041
Shifted positions: 537,554–537,565
Matching rows: 12

Original block:
Invoice range: 537366 to 537366
Unique invoices: 1
Date range: 2010-12-06 12:39:00 to 2010-12-06 12:39:00

Shifted block:
Invoice range: 537366 to 537366
Unique invoices: 1
Date range: 2010-12-06 12:39:00 to 2010-12-06 12:39:00

Boundary records:


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
515030,537366,22569,FELTCRAFT CUSHION BUTTERFLY,10,2010-12-06 12:39:00,3.75,14085.0,United Kingdom
515041,537366,22144,CHRISTMAS CRAFT LITTLE FRIENDS,17,2010-12-06 12:39:00,2.10,14085.0,United Kingdom
537554,537366,22569,FELTCRAFT CUSHION BUTTERFLY,10,2010-12-06 12:39:00,3.75,14085.0,United Kingdom
537565,537366,22144,CHRISTMAS CRAFT LITTLE FRIENDS,17,2010-12-06 12:39:00,2.10,14085.0,United Kingdom



OFFSET: 22,682

-----------------------------------------------------------------
Original positions: 513,791–514,191
Shifted positions: 536,473–536,873
Matching rows: 401

Original block:
Invoice range: 537240 to 537240
Unique invoices: 1
Date range: 2010-12-06 10:08:00 to 2010-12-06 10:08:00

Shifted block:
Invoice range: 537240 to 537240
Unique invoices: 1
Date range: 2010-12-06 10:08:00 to 2010-12-06 10:08:00

Boundary records:


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
513791,537240,20972,PINK CREAM FELT CRAFT TRINKET BOX,8,2010-12-06 10:08:00,2.51,NaN,United Kingdom
514191,537240,22386,JUMBO BAG PINK POLKADOT,9,2010-12-06 10:08:00,1.95,NaN,United Kingdom
536473,537240,20972,PINK CREAM FELT CRAFT TRINKET BOX,8,2010-12-06 10:08:00,2.51,NaN,United Kingdom
536873,537240,22386,JUMBO BAG PINK POLKADOT,9,2010-12-06 10:08:00,1.95,NaN,United Kingdom


In [36]:
# Validate invoice-level duplicate positions against offset matching results

invoices_to_validate = {
    537240: 22682,
    536544: 22524
}

for invoice_id, offset in invoices_to_validate.items():

    print("\n" + "=" * 70)
    print(f"INVOICE: {invoice_id} | OFFSET: {offset:,}")
    print("=" * 70)

    invoice_rows = df_work[
        df_work["Invoice"] == invoice_id
    ].copy()

    invoice_rows["source_position"] = invoice_rows.index

    # Reconstruct exact duplicate groups within this invoice
    invoice_groups = (
        invoice_rows
        .groupby(source_columns, dropna=False)
        .agg(
            occurrences=("source_position", "size"),
            source_positions=("source_position", list)
        )
        .reset_index()
    )

    invoice_groups = invoice_groups[
        invoice_groups["occurrences"] > 1
    ].copy()

    # Extract original and repeated positions
    invoice_groups["original_position"] = (
        invoice_groups["source_positions"].apply(min)
    )

    invoice_groups["repeated_position"] = (
        invoice_groups["source_positions"].apply(max)
    )

    invoice_groups["observed_gap"] = (
        invoice_groups["repeated_position"]
        - invoice_groups["original_position"]
    )

    # Check whether the observed gap equals the candidate offset
    invoice_groups["gap_matches_candidate"] = (
        invoice_groups["observed_gap"] == offset
    )

    # Check whether each pair is identified by the dataset-wide test
    matched_indices = set(
        offset_results[offset]["matching_indices"]
    )

    invoice_groups["found_in_global_offset_test"] = (
        invoice_groups["original_position"].isin(matched_indices)
    )

    print("Total invoice rows:", len(invoice_rows))
    print("Duplicated record groups:", len(invoice_groups))

    print(
        "Groups with expected positional gap:",
        invoice_groups["gap_matches_candidate"].sum()
    )

    print(
        "Groups found in global offset test:",
        invoice_groups["found_in_global_offset_test"].sum()
    )

    print(
        "Groups with expected gap but not found globally:",
        (
            invoice_groups["gap_matches_candidate"]
            & ~invoice_groups["found_in_global_offset_test"]
        ).sum()
    )

    print("\nObserved positional gap distribution:")

    display(
        invoice_groups["observed_gap"]
        .value_counts()
        .head(10)
        .rename_axis("row_gap")
        .reset_index(name="frequency")
    )

    print("\nGroups not found in global offset test:")

    display(
        invoice_groups[
            invoice_groups["gap_matches_candidate"]
            & ~invoice_groups["found_in_global_offset_test"]
        ][[
            "original_position",
            "repeated_position",
            "observed_gap"
        ]].head(20)
    )


INVOICE: 537240 | OFFSET: 22,682
Total invoice rows: 1136
Duplicated record groups: 568
Groups with expected positional gap: 401
Groups found in global offset test: 401
Groups with expected gap but not found globally: 0

Observed positional gap distribution:


,row_gap,frequency
0,22682,401
1,22122,159
2,22523,8



Groups not found in global offset test:


,original_position,repeated_position,observed_gap



INVOICE: 536544 | OFFSET: 22,524
Total invoice rows: 1054
Duplicated record groups: 527
Groups with expected positional gap: 460
Groups found in global offset test: 460
Groups with expected gap but not found globally: 0

Observed positional gap distribution:


,row_gap,frequency
0,22524,460
1,22523,67



Groups not found in global offset test:


,original_position,repeated_position,observed_gap


In [40]:
def get_consecutive_runs(matching_positions):
    """
    Identify consecutive sequences of True values.

    Returns:
        List of tuples:
        (start_position, end_position, matching_rows)
    """
    matching_indices = np.flatnonzero(matching_positions)

    if len(matching_indices) == 0:
        return []

    split_points = np.where(
        np.diff(matching_indices) != 1
    )[0] + 1

    groups = np.split(matching_indices, split_points)

    runs = [
        (
            group[0],
            group[-1],
            len(group)
        )
        for group in groups
    ]

    return runs

In [41]:
# Test another candidate positional offset

candidate_offset = 22122

matching_results = (
    df_work[source_columns]
    .eq(df_work[source_columns].shift(-candidate_offset))
    |
    (
        df_work[source_columns].isna()
        & df_work[source_columns].shift(-candidate_offset).isna()
    )
).all(axis=1)

matching_indices = matching_results[
    matching_results
].index

matching_positions = matching_results.to_numpy()
runs = get_consecutive_runs(matching_positions)

print("Tested offset:", candidate_offset)
print("Total row comparisons:", len(df_work) - candidate_offset)
print("Matching row pairs:", len(matching_indices))
print(
    "Matching percentage:",
    f"{len(matching_indices) / (len(df_work) - candidate_offset) * 100:.4f}%"
)
print("Consecutive matching sequences:", len(runs))

display(
    pd.DataFrame(
        runs,
        columns=["start_position", "end_position", "matching_rows"]
    )
    .sort_values("matching_rows", ascending=False)
    .head(20)
)

Tested offset: 22122
Total row comparisons: 1045249
Matching row pairs: 159
Matching percentage: 0.0152%
Consecutive matching sequences: 1


,start_position,end_position,matching_rows
0,514192,514350,159


In [42]:
candidate_offset = 22122

sequence_start = 514192
sequence_end = 514350

for label, start, end in [
    ("Original sequence", sequence_start, sequence_end),
    (
        "Corresponding sequence",
        sequence_start + candidate_offset,
        sequence_end + candidate_offset
    )
]:
    print("\n" + "=" * 70)
    print(label)
    print("=" * 70)

    display(
        df_work.iloc[start:end + 1][source_columns]
        .head(5)
    )

    print("\nFirst and last records:")

    display(
        df_work.iloc[[start, end]][source_columns]
    )


Original sequence


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
514192,537240,22358,KINGS CHOICE TEA CADDY,1,2010-12-06 10:08:00,5.91,NaN,United Kingdom
514193,537240,22360,GLASS JAR ENGLISH CONFECTIONERY,1,2010-12-06 10:08:00,5.91,NaN,United Kingdom
514194,537240,22372,AIRLINE BAG VINTAGE WORLD CHAMPION,1,2010-12-06 10:08:00,8.47,NaN,United Kingdom
514195,537240,22375,AIRLINE BAG VINTAGE JET SET BROWN,1,2010-12-06 10:08:00,8.47,NaN,United Kingdom
514196,537240,22377,BOTTLE BAG RETROSPOT,1,2010-12-06 10:08:00,4.21,NaN,United Kingdom



First and last records:


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
514192,537240,22358,KINGS CHOICE TEA CADDY,1,2010-12-06 10:08:00,5.91,NaN,United Kingdom
514350,537240,22942,CHRISTMAS LIGHTS 10 SANTAS,1,2010-12-06 10:08:00,16.98,NaN,United Kingdom



Corresponding sequence


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
536314,537240,22358,KINGS CHOICE TEA CADDY,1,2010-12-06 10:08:00,5.91,NaN,United Kingdom
536315,537240,22360,GLASS JAR ENGLISH CONFECTIONERY,1,2010-12-06 10:08:00,5.91,NaN,United Kingdom
536316,537240,22372,AIRLINE BAG VINTAGE WORLD CHAMPION,1,2010-12-06 10:08:00,8.47,NaN,United Kingdom
536317,537240,22375,AIRLINE BAG VINTAGE JET SET BROWN,1,2010-12-06 10:08:00,8.47,NaN,United Kingdom
536318,537240,22377,BOTTLE BAG RETROSPOT,1,2010-12-06 10:08:00,4.21,NaN,United Kingdom



First and last records:


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
536314,537240,22358,KINGS CHOICE TEA CADDY,1,2010-12-06 10:08:00,5.91,NaN,United Kingdom
536472,537240,22942,CHRISTMAS LIGHTS 10 SANTAS,1,2010-12-06 10:08:00,16.98,NaN,United Kingdom


In [43]:
# Inspect the boundary between the two repeated sequences

boundary_positions = [
    514190, 514191, 514192, 514193,
    536312, 536313, 536314, 536315
]

display(
    df_work.iloc[boundary_positions][source_columns]
)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
514190,537240,15056N,EDWARDIAN PARASOL NATURAL,3,2010-12-06 10:08:00,5.95,NaN,United Kingdom
514191,537240,22386,JUMBO BAG PINK POLKADOT,9,2010-12-06 10:08:00,1.95,NaN,United Kingdom
514192,537240,22358,KINGS CHOICE TEA CADDY,1,2010-12-06 10:08:00,5.91,NaN,United Kingdom
514193,537240,22360,GLASS JAR ENGLISH CONFECTIONERY,1,2010-12-06 10:08:00,5.91,NaN,United Kingdom
536312,537240,20969,RED FLORAL FELTCRAFT SHOULDER BAG,2,2010-12-06 10:08:00,7.62,NaN,United Kingdom
536313,537240,20970,PINK FLORAL FELTCRAFT SHOULDER BAG,3,2010-12-06 10:08:00,7.62,NaN,United Kingdom
536314,537240,22358,KINGS CHOICE TEA CADDY,1,2010-12-06 10:08:00,5.91,NaN,United Kingdom
536315,537240,22360,GLASS JAR ENGLISH CONFECTIONERY,1,2010-12-06 10:08:00,5.91,NaN,United Kingdom


In [45]:
# Compare the two 560-row regions using one constant positional offset

original_start = 513791
original_end = 514350

corresponding_start = 536473
corresponding_end = 537032

# Extract both blocks
original_block = (
    df_work.iloc[original_start:original_end + 1][source_columns]
    .reset_index(drop=True)
)

corresponding_block = (
    df_work.iloc[corresponding_start:corresponding_end + 1][source_columns]
    .reset_index(drop=True)
)

# Display dimensions
print("Original block dimensions:", original_block.shape)
print("Corresponding block dimensions:", corresponding_block.shape)

# Compare all eight columns, treating NaN values as equal
comparison = (
    original_block.eq(corresponding_block)
    |
    (
        original_block.isna()
        & corresponding_block.isna()
    )
).all(axis=1)

# Summarize comparison
print("\nComparison results:")
print("Matching rows:", comparison.sum())
print("Non-matching rows:", (~comparison).sum())
print(
    "Matching percentage:",
    f"{comparison.mean() * 100:.2f}%"
)

# Inspect non-matching records
print("\nFirst 20 non-matching record pairs:")

display(
    pd.concat(
        [
            original_block.loc[~comparison].add_prefix("original_"),
            corresponding_block.loc[~comparison].add_prefix("corresponding_")
        ],
        axis=1
    ).head(20)
)

Original block dimensions: (560, 8)
Corresponding block dimensions: (560, 8)

Comparison results:
Matching rows: 401
Non-matching rows: 159
Matching percentage: 71.61%

First 20 non-matching record pairs:


,original_Invoice,original_StockCode,original_Description,original_Quantity,original_InvoiceDate,original_Price,original_Customer ID,original_Country,corresponding_Invoice,corresponding_StockCode,corresponding_Description,corresponding_Quantity,corresponding_InvoiceDate,corresponding_Price,corresponding_Customer ID,corresponding_Country
401,537240,22358,KINGS CHOICE TEA CADDY,1,2010-12-06 10:08:00,5.91,NaN,United Kingdom,537241,22111,SCOTTIE DOG HOT WATER BOTTLE,9,2010-12-06 10:18:00,4.95,14293.0,United Kingdom
402,537240,22360,GLASS JAR ENGLISH CONFECTIONERY,1,2010-12-06 10:08:00,5.91,NaN,United Kingdom,537241,21485,RETROSPOT HEART HOT WATER BOTTLE,9,2010-12-06 10:18:00,4.95,14293.0,United Kingdom
403,537240,22372,AIRLINE BAG VINTAGE WORLD CHAMPION,1,2010-12-06 10:08:00,8.47,NaN,United Kingdom,537241,22837,HOT WATER BOTTLE BABUSHKA,8,2010-12-06 10:18:00,4.65,14293.0,United Kingdom
404,537240,22375,AIRLINE BAG VINTAGE JET SET BROWN,1,2010-12-06 10:08:00,8.47,NaN,United Kingdom,537241,84032A,CHARLIE+LOLA PINK HOT WATER BOTTLE,6,2010-12-06 10:18:00,2.95,14293.0,United Kingdom
405,537240,22377,BOTTLE BAG RETROSPOT,1,2010-12-06 10:08:00,4.21,NaN,United Kingdom,537241,18007,ESSENTIAL BALM 3.5g TIN IN ENVELOPE,24,2010-12-06 10:18:00,0.18,14293.0,United Kingdom
406,537240,22379,RECYCLING BAG RETROSPOT,6,2010-12-06 10:08:00,4.21,NaN,United Kingdom,537241,84006,MAGIC TREE -PAPER FLOWERS,12,2010-12-06 10:18:00,0.85,14293.0,United Kingdom
407,537240,22382,LUNCH BAG SPACEBOY DESIGN,1,2010-12-06 10:08:00,4.21,NaN,United Kingdom,537241,22632,HAND WARMER RED RETROSPOT,12,2010-12-06 10:18:00,2.10,14293.0,United Kingdom
408,537240,22395,PAPERWEIGHT VINTAGE PAISLEY,1,2010-12-06 10:08:00,5.06,NaN,United Kingdom,537241,22866,HAND WARMER SCOTTY DOG DESIGN,12,2010-12-06 10:18:00,2.10,14293.0,United Kingdom
409,537240,22403,MAGNETS PACK OF 4 VINTAGE LABELS,1,2010-12-06 10:08:00,2.51,NaN,United Kingdom,537241,22834,HAND WARMER BABUSHKA DESIGN,12,2010-12-06 10:18:00,2.10,14293.0,United Kingdom
410,537240,22411,JUMBO SHOPPER VINTAGE RED PAISLEY,1,2010-12-06 10:08:00,4.21,NaN,United Kingdom,537241,22867,HAND WARMER BIRD DESIGN,12,2010-12-06 10:18:00,2.10,14293.0,United Kingdom


In [46]:
# Inspect the transition between corresponding sequences

boundary_positions = [
    536312,
    536313,
    536314,
    536315,
    536871,
    536872,
    536873,
    536874
]

display(
    df_work.iloc[boundary_positions][source_columns]
)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
536312,537240,20969,RED FLORAL FELTCRAFT SHOULDER BAG,2,2010-12-06 10:08:00,7.62,NaN,United Kingdom
536313,537240,20970,PINK FLORAL FELTCRAFT SHOULDER BAG,3,2010-12-06 10:08:00,7.62,NaN,United Kingdom
536314,537240,22358,KINGS CHOICE TEA CADDY,1,2010-12-06 10:08:00,5.91,NaN,United Kingdom
536315,537240,22360,GLASS JAR ENGLISH CONFECTIONERY,1,2010-12-06 10:08:00,5.91,NaN,United Kingdom
536871,537240,15056BL,EDWARDIAN PARASOL BLACK,3,2010-12-06 10:08:00,5.95,NaN,United Kingdom
536872,537240,15056N,EDWARDIAN PARASOL NATURAL,3,2010-12-06 10:08:00,5.95,NaN,United Kingdom
536873,537240,22386,JUMBO BAG PINK POLKADOT,9,2010-12-06 10:08:00,1.95,NaN,United Kingdom
536874,537241,22111,SCOTTIE DOG HOT WATER BOTTLE,9,2010-12-06 10:18:00,4.95,14293.0,United Kingdom


In [47]:
# Verify the two repeated sequences independently

sequence_checks = [
    {
        "name": "401-row sequence",
        "original_start": 513791,
        "original_end": 514191,
        "offset": 22682
    },
    {
        "name": "159-row sequence",
        "original_start": 514192,
        "original_end": 514350,
        "offset": 22122
    }
]

for sequence in sequence_checks:

    start = sequence["original_start"]
    end = sequence["original_end"]
    offset = sequence["offset"]

    original = (
        df_work.iloc[start:end + 1][source_columns]
        .reset_index(drop=True)
    )

    corresponding = (
        df_work.iloc[start + offset:end + offset + 1][source_columns]
        .reset_index(drop=True)
    )

    matches = (
        original.eq(corresponding)
        |
        (
            original.isna()
            & corresponding.isna()
        )
    ).all(axis=1)

    print("\n" + "=" * 65)
    print(sequence["name"])
    print("=" * 65)
    print("Original positions:", start, "-", end)
    print("Corresponding positions:", start + offset, "-", end + offset)
    print("Offset:", offset)
    print("Expected matching rows:", len(original))
    print("Actual matching rows:", matches.sum())
    print("All records match:", matches.all())


401-row sequence
Original positions: 513791 - 514191
Corresponding positions: 536473 - 536873
Offset: 22682
Expected matching rows: 401
Actual matching rows: 401
All records match: True

159-row sequence
Original positions: 514192 - 514350
Corresponding positions: 536314 - 536472
Offset: 22122
Expected matching rows: 159
Actual matching rows: 159
All records match: True


### Investigation of Repeated Record Sequences

The positional analysis identified multiple repeated sequences in the December 2010 transaction records.

A particularly notable case involves invoice `537240`, which contains 568 exact duplicated record groups.

Further investigation identified two consecutive sequences in the original dataset:

| Sequence | Original positions | Corresponding positions | Positional offset | Matching rows |
|---|---|---|---:|---:|
| A | 513,791–514,191 | 536,473–536,873 | 22,682 | 401 |
| B | 514,192–514,350 | 536,314–536,472 | 22,122 | 159 |

Both sequences were independently validated. Every corresponding record matched across all eight original columns.

An additional observation is that the corresponding sequences appear in reverse order relative to their original positions. The two sequences are adjacent in the original dataset, but their corresponding copies appear in the opposite order.

These findings provide strong evidence of systematic record repetition in the source data. However, the precise mechanism responsible for this pattern has not been established.

**Data handling decision:** No records are removed at this stage. The findings will be considered alongside the broader duplicate investigation before defining the final duplicate-handling strategy.

In [49]:
# Identify excess duplicate rows
excess_duplicate_mask = df_work.duplicated(
    subset=source_columns,
    keep="first"
)

excess_duplicate_positions = set(
    df_work.index[excess_duplicate_mask]
)

# Reconstruct matching positions for offset 22,523
offset = 22523

matching_mask = (
    df_work[source_columns]
    .eq(df_work[source_columns].shift(offset))
    .all(axis=1)
)

matching_positions_22523 = set(
    df_work.index[matching_mask]
)

# Keep only the shifted positions that are excess duplicates
matched_excess_positions_22523 = (
    excess_duplicate_positions
    & matching_positions_22523
)

# Identify remaining excess duplicate positions
remaining_excess_positions = (
    excess_duplicate_positions
    - matched_excess_positions_22523
)

print("Total excess duplicate rows:", len(excess_duplicate_positions))
print("Explained by offset 22,523:", len(matched_excess_positions_22523))
print("Remaining excess duplicate rows:", len(remaining_excess_positions))

print(
    "Percentage remaining:",
    f"{len(remaining_excess_positions) / len(excess_duplicate_positions) * 100:.2f}%"
)

Total excess duplicate rows: 34335
Explained by offset 22,523: 12749
Remaining excess duplicate rows: 21586
Percentage remaining: 62.87%


In [50]:
# Identify excess duplicate rows
excess_duplicate_mask = df_work.duplicated(
    subset=source_columns,
    keep="first"
)

excess_duplicate_positions = set(
    df_work.index[excess_duplicate_mask]
)

# Generate row hashes, treating identical rows consistently
row_hashes = pd.util.hash_pandas_object(
    df_work[source_columns],
    index=False
)

# Compare rows at offset 22,523
offset = 22523

matching_mask = (
    row_hashes
    .eq(row_hashes.shift(offset))
)

matching_positions_22523 = set(
    df_work.index[matching_mask]
)

# Identify excess duplicate positions explained by the offset
matched_excess_positions_22523 = (
    excess_duplicate_positions
    & matching_positions_22523
)

# Identify remaining excess duplicate positions
remaining_excess_positions = (
    excess_duplicate_positions
    - matched_excess_positions_22523
)

print("Total excess duplicate rows:", len(excess_duplicate_positions))
print("Explained by offset 22,523:", len(matched_excess_positions_22523))
print("Remaining excess duplicate rows:", len(remaining_excess_positions))

print(
    "Percentage remaining:",
    f"{len(remaining_excess_positions) / len(excess_duplicate_positions) * 100:.2f}%"
)

Total excess duplicate rows: 34335
Explained by offset 22,523: 18972
Remaining excess duplicate rows: 15363
Percentage remaining: 44.74%


In [51]:
# Candidate offsets identified during the investigation
candidate_offsets = [22522, 22523, 22524, 22682, 22122]

# Store row hashes for exact comparison
offset_explanation = {}
all_explained_positions = set()

for offset in candidate_offsets:

    # Identify exact matches at this offset
    matching_mask = row_hashes.eq(
        row_hashes.shift(offset)
    )

    matching_positions = set(
        df_work.index[matching_mask]
    )

    # Keep only later-side positions that are excess duplicates
    explained_positions = (
        matching_positions
        & excess_duplicate_positions
    )

    # Identify newly explained positions
    newly_explained_positions = (
        explained_positions
        - all_explained_positions
    )

    # Store results
    offset_explanation[offset] = {
        "matching_pairs": len(matching_positions),
        "excess_positions_matched": len(explained_positions),
        "newly_explained": len(newly_explained_positions)
    }

    all_explained_positions.update(explained_positions)

# Display offset comparison
offset_summary = pd.DataFrame.from_dict(
    offset_explanation,
    orient="index"
)

offset_summary.index.name = "row_offset"

display(offset_summary)

# Calculate cumulative results
remaining_after_all_offsets = (
    excess_duplicate_positions
    - all_explained_positions
)

print("Total excess duplicate rows:", len(excess_duplicate_positions))
print("Unique excess positions explained:", len(all_explained_positions))
print("Remaining unexplained excess positions:", len(remaining_after_all_offsets))

,matching_pairs,excess_positions_matched,newly_explained
row_offset,,,
22522,209,209,209
22523,18972,18972,18958
22524,791,791,777
22682,401,401,401
22122,159,159,159


Total excess duplicate rows: 34335
Unique excess positions explained: 20504
Remaining unexplained excess positions: 13831


In [52]:
# Reconstruct the remaining excess duplicate positions
remaining_after_all_offsets = (
    excess_duplicate_positions
    - all_explained_positions
)

# Create a DataFrame containing the remaining positions
remaining_excess_df = df_work.loc[
    sorted(remaining_after_all_offsets)
].copy()

# Add the original row position
remaining_excess_df["RowPosition"] = (
    remaining_excess_df.index
)

# Extract year-month
remaining_excess_df["YearMonth"] = (
    remaining_excess_df["InvoiceDate"]
    .dt.to_period("M")
)

# Monthly distribution
monthly_remaining = (
    remaining_excess_df
    .groupby("YearMonth")
    .size()
    .sort_values(ascending=False)
)

print("Remaining excess duplicates by month:")
display(monthly_remaining.to_frame("excess_duplicate_rows"))

# Invoice-level distribution
invoice_remaining = (
    remaining_excess_df
    .groupby("Invoice", dropna=False)
    .agg(
        excess_duplicate_rows=("RowPosition", "size"),
        first_position=("RowPosition", "min"),
        last_position=("RowPosition", "max"),
        unique_products=("StockCode", "nunique")
    )
    .sort_values(
        "excess_duplicate_rows",
        ascending=False
    )
)

print("\nTop 20 invoices by remaining excess duplicates:")
display(invoice_remaining.head(20))

print(
    "\nUnique invoices affected:",
    remaining_excess_df["Invoice"].nunique(dropna=False)
)

Remaining excess duplicates by month:


,excess_duplicate_rows
YearMonth,
2010-12,2519
2010-11,1431
2011-11,1368
2010-10,849
2011-10,773
2010-03,526
2010-06,514
2009-12,506
2010-09,484



Top 20 invoices by remaining excess duplicates:


,excess_duplicate_rows,first_position,last_position,unique_products
Invoice,,,,
537126,119,511095,533622,116
537196,96,512164,534714,74
537374,92,515297,537813,74
536862,81,530831,530911,81
537154,73,511801,534360,54
537382,62,515455,537974,53
537208,61,512505,535091,51
537265,59,514657,537165,38
537133,59,511236,533776,50



Unique invoices affected: 4507


In [53]:
# Inspect two invoices with different positional patterns
invoices_to_inspect = ["537126", "536862"]

for invoice_id in invoices_to_inspect:

    print("\n" + "=" * 70)
    print(f"INVOICE: {invoice_id}")
    print("=" * 70)

    invoice_data = df_work[
        df_work["Invoice"].astype(str) == invoice_id
    ].copy()

    # Group identical records and retain their original positions
    invoice_groups = (
        invoice_data
        .groupby(source_columns, dropna=False)
        .apply(
            lambda group: pd.Series({
                "occurrences": len(group),
                "positions": list(group.index)
            }),
            include_groups=False
        )
        .reset_index()
    )

    # Keep duplicated record groups
    invoice_groups = invoice_groups[
        invoice_groups["occurrences"] > 1
    ].copy()

    # Calculate positional gaps
    invoice_groups["position_gaps"] = (
        invoice_groups["positions"]
        .apply(
            lambda positions: [
                positions[i] - positions[i - 1]
                for i in range(1, len(positions))
            ]
        )
    )

    print("Total invoice rows:", len(invoice_data))
    print("Duplicated record groups:", len(invoice_groups))
    print(
        "Excess duplicate occurrences:",
        (invoice_groups["occurrences"] - 1).sum()
    )

    print("\nPositional gap distribution:")

    all_gaps = [
        gap
        for gaps in invoice_groups["position_gaps"]
        for gap in gaps
    ]

    display(
        pd.Series(all_gaps)
        .value_counts()
        .head(15)
        .rename_axis("row_gap")
        .to_frame("frequency")
    )

    print("\nSample duplicated groups:")
    display(
        invoice_groups[
            ["StockCode", "Description", "occurrences", "positions", "position_gaps"]
        ].head(15)
    )


INVOICE: 537126
Total invoice rows: 236
Duplicated record groups: 117
Excess duplicate occurrences: 119

Positional gap distribution:


,frequency
row_gap,
22521,116
93,2
22428,1



Sample duplicated groups:


,StockCode,Description,occurrences,positions,position_gaps
0,10133,COLOURING PENCILS BROWN TUBE,2,"[511029, 533550]",[22521]
1,10135,COLOURING PENCILS BROWN TUBE,2,"[511028, 533549]",[22521]
2,20674,GREEN POLKADOT BOWL,2,"[511052, 533573]",[22521]
3,20675,BLUE POLKADOT BOWL,2,"[511051, 533572]",[22521]
4,20676,RED RETROSPOT BOWL,2,"[511050, 533571]",[22521]
5,20677,PINK POLKADOT BOWL,2,"[511053, 533574]",[22521]
6,20684,STRAWBERRY DREAM CHILDS UMBRELLA,2,"[511024, 533545]",[22521]
7,20724,RED RETROSPOT CHARLOTTE BAG,2,"[511080, 533601]",[22521]
8,20751,FUNKY WASHING UP GLOVES ASSORTED,2,"[511000, 533521]",[22521]
9,20754,RETROSPOT RED WASHING UP GLOVES,2,"[510999, 533520]",[22521]



INVOICE: 536862
Total invoice rows: 162
Duplicated record groups: 81
Excess duplicate occurrences: 81

Positional gap distribution:


,frequency
row_gap,
22510,68
22591,13



Sample duplicated groups:


,StockCode,Description,occurrences,positions,position_gaps
0,20658,RED RETROSPOT LUGGAGE TAG,2,"[508323, 530833]",[22510]
1,20725,LUNCH BAG RED RETROSPOT,2,"[508324, 530834]",[22510]
2,20941,FROG CANDLE,2,"[508325, 530835]",[22510]
3,21066,VINTAGE RED MUG,2,"[508326, 530836]",[22510]
4,21071,VINTAGE BILLBOARD DRINK ME MUG,2,"[508327, 530837]",[22510]
5,21086,SET/6 RED SPOTTY PAPER CUPS,2,"[508328, 530838]",[22510]
6,21116,OWL DOORSTOP,2,"[508329, 530839]",[22510]
7,21156,RETROSPOT CHILDRENS APRON,2,"[508330, 530840]",[22510]
8,21166,COOK WITH WINE METAL SIGN,2,"[508331, 530841]",[22510]
9,21174,POTTERING IN THE SHED METAL SIGN,2,"[508332, 530842]",[22510]


In [54]:
# Add newly discovered candidate offsets
candidate_offsets = [
    22510,
    22521,
    22522,
    22523,
    22524,
    22591,
    22682,
    22122
]

# Recalculate unique explained positions
offset_explanation = {}
all_explained_positions = set()

for offset in candidate_offsets:

    matching_mask = row_hashes.eq(
        row_hashes.shift(offset)
    )

    matching_positions = set(
        df_work.index[matching_mask]
    )

    explained_positions = (
        matching_positions
        & excess_duplicate_positions
    )

    newly_explained_positions = (
        explained_positions
        - all_explained_positions
    )

    offset_explanation[offset] = {
        "matching_pairs": len(matching_positions),
        "excess_positions_matched": len(explained_positions),
        "newly_explained": len(newly_explained_positions)
    }

    all_explained_positions.update(explained_positions)

# Display results
offset_summary = pd.DataFrame.from_dict(
    offset_explanation,
    orient="index"
)

offset_summary.index.name = "row_offset"

display(
    offset_summary.sort_index()
)

# Updated investigation totals
remaining_after_all_offsets = (
    excess_duplicate_positions
    - all_explained_positions
)

print("Total excess duplicate rows:", len(excess_duplicate_positions))
print("Unique excess positions explained:", len(all_explained_positions))
print("Remaining unexplained excess positions:", len(remaining_after_all_offsets))

print(
    "Percentage explained:",
    f"{len(all_explained_positions) / len(excess_duplicate_positions) * 100:.2f}%"
)

,matching_pairs,excess_positions_matched,newly_explained
row_offset,,,
22122,159,159,159
22510,73,73,73
22521,216,216,216
22522,209,209,209
22523,18972,18972,18929
22524,791,791,777
22591,13,13,13
22682,401,401,401


Total excess duplicate rows: 34335
Unique excess positions explained: 20777
Remaining unexplained excess positions: 13558
Percentage explained: 60.51%


In [55]:
# Investigate consecutive matching sequences for new offsets

new_offsets = [22510, 22521, 22591]

for offset in new_offsets:

    print("\n" + "=" * 70)
    print(f"OFFSET: {offset:,}")
    print("=" * 70)

    # Identify matching row positions
    matching_mask = row_hashes.eq(
        row_hashes.shift(offset)
    )

    matching_positions = sorted(
        df_work.index[matching_mask].tolist()
    )

    # Identify consecutive matching runs
    runs = []

    if matching_positions:

        run_start = matching_positions[0]
        previous_position = matching_positions[0]

        for position in matching_positions[1:]:

            if position == previous_position + 1:
                previous_position = position

            else:
                runs.append({
                    "start_position": run_start,
                    "end_position": previous_position,
                    "matching_rows": (
                        previous_position - run_start + 1
                    )
                })

                run_start = position
                previous_position = position

        # Add the final run
        runs.append({
            "start_position": run_start,
            "end_position": previous_position,
            "matching_rows": (
                previous_position - run_start + 1
            )
        })

    runs_df = pd.DataFrame(runs)

    if not runs_df.empty:

        runs_df = runs_df.sort_values(
            "matching_rows",
            ascending=False
        )

        print("Total matching pairs:", len(matching_positions))
        print("Number of consecutive sequences:", len(runs_df))
        print(
            "Longest consecutive sequence:",
            runs_df["matching_rows"].max()
        )

        print("\nLargest matching sequences:")
        display(runs_df.head(15))

    else:
        print("No matching sequences found.")


OFFSET: 22,510
Total matching pairs: 73
Number of consecutive sequences: 6
Longest consecutive sequence: 68

Largest matching sequences:


,start_position,end_position,matching_rows
2,530831,530898,68
0,529043,529043,1
1,530089,530089,1
3,534176,534176,1
4,535485,535485,1
5,538226,538226,1



OFFSET: 22,521
Total matching pairs: 216
Number of consecutive sequences: 38
Longest consecutive sequence: 118

Largest matching sequences:


,start_position,end_position,matching_rows
15,533505,533622,118
2,527448,527483,36
35,545537,545546,10
3,528593,528602,10
27,538474,538479,6
31,541891,541893,3
37,547251,547252,2
4,530057,530057,1
0,526064,526064,1
1,526290,526290,1



OFFSET: 22,591
Total matching pairs: 13
Number of consecutive sequences: 1
Longest consecutive sequence: 13

Largest matching sequences:


,start_position,end_position,matching_rows
0,530899,530911,13


In [56]:
# Verify the three newly discovered sequences

sequences_to_verify = [
    {
        "name": "Invoice 536862 - 68-row sequence",
        "offset": 22510,
        "start": 530831,
        "end": 530898
    },
    {
        "name": "Invoice 536862 - 13-row sequence",
        "offset": 22591,
        "start": 530899,
        "end": 530911
    },
    {
        "name": "Invoice 537126 - 118-row sequence",
        "offset": 22521,
        "start": 533505,
        "end": 533622
    }
]

for sequence in sequences_to_verify:

    start = sequence["start"]
    end = sequence["end"]
    offset = sequence["offset"]

    original = df_work.iloc[
        start - offset:end - offset + 1
    ][source_columns]

    corresponding = df_work.iloc[
        start:end + 1
    ][source_columns]

    matches = (
        original.reset_index(drop=True)
        .equals(corresponding.reset_index(drop=True))
    )

    print("\n" + "=" * 70)
    print(sequence["name"])
    print("=" * 70)

    print("Offset:", offset)
    print("Original positions:", start - offset, "-", end - offset)
    print("Corresponding positions:", start, "-", end)
    print("Expected matching rows:", end - start + 1)
    print("Actual matching rows:", len(original))
    print("All records match:", matches)

    print("\nOriginal sequence summary:")
    display(
        df_work.iloc[start - offset:end - offset + 1]
        .groupby(["Invoice", "InvoiceDate"], dropna=False)
        .size()
        .reset_index(name="rows")
        .head(10)
    )

    print("\nCorresponding sequence summary:")
    display(
        df_work.iloc[start:end + 1]
        .groupby(["Invoice", "InvoiceDate"], dropna=False)
        .size()
        .reset_index(name="rows")
        .head(10)
    )


Invoice 536862 - 68-row sequence
Offset: 22510
Original positions: 508321 - 508388
Corresponding positions: 530831 - 530898
Expected matching rows: 68
Actual matching rows: 68
All records match: True

Original sequence summary:


,Invoice,InvoiceDate,rows
0,536862,2010-12-03 11:13:00,68



Corresponding sequence summary:


,Invoice,InvoiceDate,rows
0,536862,2010-12-03 11:13:00,68



Invoice 536862 - 13-row sequence
Offset: 22591
Original positions: 508308 - 508320
Corresponding positions: 530899 - 530911
Expected matching rows: 13
Actual matching rows: 13
All records match: True

Original sequence summary:


,Invoice,InvoiceDate,rows
0,536862,2010-12-03 11:13:00,13


... (2 outputs left out)


Invoice 537126 - 118-row sequence
Offset: 22521
Original positions: 510984 - 511101
Corresponding positions: 533505 - 533622
Expected matching rows: 118
Actual matching rows: 118
All records match: True

Original sequence summary:


,Invoice,InvoiceDate,rows
0,537126,2010-12-05 12:13:00,118



Corresponding sequence summary:


,Invoice,InvoiceDate,rows
0,537126,2010-12-05 12:13:00,118


## Duplicate Record Investigation — Findings and Conclusion

### 1. Investigation Objective

The objective of this investigation was to assess exact duplicate records in the Online Retail II dataset, understand their distribution, and determine whether repeated records exhibit systematic positional patterns.

The investigation was conducted before implementing any duplicate-removal rules. This distinction is important because identical transaction lines do not automatically represent erroneous transactions.

### 2. Initial Duplicate Assessment

The combined dataset contains 1,067,371 transaction line items across eight original columns.

The initial exact duplicate assessment produced the following results:

| Measurement | Result |
|---|---:|
| Total records | 1,067,371 |
| Unique duplicated record groups | 32,907 |
| Rows belonging to duplicated groups | 67,242 |
| Excess duplicate occurrences | 34,335 |
| Excess duplicate percentage | 3.2168% |

An excess duplicate occurrence represents an additional instance of an identical record beyond its first occurrence.

For example, if an identical transaction line appears three times, it contributes two excess duplicate occurrences.

### 3. Monthly and Invoice-Level Distribution

Duplicate records were not distributed uniformly across the dataset.

December 2010 showed an unusually high concentration:

| Month | Excess duplicate rows |
|---|---:|
| December 2010 | 23,023 |
| November 2010 | 1,431 |
| November 2011 | 1,368 |
| October 2010 | 849 |

December 2010 alone accounted for approximately 67.05% of all excess duplicate occurrences.

At the invoice level:

- 53,628 unique invoice identifiers were identified.
- 5,391 invoices contained exact duplicate records.
- Approximately 10.05% of invoices were affected.

Several invoices contained extensive repeated product lines, including invoices 537434, 538071, and 537240.

### 4. Investigation of Systematic Positional Repetition

Further investigation revealed that many exact duplicate records appeared at consistent offsets within the original dataset.

The first major finding was an offset of 22,523 rows.

| Measurement | Result |
|---|---:|
| Tested offset | 22,523 |
| Total row comparisons | 1,044,848 |
| Matching row pairs | 18,972 |
| Matching percentage | 1.8158% |
| Largest consecutive matching sequence | 1,207 rows |

The matching sequences were distributed across multiple sections of the dataset, particularly during December 2010.

All 18,972 later-side matching positions were classified as excess duplicate occurrences.

The largest consecutive matching sequence covered original DataFrame positions 512,584–513,790 and corresponding positions 535,107–536,313.

These observations provided strong evidence of repeated sequences within the source data.

### 5. Additional Positional Patterns

The initial offset did not explain all excess duplicate occurrences.

Further analysis identified seven additional candidate offsets.

The following table summarizes the final eight-offset investigation.

| Row offset | Matching pairs | Newly explained excess positions |
|---:|---:|---:|
| 22,122 | 159 | 159 |
| 22,510 | 73 | 73 |
| 22,521 | 216 | 216 |
| 22,522 | 209 | 209 |
| 22,523 | 18,972 | 18,929 |
| 22,524 | 791 | 777 |
| 22,591 | 13 | 13 |
| 22,682 | 401 | 401 |
| **Total unique positions** | | **20,777** |

The newly explained column accounts for overlapping matches, ensuring that each excess duplicate position is counted only once.

The eight offsets collectively explained 20,777 of the 34,335 excess duplicate occurrences.

| Measurement | Result |
|---|---:|
| Total excess duplicate occurrences | 34,335 |
| Unique positions matched at tested offsets | 20,777 |
| Remaining positions not matched at tested offsets | 13,558 |
| Percentage explained | 60.51% |
| Percentage remaining | 39.49% |

The remaining 13,558 positions should not be interpreted as unrelated duplicates. They simply have not been matched through the eight offsets investigated.

### 6. Verified Repeated Sequences

Several sequences were independently verified by comparing the complete original and corresponding DataFrame rows.

#### 6.1 Invoice 537240

Two consecutive sequences were identified:

| Original positions | Corresponding positions | Offset | Matching rows |
|---|---|---:|---:|
| 513,791–514,191 | 536,473–536,873 | 22,682 | 401 |
| 514,192–514,350 | 536,314–536,472 | 22,122 | 159 |

Both sequences matched exactly.

An important observation was that the corresponding sequences appeared in reverse order relative to the original sequences.

The original sequence contained 401 rows followed by 159 rows, while the corresponding section contained 159 rows followed by 401 rows.

Both sections belonged to invoice 537240, with the same timestamp and missing Customer ID.

#### 6.2 Invoice 536862

Two additional consecutive sequences were verified:

| Original positions | Corresponding positions | Offset | Matching rows |
|---|---|---:|---:|
| 508,321–508,388 | 530,831–530,898 | 22,510 | 68 |
| 508,308–508,320 | 530,899–530,911 | 22,591 | 13 |

All 81 row pairs matched exactly.

Both sequences belonged to invoice 536862, with the same timestamp: December 3, 2010, at 11:13.

The original order consisted of 13 rows followed by 68 rows. The corresponding order consisted of 68 rows followed by 13 rows.

This represents another verified example of reordered repeated sequences.

#### 6.3 Invoice 537126

A further sequence was verified:

| Original positions | Corresponding positions | Offset | Matching rows |
|---|---|---:|---:|
| 510,984–511,101 | 533,505–533,622 | 22,521 | 118 |

All 118 row pairs matched exactly.

Both sequences belonged to invoice 537126, with the same timestamp: December 5, 2010, at 12:13.

### 7. Remaining Duplicate Investigation

After accounting for the eight tested offsets, 13,558 excess duplicate positions remained unmatched.

The remaining duplicates were distributed across multiple months.

December 2010 still contained the largest number, with 2,519 remaining excess occurrences.

Other notable months included:

- November 2010: 1,431
- November 2011: 1,368
- October 2010: 849
- October 2011: 773
- March 2010: 526
- June 2010: 514

Additional invoice-level investigation revealed further positional patterns, including:

- Invoice 537126: 116 duplicated groups with a gap of 22,521.
- Invoice 536862: 68 duplicated groups with a gap of 22,510 and 13 groups with a gap of 22,591.

These findings were consistent with the subsequent global offset tests and sequence verification.

### 8. Interpretation and Limitations

The investigation provides strong evidence that a substantial portion of the exact duplicate records is associated with systematic repeated sequences in the source dataset.

The evidence includes:

1. Multiple exact matching sequences at consistent positional offsets.
2. Long consecutive runs of identical records.
3. Repeated sequences belonging to the same invoice and timestamp.
4. Corresponding sequences appearing in different orders.
5. Additional positional patterns identified across multiple invoices.

However, the exact mechanism responsible for these patterns has not been established.

The investigation cannot conclusively determine whether the repetition originated from an export process, data concatenation, source-system behavior, or another process.

Furthermore, a positional match alone does not establish that every repeated transaction line is invalid.

The 13,558 remaining excess duplicate positions also require separate consideration. They have not been explained by the tested offsets, but this does not prove that they represent legitimate transactions or unrelated errors.

### 9. Cleaning Decision

No duplicate records have been removed during this investigation.

The original dataset remains preserved in `df_combined_raw`, and `df_work` remains unchanged.

The investigation establishes the evidence necessary to make informed cleaning decisions later, but it does not justify automatically deleting every exact duplicate occurrence.

Duplicate handling will be determined during the cleaning implementation stage, with attention to transaction semantics, invoice consistency, and the risk of removing legitimate records.

### 10. Conclusion

The initial assessment identified 34,335 excess duplicate occurrences, representing approximately 3.22% of the dataset.

Further positional analysis identified 20,777 unique excess duplicate positions matching eight tested offsets, equivalent to 60.51% of all excess duplicate occurrences.

Several repeated sequences were independently verified, including reordered sequences within invoices 537240 and 536862.

The investigation has therefore established substantial evidence of systematic repetition in the source data while preserving uncertainty about its exact cause.

The duplicate investigation is considered complete for the current assessment stage. No records have been deleted.

# Establish the missing-value baseline

In [57]:
# Missing-value assessment

missing_summary = pd.DataFrame({
    "missing_count": df_work.isna().sum(),
    "missing_percentage": (
        df_work.isna().sum() / len(df_work) * 100
    )
})

missing_summary = missing_summary.sort_values(
    "missing_count",
    ascending=False
)

print("Missing-value summary:")
display(missing_summary)

# Investigate overlap between missing Customer ID and Description
missing_customer_id = df_work["Customer ID"].isna()
missing_description = df_work["Description"].isna()

missing_overlap = pd.Series({
    "Both Customer ID and Description missing": (
        missing_customer_id & missing_description
    ).sum(),

    "Only Customer ID missing": (
        missing_customer_id & ~missing_description
    ).sum(),

    "Only Description missing": (
        ~missing_customer_id & missing_description
    ).sum(),

    "Neither missing": (
        ~missing_customer_id & ~missing_description
    ).sum()
})

print("\nMissing-value overlap:")
display(missing_overlap.to_frame("record_count"))

print(
    "\nTotal records:",
    len(df_work)
)

Missing-value summary:


,missing_count,missing_percentage
Customer ID,243007,22.766873
Description,4382,0.410541
Invoice,0,0.000000
Quantity,0,0.000000
StockCode,0,0.000000
InvoiceDate,0,0.000000
Price,0,0.000000
Country,0,0.000000
YearMonth,0,0.000000



Missing-value overlap:


,record_count
Both Customer ID and Description missing,4382
Only Customer ID missing,238625
Only Description missing,0
Neither missing,824364



Total records: 1067371


In [58]:
# Isolate records with missing Customer IDs
missing_id_df = df_work[df_work["Customer ID"].isna()].copy()

# Classify transaction characteristics
missing_id_df["Invoice_Prefix_C"] = (
    missing_id_df["Invoice"].astype(str).str.startswith("C")
)

missing_id_df["Negative_Quantity"] = (
    missing_id_df["Quantity"] < 0
)

missing_id_df["Zero_Quantity"] = (
    missing_id_df["Quantity"] == 0
)

missing_id_df["Negative_Price"] = (
    missing_id_df["Price"] < 0
)

missing_id_df["Zero_Price"] = (
    missing_id_df["Price"] == 0
)

# Summarize the characteristics
characteristics = pd.DataFrame({
    "record_count": [
        missing_id_df["Invoice_Prefix_C"].sum(),
        (~missing_id_df["Invoice_Prefix_C"]).sum(),
        missing_id_df["Negative_Quantity"].sum(),
        missing_id_df["Zero_Quantity"].sum(),
        missing_id_df["Negative_Price"].sum(),
        missing_id_df["Zero_Price"].sum()
    ]
}, index=[
    "Invoice starts with C",
    "Invoice does not start with C",
    "Negative quantity",
    "Zero quantity",
    "Negative price",
    "Zero price"
])

characteristics["percentage"] = (
    characteristics["record_count"]
    / len(missing_id_df) * 100
)

print("Characteristics of records with missing Customer ID:")
display(characteristics)

print("\nTop 10 countries:")
display(
    missing_id_df["Country"]
    .value_counts()
    .head(10)
    .to_frame("record_count")
)

print("\nMonthly distribution:")
monthly_missing_ids = (
    missing_id_df
    .groupby(missing_id_df["InvoiceDate"].dt.to_period("M"))
    .size()
    .to_frame("missing_customer_id_records")
)

display(monthly_missing_ids)

Characteristics of records with missing Customer ID:


,record_count,percentage
Invoice starts with C,750,0.308633
Invoice does not start with C,242257,99.691367
Negative quantity,4206,1.730814
Zero quantity,0,0.000000
Negative price,5,0.002058
Zero price,6131,2.522973



Top 10 countries:


,record_count
Country,
United Kingdom,240029
EIRE,1671
Hong Kong,364
Unspecified,232
France,128
Switzerland,125
Portugal,116
United Arab Emirates,114
Bahrain,67



Monthly distribution:


,missing_customer_id_records
InvoiceDate,
2009-12,13468
2010-01,9116
2010-02,5482
2010-03,8397
2010-04,6224
2010-05,5719
2010-06,8033
2010-07,5637
2010-08,6364


In [60]:
# Calculate monthly missing Customer ID rates

monthly_customer_id = (
    df_work
    .groupby(df_work["InvoiceDate"].dt.to_period("M"))
    .agg(
        total_records=("Customer ID", "size"),
        missing_customer_ids=(
            "Customer ID",
            lambda x: x.isna().sum()
        )
    )
)

monthly_customer_id["known_customer_ids"] = (
    monthly_customer_id["total_records"]
    - monthly_customer_id["missing_customer_ids"]
)

monthly_customer_id["missing_percentage"] = (
    monthly_customer_id["missing_customer_ids"]
    / monthly_customer_id["total_records"]
    * 100
)

monthly_customer_id["known_percentage"] = (
    monthly_customer_id["known_customer_ids"]
    / monthly_customer_id["total_records"]
    * 100
)

print("Monthly Customer ID completeness:")
display(monthly_customer_id.round(2))

print("\nHighest monthly missing-ID rate:")
display(
    monthly_customer_id
    .sort_values("missing_percentage", ascending=False)
    .head(5)
    .round(2)
)

Monthly Customer ID completeness:


,total_records,missing_customer_ids,known_customer_ids,missing_percentage,known_percentage
InvoiceDate,,,,,
2009-12,45228,13468,31760,29.78,70.22
2010-01,31555,9116,22439,28.89,71.11
2010-02,29388,5482,23906,18.65,81.35
2010-03,41511,8397,33114,20.23,79.77
2010-04,34057,6224,27833,18.28,81.72
2010-05,35323,5719,29604,16.19,83.81
2010-06,39983,8033,31950,20.09,79.91
2010-07,33383,5637,27746,16.89,83.11
2010-08,33306,6364,26942,19.11,80.89



Highest monthly missing-ID rate:


,total_records,missing_customer_ids,known_customer_ids,missing_percentage,known_percentage
InvoiceDate,,,,,
2011-01,35147,13235,21912,37.66,62.34
2010-12,65004,23351,41653,35.92,64.08
2011-12,25526,7864,17662,30.81,69.19
2011-07,39518,12016,27502,30.41,69.59
2009-12,45228,13468,31760,29.78,70.22


In [61]:
# Calculate Customer ID completeness by country

country_customer_id = (
    df_work
    .groupby("Country")
    .agg(
        total_records=("Customer ID", "size"),
        missing_customer_ids=(
            "Customer ID",
            lambda x: x.isna().sum()
        )
    )
)

country_customer_id["known_customer_ids"] = (
    country_customer_id["total_records"]
    - country_customer_id["missing_customer_ids"]
)

country_customer_id["missing_percentage"] = (
    country_customer_id["missing_customer_ids"]
    / country_customer_id["total_records"]
    * 100
)

country_customer_id["known_percentage"] = (
    country_customer_id["known_customer_ids"]
    / country_customer_id["total_records"]
    * 100
)

# Focus on countries with at least 1,000 records
country_customer_id_filtered = (
    country_customer_id[
        country_customer_id["total_records"] >= 1000
    ]
    .sort_values("missing_percentage", ascending=False)
)

print("Customer ID completeness by country")
display(country_customer_id_filtered.round(2))

print("\nCountries with the highest missing-ID rates:")
display(
    country_customer_id_filtered
    .head(10)
    .round(2)
)

Customer ID completeness by country


,total_records,missing_customer_ids,known_customer_ids,missing_percentage,known_percentage
Country,,,,,
United Kingdom,981330,240029,741301,24.46,75.54
EIRE,17866,1671,16195,9.35,90.65
Portugal,2620,116,2504,4.43,95.57
Switzerland,3189,125,3064,3.92,96.08
Sweden,1364,19,1345,1.39,98.61
France,14330,128,14202,0.89,99.11
Australia,1913,0,1913,0.00,100.00
Belgium,3123,0,3123,0.00,100.00
Germany,17624,0,17624,0.00,100.00



Countries with the highest missing-ID rates:


,total_records,missing_customer_ids,known_customer_ids,missing_percentage,known_percentage
Country,,,,,
United Kingdom,981330,240029,741301,24.46,75.54
EIRE,17866,1671,16195,9.35,90.65
Portugal,2620,116,2504,4.43,95.57
Switzerland,3189,125,3064,3.92,96.08
Sweden,1364,19,1345,1.39,98.61
France,14330,128,14202,0.89,99.11
Australia,1913,0,1913,0.00,100.00
Belgium,3123,0,3123,0.00,100.00
Germany,17624,0,17624,0.00,100.00


In [62]:
# Analyze Customer ID completeness at invoice level

invoice_customer_id = (
    df_work
    .groupby("Invoice")
    .agg(
        total_lines=("Customer ID", "size"),
        missing_id_lines=(
            "Customer ID",
            lambda x: x.isna().sum()
        ),
        known_id_lines=(
            "Customer ID",
            lambda x: x.notna().sum()
        ),
        unique_known_customer_ids=(
            "Customer ID",
            "nunique"
        )
    )
)

# Classify invoices by Customer ID completeness
invoice_customer_id["id_status"] = "All IDs known"

invoice_customer_id.loc[
    invoice_customer_id["missing_id_lines"]
    == invoice_customer_id["total_lines"],
    "id_status"
] = "All IDs missing"

invoice_customer_id.loc[
    (invoice_customer_id["missing_id_lines"] > 0)
    & (invoice_customer_id["known_id_lines"] > 0),
    "id_status"
] = "Mixed: known and missing IDs"

# Summarize invoice-level patterns
invoice_status_summary = (
    invoice_customer_id
    .groupby("id_status")
    .agg(
        invoice_count=("total_lines", "size"),
        transaction_lines=("total_lines", "sum"),
        missing_id_lines=("missing_id_lines", "sum")
    )
)

invoice_status_summary["invoice_percentage"] = (
    invoice_status_summary["invoice_count"]
    / len(invoice_customer_id) * 100
)

print("Invoice-level Customer ID completeness:")
display(invoice_status_summary.round(2))

# Investigate invoices containing multiple known Customer IDs
multiple_customer_invoices = invoice_customer_id[
    invoice_customer_id["unique_known_customer_ids"] > 1
]

print(
    "\nInvoices containing multiple known Customer IDs:",
    len(multiple_customer_invoices)
)

display(
    multiple_customer_invoices
    .sort_values("unique_known_customer_ids", ascending=False)
    .head(10)
)

# Inspect invoices with both known and missing IDs
mixed_invoices = invoice_customer_id[
    invoice_customer_id["id_status"]
    == "Mixed: known and missing IDs"
]

print("\nNumber of mixed invoices:", len(mixed_invoices))

display(mixed_invoices.head(10))

Invoice-level Customer ID completeness:


,invoice_count,transaction_lines,missing_id_lines,invoice_percentage
id_status,,,,
All IDs known,44876,824364,0,83.68
All IDs missing,8752,243007,243007,16.32



Invoices containing multiple known Customer IDs: 0


,total_lines,missing_id_lines,known_id_lines,unique_known_customer_ids,id_status
Invoice,,,,,



Number of mixed invoices: 0


,total_lines,missing_id_lines,known_id_lines,unique_known_customer_ids,id_status
Invoice,,,,,


In [63]:
# Isolate records with missing descriptions
missing_description_df = df_work[
    df_work["Description"].isna()
].copy()

# Build a StockCode-level description profile
stockcode_description_profile = (
    df_work
    .groupby("StockCode")
    .agg(
        total_records=("Description", "size"),
        known_description_records=("Description", "count"),
        unique_descriptions=("Description", "nunique")
    )
)

# Calculate missing description records per StockCode
missing_description_by_stockcode = (
    missing_description_df
    .groupby("StockCode")
    .size()
    .to_frame("missing_description_records")
)

# Combine the two analyses
missing_description_analysis = (
    missing_description_by_stockcode
    .join(stockcode_description_profile)
)

# Classify whether descriptions can potentially be recovered
missing_description_analysis["recovery_category"] = "No known description"

missing_description_analysis.loc[
    missing_description_analysis["unique_descriptions"] == 1,
    "recovery_category"
] = "One consistent known description"

missing_description_analysis.loc[
    missing_description_analysis["unique_descriptions"] > 1,
    "recovery_category"
] = "Multiple known descriptions"

# Summarize potential recovery opportunities
recovery_summary = (
    missing_description_analysis
    .groupby("recovery_category")
    .agg(
        stockcode_count=("missing_description_records", "size"),
        affected_records=("missing_description_records", "sum")
    )
)

print("Potential description recovery:")
display(recovery_summary)

print("\nStockCodes with the most missing descriptions:")
display(
    missing_description_analysis
    .sort_values(
        "missing_description_records",
        ascending=False
    )
    .head(20)
)

Potential description recovery:


,stockcode_count,affected_records
recovery_category,,
Multiple known descriptions,480,957
No known description,355,363
One consistent known description,1616,3062



StockCodes with the most missing descriptions:


,missing_description_records,total_records,known_description_records,unique_descriptions,recovery_category
StockCode,,,,,
22139,12,2196,2184,4,Multiple known descriptions
84990,12,218,206,2,Multiple known descriptions
35965,11,198,187,4,Multiple known descriptions
79321,11,1222,1211,1,One consistent known description
22950,10,372,362,3,Multiple known descriptions
23084,10,1067,1057,7,Multiple known descriptions
22087,9,910,901,2,Multiple known descriptions
22084,9,852,843,1,One consistent known description
71477,8,940,932,4,Multiple known descriptions


In [64]:
# Select StockCodes with multiple known descriptions
ambiguous_stockcodes = (
    missing_description_analysis[
        missing_description_analysis["recovery_category"]
        == "Multiple known descriptions"
    ]
    .sort_values(
        "missing_description_records",
        ascending=False
    )
    .head(15)
    .index
)

# Inspect known descriptions and their frequencies
for stockcode in ambiguous_stockcodes:

    descriptions = (
        df_work.loc[
            df_work["StockCode"] == stockcode,
            "Description"
        ]
        .dropna()
        .value_counts()
    )

    print(f"\nStockCode: {stockcode}")
    print(
        "Missing description records:",
        missing_description_analysis.loc[
            stockcode,
            "missing_description_records"
        ]
    )
    print("Known descriptions and frequencies:")

    display(
        descriptions.to_frame("record_count")
    )


StockCode: 84990
Missing description records: 12
Known descriptions and frequencies:


,record_count
Description,
60 GOLD AND SILVER FAIRY CAKE CASES,205
?,1



StockCode: 22139
Missing description records: 12
Known descriptions and frequencies:


,record_count
Description,
RETROSPOT TEA SET CERAMIC 11 PC,1368
RETRO SPOT TEA SET CERAMIC 11 PC,813
amazon,2
11 PC CERAMIC TEA SET POLKADOT,1



StockCode: 35965
Missing description records: 11
Known descriptions and frequencies:


,record_count
Description,
FOLKART HEART NAPKIN RINGS,184
found some more on shelf,1
found,1
dotcom,1


... (20 outputs left out)


StockCode: 23348
Missing description records: 5
Known descriptions and frequencies:


,record_count
Description,
CHILDRENS TOY COOKING UTENSIL SET,181
sold with wrong barcode,1
check,1



StockCode: 22719
Missing description records: 5
Known descriptions and frequencies:


,record_count
Description,
GUMBALL MONOCHROME COAT RACK,251
"GUMBALL COATHOOK., BLACK & WHITE",5
22467,1
wrong barcode (22467),1
sold as 22467,1
wrong code,1


In [65]:
# Identify StockCodes with exactly one known description
recoverable_stockcodes = (
    missing_description_analysis[
        missing_description_analysis["recovery_category"]
        == "One consistent known description"
    ].index
)

# Retrieve all records associated with those StockCodes
recoverable_records = df_work[
    df_work["StockCode"].isin(recoverable_stockcodes)
].copy()

# Extract unique known descriptions per StockCode
description_consistency = (
    recoverable_records
    .dropna(subset=["Description"])
    .groupby("StockCode")
    .agg(
        unique_descriptions=("Description", "nunique"),
        first_seen=("InvoiceDate", "min"),
        last_seen=("InvoiceDate", "max"),
        total_records=("Description", "size")
    )
)

# Verify consistency
print(
    "StockCodes with exactly one known description:",
    len(description_consistency)
)

print(
    "StockCodes with more than one known description:",
    (description_consistency["unique_descriptions"] > 1).sum()
)

print(
    "StockCodes with exactly one known description:",
    (description_consistency["unique_descriptions"] == 1).sum()
)

display(description_consistency.head(20))

StockCodes with exactly one known description: 1616
StockCodes with more than one known description: 0
StockCodes with exactly one known description: 1616


,unique_descriptions,first_seen,last_seen,total_records
StockCode,,,,
10002,1,2009-12-01 09:08:00,2011-04-18 12:56:00,398
10109,1,2009-12-03 12:31:00,2009-12-03 12:31:00,1
10134,1,2009-12-03 14:28:00,2010-11-12 13:43:00,57
10138,1,2009-12-02 12:43:00,2010-04-20 12:46:00,25
15030,1,2009-12-13 14:38:00,2011-12-06 12:21:00,27
15036,1,2009-12-01 16:05:00,2011-12-09 10:03:00,1018
16010,1,2010-01-17 12:12:00,2011-03-28 14:27:00,5
16015,1,2009-12-03 14:36:00,2011-11-17 17:34:00,66
16033,1,2009-12-08 12:25:00,2011-02-17 15:40:00,54


In [66]:
# Retrieve one known description for each candidate StockCode

candidate_descriptions = (
    recoverable_records
    .dropna(subset=["Description"])
    .groupby("StockCode")
    .agg(
        known_description=("Description", "first"),
        known_description_count=("Description", "count"),
        total_records=("StockCode", "size")
    )
)

# Add missing description counts
candidate_descriptions = candidate_descriptions.join(
    missing_description_analysis[
        ["missing_description_records"]
    ],
    how="left"
)

candidate_descriptions["missing_description_records"] = (
    candidate_descriptions["missing_description_records"]
    .fillna(0)
    .astype(int)
)

# Sort by number of affected missing descriptions
candidate_descriptions = candidate_descriptions.sort_values(
    "missing_description_records",
    ascending=False
)

print("Total candidate StockCodes:", len(candidate_descriptions))

print(
    "Total potentially recoverable records:",
    candidate_descriptions["missing_description_records"].sum()
)

print("\nCandidate descriptions with missing records:")
display(
    candidate_descriptions[
        candidate_descriptions["missing_description_records"] > 0
    ].head(50)
)

Total candidate StockCodes: 1616
Total potentially recoverable records: 3062

Candidate descriptions with missing records:


,known_description,known_description_count,total_records,missing_description_records
StockCode,,,,
79321,CHILLI LIGHTS,1211,1211,11
22084,PAPER CHAIN KIT EMPIRE,843,843,9
37461,FUNKY MONKEY MUG,98,98,8
37509,NEW ENGLAND MUG W GIFT BOX,18,18,7
84977,WIRE FLOWER T-LIGHT HOLDER,45,45,7
35970,ZINC FOLKART SLEIGH BELLS,729,729,7
72803B,OCEAN SCENT CANDLE JEWELLED DRAWER,42,42,7
22528,GARDENERS KNEELING PAD,568,568,7
POST,POSTAGE,2115,2115,7


In [67]:
# Examine candidate StockCode description evidence

candidate_validation = (
    recoverable_records
    .dropna(subset=["Description"])
    .groupby(["StockCode", "Description"])
    .agg(
        record_count=("Invoice", "size"),
        first_seen=("InvoiceDate", "min"),
        last_seen=("InvoiceDate", "max")
    )
    .reset_index()
)

# Add missing description counts
candidate_validation = candidate_validation.merge(
    missing_description_analysis[
        ["missing_description_records"]
    ],
    on="StockCode",
    how="left"
)

# Sort by number of missing records
candidate_validation = candidate_validation.sort_values(
    ["missing_description_records", "record_count"],
    ascending=[False, False]
)

print("Candidate description validation:")
display(candidate_validation.head(30))

# Check whether any candidate description has very limited evidence
print("\nDescription evidence distribution:")

display(
    candidate_validation["record_count"]
    .describe()
    .to_frame("record_count")
)

Candidate description validation:


,StockCode,Description,record_count,first_seen,last_seen,missing_description_records
776,79321,CHILLI LIGHTS,1211,2009-12-01 14:56:00,2011-12-09 12:20:00,11
506,22084,PAPER CHAIN KIT EMPIRE,843,2009-12-02 13:23:00,2011-12-08 15:01:00,9
709,37461,FUNKY MONKEY MUG,98,2009-12-01 12:18:00,2010-12-05 15:12:00,8
1607,POST,POSTAGE,2115,2009-12-01 09:28:00,2011-12-09 12:50:00,7
681,35970,ZINC FOLKART SLEIGH BELLS,729,2010-01-19 12:13:00,2011-12-09 10:26:00,7
579,22528,GARDENERS KNEELING PAD,568,2010-03-26 13:03:00,2010-12-07 15:31:00,7
429,21768,WALL MOUNTED VINTAGE ORGANISER,168,2009-12-01 13:47:00,2010-11-15 09:32:00,7
918,84977,WIRE FLOWER T-LIGHT HOLDER,45,2009-12-01 12:22:00,2011-03-18 11:22:00,7
1233,72803B,OCEAN SCENT CANDLE JEWELLED DRAWER,42,2009-12-03 14:28:00,2010-11-18 14:34:00,7
723,37509,NEW ENGLAND MUG W GIFT BOX,18,2010-03-22 14:30:00,2010-12-06 11:01:00,7



Description evidence distribution:


,record_count
count,1616.000000
mean,131.072401
std,259.791510
min,1.000000
25%,11.000000
50%,38.500000
75%,113.250000
max,2215.000000


In [68]:
# Evaluate candidate descriptions using a minimum evidence threshold

MIN_DESCRIPTION_EVIDENCE = 5

candidate_descriptions["evidence_category"] = "Below threshold"

candidate_descriptions.loc[
    candidate_descriptions["known_description_count"]
    >= MIN_DESCRIPTION_EVIDENCE,
    "evidence_category"
] = "Meets threshold"

# Summarize the results
evidence_summary = (
    candidate_descriptions
    .groupby("evidence_category")
    .agg(
        stockcode_count=("known_description", "size"),
        recoverable_records=("missing_description_records", "sum")
    )
)

print("Description imputation evidence assessment:")
display(evidence_summary)

print("\nTotal candidate missing descriptions:")
print(
    candidate_descriptions["missing_description_records"].sum()
)

print("\nMinimum evidence threshold:")
print(MIN_DESCRIPTION_EVIDENCE)

Description imputation evidence assessment:


,stockcode_count,recoverable_records
evidence_category,,
Below threshold,229,322
Meets threshold,1387,2740



Total candidate missing descriptions:
3062

Minimum evidence threshold:
5


In [69]:
# Create an approved StockCode-to-Description mapping

description_mapping = (
    candidate_descriptions[
        candidate_descriptions["known_description_count"]
        >= MIN_DESCRIPTION_EVIDENCE
    ][["known_description", "known_description_count"]]
    .rename(columns={
        "known_description": "imputed_description",
        "known_description_count": "evidence_count"
    })
    .copy()
)

# Validate mapping uniqueness
print(
    "Mapping contains unique StockCodes:",
    description_mapping.index.is_unique
)

print(
    "Number of StockCodes in mapping:",
    len(description_mapping)
)

print(
    "Total records eligible for description imputation:",
    description_mapping.join(
        missing_description_analysis[
            ["missing_description_records"]
        ]
    )["missing_description_records"].sum()
)

print("\nSample mapping:")
display(description_mapping.head(20))

Mapping contains unique StockCodes: True
Number of StockCodes in mapping: 1387
Total records eligible for description imputation: 2740

Sample mapping:


,imputed_description,evidence_count
StockCode,,
79321,CHILLI LIGHTS,1211
22084,PAPER CHAIN KIT EMPIRE,843
37461,FUNKY MONKEY MUG,98
37509,NEW ENGLAND MUG W GIFT BOX,18
84977,WIRE FLOWER T-LIGHT HOLDER,45
35970,ZINC FOLKART SLEIGH BELLS,729
72803B,OCEAN SCENT CANDLE JEWELLED DRAWER,42
22528,GARDENERS KNEELING PAD,568
POST,POSTAGE,2115


In [70]:
# Create a separate dataset for the missing-description transformation

df_description_cleaning = df_work.copy(deep=True)

# Track which descriptions were originally missing
df_description_cleaning["Description_was_missing"] = (
    df_description_cleaning["Description"].isna()
)

# Map approved descriptions using StockCode
mapped_descriptions = (
    df_description_cleaning["StockCode"]
    .map(description_mapping["imputed_description"])
)

# Fill only missing descriptions with approved mappings
imputation_mask = (
    df_description_cleaning["Description"].isna()
    & mapped_descriptions.notna()
)

df_description_cleaning.loc[
    imputation_mask, "Description"
] = mapped_descriptions[imputation_mask]

# Report results
print("Original missing descriptions:", missing_description.sum())
print("Descriptions imputed:", imputation_mask.sum())
print(
    "Remaining missing descriptions:",
    df_description_cleaning["Description"].isna().sum()
)

print(
    "Original dataset unchanged:",
    df_work["Description"].isna().sum() == missing_description.sum()
)

Original missing descriptions: 4382
Descriptions imputed: 2740
Remaining missing descriptions: 1642
Original dataset unchanged: True


In [73]:
# Validate description imputation

imputed_mask = (
    df_description_cleaning["Description_was_missing"]
    & df_description_cleaning["Description"].notna()
)

# Check 1: Original known descriptions remain unchanged
original_nonmissing = df_work["Description"].notna()

unchanged_originals = (
    df_description_cleaning.loc[original_nonmissing, "Description"]
    .equals(df_work.loc[original_nonmissing, "Description"])
)

# Check 2: Imputed descriptions match the approved mapping
expected_descriptions = (
    df_description_cleaning.loc[imputed_mask, "StockCode"]
    .map(description_mapping["imputed_description"])
)

actual_descriptions = df_description_cleaning.loc[
    imputed_mask, "Description"
]

imputed_values_match = (
    expected_descriptions.notna()
    & actual_descriptions.eq(expected_descriptions)
).all()

# Check 3: Count imputed records
imputed_count = imputed_mask.sum()

print("Original descriptions preserved:", unchanged_originals)
print("Imputed descriptions match mapping:", imputed_values_match)
print("Total imputed descriptions:", imputed_count)
print(
    "Remaining missing descriptions:",
    df_description_cleaning["Description"].isna().sum()
)

Original descriptions preserved: True
Imputed descriptions match mapping: True
Total imputed descriptions: 2740
Remaining missing descriptions: 1642


## Missing-Value Investigation and Treatment

### 1. Investigation Objective

The objective of this investigation was to identify missing values, understand their distribution and potential causes, and establish defensible treatment strategies before proceeding with further data analysis.

The investigation focused on two columns containing missing values:

- `Customer ID`
- `Description`

The original dataset was preserved throughout the investigation. No missing values were imputed without supporting evidence.

### 2. Initial Missing-Value Assessment

The combined dataset contained 1,067,371 transaction line items.

The initial missing-value assessment identified the following:

| Column | Missing Records | Missing Percentage |
|---|---:|---:|
| Customer ID | 243,007 | 22.7669% |
| Description | 4,382 | 0.4105% |
| Invoice | 0 | 0.0000% |
| Quantity | 0 | 0.0000% |
| StockCode | 0 | 0.0000% |
| InvoiceDate | 0 | 0.0000% |
| Price | 0 | 0.0000% |
| Country | 0 | 0.0000% |

An overlap analysis established that all 4,382 records with missing descriptions also had missing Customer IDs.

| Missing-Value Category | Records |
|---|---:|
| Both Customer ID and Description missing | 4,382 |
| Only Customer ID missing | 238,625 |
| Only Description missing | 0 |
| Neither missing | 824,364 |

This indicates that missing descriptions are entirely contained within the records with missing customer identifiers.

### 3. Customer ID Investigation

#### 3.1 Distribution and Characteristics

A total of 243,007 records (22.77%) had missing Customer IDs.

Further investigation revealed:

- 240,029 missing-ID records originated from the United Kingdom.
- Missing Customer IDs were observed across all 25 calendar months represented in the dataset.
- Monthly missing-ID rates varied considerably, ranging from approximately 14.45% to 37.66%.
- Missing Customer IDs were not exclusive to cancellation transactions. Only 750 missing-ID records had invoice identifiers beginning with `C`.

These findings demonstrate that missing Customer IDs cannot simply be interpreted as cancelled transactions.

#### 3.2 Invoice-Level Investigation

Transactions were grouped by invoice to determine whether invoices contained a mixture of known and missing Customer IDs.

| Invoice Identification | Number of Invoices | Transaction Lines |
|---|---:|---:|
| All Customer IDs known | 44,876 | 824,364 |
| All Customer IDs missing | 8,752 | 243,007 |
| Mixed known and missing Customer IDs | 0 | 0 |
| **Total** | **53,628** | **1,067,371** |

No invoices contained a mixture of identified and unidentified customer records.

This is important because there are no known Customer IDs within an affected invoice that could provide a direct basis for recovering the missing identifier.

#### 3.3 Customer ID Treatment Decision

Customer IDs were not imputed.

Assigning customer identities without reliable supporting evidence could incorrectly associate transactions with customers and distort customer-level metrics, segmentation, and predictive modeling.

The treatment policy is therefore:

- Preserve anonymous transactions in the dataset.
- Exclude records without Customer IDs from analyses requiring reliable customer identity.
- Retain them where appropriate for transaction-level and aggregate revenue analysis.
- Do not infer or fabricate customer identifiers.

Missing Customer IDs are treated as an identity limitation rather than automatically as invalid transaction records.

### 4. Description Investigation

#### 4.1 StockCode-to-Description Relationship

A total of 4,382 records had missing descriptions.

To investigate whether these values could be recovered, the relationship between `StockCode` and known `Description` values was examined.

The investigation identified 2,451 distinct StockCodes associated with missing descriptions.

| Recovery Evidence Category | StockCodes | Missing Description Records |
|---|---:|---:|
| One consistent known description | 1,616 | 3,062 |
| Multiple known descriptions | 480 | 957 |
| No known description | 355 | 363 |
| **Total** | **2,451** | **4,382** |

The results showed that many missing descriptions could potentially be recovered using the corresponding StockCode.

However, further inspection revealed that the `Description` column does not exclusively contain product names. It may also contain operational annotations, corrections, and service descriptions.

Examples included:

- `POSTAGE`
- `damages`
- `found some more on shelf`
- `wrongly sold (22719) barcode`
- `temp adjustment`

Consequently, a simple mode-based imputation strategy was not considered sufficiently reliable.

#### 4.2 Description Evidence Assessment

For StockCodes with exactly one known description, the number of supporting records was evaluated.

The investigation identified 1,616 StockCodes with exactly one distinct known description.

Their supporting record counts were:

| Statistic | Known Records per StockCode |
|---|---:|
| Mean | 131.07 |
| Median | 38.5 |
| Minimum | 1 |
| Maximum | 2,215 |

A minimum evidence threshold of five known records per StockCode was established as a project-specific quality-control criterion.

The resulting assessment was:

| Evidence Category | StockCodes | Missing Records |
|---|---:|---:|
| Meets threshold (5 or more known records) | 1,387 | 2,740 |
| Below threshold (fewer than 5 known records) | 229 | 322 |
| **Total** | **1,616** | **3,062** |

The threshold was used to distinguish stronger description-to-StockCode evidence from mappings supported by very few observations. It is an operational decision for this project, not a universally established statistical threshold.

#### 4.3 Description Imputation Policy

The following treatment rules were established:

| Condition | Treatment |
|---|---|
| One consistent known description with at least 5 supporting records | Impute |
| One consistent known description with fewer than 5 supporting records | Retain as missing |
| Multiple known descriptions | Retain as missing |
| No known description | Retain as missing |

The approved mapping contained 1,387 unique StockCodes and supported the recovery of 2,740 missing descriptions.

Descriptions representing services or other non-merchandise transactions were not automatically excluded, provided the StockCode-to-Description relationship met the evidence criterion.

#### 4.4 Imputation Implementation

A separate working dataset, `df_description_cleaning`, was created from `df_work`.

An additional indicator column, `Description_was_missing`, was introduced to preserve the provenance of each description.

The approved mapping was applied only to records satisfying both conditions:

1. The original Description was missing.
2. The corresponding StockCode existed in the approved mapping.

No originally populated descriptions were overwritten.

#### 4.5 Imputation Validation

The transformation was validated using three checks.

| Validation | Result |
|---|---|
| Original non-missing descriptions preserved | Passed |
| Imputed descriptions match the approved mapping | Passed |
| Total imputed descriptions equal 2,740 | Passed |

The final missing-description distribution was:

| Measurement | Records |
|---|---:|
| Originally missing descriptions | 4,382 |
| Successfully imputed descriptions | 2,740 |
| Remaining missing descriptions | 1,642 |

The imputation recovered approximately 62.53% of the originally missing descriptions.

The remaining 1,642 records were intentionally retained as missing because their StockCode-to-Description relationships did not satisfy the established evidence criteria.

### 5. Final Missing-Value Treatment Summary

| Column | Original Missing Records | Treatment | Current Result |
|---|---:|---|---|
| Customer ID | 243,007 | No imputation; retain anonymous transactions | 243,007 missing |
| Description | 4,382 | Impute using approved StockCode mapping | 1,642 missing |

The investigation established that missing values should not automatically be deleted or imputed.

Customer IDs require reliable customer identity evidence, while descriptions can be recovered when a sufficiently consistent StockCode relationship exists.

### 6. Conclusion

The missing-value investigation established separate treatment strategies for the two affected columns.

Customer IDs were left missing because no reliable evidence supported recovering customer identities. These anonymous transactions remain available for appropriate transaction-level analysis but will not be used in customer-level analyses requiring identified customers.

Descriptions were partially recovered using a conservative StockCode mapping strategy. A total of 2,740 descriptions were successfully imputed, while 1,642 uncertain descriptions were retained as missing.

# Transaction Anomaly Investigation

In [75]:
# Identify cancellation transactions

df_anomaly = df_description_cleaning.copy()

df_anomaly["is_cancellation"] = (
    df_anomaly["Invoice"]
    .astype(str)
    .str.startswith("C")
)

# Summarize cancellation records
cancellation_summary = pd.DataFrame({
    "record_count": [
        df_anomaly["is_cancellation"].sum(),
        (~df_anomaly["is_cancellation"]).sum()
    ],
    "unique_invoices": [
        df_anomaly.loc[
            df_anomaly["is_cancellation"], "Invoice"
        ].nunique(),
        df_anomaly.loc[
            ~df_anomaly["is_cancellation"], "Invoice"
        ].nunique()
    ]
}, index=["Cancellation", "Non-cancellation"])

print("Cancellation transaction summary:")
display(cancellation_summary)

print("\nCancellation percentage:")
print(
    f"{df_anomaly['is_cancellation'].mean() * 100:.2f}%"
)

Cancellation transaction summary:


,record_count,unique_invoices
Cancellation,19494,8292
Non-cancellation,1047877,45336



Cancellation percentage:
1.83%


In [76]:
# Compare quantity characteristics between cancellation
# and non-cancellation transactions

quantity_analysis = (
    df_anomaly
    .groupby("is_cancellation")
    .agg(
        total_records=("Quantity", "size"),
        negative_quantity_records=(
            "Quantity",
            lambda x: (x < 0).sum()
        ),
        zero_quantity_records=(
            "Quantity",
            lambda x: (x == 0).sum()
        ),
        positive_quantity_records=(
            "Quantity",
            lambda x: (x > 0).sum()
        ),
        total_quantity=("Quantity", "sum")
    )
)

quantity_analysis.index = [
    "Non-cancellation",
    "Cancellation"
]

quantity_analysis["negative_quantity_percentage"] = (
    quantity_analysis["negative_quantity_records"]
    / quantity_analysis["total_records"] * 100
)

display(quantity_analysis)

,total_records,negative_quantity_records,zero_quantity_records,positive_quantity_records,total_quantity,negative_quantity_percentage
Non-cancellation,1047877,3457,0,1044420,11099484,0.329905
Cancellation,19494,19493,0,1,-490992,99.994870


In [77]:
# Isolate negative quantities under non-cancellation invoices

negative_non_cancellation = df_anomaly[
    (~df_anomaly["is_cancellation"])
    & (df_anomaly["Quantity"] < 0)
].copy()

print(
    "Negative-quantity records outside cancellations:",
    len(negative_non_cancellation)
)

print(
    "Unique affected invoices:",
    negative_non_cancellation["Invoice"].nunique()
)

print(
    "Unique affected StockCodes:",
    negative_non_cancellation["StockCode"].nunique()
)

print("\nQuantity distribution:")
display(
    negative_non_cancellation["Quantity"].describe()
)

print("\nPrice distribution:")
display(
    negative_non_cancellation["Price"].describe()
)

print("\nMost frequent StockCodes:")
display(
    negative_non_cancellation["StockCode"]
    .value_counts()
    .head(20)
)

Negative-quantity records outside cancellations: 3457
Unique affected invoices: 3393
Unique affected StockCodes: 2473

Quantity distribution:


count    3457.000000
mean     -165.775239
std       673.108503
min     -9600.000000
25%       -82.000000
50%       -26.000000
75%        -8.000000
max        -1.000000
Name: Quantity, dtype: float64


Price distribution:


count    3457.0
mean        0.0
std         0.0
min         0.0
25%         0.0
50%         0.0
75%         0.0
max         0.0
Name: Price, dtype: float64


Most frequent StockCodes:


StockCode
22423     12
46000M     6
82494L     6
22719      6
46000S     5
84016      5
85017A     5
47566B     5
20852      5
85175      5
21830      5
71477      4
21768      4
84559D     4
84990      4
22028      4
35972      4
21040      4
79000      4
20713      4
Name: count, dtype: int64

In [78]:
# Examine descriptions associated with negative quantities
# outside cancellation invoices

description_analysis = (
    negative_non_cancellation
    .groupby("Description", dropna=False)
    .agg(
        record_count=("Invoice", "size"),
        unique_invoices=("Invoice", "nunique"),
        unique_stockcodes=("StockCode", "nunique"),
        total_quantity=("Quantity", "sum")
    )
    .sort_values("record_count", ascending=False)
)

print(
    "Number of distinct descriptions:",
    len(description_analysis)
)

print("\nTop 30 descriptions:")
display(description_analysis.head(30))

Number of distinct descriptions: 1433

Top 30 descriptions:


,record_count,unique_invoices,unique_stockcodes,total_quantity
Description,,,,
NaN,966,957,834,-96295
check,123,121,114,-13329
damages,84,83,80,-8126
?,83,81,79,-37641
damaged,78,78,73,-4625
missing,27,27,26,-16467
sold as set on dotcom,20,20,20,-2393
Damaged,17,17,17,-8013
"Unsaleable, destroyed.",9,9,9,-15644


In [79]:
# Investigate customer identification and invoice context

print("CUSTOMER IDENTIFICATION")
print("-----------------------")

customer_summary = pd.DataFrame({
    "record_count": [
        negative_non_cancellation["Customer ID"].isna().sum(),
        negative_non_cancellation["Customer ID"].notna().sum()
    ],
    "unique_invoices": [
        negative_non_cancellation.loc[
            negative_non_cancellation["Customer ID"].isna(),
            "Invoice"
        ].nunique(),
        negative_non_cancellation.loc[
            negative_non_cancellation["Customer ID"].notna(),
            "Invoice"
        ].nunique()
    ]
}, index=["Missing Customer ID", "Known Customer ID"])

display(customer_summary)

print("\nUnique identified customers:")
print(
    negative_non_cancellation["Customer ID"].nunique()
)

print("\nINVOICE CONTEXT")
print("---------------")

invoice_summary = (
    negative_non_cancellation
    .groupby("Invoice")
    .agg(
        record_count=("Invoice", "size"),
        unique_stockcodes=("StockCode", "nunique"),
        customer_count=("Customer ID", "nunique"),
        date=("InvoiceDate", "first"),
        total_quantity=("Quantity", "sum")
    )
    .sort_values("record_count", ascending=False)
)

print("Number of invoices:", len(invoice_summary))

print("\nInvoices with the most negative-quantity records:")
display(invoice_summary.head(15))

print("\nSAMPLE TRANSACTION RECORDS")

display(
    negative_non_cancellation[
        [
            "Invoice",
            "StockCode",
            "Description",
            "Quantity",
            "InvoiceDate",
            "Customer ID",
            "Country"
        ]
    ].head(15)
)

CUSTOMER IDENTIFICATION
-----------------------


,record_count,unique_invoices
Missing Customer ID,3457,3393
Known Customer ID,0,0



Unique identified customers:
0

INVOICE CONTEXT
---------------
Number of invoices: 3393

Invoices with the most negative-quantity records:


,record_count,unique_stockcodes,customer_count,date,total_quantity
Invoice,,,,,
538162,2,1,0,2010-12-09 17:25:00,-200
536589,2,1,0,2010-12-01 16:50:00,-20
536764,2,1,0,2010-12-02 14:42:00,-76
536996,2,1,0,2010-12-03 15:30:00,-40
536997,2,1,0,2010-12-03 15:30:00,-40
536998,2,1,0,2010-12-03 15:30:00,-12
537000,2,1,0,2010-12-03 15:32:00,-44
537001,2,1,0,2010-12-03 15:33:00,-12
537003,2,1,0,2010-12-03 15:33:00,-4



SAMPLE TRANSACTION RECORDS


,Invoice,StockCode,Description,Quantity,InvoiceDate,Customer ID,Country
263,489464,21733,85123a mixed,-96,2009-12-01 10:52:00,NaN,United Kingdom
283,489463,71477,short,-240,2009-12-01 10:52:00,NaN,United Kingdom
284,489467,85123A,21733 mixed,-192,2009-12-01 10:53:00,NaN,United Kingdom
470,489521,21646,NaN,-50,2009-12-01 11:44:00,NaN,United Kingdom
3114,489655,20683,NaN,-44,2009-12-01 17:26:00,NaN,United Kingdom
3162,489660,35956,lost,-1043,2009-12-01 17:43:00,NaN,United Kingdom
3168,489663,35605A,damages,-117,2009-12-01 18:02:00,NaN,United Kingdom
4296,489806,18010,ASSORTED FRAGRANCE BATH CONFETTI,-770,2009-12-02 12:42:00,NaN,United Kingdom
4538,489820,21133,invcd as 84879?,-720,2009-12-02 13:23:00,NaN,United Kingdom
4566,489821,85049G,CHOCOLATE BOX RIBBONS,-240,2009-12-02 13:25:00,NaN,United Kingdom


In [80]:
# Investigate repeated StockCodes within affected invoices

invoice_stockcode_groups = (
    negative_non_cancellation
    .groupby(["Invoice", "StockCode"], dropna=False)
    .agg(
        record_count=("Quantity", "size"),
        distinct_descriptions=("Description", "nunique"),
        distinct_quantities=("Quantity", "nunique"),
        total_quantity=("Quantity", "sum")
    )
    .reset_index()
)

repeated_groups = invoice_stockcode_groups[
    invoice_stockcode_groups["record_count"] > 1
]

print(
    "Invoice-StockCode groups:",
    len(invoice_stockcode_groups)
)

print(
    "Groups with repeated StockCodes:",
    len(repeated_groups)
)

print(
    "Records belonging to repeated groups:",
    repeated_groups["record_count"].sum()
)

print("\nRepeated StockCode groups:")
display(
    repeated_groups
    .sort_values("record_count", ascending=False)
    .head(20)
)

print("\nExact duplicate records in this subset:")
print(
    negative_non_cancellation.duplicated().sum()
)

Invoice-StockCode groups: 3393
Groups with repeated StockCodes: 64
Records belonging to repeated groups: 128

Repeated StockCode groups:


,Invoice,StockCode,record_count,distinct_descriptions,distinct_quantities,total_quantity
2057,536589,21777,2,1,1,-20
2058,536764,84952C,2,1,1,-76
2059,536996,22712,2,1,1,-40
2060,536997,22028,2,1,1,-40
2061,536998,85067,2,1,1,-12
2062,537000,21414,2,1,1,-44
2063,537001,21653,2,0,1,-12
2064,537003,85126,2,1,1,-4
2065,537004,21814,2,1,1,-60
2066,537005,21692,2,1,1,-140



Exact duplicate records in this subset:
64


In [81]:
# Identify transactions with negative unit prices

negative_price_records = df_description_cleaning[
    df_description_cleaning["Price"] < 0
].copy()

print(
    "Negative-price records:",
    len(negative_price_records)
)

print(
    "Unique affected invoices:",
    negative_price_records["Invoice"].nunique()
)

print(
    "Unique affected StockCodes:",
    negative_price_records["StockCode"].nunique()
)

print("\nNegative-price transaction details:")

display(
    negative_price_records[
        [
            "Invoice",
            "StockCode",
            "Description",
            "Quantity",
            "InvoiceDate",
            "Price",
            "Customer ID",
            "Country"
        ]
    ]
)

Negative-price records: 5
Unique affected invoices: 5
Unique affected StockCodes: 1

Negative-price transaction details:


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
179403,A506401,B,Adjust bad debt,1,2010-04-29 13:36:00,-53594.36,NaN,United Kingdom
276274,A516228,B,Adjust bad debt,1,2010-07-19 11:24:00,-44031.79,NaN,United Kingdom
403472,A528059,B,Adjust bad debt,1,2010-10-20 12:04:00,-38925.87,NaN,United Kingdom
825444,A563186,B,Adjust bad debt,1,2011-08-12 14:51:00,-11062.06,NaN,United Kingdom
825445,A563187,B,Adjust bad debt,1,2011-08-12 14:52:00,-11062.06,NaN,United Kingdom


### Investigation of Negative Unit Prices

#### 1. Identification

Five records were identified with negative unit prices. They belong to five unique invoices and share the same StockCode (`B`).

#### 2. Transaction Characteristics

All five records have the following characteristics:

- StockCode: `B`
- Description: `Adjust bad debt`
- Quantity: 1
- Customer ID: Missing
- Country: United Kingdom
- Unit price: Negative

The recorded negative prices range from -11,062.06 to -53,594.36.

#### 3. Interpretation

The consistent StockCode and description suggest that these records represent bad-debt financial adjustments rather than ordinary merchandise transactions.

Although their calculated transaction values are negative, they should not automatically be interpreted as merchandise returns or sales refunds.

#### 4. Treatment Decision

These records will be retained in the source and working datasets but excluded from merchandise revenue calculations and customer-level purchasing analysis.

They may be considered separately when reconciling financial adjustments.

**Conclusion:** Negative unit prices in these five records are associated with a specific financial adjustment category. They should be classified separately rather than treated as ordinary product transactions or indiscriminately deleted.

# Zero Unit Prices

In [82]:
# Identify transactions with zero unit prices

zero_price_records = df_description_cleaning[
    df_description_cleaning["Price"] == 0
].copy()

print("Zero-price records:", len(zero_price_records))

print(
    "Unique affected invoices:",
    zero_price_records["Invoice"].nunique()
)

print(
    "Unique affected StockCodes:",
    zero_price_records["StockCode"].nunique()
)

print("\nQuantity characteristics:")

display(
    zero_price_records["Quantity"].describe()
)

print("\nMost frequent descriptions:")

display(
    zero_price_records["Description"]
    .value_counts(dropna=False)
    .head(30)
)

Zero-price records: 6202
Unique affected invoices: 5379
Unique affected StockCodes: 2971

Quantity characteristics:


count     6202.000000
mean       -51.729442
std        632.376662
min      -9600.000000
25%        -32.000000
50%         -2.000000
75%          3.000000
max      12540.000000
Name: Quantity, dtype: float64


Most frequent descriptions:


Description
NaN                                    1642
check                                   162
?                                        92
damages                                  84
damaged                                  81
found                                    28
missing                                  27
sold as set on dotcom                    20
CHILLI LIGHTS                            17
Damaged                                  17
OWL DOORSTOP                             16
adjustment                               16
POLYESTER FILLER PAD 45x45cm             12
dotcom                                   12
MINT KITCHEN SCALES                      11
amazon                                   11
POLYESTER FILLER PAD 40x40cm             10
RED KITCHEN SCALES                       10
IVORY KITCHEN SCALES                     10
FRENCH BLUE METAL DOOR SIGN 1            10
smashed                                   9
CHILDRENS GARDEN GLOVES BLUE              9
Found               

In [83]:
# Examine zero-price transactions by quantity direction
# and cancellation status

zero_price_analysis = zero_price_records.copy()

zero_price_analysis["is_cancellation"] = (
    zero_price_analysis["Invoice"]
    .astype(str)
    .str.startswith("C")
)

zero_price_analysis["quantity_direction"] = (
    zero_price_analysis["Quantity"]
    .apply(
        lambda x: (
            "Negative" if x < 0
            else "Positive" if x > 0
            else "Zero"
        )
    )
)

zero_price_summary = (
    zero_price_analysis
    .groupby(
        ["quantity_direction", "is_cancellation"],
        observed=True
    )
    .agg(
        record_count=("Invoice", "size"),
        unique_invoices=("Invoice", "nunique"),
        unique_stockcodes=("StockCode", "nunique"),
        known_customer_records=(
            "Customer ID",
            lambda x: x.notna().sum()
        ),
        missing_customer_records=(
            "Customer ID",
            lambda x: x.isna().sum()
        ),
        total_quantity=("Quantity", "sum")
    )
    .reset_index()
)

zero_price_summary["invoice_type"] = (
    zero_price_summary["is_cancellation"]
    .map({
        True: "Cancellation",
        False: "Non-cancellation"
    })
)

display(
    zero_price_summary[
        [
            "quantity_direction",
            "invoice_type",
            "record_count",
            "unique_invoices",
            "unique_stockcodes",
            "known_customer_records",
            "missing_customer_records",
            "total_quantity"
        ]
    ]
)

,quantity_direction,invoice_type,record_count,unique_invoices,unique_stockcodes,known_customer_records,missing_customer_records,total_quantity
0,Negative,Non-cancellation,3457,3393,2473,0,3457,-573085
1,Positive,Non-cancellation,2745,1986,1395,71,2674,252259


In [84]:
# Isolate positive-quantity, zero-price transactions

zero_price_positive = zero_price_records[
    zero_price_records["Quantity"] > 0
].copy()

print(
    "Positive-quantity, zero-price records:",
    len(zero_price_positive)
)

print(
    "Unique affected invoices:",
    zero_price_positive["Invoice"].nunique()
)

print(
    "Unique identified customers:",
    zero_price_positive["Customer ID"].nunique()
)

print("\nCustomer identification:")
display(
    zero_price_positive["Customer ID"]
    .isna()
    .value_counts()
    .rename(index={
        True: "Missing Customer ID",
        False: "Known Customer ID"
    })
    .to_frame("record_count")
)

print("\nMost frequent descriptions:")

positive_zero_description = (
    zero_price_positive
    .groupby("Description", dropna=False)
    .agg(
        record_count=("Invoice", "size"),
        unique_invoices=("Invoice", "nunique"),
        unique_stockcodes=("StockCode", "nunique"),
        known_customer_records=(
            "Customer ID",
            lambda x: x.notna().sum()
        ),
        total_quantity=("Quantity", "sum")
    )
    .sort_values("record_count", ascending=False)
)

display(positive_zero_description.head(30))

Positive-quantity, zero-price records: 2745
Unique affected invoices: 1986
Unique identified customers: 51

Customer identification:


,record_count
Customer ID,
Missing Customer ID,2674
Known Customer ID,71



Most frequent descriptions:


,record_count,unique_invoices,unique_stockcodes,known_customer_records,total_quantity
Description,,,,,
NaN,676,665,422,0,68149
check,39,39,36,0,1217
found,28,28,28,0,2136
OWL DOORSTOP,15,14,1,0,28
CHILLI LIGHTS,15,15,1,0,147
adjustment,14,14,13,0,365
POLYESTER FILLER PAD 45x45cm,12,12,1,1,1688
amazon,11,8,5,0,226
POLYESTER FILLER PAD 40x40cm,10,10,1,0,1790


In [85]:
# Examine the full invoice context of zero-price records
# associated with identified customers

known_customer_zero_price = zero_price_positive[
    zero_price_positive["Customer ID"].notna()
].copy()

known_customer_invoices = (
    known_customer_zero_price["Invoice"].unique()
)

invoice_context = df_description_cleaning[
    df_description_cleaning["Invoice"].isin(
        known_customer_invoices
    )
].copy()

invoice_context["line_value"] = (
    invoice_context["Quantity"] * invoice_context["Price"]
)

invoice_context_summary = (
    invoice_context
    .groupby("Invoice")
    .agg(
        customer_count=("Customer ID", "nunique"),
        total_records=("Invoice", "size"),
        zero_price_records=(
            "Price",
            lambda x: (x == 0).sum()
        ),
        positive_price_records=(
            "Price",
            lambda x: (x > 0).sum()
        ),
        negative_price_records=(
            "Price",
            lambda x: (x < 0).sum()
        ),
        invoice_value=("line_value", "sum")
    )
)

invoice_context_summary["has_positive_price_purchase"] = (
    invoice_context_summary["positive_price_records"] > 0
)

print("Identified customer invoices:", len(known_customer_invoices))

print("\nInvoice-level context:")
display(
    invoice_context_summary
    .groupby("has_positive_price_purchase")
    .agg(
        invoice_count=("total_records", "size"),
        total_records=("total_records", "sum"),
        zero_price_records=("zero_price_records", "sum"),
        positive_price_records=("positive_price_records", "sum")
    )
)

print("\nSample invoice details:")
display(
    invoice_context[
        [
            "Invoice",
            "StockCode",
            "Description",
            "Quantity",
            "Price",
            "Customer ID"
        ]
    ].head(30)
)

Identified customer invoices: 61

Invoice-level context:


,invoice_count,total_records,zero_price_records,positive_price_records
has_positive_price_purchase,,,,
False,6,9,9,0
True,55,1004,62,942



Sample invoice details:


,Invoice,StockCode,Description,Quantity,Price,Customer ID
4671,489825,20972,PINK CREAM FELT CRAFT TRINKET BOX,24,1.25,16126.0
4672,489825,20971,PINK BLUE FELT CRAFT TRINKET BOX,24,1.25,16126.0
4673,489825,22147,FELTCRAFT BUTTERFLY HEARTS,12,1.45,16126.0
4674,489825,22076,6 RIBBONS EMPIRE,12,0.00,16126.0
4675,489825,22322,BIRD DECORATION GREEN SPOT,12,0.85,16126.0
4676,489825,22321,BIRD DECORATION RED SPOT,12,0.85,16126.0
4677,489825,21843,RETRO SPOT CAKE STAND,1,10.95,16126.0
4678,489825,21632,VINTAGE PINK DECORATIVE PARASOL,1,12.75,16126.0
4679,489825,72760B,VINTAGE CREAM 3 BASKET CAKE STAND,2,9.95,16126.0
4680,489825,21837,CROSS STITCH ALPHABET CUSHION COVER,3,5.95,16126.0


In [86]:
# Inspect all customer-identified, positive-quantity,
# zero-price transaction records

zero_price_customer_detail = (
    known_customer_zero_price[
        [
            "Invoice",
            "StockCode",
            "Description",
            "Quantity",
            "InvoiceDate",
            "Customer ID",
            "Country"
        ]
    ]
    .copy()
)

# Add invoice-level context
zero_price_customer_detail["has_positive_price_purchase"] = (
    zero_price_customer_detail["Invoice"]
    .map(
        invoice_context_summary[
            "has_positive_price_purchase"
        ]
    )
)

zero_price_customer_detail = (
    zero_price_customer_detail
    .sort_values(["Customer ID", "Invoice", "StockCode"])
)

print(
    "Total identified zero-price records:",
    len(zero_price_customer_detail)
)

print("\nInvoice context distribution:")

display(
    zero_price_customer_detail[
        "has_positive_price_purchase"
    ]
    .value_counts()
    .rename(index={
        True: "Mixed invoice",
        False: "Zero-price-only invoice"
    })
    .to_frame("record_count")
)

print("\nAll identified zero-price records:")

display(zero_price_customer_detail)

Total identified zero-price records: 71

Invoice context distribution:


,record_count
has_positive_price_purchase,
Mixed invoice,62
Zero-price-only invoice,9



All identified zero-price records:


,Invoice,StockCode,Description,Quantity,InvoiceDate,Customer ID,Country,has_positive_price_purchase
723844,554037,22619,SET OF 6 SOLDIER SKITTLES,80,2011-05-20 14:13:00,12415.0,Australia,True
961889,574138,23234,BISCUIT TIN VINTAGE CHRISTMAS,216,2011-11-03 11:26:00,12415.0,Australia,True
48342,493899,22355,"CHARLOTTE BAG , SUKI DESIGN",10,2010-01-08 10:43:00,12417.0,Belgium,True
964822,574469,22385,JUMBO BAG SPACEBOY DESIGN,12,2011-11-04 11:55:00,12431.0,Australia,True
962422,574252,M,Manual,1,2011-11-03 13:24:00,12437.0,France,True
...,...,...,...,...,...,...,...,...
612250,543599,84535B,FAIRY CAKES NOTEBOOK A6 SIZE,16,2011-02-10 13:08:00,17560.0,United Kingdom,False
364333,524701,22218,CAKE STAND LACE WHITE,2,2010-09-30 12:19:00,17667.0,United Kingdom,True
713074,553000,47566,PARTY BUNTING,4,2011-05-12 15:21:00,17667.0,United Kingdom,True
945865,572893,21208,PASTEL COLOUR HONEYCOMB FAN,5,2011-10-26 14:36:00,18059.0,United Kingdom,True


In [87]:
# Identify invoices containing known-customer zero-price records
# but no positive-price transaction lines

zero_price_only_invoices = (
    invoice_context_summary[
        ~invoice_context_summary[
            "has_positive_price_purchase"
        ]
    ].index
)

zero_price_only_detail = (
    invoice_context[
        invoice_context["Invoice"].isin(
            zero_price_only_invoices
        )
    ][
        [
            "Invoice",
            "StockCode",
            "Description",
            "Quantity",
            "Price",
            "InvoiceDate",
            "Customer ID",
            "Country"
        ]
    ]
    .sort_values(["Invoice", "StockCode"])
)

print(
    "Zero-price-only invoices:",
    len(zero_price_only_invoices)
)

print(
    "Total records in these invoices:",
    len(zero_price_only_detail)
)

display(zero_price_only_detail)

Zero-price-only invoices: 6
Total records in these invoices: 9


,Invoice,StockCode,Description,Quantity,Price,InvoiceDate,Customer ID,Country
89084,497819,TEST001,This is a test product.,5,0.0,2010-02-12 14:58:00,14103.0,United Kingdom
89180,497843,TEST001,This is a test product.,5,0.0,2010-02-12 15:47:00,14827.0,United Kingdom
612250,543599,84535B,FAIRY CAKES NOTEBOOK A6 SIZE,16,0.0,2011-02-10 13:08:00,17560.0,United Kingdom
840209,564651,21786,POLKADOT RAIN HAT,144,0.0,2011-08-26 14:19:00,14646.0,Netherlands
840208,564651,22955,36 FOIL STAR CAKE CASES,144,0.0,2011-08-26 14:19:00,14646.0,Netherlands
840207,564651,23268,SET OF 2 CERAMIC CHRISTMAS REINDEER,192,0.0,2011-08-26 14:19:00,14646.0,Netherlands
840206,564651,23270,SET OF 2 CERAMIC PAINTED HEARTS,96,0.0,2011-08-26 14:19:00,14646.0,Netherlands
887286,568384,M,Manual,1,0.0,2011-09-27 09:46:00,12748.0,United Kingdom
1027583,578841,84826,ASSTD DESIGN 3D PAPER STICKERS,12540,0.0,2011-11-25 15:57:00,13256.0,United Kingdom


In [88]:
# Investigate extreme transaction quantities

print("Largest positive quantities:")

display(
    df_description_cleaning[
        df_description_cleaning["Quantity"] > 0
    ]
    .sort_values("Quantity", ascending=False)
    [
        [
            "Invoice",
            "StockCode",
            "Description",
            "Quantity",
            "InvoiceDate",
            "Price",
            "Customer ID",
            "Country"
        ]
    ]
    .head(30)
)

print("\nLargest negative quantities:")

display(
    df_description_cleaning[
        df_description_cleaning["Quantity"] < 0
    ]
    .sort_values("Quantity")
    [
        [
            "Invoice",
            "StockCode",
            "Description",
            "Quantity",
            "InvoiceDate",
            "Price",
            "Customer ID",
            "Country"
        ]
    ]
    .head(30)
)

Largest positive quantities:


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
1065882,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2011-12-09 09:15:00,2.08,16446.0,United Kingdom
587080,541431,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,2011-01-18 10:01:00,1.04,12346.0,United Kingdom
90857,497946,37410,BLACK AND WHITE PAISLEY FLOWER MUG,19152,2010-02-15 11:57:00,0.10,13902.0,Denmark
127166,501534,21099,SET/6 STRAWBERRY PAPER CUPS,12960,2010-03-17 13:09:00,0.10,13902.0,Denmark
127168,501534,21091,SET/6 WOODLAND PAPER PLATES,12960,2010-03-17 13:09:00,0.10,13902.0,Denmark
127169,501534,21085,SET/6 WOODLAND PAPER CUPS,12744,2010-03-17 13:09:00,0.10,13902.0,Denmark
1027583,578841,84826,ASSTD DESIGN 3D PAPER STICKERS,12540,2011-11-25 15:57:00,0.00,13256.0,United Kingdom
127167,501534,21092,SET/6 STRAWBERRY PAPER PLATES,12480,2010-03-17 13:09:00,0.10,13902.0,Denmark
192197,507637,84016,FLAG OF ST GEORGE CAR FLAG,10200,2010-05-10 14:55:00,0.00,NaN,United Kingdom
135029,502269,21980,PACK OF 12 RED SPOTTY TISSUES,10000,2010-03-23 15:36:00,0.25,17940.0,United Kingdom



Largest negative quantities:


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
1065883,C581484,23843,"PAPER CRAFT , LITTLE BIRDIE",-80995,2011-12-09 09:27:00,2.08,16446.0,United Kingdom
587085,C541433,23166,MEDIUM CERAMIC TOP STORAGE JAR,-74215,2011-01-18 10:17:00,1.04,12346.0,United Kingdom
750990,556690,23005,printing smudges/thrown away,-9600,2011-06-14 10:37:00,0.00,NaN,United Kingdom
303996,519017,22759,NaN,-9600,2010-08-13 09:14:00,0.00,NaN,United Kingdom
750991,556691,23005,printing smudges/thrown away,-9600,2011-06-14 10:37:00,0.00,NaN,United Kingdom
507225,C536757,84347,ROTATING SILVER ANGELS T-LIGHT HLDR,-9360,2010-12-02 14:23:00,0.03,15838.0,United Kingdom
529729,C536757,84347,ROTATING SILVER ANGELS T-LIGHT HLDR,-9360,2010-12-02 14:23:00,0.03,15838.0,United Kingdom
156488,504311,22197,NaN,-9200,2010-04-12 14:39:00,0.00,NaN,United Kingdom
750989,556687,23003,Printing smudges/thrown away,-9058,2011-06-14 10:36:00,0.00,NaN,United Kingdom
194372,507913,10120,Zebra invcing error,-9000,2010-05-11 17:16:00,0.00,NaN,United Kingdom


In [91]:
# Match the 30 largest positive quantities against
# negative quantities on cancellation invoices

positive_extremes = (
    df_description_cleaning[
        df_description_cleaning["Quantity"] > 0
    ]
    .nlargest(30, "Quantity")
    .copy()
)

cancellation_candidates = (
    df_description_cleaning[
        df_description_cleaning["Invoice"]
        .astype(str).str.startswith("C")
        & (df_description_cleaning["Quantity"] < 0)
    ]
    .copy()
)

# Create matching keys without renaming original columns
positive_extremes["absolute_quantity"] = (
    positive_extremes["Quantity"]
)

cancellation_candidates["absolute_quantity"] = (
    cancellation_candidates["Quantity"].abs()
)

matched_candidates = positive_extremes.merge(
    cancellation_candidates,
    on=[
        "StockCode",
        "absolute_quantity",
        "Customer ID"
    ],
    how="left",
    suffixes=("_original", "_cancellation")
)

# Retain only cancellation records occurring on or after
# the original transaction date
matched_candidates = matched_candidates[
    matched_candidates["Invoice_cancellation"].notna()
    & (
        matched_candidates["InvoiceDate_cancellation"]
        >= matched_candidates["InvoiceDate_original"]
    )
].copy()

matched_candidates["time_difference"] = (
    matched_candidates["InvoiceDate_cancellation"]
    - matched_candidates["InvoiceDate_original"]
)

print(
    "Largest positive records examined:",
    len(positive_extremes)
)

print(
    "Positive records with matching cancellation candidates:",
    matched_candidates["Invoice_original"].nunique()
)

print("\nPotential matching records:")

display(
    matched_candidates[
        [
            "Invoice_original",
            "Invoice_cancellation",
            "StockCode",
            "Description_original",
            "Quantity_original",
            "Quantity_cancellation",
            "InvoiceDate_original",
            "InvoiceDate_cancellation",
            "Price_original",
            "Price_cancellation",
            "Customer ID",
            "time_difference"
        ]
    ].sort_values(
        "Quantity_original",
        ascending=False
    )
)

Largest positive records examined: 30
Positive records with matching cancellation candidates: 4

Potential matching records:


,Invoice_original,Invoice_cancellation,StockCode,Description_original,Quantity_original,Quantity_cancellation,InvoiceDate_original,InvoiceDate_cancellation,Price_original,Price_cancellation,Customer ID,time_difference
0,581483,C581484,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,-80995.0,2011-12-09 09:15:00,2011-12-09 09:27:00,2.08,2.08,16446.0,0 days 00:12:00
1,541431,C541433,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,-74215.0,2011-01-18 10:01:00,2011-01-18 10:17:00,1.04,1.04,12346.0,0 days 00:16:00
15,530715,C536757,84347,ROTATING SILVER ANGELS T-LIGHT HLDR,9360,-9360.0,2010-11-04 11:36:00,2010-12-02 14:23:00,1.69,0.03,15838.0,28 days 02:47:00
16,530715,C536757,84347,ROTATING SILVER ANGELS T-LIGHT HLDR,9360,-9360.0,2010-11-04 11:36:00,2010-12-02 14:23:00,1.69,0.03,15838.0,28 days 02:47:00
18,518505,C524235,21088,SET/6 FRUIT SALAD PAPER CUPS,7128,-7128.0,2010-08-09 13:10:00,2010-09-28 11:02:00,0.08,0.08,14277.0,49 days 21:52:00
21,518505,C524235,21096,SET/6 FRUIT SALAD PAPER PLATES,7008,-7008.0,2010-08-09 13:10:00,2010-09-28 11:02:00,0.13,0.13,14277.0,49 days 21:52:00


In [92]:
# Inspect complete original and cancellation invoices

invoices_to_inspect = [
    "530715",
    "C536757",
    "518505",
    "C524235",
    "541431",
    "C541433"
]

invoice_detail = df_description_cleaning[
    df_description_cleaning["Invoice"]
    .astype(str)
    .isin(invoices_to_inspect)
].copy()

invoice_detail["line_value"] = (
    invoice_detail["Quantity"] * invoice_detail["Price"]
)

invoice_detail = invoice_detail.sort_values(
    ["Invoice", "StockCode", "Quantity"]
)

print("Number of records:", len(invoice_detail))

print("\nInvoice-level summary:")

display(
    invoice_detail
    .groupby("Invoice")
    .agg(
        record_count=("Invoice", "size"),
        unique_stockcodes=("StockCode", "nunique"),
        total_quantity=("Quantity", "sum"),
        total_line_value=("line_value", "sum"),
        customer_count=("Customer ID", "nunique"),
        invoice_date=("InvoiceDate", "first")
    )
)

print("\nInvoice transaction details:")

display(
    invoice_detail[
        [
            "Invoice",
            "StockCode",
            "Description",
            "Quantity",
            "Price",
            "line_value",
            "InvoiceDate",
            "Customer ID"
        ]
    ]
)

Number of records: 96

Invoice-level summary:


,record_count,unique_stockcodes,total_quantity,total_line_value,customer_count,invoice_date
Invoice,,,,,,
518505,45,45,87167,11880.84,1,2010-08-09 13:10:00
530715,2,2,15696,17022.24,1,2010-11-04 11:36:00
541431,1,1,74215,77183.60,1,2011-01-18 10:01:00
C524235,45,45,-87167,-11880.84,1,2010-09-28 11:02:00
C536757,2,1,-18720,-561.60,1,2010-12-02 14:23:00
C541433,1,1,-74215,-77183.60,1,2011-01-18 10:17:00



Invoice transaction details:


,Invoice,StockCode,Description,Quantity,Price,line_value,InvoiceDate,Customer ID
298955,518505,16043,POP ART PUSH DOWN RUBBER,960,0.06,57.60,2010-08-09 13:10:00,14277.0
298954,518505,16046,TEATIME PEN CASE & PENS,4608,0.08,368.64,2010-08-09 13:10:00,14277.0
298953,518505,16047,POP ART PEN CASE & PENS,5184,0.08,414.72,2010-08-09 13:10:00,14277.0
298952,518505,16053,POPART COL BALLPOINT PEN ASST,2000,0.03,60.00,2010-08-09 13:10:00,14277.0
298948,518505,20661,BLUE SPOTTY PURSE,300,0.17,51.00,2010-08-09 13:10:00,14277.0
...,...,...,...,...,...,...,...,...
359673,C524235,85160B,BLACK BIRD GARDEN DESIGN MUG,-2976,0.13,-386.88,2010-09-28 11:02:00,14277.0
359674,C524235,85184D,S/4 BLUE ROUND DECOUPAGE BOXES,-3936,0.42,-1653.12,2010-09-28 11:02:00,14277.0
507225,C536757,84347,ROTATING SILVER ANGELS T-LIGHT HLDR,-9360,0.03,-280.80,2010-12-02 14:23:00,15838.0
529729,C536757,84347,ROTATING SILVER ANGELS T-LIGHT HLDR,-9360,0.03,-280.80,2010-12-02 14:23:00,15838.0


In [93]:
# Reconcile original invoices against their cancellation invoices

invoice_pairs = [
    ("518505", "C524235"),
    ("541431", "C541433"),
    ("530715", "C536757")
]

def reconcile_invoice_pair(original_invoice, cancellation_invoice):

    original = df_description_cleaning[
        df_description_cleaning["Invoice"].astype(str)
        == original_invoice
    ]

    cancellation = df_description_cleaning[
        df_description_cleaning["Invoice"].astype(str)
        == cancellation_invoice
    ]

    # Aggregate records by StockCode to expose repeated lines
    original_summary = (
        original.groupby("StockCode")
        .agg(
            original_record_count=("Quantity", "size"),
            original_quantity=("Quantity", "sum"),
            original_price=("Price", "first"),
            original_price_variations=("Price", "nunique")
        )
        .reset_index()
    )

    cancellation_summary = (
        cancellation.groupby("StockCode")
        .agg(
            cancellation_record_count=("Quantity", "size"),
            cancellation_quantity=("Quantity", "sum"),
            cancellation_price=("Price", "first"),
            cancellation_price_variations=("Price", "nunique")
        )
        .reset_index()
    )

    comparison = original_summary.merge(
        cancellation_summary,
        on="StockCode",
        how="outer",
        indicator=True
    )

    comparison["quantity_reversed"] = (
        comparison["original_quantity"]
        + comparison["cancellation_quantity"]
    ) == 0

    comparison["price_matches"] = (
        comparison["original_price"]
        == comparison["cancellation_price"]
    )

    print(
        f"\nOriginal invoice: {original_invoice} | "
        f"Cancellation invoice: {cancellation_invoice}"
    )

    print("Original StockCodes:", len(original_summary))
    print("Cancellation StockCodes:", len(cancellation_summary))
    print(
        "StockCodes present in both:",
        (comparison["_merge"] == "both").sum()
    )
    print(
        "StockCodes with reversed quantities:",
        comparison["quantity_reversed"].sum()
    )
    print(
        "StockCodes with matching prices:",
        comparison["price_matches"].sum()
    )

    print("\nExceptions requiring investigation:")

    exceptions = comparison[
        (comparison["_merge"] != "both")
        | (~comparison["quantity_reversed"])
        | (~comparison["price_matches"])
        | (comparison["original_record_count"].fillna(0) > 1)
        | (comparison["cancellation_record_count"].fillna(0) > 1)
    ]

    display(exceptions)

for original_invoice, cancellation_invoice in invoice_pairs:
    reconcile_invoice_pair(
        original_invoice,
        cancellation_invoice
    )


Original invoice: 518505 | Cancellation invoice: C524235
Original StockCodes: 45
Cancellation StockCodes: 45
StockCodes present in both: 44
StockCodes with reversed quantities: 44
StockCodes with matching prices: 44

Exceptions requiring investigation:


,StockCode,original_record_count,original_quantity,original_price,original_price_variations,cancellation_record_count,cancellation_quantity,cancellation_price,cancellation_price_variations,_merge,quantity_reversed,price_matches
37,84924D,1.0,372.0,0.25,1.0,NaN,NaN,NaN,NaN,left_only,False,False
39,84925D,NaN,NaN,NaN,NaN,1.0,-372.0,0.25,1.0,right_only,False,False



Original invoice: 541431 | Cancellation invoice: C541433
Original StockCodes: 1
Cancellation StockCodes: 1
StockCodes present in both: 1
StockCodes with reversed quantities: 1
StockCodes with matching prices: 1

Exceptions requiring investigation:


,StockCode,original_record_count,original_quantity,original_price,original_price_variations,cancellation_record_count,cancellation_quantity,cancellation_price,cancellation_price_variations,_merge,quantity_reversed,price_matches



Original invoice: 530715 | Cancellation invoice: C536757
Original StockCodes: 2
Cancellation StockCodes: 1
StockCodes present in both: 1
StockCodes with reversed quantities: 0
StockCodes with matching prices: 0

Exceptions requiring investigation:


,StockCode,original_record_count,original_quantity,original_price,original_price_variations,cancellation_record_count,cancellation_quantity,cancellation_price,cancellation_price_variations,_merge,quantity_reversed,price_matches
0,17003,1,6336,0.19,1,NaN,NaN,NaN,NaN,left_only,False,False
1,84347,1,9360,1.69,1,2.0,-18720.0,0.03,1.0,both,False,False


In [94]:
# Verify whether the repeated C536757 cancellation lines
# are exact duplicates in the original dataset

cancellation_invoice = "C536757"

original_columns = df_combined_raw.columns.tolist()

raw_cancellation_lines = df_combined_raw[
    df_combined_raw["Invoice"].astype(str)
    == cancellation_invoice
].copy()

working_cancellation_lines = df_description_cleaning[
    df_description_cleaning["Invoice"].astype(str)
    == cancellation_invoice
].copy()

print("Original dataset — cancellation invoice details:")

display(raw_cancellation_lines)

print("\nExact duplicate flags in original dataset:")

print(
    raw_cancellation_lines.duplicated(
        subset=original_columns,
        keep=False
    )
)

print(
    "\nExcess duplicate rows in original invoice:",
    raw_cancellation_lines.duplicated(
        subset=original_columns
    ).sum()
)

print(
    "\nExact duplicate flags in working dataset:"
)

print(
    working_cancellation_lines.duplicated(
        subset=original_columns,
        keep=False
    )
)

print(
    "\nOriginal cancellation lines preserved in working dataset:",
    raw_cancellation_lines[original_columns].reset_index(drop=True).equals(
        working_cancellation_lines[original_columns].reset_index(drop=True)
    )
)

Original dataset — cancellation invoice details:


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
507225,C536757,84347,ROTATING SILVER ANGELS T-LIGHT HLDR,-9360,2010-12-02 14:23:00,0.03,15838.0,United Kingdom
529729,C536757,84347,ROTATING SILVER ANGELS T-LIGHT HLDR,-9360,2010-12-02 14:23:00,0.03,15838.0,United Kingdom



Exact duplicate flags in original dataset:
507225    True
529729    True
dtype: bool

Excess duplicate rows in original invoice: 1

Exact duplicate flags in working dataset:
507225    True
529729    True
dtype: bool

Original cancellation lines preserved in working dataset: True


### Investigation of Extreme Transaction Quantities and Order Reversals

#### Objective

Investigate extreme positive and negative transaction quantities to determine whether they represent potential data errors, wholesale transactions, cancellation activity, or operational adjustments.

The investigation focused on matching large positive quantities against negative quantities recorded on cancellation invoices (invoice identifiers beginning with `C`).

#### 1. Matching Positive Transactions with Cancellation Records

The 30 largest positive-quantity records were compared against cancellation records using StockCode, absolute quantity, and Customer ID.

Four original invoice records had matching cancellation candidates, producing six matching rows in the merged results.

Three notable transaction patterns were investigated:

| Original invoice | Cancellation invoice | Finding |
|---|---|---|
| 581483 | C581484 | Exact reversal; quantity 80,995, same unit price, cancellation issued 12 minutes later |
| 541431 | C541433 | Exact reversal; quantity 74,215, same unit price, cancellation issued 16 minutes later |
| 518505 | C524235 | Near-complete invoice reversal; one StockCode discrepancy |
| 530715 | C536757 | Unresolved cancellation anomaly involving duplicate records and a price discrepancy |

#### 2. Invoice-Level Reconciliation

**Invoice 518505 and cancellation C524235**

Both invoices contain 45 records and 45 distinct StockCodes.

- Original total quantity: 87,167
- Cancellation total quantity: -87,167
- Original total line value: £11,880.84
- Cancellation total line value: -£11,880.84

A line-level comparison found 44 StockCodes present in both invoices. All 44 had exactly reversed quantities and matching unit prices.

One discrepancy remained:

- Original: StockCode 84924D, quantity +372, price £0.25.
- Cancellation: StockCode 84925D, quantity -372, price £0.25.

This represents a near-complete invoice reversal with one StockCode discrepancy. The available data does not establish whether this resulted from product substitution, coding inconsistency, or another transaction adjustment.

**Invoice 541431 and cancellation C541433**

This transaction represents an exact line-level reversal:

- StockCode: 23166.
- Original quantity: +74,215.
- Cancellation quantity: -74,215.
- Unit price: £1.04 on both records.
- Customer ID: 12346.
- Time difference: 16 minutes.

**Invoice 530715 and cancellation C536757**

The original invoice contains two distinct StockCodes, while the cancellation invoice contains two identical records for StockCode 84347.

The original transaction for StockCode 84347 has quantity +9,360 and unit price £1.69. The cancellation records each have quantity -9,360 and unit price £0.03.

The second original product, StockCode 17003, has no corresponding cancellation line.

#### 3. Verification of Exact Duplicate Records

The two records in cancellation invoice C536757 were checked against the original, unmodified dataset.

Both records are identical across all eight original columns:

- Invoice: C536757.
- StockCode: 84347.
- Description: ROTATING SILVER ANGELS T-LIGHT HLDR.
- Quantity: -9,360.
- InvoiceDate: 2010-12-02 14:23:00.
- Price: £0.03.
- Customer ID: 15838.
- Country: United Kingdom.

The duplicate was already present in the original dataset, and both records were preserved in the working dataset.

There is one excess duplicate occurrence in this invoice.

Removing one occurrence would make the cancellation quantity equal to the original quantity, but the price discrepancy would remain unresolved.

#### 4. Data Quality Decisions

The investigation demonstrates that extreme transaction quantities cannot automatically be classified as invalid records.

Some extreme quantities are associated with complete order reversals, while others may represent wholesale purchases or operational adjustments.

The following decisions were established:

1. Preserve extreme positive and negative quantities during the investigation.
2. Retain cancellation transactions in the source dataset because they represent meaningful business events.
3. Record verified exact duplicates for consideration during the final cleaning stage.
4. Do not automatically correct StockCode or price discrepancies.
5. Investigate transaction reversals at both invoice and line-item levels.
6. Distinguish cancellation activity from ordinary purchasing activity when constructing customer features and revenue metrics.

#### Conclusion

The investigation identified multiple legitimate-looking transaction reversals, a near-complete invoice reversal with a StockCode discrepancy, and an exact duplicate cancellation record in the original dataset.

These findings reinforce the importance of investigating transaction context before removing extreme values or duplicate records.

# Extreme Quantities

In [96]:
# Identify the 30 largest positive-quantity records
# and exclude records with previously identified cancellation candidates

positive_extremes = (
    df_description_cleaning[
        df_description_cleaning["Quantity"] > 0
    ]
    .nlargest(30, "Quantity")
    .copy()
)

matched_original_invoices = (
    matched_candidates["Invoice_original"]
    .dropna()
    .unique()
)

unmatched_positive_extremes = positive_extremes[
    ~positive_extremes["Invoice"].isin(matched_original_invoices)
].copy()

# Calculate transaction line values
unmatched_positive_extremes["line_value"] = (
    unmatched_positive_extremes["Quantity"]
    * unmatched_positive_extremes["Price"]
)

# Summarize the complete invoices containing these records
invoice_summary = (
    df_description_cleaning
    .groupby("Invoice")
    .agg(
        invoice_record_count=("Invoice", "size"),
        unique_stockcodes=("StockCode", "nunique"),
        invoice_total_quantity=("Quantity", "sum"),
        invoice_total_value=(
            "Quantity",
            lambda x: (
                x * df_description_cleaning.loc[x.index, "Price"]
            ).sum()
        ),
        customer_id=("Customer ID", "first")
    )
    .reset_index()
)

unmatched_positive_extremes = unmatched_positive_extremes.merge(
    invoice_summary,
    on="Invoice",
    how="left"
)

print(
    "Remaining extreme positive records:",
    len(unmatched_positive_extremes)
)

print(
    "\nUnique invoices:",
    unmatched_positive_extremes["Invoice"].nunique()
)

print(
    "\nUnique identified customers:",
    unmatched_positive_extremes["Customer ID"].nunique()
)

print("\nRemaining extreme positive transactions:")

display(
    unmatched_positive_extremes[
        [
            "Invoice",
            "StockCode",
            "Description",
            "Quantity",
            "Price",
            "line_value",
            "InvoiceDate",
            "Customer ID",
            "Country",
            "invoice_record_count",
            "unique_stockcodes",
            "invoice_total_value"
        ]
    ].sort_values(
        "Quantity",
        ascending=False
    )
)

Remaining extreme positive records: 24

Unique invoices: 12

Unique identified customers: 5

Remaining extreme positive transactions:


,Invoice,StockCode,Description,Quantity,Price,line_value,InvoiceDate,Customer ID,Country,invoice_record_count,unique_stockcodes,invoice_total_value
0,497946,37410,BLACK AND WHITE PAISLEY FLOWER MUG,19152,0.10,1915.20,2010-02-15 11:57:00,13902.0,Denmark,30,30,16973.10
1,501534,21099,SET/6 STRAWBERRY PAPER CUPS,12960,0.10,1296.00,2010-03-17 13:09:00,13902.0,Denmark,10,10,6866.30
2,501534,21091,SET/6 WOODLAND PAPER PLATES,12960,0.10,1296.00,2010-03-17 13:09:00,13902.0,Denmark,10,10,6866.30
3,501534,21085,SET/6 WOODLAND PAPER CUPS,12744,0.10,1274.40,2010-03-17 13:09:00,13902.0,Denmark,10,10,6866.30
4,578841,84826,ASSTD DESIGN 3D PAPER STICKERS,12540,0.00,0.00,2011-11-25 15:57:00,13256.0,United Kingdom,1,1,0.00
5,501534,21092,SET/6 STRAWBERRY PAPER PLATES,12480,0.10,1248.00,2010-03-17 13:09:00,13902.0,Denmark,10,10,6866.30
6,507637,84016,FLAG OF ST GEORGE CAR FLAG,10200,0.00,0.00,2010-05-10 14:55:00,NaN,United Kingdom,2,1,1157.15
7,502269,21984,PACK OF 12 PINK PAISLEY TISSUES,10000,0.25,2500.00,2010-03-23 15:36:00,17940.0,United Kingdom,4,4,10000.00
8,502269,21982,PACK OF 12 SUKI TISSUES,10000,0.25,2500.00,2010-03-23 15:36:00,17940.0,United Kingdom,4,4,10000.00
9,502269,21980,PACK OF 12 RED SPOTTY TISSUES,10000,0.25,2500.00,2010-03-23 15:36:00,17940.0,United Kingdom,4,4,10000.00


In [97]:
# Inspect invoices containing extreme positive quantities
# with zero prices and missing Customer IDs

anonymous_zero_price_invoices = [
    "507637",
    "518241",
    "518234",
    "542504"
]

anonymous_extreme_details = df_description_cleaning[
    df_description_cleaning["Invoice"].astype(str).isin(
        anonymous_zero_price_invoices
    )
].copy()

# Calculate transaction line values
anonymous_extreme_details["line_value"] = (
    anonymous_extreme_details["Quantity"]
    * anonymous_extreme_details["Price"]
)

print(
    "Number of records:",
    len(anonymous_extreme_details)
)

print("\nInvoice-level summary:")

display(
    anonymous_extreme_details
    .groupby("Invoice")
    .agg(
        record_count=("Invoice", "size"),
        unique_stockcodes=("StockCode", "nunique"),
        total_quantity=("Quantity", "sum"),
        total_line_value=("line_value", "sum"),
        customer_count=("Customer ID", "nunique"),
        invoice_date=("InvoiceDate", "first")
    )
)

print("\nComplete invoice details:")

display(
    anonymous_extreme_details[
        [
            "Invoice",
            "StockCode",
            "Description",
            "Quantity",
            "Price",
            "line_value",
            "InvoiceDate",
            "Customer ID",
            "Country"
        ]
    ].sort_values(
        ["Invoice", "StockCode"]
    )
)

Number of records: 5

Invoice-level summary:


,record_count,unique_stockcodes,total_quantity,total_line_value,customer_count,invoice_date
Invoice,,,,,,
507637,2,1,10201,1157.15,0,2010-05-10 14:55:00
518234,1,1,6000,0.00,0,2010-08-05 15:51:00
518241,1,1,9600,0.00,0,2010-08-05 15:55:00
542504,1,1,5568,0.00,0,2011-01-28 12:03:00



Complete invoice details:


,Invoice,StockCode,Description,Quantity,Price,line_value,InvoiceDate,Customer ID,Country
192196,507637,84016,FLAG OF ST GEORGE CAR FLAG,1,1157.15,1157.15,2010-05-10 14:55:00,NaN,United Kingdom
192197,507637,84016,FLAG OF ST GEORGE CAR FLAG,10200,0.00,0.00,2010-05-10 14:55:00,NaN,United Kingdom
296414,518234,22752,SET 7 BABUSHKA NESTING BOXES,6000,0.00,0.00,2010-08-05 15:51:00,NaN,United Kingdom
296420,518241,22759,NaN,9600,0.00,0.00,2010-08-05 15:55:00,NaN,United Kingdom
600075,542504,37413,ICON MUG REVOLUTIONARY,5568,0.00,0.00,2011-01-28 12:03:00,NaN,United Kingdom


### Investigation of Extreme Positive Quantities with Zero Prices and Missing Customer IDs

#### Objective

Investigate extreme positive-quantity transactions with zero unit prices and missing Customer IDs to understand their invoice context and determine their suitability for customer-level purchasing and revenue analysis.

#### Findings

Four invoices containing five transaction records were examined.

**Invoice 507637**

This invoice contains two records for StockCode 84016 (FLAG OF ST GEORGE CAR FLAG):

- Quantity 1 at £1,157.15, resulting in a line value of £1,157.15.
- Quantity 10,200 at £0.00, resulting in a line value of £0.00.

Both records share the same invoice, StockCode, timestamp, description, and missing Customer ID.

The unusually high positive price and the separate zero-price quantity require further investigation. The available data does not establish whether these records represent an adjustment, recording issue, or another transaction process.

**Invoice 518234**

- StockCode: 22752.
- Description: SET 7 BABUSHKA NESTING BOXES.
- Quantity: 6,000.
- Unit price: £0.00.
- Customer ID: missing.

The invoice contains only one zero-price transaction record.

**Invoice 518241**

- StockCode: 22759.
- Description: missing.
- Quantity: 9,600.
- Unit price: £0.00.
- Customer ID: missing.

The invoice contains only one zero-price transaction record. Its description remains unresolved.

**Invoice 542504**

- StockCode: 37413.
- Description: ICON MUG REVOLUTIONARY.
- Quantity: 5,568.
- Unit price: £0.00.
- Customer ID: missing.

The invoice contains only one zero-price transaction record.

#### Data Quality Decisions

1. Preserve all five records in the transaction-level dataset.
2. Do not infer customer identities for records with missing Customer IDs.
3. Do not count zero-price-only invoices as paid purchases when constructing customer inactivity targets.
4. Exclude records with missing Customer IDs from customer-level purchasing analysis.
5. Flag invoice 507637 for further investigation because of its unusually high positive unit price and mixed-price transaction lines.
6. Retain the unresolved Description in invoice 518241 rather than imputing it without sufficient evidence.

#### Conclusion

The investigation identified four invoices containing extreme positive quantities, all with missing Customer IDs. Three invoices contain only zero-price transactions, while one contains both a positive-price and a zero-price line for the same StockCode.

These records are preserved in the source and working datasets. Their treatment will depend on the analytical purpose, particularly the distinction between recorded transaction activity, paid purchasing activity, and customer-attributable revenue.

# Extreme Negative Quantities

In [98]:
# Inspect the 30 largest negative-quantity records

negative_extremes = (
    df_description_cleaning[
        df_description_cleaning["Quantity"] < 0
    ]
    .nsmallest(30, "Quantity")
    .copy()
)

negative_extremes["line_value"] = (
    negative_extremes["Quantity"]
    * negative_extremes["Price"]
)

print(
    "Extreme negative records examined:",
    len(negative_extremes)
)

print(
    "\nCancellation records:",
    negative_extremes["Invoice"]
    .astype(str)
    .str.startswith("C")
    .sum()
)

print(
    "\nNon-cancellation records:",
    (~negative_extremes["Invoice"]
    .astype(str)
    .str.startswith("C"))
    .sum()
)

print("\nExtreme negative transactions:")

display(
    negative_extremes[
        [
            "Invoice",
            "StockCode",
            "Description",
            "Quantity",
            "Price",
            "line_value",
            "InvoiceDate",
            "Customer ID",
            "Country"
        ]
    ].sort_values(
        "Quantity"
    )
)

Extreme negative records examined: 30

Cancellation records: 7

Non-cancellation records: 23

Extreme negative transactions:


,Invoice,StockCode,Description,Quantity,Price,line_value,InvoiceDate,Customer ID,Country
1065883,C581484,23843,"PAPER CRAFT , LITTLE BIRDIE",-80995,2.08,-168469.60,2011-12-09 09:27:00,16446.0,United Kingdom
587085,C541433,23166,MEDIUM CERAMIC TOP STORAGE JAR,-74215,1.04,-77183.60,2011-01-18 10:17:00,12346.0,United Kingdom
303996,519017,22759,NaN,-9600,0.00,-0.00,2010-08-13 09:14:00,NaN,United Kingdom
750990,556690,23005,printing smudges/thrown away,-9600,0.00,-0.00,2011-06-14 10:37:00,NaN,United Kingdom
750991,556691,23005,printing smudges/thrown away,-9600,0.00,-0.00,2011-06-14 10:37:00,NaN,United Kingdom
507225,C536757,84347,ROTATING SILVER ANGELS T-LIGHT HLDR,-9360,0.03,-280.80,2010-12-02 14:23:00,15838.0,United Kingdom
529729,C536757,84347,ROTATING SILVER ANGELS T-LIGHT HLDR,-9360,0.03,-280.80,2010-12-02 14:23:00,15838.0,United Kingdom
156488,504311,22197,NaN,-9200,0.00,-0.00,2010-04-12 14:39:00,NaN,United Kingdom
750989,556687,23003,Printing smudges/thrown away,-9058,0.00,-0.00,2011-06-14 10:36:00,NaN,United Kingdom
194372,507913,10120,Zebra invcing error,-9000,0.00,-0.00,2010-05-11 17:16:00,NaN,United Kingdom


In [100]:
# Identify non-cancellation extreme negative invoices

non_cancellation_extreme_invoices = (
    negative_extremes.loc[
        ~negative_extremes["Invoice"]
        .astype(str)
        .str.startswith("C"),
        "Invoice"
    ]
    .unique()
)

# Retrieve every transaction line from those invoices

non_cancellation_extreme_details = (
    df_description_cleaning[
        df_description_cleaning["Invoice"].isin(
            non_cancellation_extreme_invoices
        )
    ]
    .copy()
)

# Calculate transaction line values

non_cancellation_extreme_details["line_value"] = (
    non_cancellation_extreme_details["Quantity"]
    * non_cancellation_extreme_details["Price"]
)

print(
    "Number of affected invoices:",
    len(non_cancellation_extreme_invoices)
)

print(
    "Total records across complete invoices:",
    len(non_cancellation_extreme_details)
)

print("\nInvoice-level summary:")

display(
    non_cancellation_extreme_details
    .groupby("Invoice")
    .agg(
        record_count=("Invoice", "size"),
        unique_stockcodes=("StockCode", "nunique"),
        total_quantity=("Quantity", "sum"),
        total_line_value=("line_value", "sum"),
        customer_count=("Customer ID", "nunique"),
        invoice_date=("InvoiceDate", "first"),
        descriptions=("Description", lambda x: " | ".join(
            x.dropna().astype(str).unique()
        ))
    )
    .sort_index()
)

print("\nComplete transaction details:")

display(
    non_cancellation_extreme_details[
        [
            "Invoice",
            "StockCode",
            "Description",
            "Quantity",
            "Price",
            "line_value",
            "InvoiceDate",
            "Customer ID",
            "Country"
        ]
    ].sort_values(
        ["Invoice", "StockCode"]
    )
)

Number of affected invoices: 23
Total records across complete invoices: 23

Invoice-level summary:


,record_count,unique_stockcodes,total_quantity,total_line_value,customer_count,invoice_date,descriptions
Invoice,,,,,,,
502457,1,1,-5200,0.0,0,2010-03-24 14:20:00,ebay sales
502458,1,1,-5000,0.0,0,2010-03-24 14:21:00,ebay sales
504311,1,1,-9200,0.0,0,2010-04-12 14:39:00,
507913,1,1,-9000,0.0,0,2010-05-11 17:16:00,Zebra invcing error
514363,1,1,-7100,0.0,0,2010-07-01 18:07:00,Ebay sales by the box.
515462,1,1,-7000,0.0,0,2010-07-12 15:47:00,given away
515463,1,1,-5000,0.0,0,2010-07-12 15:47:00,given away
515464,1,1,-5000,0.0,0,2010-07-12 15:48:00,given away
519016,1,1,-6000,0.0,0,2010-08-13 09:13:00,SET 7 BABUSHKA NESTING BOXES



Complete transaction details:


,Invoice,StockCode,Description,Quantity,Price,line_value,InvoiceDate,Customer ID,Country
136409,502457,84016,ebay sales,-5200,0.0,-0.0,2010-03-24 14:20:00,NaN,United Kingdom
136410,502458,84016,ebay sales,-5000,0.0,-0.0,2010-03-24 14:21:00,NaN,United Kingdom
156488,504311,22197,NaN,-9200,0.0,-0.0,2010-04-12 14:39:00,NaN,United Kingdom
194372,507913,10120,Zebra invcing error,-9000,0.0,-0.0,2010-05-11 17:16:00,NaN,United Kingdom
258814,514363,84016,Ebay sales by the box.,-7100,0.0,-0.0,2010-07-01 18:07:00,NaN,United Kingdom
269558,515462,20852,given away,-7000,0.0,-0.0,2010-07-12 15:47:00,NaN,United Kingdom
269557,515463,22126,given away,-5000,0.0,-0.0,2010-07-12 15:47:00,NaN,United Kingdom
269559,515464,22126,given away,-5000,0.0,-0.0,2010-07-12 15:48:00,NaN,United Kingdom
303995,519016,22752,SET 7 BABUSHKA NESTING BOXES,-6000,0.0,-0.0,2010-08-13 09:13:00,NaN,United Kingdom
303996,519017,22759,NaN,-9600,0.0,-0.0,2010-08-13 09:14:00,NaN,United Kingdom


### Investigation of Extreme Negative Quantities Outside Cancellation Invoices

#### Objective

Investigate the largest negative-quantity transactions recorded under non-cancellation invoice identifiers to determine whether they represent ordinary customer returns, cancellations, or other transaction adjustments.

#### Findings

The 30 largest negative-quantity records contained:

- 7 records associated with cancellation invoices.
- 23 records associated with non-cancellation invoices.

The seven cancellation records were examined in the earlier cancellation and invoice-reconciliation investigation.

The remaining 23 records belonged to 23 distinct invoices. Each invoice contained exactly one transaction line.

All 23 records shared the following characteristics:

- Negative quantities.
- Zero unit prices.
- Missing Customer IDs.
- United Kingdom as the recorded country.

Consequently, all 23 records had zero calculated line values.

#### Description-Based Investigation

The transaction descriptions revealed several recurring patterns:

**1. eBay-related adjustments**

Invoices 502457, 502458, and 514363 contained descriptions such as `ebay sales` and `Ebay sales by the box.`

These descriptions suggest adjustments associated with eBay-related sales, although the precise transaction process cannot be established from the available data.

**2. Giveaways**

Invoices 515462, 515463, 515464, and 531177 contained descriptions indicating that products were given away.

These records are consistent with non-revenue-generating stock adjustments.

**3. Invoicing errors**

Invoice 507913 contained the description `Zebra invcing error`, indicating a possible invoicing-related correction.

**4. Damaged or discarded inventory**

Invoices 546152, 556687, 556690, and 556691 contained descriptions such as `throw away` and `printing smudges/thrown away`.

These records are consistent with damaged or discarded inventory adjustments.

**5. Babushka product adjustments**

Invoices 519016 through 519023 were recorded consecutively between 09:13 and 09:17 on August 13, 2010.

They contained multiple Babushka product variants, including nesting boxes and notebooks, with negative quantities ranging from 5,000 to 9,600 units.

All records had zero unit prices and missing Customer IDs.

The coordinated timing and product descriptions suggest a related series of stock adjustments, although their exact operational purpose remains unverified.

**6. Unclear descriptions**

Invoices 504311, 530348, and 531178 contained missing or ambiguous descriptions. Their precise purpose could not be established.

#### Data Quality Decisions

1. Preserve all 23 records in the transaction-level dataset.
2. Exclude these records from customer-level purchasing and revenue analysis.
3. Do not classify them as customer cancellations or returns solely because their quantities are negative.
4. Do not impute Customer IDs.
5. Retain the original descriptions as contextual evidence.
6. Do not assign definitive operational explanations where the available evidence is insufficient.

#### Conclusion

The 23 investigated records are consistent with non-customer-attributable, zero-value negative adjustments. Their descriptions suggest several possible operational contexts, including giveaways, eBay-related adjustments, invoicing errors, and damaged or discarded inventory.

The findings reinforce the distinction between negative quantities and genuine customer cancellations or returns.

These records will be preserved in the transaction-level dataset but excluded from customer-level purchasing and revenue calculations.

In [101]:
# Create a separate DataFrame for transaction classification

df_transaction_review = df_description_cleaning.copy()

# Identify cancellation invoices
df_transaction_review["is_cancellation"] = (
    df_transaction_review["Invoice"]
    .astype(str)
    .str.startswith("C")
)

# Identify transaction characteristics
df_transaction_review["has_customer_id"] = (
    df_transaction_review["Customer ID"].notna()
)

df_transaction_review["has_positive_quantity"] = (
    df_transaction_review["Quantity"] > 0
)

df_transaction_review["has_positive_price"] = (
    df_transaction_review["Price"] > 0
)

df_transaction_review["has_positive_line_value"] = (
    df_transaction_review["Quantity"]
    * df_transaction_review["Price"] > 0
)

# Candidate eligibility for customer-attributable paid purchasing
df_transaction_review["candidate_paid_purchase"] = (
    ~df_transaction_review["is_cancellation"]
    & df_transaction_review["has_customer_id"]
    & df_transaction_review["has_positive_quantity"]
    & df_transaction_review["has_positive_price"]
)

# Summarize transaction classification
print("Transaction classification summary:")

classification_summary = pd.Series({
    "Total records": len(df_transaction_review),
    "Cancellation records": (
        df_transaction_review["is_cancellation"].sum()
    ),
    "Records with known Customer ID": (
        df_transaction_review["has_customer_id"].sum()
    ),
    "Records with positive quantity": (
        df_transaction_review["has_positive_quantity"].sum()
    ),
    "Records with positive unit price": (
        df_transaction_review["has_positive_price"].sum()
    ),
    "Records with positive line value": (
        df_transaction_review["has_positive_line_value"].sum()
    ),
    "Candidate paid purchase records": (
        df_transaction_review["candidate_paid_purchase"].sum()
    )
})

display(classification_summary.to_frame("record_count"))

print("\nCandidate paid purchase invoice summary:")

candidate_invoices = (
    df_transaction_review.loc[
        df_transaction_review["candidate_paid_purchase"]
    ]
    .groupby("Invoice")
    .agg(
        record_count=("Invoice", "size"),
        unique_stockcodes=("StockCode", "nunique"),
        total_quantity=("Quantity", "sum"),
        total_line_value=(
            "Quantity",
            lambda x: (
                x * df_transaction_review.loc[
                    x.index, "Price"
                ]
            ).sum()
        ),
        customer_id=("Customer ID", "first")
    )
)

print("Candidate invoice count:", len(candidate_invoices))

display(candidate_invoices.head())

Transaction classification summary:


,record_count
Total records,1067371
Cancellation records,19494
Records with known Customer ID,824364
Records with positive quantity,1044421
Records with positive unit price,1061164
Records with positive line value,1041671
Candidate paid purchase records,805549



Candidate paid purchase invoice summary:
Candidate invoice count: 36969


,record_count,unique_stockcodes,total_quantity,total_line_value,customer_id
Invoice,,,,,
489434,8,8,166,505.30,13085.0
489435,4,4,60,145.80,13085.0
489436,19,19,193,630.33,13078.0
489437,23,23,145,310.75,15362.0
489438,17,17,826,2286.24,18102.0


In [102]:
# Profile StockCodes in the candidate paid-purchase population

candidate_paid = df_transaction_review[
    df_transaction_review["candidate_paid_purchase"]
].copy()

stockcode_profile = (
    candidate_paid
    .groupby("StockCode")
    .agg(
        record_count=("StockCode", "size"),
        unique_descriptions=("Description", "nunique"),
        total_quantity=("Quantity", "sum"),
        total_revenue=(
            "Quantity",
            lambda x: (
                x * candidate_paid.loc[x.index, "Price"]
            ).sum()
        ),
        unique_customers=("Customer ID", "nunique")
    )
    .sort_values("record_count", ascending=False)
)

print("Unique candidate StockCodes:", len(stockcode_profile))

print("\nMost frequently occurring StockCodes:")

display(stockcode_profile.head(30))

print("\nStockCodes with multiple descriptions:")

display(
    stockcode_profile[
        stockcode_profile["unique_descriptions"] > 1
    ].sort_values(
        "unique_descriptions",
        ascending=False
    ).head(20)
)

print("\nCandidate records with potentially special descriptions:")

special_description_keywords = (
    r"POSTAGE|CARRIAGE|BANK CHARGES|AMAZON FEE|"
    r"ADJUST|MANUAL|DOTCOM|FEE|VOUCHER|"
    r"DISCOUNT|CRUK|COMMISSION|CREDIT"
)

special_description_records = candidate_paid[
    candidate_paid["Description"]
    .astype("string")
    .str.contains(
        special_description_keywords,
        case=False,
        na=False,
        regex=True
    )
]

display(
    special_description_records[
        [
            "Invoice",
            "StockCode",
            "Description",
            "Quantity",
            "Price",
            "Customer ID",
            "Country"
        ]
    ].head(50)
)

print(
    "\nRecords matching special-description keywords:",
    len(special_description_records)
)

Unique candidate StockCodes: 4631

Most frequently occurring StockCodes:


,record_count,unique_descriptions,total_quantity,total_revenue,unique_customers
StockCode,,,,,
85123A,5188,2,93697,252227.81,1490
22423,3428,1,24899,286486.30,1314
85099B,3360,3,94983,170616.68,978
84879,2777,1,79913,127074.17,1010
20725,2678,2,38314,60271.70,845
21212,2654,2,91263,44067.34,1115
21034,2141,1,4120,3914.00,800
47566,2121,1,23607,103880.23,894
20727,2117,1,25820,40586.20,705



StockCodes with multiple descriptions:


,record_count,unique_descriptions,total_quantity,total_revenue,unique_customers
StockCode,,,,,
23236,296,4,2459,6828.31,203
20685,1515,4,9545,61829.13,626
23196,243,4,1760,2484.80,170
22384,1922,4,20494,32681.50,681
21955,601,4,3744,21288.20,279
22345,86,4,1314,609.18,62
22344,86,4,1359,636.81,65
22346,65,4,1145,521.33,43
22383,2100,3,24262,38632.70,664



Candidate records with potentially special descriptions:


,Invoice,StockCode,Description,Quantity,Price,Customer ID,Country
40,489437,37370,RETRO COFFEE MUGS ASSORTED,12,1.25,15362.0,United Kingdom
89,489439,POST,POSTAGE,3,18.00,12682.0,France
126,489444,POST,POSTAGE,1,141.00,12636.0,USA
173,489447,POST,POSTAGE,1,130.00,12362.0,Belgium
306,489488,22301,COFFEE MUG CAT + BIRD DESIGN,2,2.55,17238.0,United Kingdom
307,489488,22300,COFFEE MUG DOG + BALL DESIGN,3,2.55,17238.0,United Kingdom
349,489514,21869,POTTING SHED COFFEE MUG,2,1.25,15311.0,United Kingdom
487,489522,21869,POTTING SHED COFFEE MUG,1,1.25,15998.0,United Kingdom
499,489522,22300,COFFEE MUG DOG + BALL DESIGN,3,2.55,15998.0,United Kingdom
500,489522,22301,COFFEE MUG CAT + BIRD DESIGN,3,2.55,15998.0,United Kingdom



Records matching special-description keywords: 8098


In [104]:
# Add line_value to the candidate transaction dataset

candidate_paid = df_transaction_review[
    df_transaction_review["candidate_paid_purchase"]
].copy()

candidate_paid["line_value"] = (
    candidate_paid["Quantity"] * candidate_paid["Price"]
)

# Recreate the special-description subset

special_description_keywords = (
    r"POSTAGE|CARRIAGE|BANK CHARGES|AMAZON FEE|"
    r"ADJUST|MANUAL|DOTCOM|FEE|VOUCHER|"
    r"DISCOUNT|CRUK|COMMISSION|CREDIT"
)

special_description_records = candidate_paid[
    candidate_paid["Description"]
    .astype("string")
    .str.contains(
        special_description_keywords,
        case=False,
        na=False,
        regex=True
    )
].copy()

print("Candidate records:", len(candidate_paid))
print("Special-description records:", len(special_description_records))
print("Line value column available:", "line_value" in candidate_paid.columns)

Candidate records: 805549
Special-description records: 8098
Line value column available: True


In [106]:
# Profile potentially special transaction descriptions

special_description_profile = (
    special_description_records
    .groupby(
        ["StockCode", "Description"],
        dropna=False
    )
    .agg(
        record_count=("Invoice", "size"),
        unique_invoices=("Invoice", "nunique"),
        total_quantity=("Quantity", "sum"),
        total_revenue=("line_value", "sum")
    )
    .sort_values(
        "record_count",
        ascending=False
    )
)

print("Distinct StockCode-description combinations:",
      len(special_description_profile))

print("\nMost frequent combinations:")
display(special_description_profile.head(60))


# Inspect postage transactions

postage_records = candidate_paid[
    candidate_paid["StockCode"].astype(str) == "POST"
].copy()

postage_summary = pd.DataFrame({
    "value": [
        len(postage_records),
        postage_records["Invoice"].nunique(),
        postage_records["Customer ID"].nunique(),
        postage_records["Quantity"].sum(),
        postage_records["line_value"].sum()
    ]
}, index=[
    "record_count",
    "unique_invoices",
    "unique_customers",
    "total_quantity",
    "total_revenue"
])

print("\nPOST transaction profile:")
display(postage_summary)


# Determine whether postage invoices also contain other candidate items

postage_invoice_ids = postage_records["Invoice"].unique()

postage_invoice_context = candidate_paid[
    candidate_paid["Invoice"].isin(postage_invoice_ids)
].copy()

postage_invoice_summary = (
    postage_invoice_context
    .groupby("Invoice")
    .agg(
        total_candidate_lines=("Invoice", "size"),
        postage_lines=(
            "StockCode",
            lambda x: (x.astype(str) == "POST").sum()
        ),
        distinct_stockcodes=("StockCode", "nunique"),
        invoice_revenue=("line_value", "sum")
    )
)

postage_invoice_summary["contains_other_stockcodes"] = (
    postage_invoice_summary["distinct_stockcodes"] > 1
)

print("\nPostage invoice context:")

display(
    postage_invoice_summary[
        "contains_other_stockcodes"
    ].value_counts().to_frame("invoice_count")
)

print("\nExamples of postage invoices with other StockCodes:")

display(
    postage_invoice_summary[
        postage_invoice_summary["contains_other_stockcodes"]
    ].head(15)
)

Distinct StockCode-description combinations: 55

Most frequent combinations:


,,record_count,unique_invoices,total_quantity,total_revenue
StockCode,Description,,,,
POST,POSTAGE,1838,1803,5333,126563.040
M,Manual,709,620,9803,152340.570
22301,COFFEE MUG CAT + BIRD DESIGN,491,466,4863,11162.790
22303,COFFEE MUG APPLES DESIGN,484,480,5323,12213.100
22300,COFFEE MUG DOG + BALL DESIGN,432,411,4025,9281.490
37370,RETRO COFFEE MUGS ASSORTED,397,344,7283,8271.450
23243,SET OF TEA COFFEE SUGAR TINS PANTRY,389,385,4663,22133.830
22302,COFFEE MUG PEARS DESIGN,356,352,4815,10061.260
21216,"SET 3 RETROSPOT TEA,COFFEE,SUGAR",301,283,2113,9617.270



POST transaction profile:


,value
record_count,1838.00
unique_invoices,1803.00
unique_customers,405.00
total_quantity,5333.00
total_revenue,126563.04



Postage invoice context:


,invoice_count
contains_other_stockcodes,
True,1693
False,110



Examples of postage invoices with other StockCodes:


,total_candidate_lines,postage_lines,distinct_stockcodes,invoice_revenue,contains_other_stockcodes
Invoice,,,,,
489439,19,1,19,426.30,True
489526,44,1,44,929.92,True
489557,21,1,21,603.94,True
489883,27,1,27,578.28,True
490152,29,1,29,682.65,True
490158,22,1,22,508.26,True
490295,17,1,17,435.88,True
490395,13,1,13,283.15,True
490397,18,1,18,309.10,True


In [107]:
# Define preliminary special transaction StockCodes

special_stockcodes = [
    "POST",
    "M",
    "C2",
    "23444",
    "DOT",
    "BANK CHARGES",
    "ADJUST",
    "ADJUST2",
    "D"
]

# Retrieve all candidate records belonging to these StockCodes

special_stockcode_details = candidate_paid[
    candidate_paid["StockCode"]
    .astype(str)
    .isin(special_stockcodes)
].copy()

print(
    "Total records:",
    len(special_stockcode_details)
)

print(
    "Unique invoices:",
    special_stockcode_details["Invoice"].nunique()
)

print("\nStockCode-level summary:")

display(
    special_stockcode_details
    .groupby("StockCode")
    .agg(
        record_count=("Invoice", "size"),
        unique_descriptions=("Description", "nunique"),
        unique_invoices=("Invoice", "nunique"),
        unique_customers=("Customer ID", "nunique"),
        total_quantity=("Quantity", "sum"),
        total_revenue=("line_value", "sum"),
        minimum_price=("Price", "min"),
        maximum_price=("Price", "max")
    )
    .sort_values("record_count", ascending=False)
)

print("\nDescription-level summary:")

display(
    special_stockcode_details
    .groupby(
        ["StockCode", "Description"],
        dropna=False
    )
    .agg(
        record_count=("Invoice", "size"),
        unique_invoices=("Invoice", "nunique"),
        total_quantity=("Quantity", "sum"),
        total_revenue=("line_value", "sum")
    )
    .sort_values(
        ["StockCode", "record_count"],
        ascending=[True, False]
    )
)

print("\nSample transaction records:")

display(
    special_stockcode_details[
        [
            "Invoice",
            "StockCode",
            "Description",
            "Quantity",
            "Price",
            "line_value",
            "InvoiceDate",
            "Customer ID",
            "Country"
        ]
    ].head(40)
)

Total records: 2967
Unique invoices: 2834

StockCode-level summary:


,record_count,unique_descriptions,unique_invoices,unique_customers,total_quantity,total_revenue,minimum_price,maximum_price
StockCode,,,,,,,,
POST,1838,1,1803,405,5333,126563.040,1.000,8142.75
M,709,1,620,438,9803,152340.570,0.060,10953.50
C2,253,1,248,45,254,12796.000,18.000,150.00
23444,79,1,79,63,79,1200.000,15.000,30.00
ADJUST,32,2,32,25,32,3538.520,4.570,387.54
BANK CHARGES,32,2,30,19,32,465.001,0.001,15.00
DOT,16,1,16,1,16,11906.360,11.170,1599.26
D,5,1,5,5,196,397.890,1.000,101.99
ADJUST2,3,1,3,3,3,731.050,72.450,358.47



Description-level summary:


record_count  \
StockCode    Description                                         
23444        Next Day Carriage                              79   
ADJUST       Adjustment by john on 26/01/2010 16            18   
             Adjustment by john on 26/01/2010 17            14   
ADJUST2      Adjustment by Peter on Jun 25 2010              3   
BANK CHARGES Bank Charges                                   30   
              Bank Charges                                   2   
C2           CARRIAGE                                      253   
D            Discount                                        5   
DOT          DOTCOM POSTAGE                                 16   
M            Manual                                        709   
POST         POSTAGE                                      1838   

                                                  unique_invoices  \
StockCode    Description                                            
23444        Next Day Carriage                                 79   
ADJUST       Adjustment by john on 26/01/2010 16               18   
             Adjustment by john on 26/01/2010 17               14   
ADJUST2      Adjustment by Peter on Jun 25 2010                 3   
BANK CHARGES Bank Charges                                      28   
              Bank Charges                                      2   
C2           CARRIAGE                                         248   
D            Discount                                           5   
DOT          DOTCOM POSTAGE                                    16   
M            Manual                                           620   
POST         POSTAGE                                         1803   

                                                  total_quantity  \
StockCode    Description                                           
23444        Next Day Carriage                                79   
ADJUST       Adjustment by john on 26/01/2010 16              18   
             Adjustment by john on 26/01/2010 17              14   
ADJUST2      Adjustment by Peter on Jun 25 2010                3   
BANK CHARGES Bank Charges                                     30   
              Bank Charges                                     2   
C2           CARRIAGE                                        254   
D            Discount                                        196   
DOT          DOTCOM POSTAGE                                   16   
M            Manual                                         9803   
POST         POSTAGE                                        5333   

                                                  total_revenue  
StockCode    Description                                         
23444        Next Day Carriage                         1200.000  
ADJUST       Adjustment by john on 26/01/2010 16       2107.410  
             Adjustment by john on 26/01/2010 17       1431.110  
ADJUST2      Adjustment by Peter on Jun 25 2010         731.050  
BANK CHARGES Bank Charges                               435.001  
              Bank Charges                               30.000  
C2           CARRIAGE                                 12796.000  
D            Discount                                   397.890  
DOT          DOTCOM POSTAGE                           11906.360  
M            Manual                                  152340.570  
POST         POSTAGE                                 126563.040


Sample transaction records:


,Invoice,StockCode,Description,Quantity,Price,line_value,InvoiceDate,Customer ID,Country
89,489439,POST,POSTAGE,3,18.00,54.00,2009-12-01 09:28:00,12682.0,France
126,489444,POST,POSTAGE,1,141.00,141.00,2009-12-01 09:55:00,12636.0,USA
173,489447,POST,POSTAGE,1,130.00,130.00,2009-12-01 10:10:00,12362.0,Belgium
625,489526,POST,POSTAGE,6,18.00,108.00,2009-12-01 11:50:00,12533.0,Germany
1244,489557,POST,POSTAGE,4,18.00,72.00,2009-12-01 12:52:00,12490.0,France
6406,489883,POST,POSTAGE,3,18.00,54.00,2009-12-02 16:24:00,12437.0,France
9292,490127,C2,CARRIAGE,1,50.00,50.00,2009-12-03 18:13:00,14156.0,EIRE
10279,490152,POST,POSTAGE,7,18.00,126.00,2009-12-04 10:00:00,12714.0,France
10344,490158,POST,POSTAGE,3,18.00,54.00,2009-12-04 10:47:00,12759.0,Netherlands
11193,490295,POST,POSTAGE,4,28.00,112.00,2009-12-04 13:47:00,12510.0,Spain


In [108]:
# Investigate manual and financial adjustment transactions

investigation_stockcodes = [
    "M",
    "ADJUST",
    "ADJUST2",
    "BANK CHARGES",
    "D"
]

investigation_records = candidate_paid[
    candidate_paid["StockCode"]
    .astype(str)
    .isin(investigation_stockcodes)
].copy()

print("Investigation record count:", len(investigation_records))

print("\nCategory summary:")

display(
    investigation_records
    .groupby("StockCode")
    .agg(
        record_count=("Invoice", "size"),
        unique_invoices=("Invoice", "nunique"),
        unique_customers=("Customer ID", "nunique"),
        total_quantity=("Quantity", "sum"),
        total_revenue=("line_value", "sum"),
        minimum_price=("Price", "min"),
        median_price=("Price", "median"),
        maximum_price=("Price", "max")
    )
)

# Identify the 15 highest-value manual entries
manual_records = investigation_records[
    investigation_records["StockCode"] == "M"
].copy()

highest_value_manual = manual_records.nlargest(
    15,
    "line_value"
)

print("\n15 highest-value Manual records:")

display(
    highest_value_manual[
        [
            "Invoice",
            "Description",
            "Quantity",
            "Price",
            "line_value",
            "InvoiceDate",
            "Customer ID",
            "Country"
        ]
    ]
)

# Retrieve all candidate lines from invoices containing these records
investigation_invoice_ids = (
    investigation_records["Invoice"].unique()
)

investigation_invoice_context = candidate_paid[
    candidate_paid["Invoice"].isin(investigation_invoice_ids)
].copy()

investigation_invoice_summary = (
    investigation_invoice_context
    .groupby("Invoice")
    .agg(
        total_candidate_lines=("Invoice", "size"),
        distinct_stockcodes=("StockCode", "nunique"),
        invoice_revenue=("line_value", "sum"),
        customer_id=("Customer ID", "first")
    )
)

investigation_invoice_summary["contains_other_stockcodes"] = (
    investigation_invoice_summary["distinct_stockcodes"] > 1
)

print("\nInvoice context summary:")

display(
    investigation_invoice_summary[
        "contains_other_stockcodes"
    ].value_counts().to_frame("invoice_count")
)

print("\nTransaction context for the highest-value Manual entries:")

selected_invoice_ids = highest_value_manual["Invoice"].unique()

display(
    investigation_invoice_context[
        investigation_invoice_context["Invoice"].isin(
            selected_invoice_ids
        )
    ][
        [
            "Invoice",
            "StockCode",
            "Description",
            "Quantity",
            "Price",
            "line_value",
            "Customer ID",
            "Country"
        ]
    ].sort_values(
        ["Invoice", "StockCode"]
    )
)

Investigation record count: 781

Category summary:


,record_count,unique_invoices,unique_customers,total_quantity,total_revenue,minimum_price,median_price,maximum_price
StockCode,,,,,,,,
ADJUST,32,32,25,32,3538.520,4.570,79.565,387.54
ADJUST2,3,3,3,3,731.050,72.450,300.130,358.47
BANK CHARGES,32,30,19,32,465.001,0.001,15.000,15.00
D,5,5,5,196,397.890,1.000,32.290,101.99
M,709,620,438,9803,152340.570,0.060,1.650,10953.50



15 highest-value Manual records:


,Invoice,Description,Quantity,Price,line_value,InvoiceDate,Customer ID,Country
135013,502263,Manual,1,10953.50,10953.50,2010-03-23 15:22:00,12918.0,United Kingdom
358639,524159,Manual,1,10468.80,10468.80,2010-09-27 16:12:00,14063.0,United Kingdom
74356,496115,Manual,1,8985.60,8985.60,2010-01-29 11:04:00,17949.0,United Kingdom
129903,501766,Manual,1,6958.17,6958.17,2010-03-19 11:35:00,15760.0,Norway
129987,501768,Manual,1,6958.17,6958.17,2010-03-19 11:45:00,15760.0,Norway
947835,573077,Manual,1,4161.06,4161.06,2011-10-27 14:13:00,12536.0,France
947838,573080,Manual,1,4161.06,4161.06,2011-10-27 14:20:00,12536.0,France
931867,571751,Manual,1,3949.32,3949.32,2011-10-19 11:18:00,12744.0,Singapore
288706,517483,Manual,1,3610.50,3610.50,2010-07-29 12:29:00,12737.0,France
900002,569382,Manual,1,3155.95,3155.95,2011-10-03 16:44:00,15502.0,United Kingdom



Invoice context summary:


,invoice_count
contains_other_stockcodes,
True,442
False,248



Transaction context for the highest-value Manual entries:


,Invoice,StockCode,Description,Quantity,Price,line_value,Customer ID,Country
74356,496115,M,Manual,1,8985.60,8985.60,17949.0,United Kingdom
129903,501766,M,Manual,1,6958.17,6958.17,15760.0,Norway
129987,501768,M,Manual,1,6958.17,6958.17,15760.0,Norway
135013,502263,M,Manual,1,10953.50,10953.50,12918.0,United Kingdom
288706,517483,M,Manual,1,3610.50,3610.50,12737.0,France
358639,524159,M,Manual,1,10468.80,10468.80,14063.0,United Kingdom
358670,524162,M,Manual,1,2769.90,2769.90,14063.0,United Kingdom
374485,525498,M,Manual,1,2545.85,2545.85,14156.0,EIRE
394037,527258,M,Manual,1,2667.88,2667.88,15202.0,United Kingdom
418886,529356,M,Manual,1,2736.16,2736.16,14156.0,EIRE


In [109]:

# Select invoices containing Manual entries and other candidate StockCodes

manual_invoice_revenue = (
    manual_records
    .groupby("Invoice")
    .agg(
        manual_record_count=("Invoice", "size"),
        manual_revenue=("line_value", "sum")
    )
)

mixed_manual_invoices = manual_invoice_revenue.join(
    investigation_invoice_summary[
        ["total_candidate_lines", "distinct_stockcodes", "invoice_revenue"]
    ],
    how="inner"
)

mixed_manual_invoices = mixed_manual_invoices[
    mixed_manual_invoices["distinct_stockcodes"] > 1
].sort_values(
    "manual_revenue",
    ascending=False
)

selected_manual_invoices = mixed_manual_invoices.head(10).index

print("Selected mixed invoices:", len(selected_manual_invoices))

print("\nSelected invoice summaries:")

display(
    mixed_manual_invoices.loc[selected_manual_invoices]
)

print("\nComplete candidate transaction details:")

display(
    investigation_invoice_context[
        investigation_invoice_context["Invoice"].isin(
            selected_manual_invoices
        )
    ][
        [
            "Invoice",
            "StockCode",
            "Description",
            "Quantity",
            "Price",
            "line_value",
            "Customer ID",
            "Country"
        ]
    ].sort_values(
        ["Invoice", "StockCode"]
    )
)


Selected mixed invoices: 10

Selected invoice summaries:


,manual_record_count,manual_revenue,total_candidate_lines,distinct_stockcodes,invoice_revenue
Invoice,,,,,
578060,1,400.00,3,3,976.00
575328,1,300.00,5,5,2640.00
533824,1,253.00,2,2,268.00
572696,1,250.00,4,4,3640.00
580645,1,219.50,3,3,1931.60
580646,1,200.00,2,2,1048.00
535799,1,192.00,80,67,467.02
512295,1,180.00,38,35,957.22
556070,1,129.38,8,8,396.14



Complete candidate transaction details:


,Invoice,StockCode,Description,Quantity,Price,line_value,Customer ID,Country
236709,512295,20665,RED SPOTTY PURSE,1,2.95,2.95,15311.0,United Kingdom
236740,512295,20750,RED/WHITE DOT MINI CASES,1,7.95,7.95,15311.0,United Kingdom
236723,512295,21085,SET/6 WOODLAND PAPER CUPS,3,0.65,1.95,15311.0,United Kingdom
236705,512295,21122,SET/10 PINK SPOTTY PARTY CANDLES,1,1.25,1.25,15311.0,United Kingdom
236714,512295,21122,SET/10 PINK SPOTTY PARTY CANDLES,1,1.25,1.25,15311.0,United Kingdom
...,...,...,...,...,...,...,...,...
1051477,580645,23177,TREASURE ISLAND BOOK BOX,490,1.95,955.50,17857.0,United Kingdom
1051478,580645,23193,BUFFALO BILL TREASURE BOOK BOX,388,1.95,756.60,17857.0,United Kingdom
1051476,580645,M,Manual,1,219.50,219.50,17857.0,United Kingdom
1051480,580646,22560,TRADITIONAL MODELLING CLAY,800,1.06,848.00,17857.0,United Kingdom


In [110]:

# Examine the distribution of Manual transaction quantities and values

print("Manual quantity distribution:")

display(
    manual_records["Quantity"].describe()
)

print("\nManual quantity frequency:")

display(
    manual_records["Quantity"]
    .value_counts()
    .sort_index()
    .to_frame("record_count")
)

print("\nManual price distribution:")

display(
    manual_records["Price"].describe()
)

# Classify Manual records by quantity

manual_records["quantity_category"] = (
    manual_records["Quantity"]
    .apply(
        lambda x:
            "Quantity = 1" if x == 1
            else "Quantity > 1" if x > 1
            else "Quantity <= 0"
    )
)

manual_quantity_profile = (
    manual_records
    .groupby("quantity_category")
    .agg(
        record_count=("Invoice", "size"),
        unique_invoices=("Invoice", "nunique"),
        unique_customers=("Customer ID", "nunique"),
        total_quantity=("Quantity", "sum"),
        total_revenue=("line_value", "sum"),
        median_price=("Price", "median"),
        maximum_price=("Price", "max")
    )
)

print("\nManual transaction profile by quantity category:")

display(manual_quantity_profile)

# Inspect Manual records with quantities greater than one

manual_multiple_quantity = manual_records[
    manual_records["Quantity"] > 1
].sort_values(
    "Quantity",
    ascending=False
)

print("\nManual records with quantity greater than one:")

display(
    manual_multiple_quantity[
        [
            "Invoice",
            "Quantity",
            "Price",
            "line_value",
            "InvoiceDate",
            "Customer ID",
            "Country"
        ]
    ].head(30)
)


Manual quantity distribution:


count     709.000000
mean       13.826516
std        89.598836
min         1.000000
25%         1.000000
50%         1.000000
75%         4.000000
max      1600.000000
Name: Quantity, dtype: float64


Manual quantity frequency:


,record_count
Quantity,
1,379
2,91
3,42
4,22
5,13
6,31
7,6
8,7
9,1



Manual price distribution:


count      709.000000
mean       206.440550
std        887.797755
min          0.060000
25%          0.650000
50%          1.650000
75%         15.000000
max      10953.500000
Name: Price, dtype: float64


Manual transaction profile by quantity category:


,record_count,unique_invoices,unique_customers,total_quantity,total_revenue,median_price,maximum_price
quantity_category,,,,,,,
Quantity = 1,379,350,251,379,145567.07,10.00,10953.5
Quantity > 1,330,286,236,9424,6773.50,0.85,275.6



Manual records with quantity greater than one:


,Invoice,Quantity,Price,line_value,InvoiceDate,Customer ID,Country
1015963,578060,1600,0.25,400.00,2011-11-22 15:22:00,17857.0,United Kingdom
977679,575328,1200,0.25,300.00,2011-11-09 13:48:00,17857.0,United Kingdom
962696,574277,832,0.25,208.00,2011-11-03 14:42:00,17857.0,United Kingdom
1051479,580646,800,0.25,200.00,2011-12-05 13:13:00,17857.0,United Kingdom
939599,572344,456,1.50,684.00,2011-10-24 10:43:00,14607.0,United Kingdom
942878,572656,288,1.50,432.00,2011-10-25 12:22:00,14607.0,United Kingdom
890751,568703,183,0.08,14.64,2011-09-28 15:21:00,12748.0,United Kingdom
236741,512295,144,1.25,180.00,2010-06-14 15:11:00,15311.0,United Kingdom
481511,534500,144,0.42,60.48,2010-11-23 11:13:00,17290.0,United Kingdom
846521,565123,96,5.00,480.00,2011-09-01 11:56:00,17949.0,United Kingdom


In [111]:

# Compare Manual transaction patterns by invoice composition

manual_invoice_context = manual_records.merge(
    investigation_invoice_summary[
        ["distinct_stockcodes", "invoice_revenue"]
    ],
    left_on="Invoice",
    right_index=True,
    how="left"
)

manual_invoice_context["invoice_composition"] = (
    manual_invoice_context["distinct_stockcodes"]
    .apply(
        lambda x: (
            "Manual only"
            if x == 1
            else "Manual and other StockCodes"
        )
    )
)

print("Manual transaction composition:")

display(
    manual_invoice_context
    .groupby(
        ["quantity_category", "invoice_composition"]
    )
    .agg(
        record_count=("Invoice", "size"),
        unique_invoices=("Invoice", "nunique"),
        unique_customers=("Customer ID", "nunique"),
        manual_revenue=("line_value", "sum")
    )
)

# Inspect the highest-value quantity-one Manual entries

highest_single_manual = (
    manual_records[
        manual_records["Quantity"] == 1
    ]
    .nlargest(15, "line_value")
)

selected_invoice_ids = highest_single_manual["Invoice"].unique()

print("\nHighest-value quantity-one Manual entries:")

display(
    highest_single_manual[
        [
            "Invoice",
            "Price",
            "line_value",
            "InvoiceDate",
            "Customer ID",
            "Country"
        ]
    ]
)

print("\nComplete invoice context for those entries:")

display(
    candidate_paid[
        candidate_paid["Invoice"].isin(selected_invoice_ids)
    ][
        [
            "Invoice",
            "StockCode",
            "Description",
            "Quantity",
            "Price",
            "line_value",
            "Customer ID",
            "Country"
        ]
    ].sort_values(
        ["Invoice", "StockCode"]
    )
)


Manual transaction composition:


record_count  unique_invoices  \
quantity_category invoice_composition                                          
Quantity = 1      Manual and other StockCodes           203              187   
                  Manual only                           176              163   
Quantity > 1      Manual and other StockCodes           309              271   
                  Manual only                            21               15   

                                               unique_customers  \
quantity_category invoice_composition                             
Quantity = 1      Manual and other StockCodes               164   
                  Manual only                                95   
Quantity > 1      Manual and other StockCodes               229   
                  Manual only                                14   

                                               manual_revenue  
quantity_category invoice_composition                          
Quantity = 1      Manual and other StockCodes         1894.43  
                  Manual only                       143672.64  
Quantity > 1      Manual and other StockCodes         3768.06  
                  Manual only                         3005.44


Highest-value quantity-one Manual entries:


,Invoice,Price,line_value,InvoiceDate,Customer ID,Country
135013,502263,10953.50,10953.50,2010-03-23 15:22:00,12918.0,United Kingdom
358639,524159,10468.80,10468.80,2010-09-27 16:12:00,14063.0,United Kingdom
74356,496115,8985.60,8985.60,2010-01-29 11:04:00,17949.0,United Kingdom
129903,501766,6958.17,6958.17,2010-03-19 11:35:00,15760.0,Norway
129987,501768,6958.17,6958.17,2010-03-19 11:45:00,15760.0,Norway
947835,573077,4161.06,4161.06,2011-10-27 14:13:00,12536.0,France
947838,573080,4161.06,4161.06,2011-10-27 14:20:00,12536.0,France
931867,571751,3949.32,3949.32,2011-10-19 11:18:00,12744.0,Singapore
288706,517483,3610.50,3610.50,2010-07-29 12:29:00,12737.0,France
900002,569382,3155.95,3155.95,2011-10-03 16:44:00,15502.0,United Kingdom



Complete invoice context for those entries:


,Invoice,StockCode,Description,Quantity,Price,line_value,Customer ID,Country
74356,496115,M,Manual,1,8985.60,8985.60,17949.0,United Kingdom
129903,501766,M,Manual,1,6958.17,6958.17,15760.0,Norway
129987,501768,M,Manual,1,6958.17,6958.17,15760.0,Norway
135013,502263,M,Manual,1,10953.50,10953.50,12918.0,United Kingdom
288706,517483,M,Manual,1,3610.50,3610.50,12737.0,France
358639,524159,M,Manual,1,10468.80,10468.80,14063.0,United Kingdom
358670,524162,M,Manual,1,2769.90,2769.90,14063.0,United Kingdom
374485,525498,M,Manual,1,2545.85,2545.85,14156.0,EIRE
394037,527258,M,Manual,1,2667.88,2667.88,15202.0,United Kingdom
418886,529356,M,Manual,1,2736.16,2736.16,14156.0,EIRE


In [112]:

# Isolate Manual-only invoices

manual_only_records = manual_invoice_context[
    manual_invoice_context["invoice_composition"] == "Manual only"
].copy()

# Customer-level concentration

manual_customer_profile = (
    manual_only_records
    .groupby("Customer ID")
    .agg(
        record_count=("Invoice", "size"),
        unique_invoices=("Invoice", "nunique"),
        total_manual_revenue=("line_value", "sum"),
        average_manual_value=("line_value", "mean"),
        maximum_manual_value=("line_value", "max")
    )
    .sort_values(
        "total_manual_revenue",
        ascending=False
    )
)

print("Manual-only customer profile:")
print("Unique customers:", len(manual_customer_profile))

display(manual_customer_profile.head(20))

# Revenue concentration

total_manual_only_revenue = (
    manual_only_records["line_value"].sum()
)

top_5_revenue = (
    manual_customer_profile
    .head(5)["total_manual_revenue"]
    .sum()
)

top_10_revenue = (
    manual_customer_profile
    .head(10)["total_manual_revenue"]
    .sum()
)

print("\nRevenue concentration:")

display(
    pd.DataFrame(
        {
            "revenue": [
                total_manual_only_revenue,
                top_5_revenue,
                top_10_revenue
            ],
            "percentage_of_manual_only_revenue": [
                100,
                top_5_revenue / total_manual_only_revenue * 100,
                top_10_revenue / total_manual_only_revenue * 100
            ]
        },
        index=[
            "All manual-only customers",
            "Top 5 customers",
            "Top 10 customers"
        ]
    )
)

# Monthly distribution

manual_only_records["invoice_month"] = (
    manual_only_records["InvoiceDate"]
    .dt.to_period("M")
)

manual_monthly_profile = (
    manual_only_records
    .groupby("invoice_month")
    .agg(
        record_count=("Invoice", "size"),
        unique_customers=("Customer ID", "nunique"),
        total_revenue=("line_value", "sum")
    )
)

print("\nMonthly Manual-only transaction profile:")

display(manual_monthly_profile)


Manual-only customer profile:
Unique customers: 108


,record_count,unique_invoices,total_manual_revenue,average_manual_value,maximum_manual_value
Customer ID,,,,,
15760.0,2,2,13916.34,6958.170000,6958.17
14063.0,2,2,13238.70,6619.350000,10468.80
12744.0,7,3,12158.90,1736.985714,3949.32
14911.0,19,19,11379.66,598.929474,1687.17
12918.0,1,1,10953.50,10953.500000,10953.50
17949.0,2,2,9465.60,4732.800000,8985.60
12536.0,2,2,8322.12,4161.060000,4161.06
14156.0,5,5,8178.45,1635.690000,2736.16
15202.0,3,3,6001.49,2000.496667,2667.88



Revenue concentration:


,revenue,percentage_of_manual_only_revenue
All manual-only customers,146678.08,100.000000
Top 5 customers,61647.10,42.028843
Top 10 customers,97225.26,66.284792



Monthly Manual-only transaction profile:


,record_count,unique_customers,total_revenue
invoice_month,,,
2009-12,7,6,2135.02
2010-01,11,6,15586.62
2010-02,6,5,3503.28
2010-03,12,9,26825.49
2010-04,7,4,2439.93
2010-05,6,5,1374.26
2010-06,7,6,2499.16
2010-07,9,9,4604.47
2010-08,9,4,3893.56


In [113]:

# Identify customers with repeated Manual-only invoices

repeated_manual_customers = (
    manual_customer_profile[
        manual_customer_profile["unique_invoices"] >= 2
    ]
    .head(15)
)

selected_customers = repeated_manual_customers.index

print("Selected customers:", len(selected_customers))

print("\nSelected customer profiles:")

display(repeated_manual_customers)

# Retrieve all Manual-only transactions for these customers

repeated_customer_transactions = (
    manual_only_records[
        manual_only_records["Customer ID"].isin(selected_customers)
    ]
    .sort_values(
        ["Customer ID", "InvoiceDate"]
    )
)

print("\nTransaction history:")

display(
    repeated_customer_transactions[
        [
            "Customer ID",
            "Invoice",
            "InvoiceDate",
            "Quantity",
            "Price",
            "line_value",
            "Country"
        ]
    ]
)

# Examine transaction frequency and value by customer

customer_manual_patterns = (
    repeated_customer_transactions
    .groupby("Customer ID")
    .agg(
        record_count=("Invoice", "size"),
        unique_invoices=("Invoice", "nunique"),
        first_transaction=("InvoiceDate", "min"),
        last_transaction=("InvoiceDate", "max"),
        total_revenue=("line_value", "sum"),
        median_transaction_value=("line_value", "median"),
        maximum_transaction_value=("line_value", "max")
    )
    .sort_values("total_revenue", ascending=False)
)

print("\nCustomer-level Manual transaction patterns:")

display(customer_manual_patterns)


Selected customers: 15

Selected customer profiles:


,record_count,unique_invoices,total_manual_revenue,average_manual_value,maximum_manual_value
Customer ID,,,,,
15760.0,2,2,13916.34,6958.170000,6958.17
14063.0,2,2,13238.70,6619.350000,10468.80
12744.0,7,3,12158.90,1736.985714,3949.32
14911.0,19,19,11379.66,598.929474,1687.17
17949.0,2,2,9465.60,4732.800000,8985.60
12536.0,2,2,8322.12,4161.060000,4161.06
14156.0,5,5,8178.45,1635.690000,2736.16
15202.0,3,3,6001.49,2000.496667,2667.88
12757.0,4,4,3599.40,899.850000,1241.98



Transaction history:


,Customer ID,Invoice,InvoiceDate,Quantity,Price,line_value,Country
167667,12482.0,505227,2010-04-21 10:33:00,1,874.84,874.84,Sweden
167668,12482.0,505229,2010-04-21 10:34:00,1,874.84,874.84,Sweden
947835,12536.0,573077,2011-10-27 14:13:00,1,4161.06,4161.06,France
947838,12536.0,573080,2011-10-27 14:20:00,1,4161.06,4161.06,France
670286,12744.0,548813,2011-04-04 13:03:00,1,162.60,162.60,Singapore
...,...,...,...,...,...,...,...
822899,15581.0,562946,2011-08-11 09:38:00,1,2500.00,2500.00,United Kingdom
129903,15760.0,501766,2010-03-19 11:35:00,1,6958.17,6958.17,Norway
129987,15760.0,501768,2010-03-19 11:45:00,1,6958.17,6958.17,Norway
74356,17949.0,496115,2010-01-29 11:04:00,1,8985.60,8985.60,United Kingdom



Customer-level Manual transaction patterns:


,record_count,unique_invoices,first_transaction,last_transaction,total_revenue,median_transaction_value,maximum_transaction_value
Customer ID,,,,,,,
15760.0,2,2,2010-03-19 11:35:00,2010-03-19 11:45:00,13916.34,6958.170,6958.17
14063.0,2,2,2010-09-27 16:12:00,2010-09-27 16:18:00,13238.70,6619.350,10468.80
12744.0,7,3,2011-04-04 13:03:00,2011-10-19 11:18:00,12158.90,2053.070,3949.32
14911.0,19,19,2009-12-17 17:43:00,2011-03-11 10:22:00,11379.66,608.500,1687.17
17949.0,2,2,2010-01-29 11:04:00,2011-09-01 11:56:00,9465.60,4732.800,8985.60
12536.0,2,2,2011-10-27 14:13:00,2011-10-27 14:20:00,8322.12,4161.060,4161.06
14156.0,5,5,2010-01-13 14:25:00,2010-10-28 09:45:00,8178.45,1193.890,2736.16
15202.0,3,3,2010-10-15 12:39:00,2010-10-15 13:05:00,6001.49,2225.170,2667.88
12757.0,4,4,2011-05-12 19:02:00,2011-05-23 14:58:00,3599.40,899.850,1241.98


In [114]:

# Inspect discount and adjustment transaction values

adjustment_stockcodes = [
    "D",
    "ADJUST",
    "ADJUST2",
    "BANK CHARGES"
]

adjustment_records = candidate_paid[
    candidate_paid["StockCode"]
    .astype(str)
    .isin(adjustment_stockcodes)
].copy()

print("Quantity distribution:")

display(
    adjustment_records
    .groupby("StockCode")["Quantity"]
    .describe()
)

print("\nPrice and line-value distribution:")

display(
    adjustment_records
    .groupby("StockCode")
    .agg(
        record_count=("Invoice", "size"),
        total_quantity=("Quantity", "sum"),
        minimum_price=("Price", "min"),
        maximum_price=("Price", "max"),
        total_revenue=("line_value", "sum"),
        minimum_line_value=("line_value", "min"),
        maximum_line_value=("line_value", "max")
    )
)

print("\nAll adjustment transaction records:")

display(
    adjustment_records[
        [
            "Invoice",
            "StockCode",
            "Description",
            "Quantity",
            "Price",
            "line_value",
            "InvoiceDate",
            "Customer ID",
            "Country"
        ]
    ].sort_values(
        ["StockCode", "InvoiceDate"]
    )
)


Quantity distribution:


,count,mean,std,min,25%,50%,75%,max
StockCode,,,,,,,,
ADJUST,32.0,1.0,0.000000,1.0,1.0,1.0,1.0,1.0
ADJUST2,3.0,1.0,0.000000,1.0,1.0,1.0,1.0,1.0
BANK CHARGES,32.0,1.0,0.000000,1.0,1.0,1.0,1.0,1.0
D,5.0,39.2,85.417797,1.0,1.0,1.0,1.0,192.0



Price and line-value distribution:


,record_count,total_quantity,minimum_price,maximum_price,total_revenue,minimum_line_value,maximum_line_value
StockCode,,,,,,,
ADJUST,32,32,4.570,387.54,3538.520,4.570,387.54
ADJUST2,3,3,72.450,358.47,731.050,72.450,358.47
BANK CHARGES,32,32,0.001,15.00,465.001,0.001,15.00
D,5,196,1.000,101.99,397.890,13.980,192.00



All adjustment transaction records:


,Invoice,StockCode,Description,Quantity,Price,line_value,InvoiceDate,Customer ID,Country
70976,495733,ADJUST,Adjustment by john on 26/01/2010 16,1,68.34,68.34,2010-01-26 16:21:00,14911.0,EIRE
70977,495735,ADJUST,Adjustment by john on 26/01/2010 16,1,201.56,201.56,2010-01-26 16:22:00,12745.0,EIRE
70978,495734,ADJUST,Adjustment by john on 26/01/2010 16,1,205.82,205.82,2010-01-26 16:22:00,14911.0,EIRE
70980,495736,ADJUST,Adjustment by john on 26/01/2010 16,1,21.00,21.00,2010-01-26 16:23:00,12606.0,Spain
70985,495742,ADJUST,Adjustment by john on 26/01/2010 16,1,63.24,63.24,2010-01-26 16:25:00,12404.0,Finland
...,...,...,...,...,...,...,...,...,...
160443,504700,D,Discount,1,57.63,57.63,2010-04-15 18:08:00,17032.0,United Kingdom
212633,509979,D,Discount,1,101.99,101.99,2010-05-26 14:07:00,12843.0,United Kingdom
312285,519808,D,Discount,192,1.00,192.00,2010-08-20 12:50:00,16422.0,United Kingdom
494226,535657,D,Discount,1,13.98,13.98,2010-11-28 11:20:00,14527.0,United Kingdom


In [117]:

# Identify invoices containing Discount (D) records

discount_invoices = (
    adjustment_records.loc[
        adjustment_records["StockCode"].astype(str) == "D",
        "Invoice"
    ]
    .unique()
)

# Retrieve all candidate paid-purchase records from those invoices

discount_invoice_context = candidate_paid[
    candidate_paid["Invoice"].isin(discount_invoices)
].copy()

# Calculate line values if not already available
if "line_value" not in discount_invoice_context.columns:
    discount_invoice_context["line_value"] = (
        discount_invoice_context["Quantity"]
        * discount_invoice_context["Price"]
    )

# Summarize each invoice

discount_invoice_summary = (
    discount_invoice_context
    .groupby("Invoice")
    .agg(
        record_count=("Invoice", "size"),
        unique_stockcodes=("StockCode", "nunique"),
        invoice_total_value=("line_value", "sum"),
        discount_records=(
            "StockCode",
            lambda x: (x.astype(str) == "D").sum()
        )
    )
    .sort_values("Invoice")
)

print("Invoices containing Discount records:")
display(discount_invoice_summary)

# Display all line items in chronological order

print("\nFull invoice line-item context:")

display(
    discount_invoice_context[
        [
            "Invoice",
            "StockCode",
            "Description",
            "Quantity",
            "Price",
            "line_value",
            "InvoiceDate",
            "Customer ID",
            "Country"
        ]
    ].sort_values(
        ["Invoice", "StockCode"]
    )
)

Invoices containing Discount records:


,record_count,unique_stockcodes,invoice_total_value,discount_records
Invoice,,,,
504700,1,1,57.63,1
509979,1,1,101.99,1
519808,1,1,192.00,1
535657,1,1,13.98,1
536158,1,1,32.29,1



Full invoice line-item context:


,Invoice,StockCode,Description,Quantity,Price,line_value,InvoiceDate,Customer ID,Country
160443,504700,D,Discount,1,57.63,57.63,2010-04-15 18:08:00,17032.0,United Kingdom
212633,509979,D,Discount,1,101.99,101.99,2010-05-26 14:07:00,12843.0,United Kingdom
312285,519808,D,Discount,192,1.00,192.00,2010-08-20 12:50:00,16422.0,United Kingdom
494226,535657,D,Discount,1,13.98,13.98,2010-11-28 11:20:00,14527.0,United Kingdom
501237,536158,D,Discount,1,32.29,32.29,2010-11-30 11:57:00,13634.0,United Kingdom


In [118]:

# Retrieve every original record from the five Discount invoices

discount_raw_context = df_transaction_review[
    df_transaction_review["Invoice"].isin(discount_invoices)
].copy()

discount_raw_context["line_value"] = (
    discount_raw_context["Quantity"]
    * discount_raw_context["Price"]
)

print("Original invoice context:")

display(
    discount_raw_context[
        [
            "Invoice",
            "StockCode",
            "Description",
            "Quantity",
            "Price",
            "line_value",
            "InvoiceDate",
            "Customer ID",
            "Country",
            "is_cancellation",
            "candidate_paid_purchase"
        ]
    ].sort_values(["Invoice", "StockCode"])
)

# Summarize the complete original invoices

discount_raw_summary = (
    discount_raw_context
    .groupby("Invoice")
    .agg(
        original_record_count=("Invoice", "size"),
        unique_stockcodes=("StockCode", "nunique"),
        total_quantity=("Quantity", "sum"),
        total_recorded_value=("line_value", "sum"),
        missing_customer_records=(
            "Customer ID",
            lambda x: x.isna().sum()
        )
    )
)

print("\nOriginal invoice summary:")

display(discount_raw_summary)


Original invoice context:


,Invoice,StockCode,Description,Quantity,Price,line_value,InvoiceDate,Customer ID,Country,is_cancellation,candidate_paid_purchase
160443,504700,D,Discount,1,57.63,57.63,2010-04-15 18:08:00,17032.0,United Kingdom,False,True
212633,509979,D,Discount,1,101.99,101.99,2010-05-26 14:07:00,12843.0,United Kingdom,False,True
312285,519808,D,Discount,192,1.00,192.00,2010-08-20 12:50:00,16422.0,United Kingdom,False,True
494226,535657,D,Discount,1,13.98,13.98,2010-11-28 11:20:00,14527.0,United Kingdom,False,True
501237,536158,D,Discount,1,32.29,32.29,2010-11-30 11:57:00,13634.0,United Kingdom,False,True



Original invoice summary:


,original_record_count,unique_stockcodes,total_quantity,total_recorded_value,missing_customer_records
Invoice,,,,,
504700,1,1,1,57.63,0
509979,1,1,1,101.99,0
519808,1,1,192,192.00,0
535657,1,1,1,13.98,0
536158,1,1,1,32.29,0


In [119]:

# StockCodes already investigated and provisionally classified

classified_special_stockcodes = [
    "POST",
    "M",
    "C2",
    "23444",
    "ADJUST",
    "BANK CHARGES",
    "DOT",
    "D",
    "ADJUST2"
]

# Identify remaining candidate StockCodes

remaining_stockcodes = stockcode_profile[
    ~stockcode_profile.index.astype(str).isin(
        classified_special_stockcodes
    )
].copy()

print("Remaining candidate StockCodes:", len(remaining_stockcodes))

print("\nStockCodes with multiple descriptions:")

display(
    remaining_stockcodes[
        remaining_stockcodes["unique_descriptions"] > 1
    ]
    .sort_values(
        ["unique_descriptions", "record_count"],
        ascending=False
    )
    .head(30)
)

print("\nMost frequent remaining StockCodes:")

display(
    remaining_stockcodes
    .sort_values("record_count", ascending=False)
    .head(30)
)


Remaining candidate StockCodes: 4622

StockCodes with multiple descriptions:


,record_count,unique_descriptions,total_quantity,total_revenue,unique_customers
StockCode,,,,,
22384,1922,4,20494,32681.50,681
20685,1515,4,9545,61829.13,626
21955,601,4,3744,21288.20,279
23236,296,4,2459,6828.31,203
23196,243,4,1760,2484.80,170
22345,86,4,1314,609.18,62
22344,86,4,1359,636.81,65
22346,65,4,1145,521.33,43
85099B,3360,3,94983,170616.68,978



Most frequent remaining StockCodes:


,record_count,unique_descriptions,total_quantity,total_revenue,unique_customers
StockCode,,,,,
85123A,5188,2,93697,252227.81,1490
22423,3428,1,24899,286486.30,1314
85099B,3360,3,94983,170616.68,978
84879,2777,1,79913,127074.17,1010
20725,2678,2,38314,60271.70,845
21212,2654,2,91263,44067.34,1115
21034,2141,1,4120,3914.00,800
47566,2121,1,23607,103880.23,894
20727,2117,1,25820,40586.20,705


In [120]:

# Select StockCodes with the most description variations

top_multidescription_stockcodes = (
    remaining_stockcodes[
        remaining_stockcodes["unique_descriptions"] > 1
    ]
    .sort_values(
        ["unique_descriptions", "record_count"],
        ascending=False
    )
    .head(30)
    .index
)

# Retrieve actual description variations from candidate transactions

description_variation_records = candidate_paid[
    candidate_paid["StockCode"].isin(
        top_multidescription_stockcodes
    )
].copy()

# Summarize each StockCode-description combination

description_variation_profile = (
    description_variation_records
    .groupby(
        ["StockCode", "Description"],
        dropna=False
    )
    .agg(
        record_count=("Invoice", "size"),
        unique_invoices=("Invoice", "nunique"),
        total_quantity=("Quantity", "sum"),
        total_revenue=("line_value", "sum"),
        unique_customers=("Customer ID", "nunique")
    )
    .reset_index()
    .sort_values(
        ["StockCode", "record_count"],
        ascending=[True, False]
    )
)

print("Description variations for selected StockCodes:")

display(description_variation_profile)


Description variations for selected StockCodes:


,StockCode,Description,record_count,unique_invoices,total_quantity,total_revenue,unique_customers
1,20685,DOORMAT RED RETROSPOT,793,758,4985,33132.45,416
0,20685,DOOR MAT RED SPOT,370,369,2427,15100.97,245
2,20685,DOORMAT RED SPOT,187,187,1035,7050.21,159
3,20685,RED SPOTTY COIR DOORMAT,165,164,1098,6545.50,132
5,20750,RED RETROSPOT MINI CASES,502,479,2103,15162.05,307
...,...,...,...,...,...,...,...
92,84997D,CHILDRENS CUTLERY POLKADOT PINK,328,323,3530,13796.60,208
94,84997D,PINK 3 PIECE POLKADOT CUTLERY SET,280,253,2652,9579.88,172
95,85099B,JUMBO BAG RED RETROSPOT,2702,2612,75759,136980.08,860
96,85099B,JUMBO BAG RED WHITE SPOTTY,473,465,13890,24345.30,321


In [121]:

# Display all description variations for the selected StockCodes

for stockcode, group in description_variation_profile.groupby(
    "StockCode",
    sort=False
):
    print(f"\nStockCode: {stockcode}")
    print("-" * 70)

    for _, row in group.iterrows():
        print(
            f"{row['record_count']:>5} records | "
            f"{row['unique_invoices']:>4} invoices | "
            f"{row['unique_customers']:>4} customers | "
            f"{row['Description']}"
        )



StockCode: 20685
----------------------------------------------------------------------
  793 records |  758 invoices |  416 customers | DOORMAT RED RETROSPOT
  370 records |  369 invoices |  245 customers | DOOR MAT RED SPOT
  187 records |  187 invoices |  159 customers | DOORMAT RED SPOT
  165 records |  164 invoices |  132 customers | RED SPOTTY COIR DOORMAT

StockCode: 20750
----------------------------------------------------------------------
  502 records |  479 invoices |  307 customers | RED RETROSPOT MINI CASES
  253 records |  250 invoices |  172 customers | RED/WHITE DOT MINI CASES
  141 records |  138 invoices |  113 customers |  RED/WHITE DOT MINI CASES

StockCode: 21243
----------------------------------------------------------------------
  207 records |  205 invoices |  128 customers | PINK  SPOTTY PLATE 
  178 records |  172 invoices |  117 customers | PINK  POLKADOT PLATE 
   18 records |   18 invoices |   18 customers | PINK POLKADOT PLATE 

StockCode: 21523
-----

In [122]:

import re

# Work on a separate copy
description_normalization_review = description_variation_records.copy()

# Preserve the original description
description_normalization_review["original_description"] = (
    description_normalization_review["Description"]
)

# Apply conservative text normalization
description_normalization_review["normalized_description"] = (
    description_normalization_review["Description"]
    .astype("string")
    .str.upper()
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

# Compare original and normalized description counts

normalization_comparison = (
    description_normalization_review
    .groupby("StockCode")
    .agg(
        original_description_count=(
            "original_description",
            "nunique"
        ),
        normalized_description_count=(
            "normalized_description",
            "nunique"
        ),
        record_count=("Invoice", "size")
    )
)

normalization_comparison["descriptions_removed"] = (
    normalization_comparison["original_description_count"]
    - normalization_comparison["normalized_description_count"]
)

print("Description normalization comparison:")

display(
    normalization_comparison.sort_values(
        "descriptions_removed",
        ascending=False
    )
)


Description normalization comparison:


,original_description_count,normalized_description_count,record_count,descriptions_removed
StockCode,,,,
20750,3,2,896,1
21243,3,2,403,1
21955,4,3,601,1
22383,3,2,2100,1
23240,3,2,498,1
20685,4,4,1515,0
22139,3,3,1873,0
22191,3,3,615,0
22197,3,3,1852,0


In [123]:

# Inspect original descriptions that map to the same normalized description

normalized_description_mapping = (
    description_normalization_review
    .groupby(
        [
            "StockCode",
            "normalized_description",
            "original_description"
        ],
        dropna=False
    )
    .agg(
        record_count=("Invoice", "size")
    )
    .reset_index()
)

# Keep only StockCodes where normalization reduced description variations

affected_stockcodes = normalization_comparison[
    normalization_comparison["descriptions_removed"] > 0
].index

consolidated_description_details = (
    normalized_description_mapping[
        normalized_description_mapping["StockCode"].isin(
            affected_stockcodes
        )
    ]
    .sort_values(
        ["StockCode", "normalized_description", "record_count"],
        ascending=[True, True, False]
    )
)

print("Descriptions consolidated through normalization:")

display(consolidated_description_details)


Descriptions consolidated through normalization:


,StockCode,normalized_description,original_description,record_count
4,20750,RED RETROSPOT MINI CASES,RED RETROSPOT MINI CASES,502
6,20750,RED/WHITE DOT MINI CASES,RED/WHITE DOT MINI CASES,253
5,20750,RED/WHITE DOT MINI CASES,RED/WHITE DOT MINI CASES,141
7,21243,PINK POLKADOT PLATE,PINK POLKADOT PLATE,178
8,21243,PINK POLKADOT PLATE,PINK POLKADOT PLATE,18
9,21243,PINK SPOTTY PLATE,PINK SPOTTY PLATE,207
16,21955,DOOR MAT UNION JACK GUNS AND ROSES,DOOR MAT UNION JACK GUNS AND ROSES,168
18,21955,DOORMAT UNION JACK GUNS AND ROSES,DOORMAT UNION JACK GUNS AND ROSES,292
17,21955,DOORMAT UNION JACK GUNS AND ROSES,DOORMAT UNION JACK GUNS AND ROSES,53
19,21955,UNION JACK GUNS & ROSES DOORMAT,UNION JACK GUNS & ROSES DOORMAT,88


In [124]:

# Create a normalized description column across the full dataset

df_transaction_review["Description_normalized"] = (
    df_transaction_review["Description"]
    .astype("string")
    .str.upper()
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

# Validate that original descriptions remain unchanged

original_description_check = (
    df_transaction_review["Description"].equals(
        df_description_cleaning["Description"]
    )
)

print("Original descriptions preserved:", original_description_check)

print(
    "Original missing descriptions:",
    df_transaction_review["Description"].isna().sum()
)

print(
    "Normalized missing descriptions:",
    df_transaction_review["Description_normalized"].isna().sum()
)


Original descriptions preserved: True
Original missing descriptions: 1642
Normalized missing descriptions: 1642


### 3.5 Description Normalization

Following the investigation of multiple descriptions associated with the same StockCode, a conservative text normalization procedure was applied.

The investigation covered the 30 most frequently occurring StockCodes with multiple descriptions among candidate paid-purchase records.

**Normalization procedure:**
- Converted descriptions to uppercase.
- Removed leading and trailing whitespace.
- Replaced consecutive whitespace characters with a single space.
- Preserved the original description in a separate column.

The normalization reduced the number of distinct descriptions for 5 of the 30 investigated StockCodes. These reductions resulted from whitespace inconsistencies rather than semantic consolidation.

Other variations, including differences in word order, spelling, product attributes, and terminology, were retained. No assumption was made that different descriptions under the same StockCode necessarily represented identical products.

**Implementation:**
- Original column: `Description`
- New column: `Description_normalized`

**Validation results:**
- Original descriptions remained unchanged.
- Original missing descriptions: 1,642.
- Normalized missing descriptions: 1,642.

**Treatment decision:**

The original descriptions were preserved for traceability, while the normalized column was introduced to improve textual consistency in subsequent analysis. No automatic canonical description mapping was applied.

This approach avoids introducing unsupported assumptions about product identity while addressing basic formatting inconsistencies.

# Special Transaction Classification

In [125]:

# Create zero-price classification flags

df_transaction_review["is_zero_price"] = (
    df_transaction_review["Price"] == 0
)

df_transaction_review["zero_price_category"] = "not_zero_price"

zero_price_mask = df_transaction_review["is_zero_price"]

df_transaction_review.loc[
    zero_price_mask & (df_transaction_review["Quantity"] > 0)
    & df_transaction_review["Customer ID"].notna(),
    "zero_price_category"
] = "positive_zero_price_identified_customer"

df_transaction_review.loc[
    zero_price_mask & (df_transaction_review["Quantity"] > 0)
    & df_transaction_review["Customer ID"].isna(),
    "zero_price_category"
] = "positive_zero_price_anonymous"

df_transaction_review.loc[
    zero_price_mask & (df_transaction_review["Quantity"] < 0),
    "zero_price_category"
] = "negative_zero_price"

df_transaction_review.loc[
    zero_price_mask & (df_transaction_review["Quantity"] == 0),
    "zero_price_category"
] = "zero_quantity_zero_price"

# Summarize the classification
zero_price_classification_summary = (
    df_transaction_review
    .groupby("zero_price_category")
    .agg(
        record_count=("Invoice", "size"),
        unique_invoices=("Invoice", "nunique"),
        identified_customers=("Customer ID", "nunique"),
        total_quantity=("Quantity", "sum")
    )
    .sort_values("record_count", ascending=False)
)

display(zero_price_classification_summary)


,record_count,unique_invoices,identified_customers,total_quantity
zero_price_category,,,,
not_zero_price,1061169,48374,5939,10929318
negative_zero_price,3457,3393,0,-573085
positive_zero_price_anonymous,2674,1925,0,237505
positive_zero_price_identified_customer,71,61,51,14754


In [126]:

# Identify invoices containing candidate paid-purchase records

paid_purchase_invoices = set(
    df_transaction_review.loc[
        df_transaction_review["candidate_paid_purchase"],
        "Invoice"
    ].unique()
)

# Isolate positive-quantity, zero-price records with identified customers

identified_zero_price = df_transaction_review[
    df_transaction_review["zero_price_category"]
    == "positive_zero_price_identified_customer"
].copy()

# Classify each invoice
identified_zero_price["invoice_type"] = (
    identified_zero_price["Invoice"]
    .isin(paid_purchase_invoices)
    .map({
        True: "Mixed invoice",
        False: "Zero-price-only invoice"
    })
)

# Summarize invoice composition
zero_price_invoice_summary = (
    identified_zero_price
    .groupby("invoice_type")
    .agg(
        record_count=("Invoice", "size"),
        unique_invoices=("Invoice", "nunique"),
        identified_customers=("Customer ID", "nunique"),
        total_quantity=("Quantity", "sum")
    )
)

display(zero_price_invoice_summary)


,record_count,unique_invoices,identified_customers,total_quantity
invoice_type,,,,
Mixed invoice,62,55,47,1611
Zero-price-only invoice,9,6,6,13143


### 3.6 Zero-Price Transaction Classification

A total of 6,202 records had a unit price of zero. These records were investigated to distinguish operational adjustments from customer-associated activity.

**Classification results:**

| Category | Records | Unique invoices | Identified customers |
|---|---:|---:|---:|
| Negative-quantity zero-price records | 3,457 | 3,393 | 0 |
| Positive-quantity zero-price records, anonymous | 2,674 | 1,925 | 0 |
| Positive-quantity zero-price records, identified customers | 71 | 61 | 51 |
| **Total** | **6,202** | — | — |

The 3,457 negative-quantity records had previously been investigated as operational or inventory-related adjustments. Their exact business purpose could not be conclusively established.

The 71 positive-quantity zero-price records associated with identified customers were further examined at invoice level:

| Invoice category | Records | Unique invoices | Identified customers |
|---|---:|---:|---:|
| Mixed invoices containing candidate paid purchases | 62 | 55 | 47 |
| Zero-price-only invoices | 9 | 6 | 6 |
| **Total** | **71** | **61** | **51** |

**Treatment decisions:**

- All zero-price records are retained in the transaction dataset.
- Zero-price lines contribute zero recorded revenue.
- Zero-price lines in mixed invoices are retained but do not independently qualify as paid purchases.
- Zero-price-only invoices do not qualify as paid purchases for customer inactivity calculations.
- Identified zero-price activity may be retained as a separate customer behavioral indicator.
- Anonymous zero-price transactions remain available for transaction-level analysis but are excluded from customer-level analysis.
- Negative-quantity zero-price records are excluded from customer purchasing and merchandise revenue calculations.

For the inactivity prediction target, a customer's last qualifying paid-purchase date will be used rather than their last recorded activity date. This prevents zero-price transactions from incorrectly resetting the customer's purchase inactivity period.

**Limitation:** A zero unit price does not necessarily establish that an item was promotional or free of commercial significance. The dataset does not provide sufficient information to determine the business reason for every zero-price transaction.

# Delivery-Charge Classification

In [129]:
# Delivery-related StockCodes identified during the investigation

delivery_stockcodes = ["POST", "C2", "23444", "DOT"]

# Extract candidate paid transactions associated with these StockCodes
delivery_records = candidate_paid[
    candidate_paid["StockCode"].astype(str).isin(delivery_stockcodes)
].copy()

# Summarize each StockCode and description
delivery_profile = (
    delivery_records
    .groupby(["StockCode", "Description"], dropna=False)
    .agg(
        record_count=("Invoice", "size"),
        unique_invoices=("Invoice", "nunique"),
        unique_customers=("Customer ID", "nunique"),
        total_quantity=("Quantity", "sum"),
        total_revenue=("line_value", "sum")
    )
    .reset_index()
    .sort_values(["StockCode", "record_count"], ascending=[True, False])
)

print("Delivery-related transaction profile:")
display(delivery_profile)

# Determine invoice composition
delivery_invoice_ids = set(delivery_records["Invoice"].unique())

other_candidate_records = candidate_paid[
    ~candidate_paid["StockCode"].astype(str).isin(delivery_stockcodes)
].copy()

other_candidate_invoice_ids = set(
    other_candidate_records["Invoice"].unique()
)

delivery_invoice_composition = pd.DataFrame({
    "invoice_type": [
        "Delivery charge with other candidate purchases",
        "Delivery-charge-only invoice"
    ],
    "unique_invoices": [
        len(delivery_invoice_ids & other_candidate_invoice_ids),
        len(delivery_invoice_ids - other_candidate_invoice_ids)
    ]
})

print("Delivery invoice composition:")
display(delivery_invoice_composition)

print(
    "Total delivery-related records:",
    len(delivery_records)
)

print(
    "Total delivery-related recorded revenue:",
    round(delivery_records["line_value"].sum(), 2)
)

Delivery-related transaction profile:


,StockCode,Description,record_count,unique_invoices,unique_customers,total_quantity,total_revenue
0,23444,Next Day Carriage,79,79,63,79,1200.00
1,C2,CARRIAGE,253,248,45,254,12796.00
2,DOT,DOTCOM POSTAGE,16,16,1,16,11906.36
3,POST,POSTAGE,1838,1803,405,5333,126563.04


Delivery invoice composition:


,invoice_type,unique_invoices
0,Delivery charge with other candidate purchases,2029
1,Delivery-charge-only invoice,116


Total delivery-related records: 2186
Total delivery-related recorded revenue: 152465.4


In [130]:

# Identify invoices with delivery-related candidate transactions
delivery_invoice_ids = set(delivery_records["Invoice"].unique())

# Other candidate paid records on those invoices
other_candidate_delivery_context = candidate_paid[
    candidate_paid["Invoice"].isin(delivery_invoice_ids)
    & ~candidate_paid["StockCode"].astype(str).isin(delivery_stockcodes)
].copy()

# Summarize other StockCodes appearing alongside delivery charges
print("Other candidate StockCodes accompanying delivery charges:")

display(
    other_candidate_delivery_context
    .groupby("StockCode")
    .agg(
        record_count=("Invoice", "size"),
        unique_invoices=("Invoice", "nunique"),
        total_revenue=("line_value", "sum")
    )
    .sort_values("record_count", ascending=False)
    .head(20)
)

# Identify delivery-only invoices within candidate_paid
other_candidate_invoice_ids = set(
    other_candidate_delivery_context["Invoice"].unique()
)

delivery_only_invoice_ids = (
    delivery_invoice_ids - other_candidate_invoice_ids
)

print(
    "\nDelivery-only candidate invoices:",
    len(delivery_only_invoice_ids)
)

# Inspect all records on these invoices, including records
# that did not qualify as candidate paid purchases
delivery_only_context = df_transaction_review[
    df_transaction_review["Invoice"].isin(delivery_only_invoice_ids)
].copy()

print("\nComplete delivery-only invoice context:")

display(
    delivery_only_context[
        [
            "Invoice",
            "StockCode",
            "Description",
            "Quantity",
            "Price",
            "Customer ID",
            "candidate_paid_purchase"
        ]
    ].sort_values(["Invoice", "StockCode"])
)

# Inspect DOT transactions in detail
print("\nDOT transaction details:")

display(
    delivery_records[
        delivery_records["StockCode"].astype(str) == "DOT"
    ][
        [
            "Invoice",
            "InvoiceDate",
            "Description",
            "Quantity",
            "Price",
            "line_value",
            "Customer ID",
            "Country"
        ]
    ].sort_values("InvoiceDate")
)


Other candidate StockCodes accompanying delivery charges:


,record_count,unique_invoices,total_revenue
StockCode,,,
22326,415,406,14747.57
21731,286,280,8801.76
22328,263,259,7889.34
22423,258,254,18022.95
22556,257,253,6512.06
22554,252,246,5891.08
20719,240,237,3324.17
21212,217,214,3129.23
20750,215,208,5637.30



Delivery-only candidate invoices: 116

Complete delivery-only invoice context:


,Invoice,StockCode,Description,Quantity,Price,Customer ID,candidate_paid_purchase
126,489444,POST,POSTAGE,1,141.0,12636.0,True
173,489447,POST,POSTAGE,1,130.0,12362.0,True
21450,491094,POST,POSTAGE,1,216.0,12709.0,True
48395,493924,POST,POSTAGE,1,180.0,12678.0,True
52961,494234,POST,POSTAGE,1,5.0,12549.0,True
...,...,...,...,...,...,...,...
985421,575888,POST,POSTAGE,3,18.0,12517.0,True
1017958,578123,POST,POSTAGE,2,40.0,12452.0,True
1023029,578425,POST,POSTAGE,4,18.0,12705.0,True
1041335,579790,POST,POSTAGE,2,28.0,12784.0,True



DOT transaction details:


,Invoice,InvoiceDate,Description,Quantity,Price,line_value,Customer ID,Country
842968,564764,2011-08-30 10:49:00,DOTCOM POSTAGE,1,11.17,11.17,14096.0,United Kingdom
849463,565383,2011-09-02 15:45:00,DOTCOM POSTAGE,1,16.46,16.46,14096.0,United Kingdom
859217,566217,2011-09-09 15:17:00,DOTCOM POSTAGE,1,13.16,13.16,14096.0,United Kingdom
864291,566566,2011-09-13 12:32:00,DOTCOM POSTAGE,1,85.58,85.58,14096.0,United Kingdom
876061,567656,2011-09-21 14:40:00,DOTCOM POSTAGE,1,878.55,878.55,14096.0,United Kingdom
887045,568346,2011-09-26 15:28:00,DOTCOM POSTAGE,1,688.08,688.08,14096.0,United Kingdom
898227,569246,2011-10-03 09:57:00,DOTCOM POSTAGE,1,490.25,490.25,14096.0,United Kingdom
915445,570488,2011-10-10 17:12:00,DOTCOM POSTAGE,1,635.01,635.01,14096.0,United Kingdom
927227,571441,2011-10-17 13:31:00,DOTCOM POSTAGE,1,743.62,743.62,14096.0,United Kingdom
941925,572552,2011-10-24 17:07:00,DOTCOM POSTAGE,1,638.30,638.30,14096.0,United Kingdom


In [131]:

# 1. Inspect every complete invoice containing DOT

dot_invoice_ids = set(
    delivery_records.loc[
        delivery_records["StockCode"].astype(str) == "DOT",
        "Invoice"
    ]
)

dot_invoice_context = df_transaction_review[
    df_transaction_review["Invoice"].isin(dot_invoice_ids)
].copy()

dot_invoice_context["line_value"] = (
    dot_invoice_context["Quantity"] * dot_invoice_context["Price"]
)

print("Complete DOT invoice context:")

display(
    dot_invoice_context[
        [
            "Invoice", "InvoiceDate", "StockCode", "Description",
            "Quantity", "Price", "line_value", "Customer ID",
            "Country", "candidate_paid_purchase"
        ]
    ].sort_values(["Invoice", "StockCode"])
)

# 2. Profile the 116 delivery-only invoices

delivery_only_profile = (
    delivery_only_context
    .groupby("Customer ID")
    .agg(
        invoice_count=("Invoice", "nunique"),
        record_count=("Invoice", "size"),
        total_delivery_value=("Price", "sum"),
        first_invoice=("InvoiceDate", "min"),
        last_invoice=("InvoiceDate", "max")
    )
    .sort_values("invoice_count", ascending=False)
)

print("\nCustomer profile for delivery-only invoices:")
display(delivery_only_profile)

# 3. Inspect delivery-only POST value distribution

print("\nDelivery-only POST transaction statistics:")

display(
    delivery_only_context[
        delivery_only_context["StockCode"].astype(str) == "POST"
    ][["Quantity", "Price"]].describe()
)


Complete DOT invoice context:


,Invoice,InvoiceDate,StockCode,Description,Quantity,Price,line_value,Customer ID,Country,candidate_paid_purchase
842964,564764,2011-08-30 10:49:00,16225,RATTLE SNAKE EGGS,1,3.29,3.29,14096.0,United Kingdom,True
842965,564764,2011-08-30 10:49:00,16254,TRANSPARENT ACRYLIC TAPE DISPENSER,1,1.63,1.63,14096.0,United Kingdom,True
842963,564764,2011-08-30 10:49:00,16207B,PINK HEART RED HANDBAG,1,2.46,2.46,14096.0,United Kingdom,True
842966,564764,2011-08-30 10:49:00,84249A,"GREETING CARD,SQUARE, DOUGHNUTS",1,0.83,0.83,14096.0,United Kingdom,True
842967,564764,2011-08-30 10:49:00,84659A,WHITE TRAVEL ALARM CLOCK,2,1.23,2.46,14096.0,United Kingdom,True
...,...,...,...,...,...,...,...,...,...,...
1053539,580727,2011-12-05 17:17:00,85179A,GREEN BITTY LIGHT CHAIN,1,9.13,9.13,14096.0,United Kingdom,True
1053540,580727,2011-12-05 17:17:00,85180A,RED HEARTS LIGHT CHAIN,1,9.13,9.13,14096.0,United Kingdom,True
1053541,580727,2011-12-05 17:17:00,85199S,SMALL HANGING IVORY/RED WOOD BIRD,4,0.83,3.32,14096.0,United Kingdom,True
1053542,580727,2011-12-05 17:17:00,85231B,CINAMMON SET OF 9 T-LIGHTS,1,1.63,1.63,14096.0,United Kingdom,True



Customer profile for delivery-only invoices:


,invoice_count,record_count,total_delivery_value,first_invoice,last_invoice
Customer ID,,,,,
12678.0,6,6,818.00,2010-01-08 11:25:00,2011-08-31 10:24:00
14646.0,6,6,93.00,2010-04-22 09:41:00,2011-10-06 10:34:00
12476.0,5,5,90.00,2010-09-14 13:05:00,2010-11-08 09:35:00
12471.0,4,4,294.00,2010-01-26 10:36:00,2011-10-05 12:24:00
12422.0,3,3,98.00,2010-05-25 11:28:00,2010-07-02 14:41:00
...,...,...,...,...,...
16029.0,1,1,8142.75,2011-05-03 13:46:00,2011-05-03 13:46:00
16828.0,1,1,15.00,2011-09-07 15:10:00,2011-09-07 15:10:00
16235.0,1,1,10.00,2011-09-19 15:04:00,2011-09-19 15:04:00



Delivery-only POST transaction statistics:


,Quantity,Price
count,110.000000,110.000000
mean,2.781818,153.818818
std,3.218016,781.204655
min,1.000000,4.500000
25%,1.000000,15.000000
50%,1.000000,18.000000
75%,3.000000,43.750000
max,21.000000,8142.750000


In [132]:

# Restrict the 116 invoices to delivery-related lines only

delivery_only_charges = delivery_only_context[
    delivery_only_context["StockCode"].astype(str).isin(
        delivery_stockcodes
    )
].copy()

delivery_only_charges["line_value"] = (
    delivery_only_charges["Quantity"]
    * delivery_only_charges["Price"]
)

# Summarize by delivery StockCode
print("Delivery-only invoice charges by StockCode:")

display(
    delivery_only_charges
    .groupby(["StockCode", "Description"], dropna=False)
    .agg(
        record_count=("Invoice", "size"),
        unique_invoices=("Invoice", "nunique"),
        unique_customers=("Customer ID", "nunique"),
        total_quantity=("Quantity", "sum"),
        total_revenue=("line_value", "sum"),
        average_line_value=("line_value", "mean"),
        maximum_line_value=("line_value", "max")
    )
    .sort_values("total_revenue", ascending=False)
)

# Correct customer-level delivery-charge profile
print("\nCorrected customer-level delivery-charge profile:")

delivery_only_customer_profile = (
    delivery_only_charges
    .groupby("Customer ID")
    .agg(
        invoice_count=("Invoice", "nunique"),
        delivery_record_count=("Invoice", "size"),
        total_delivery_revenue=("line_value", "sum"),
        first_invoice=("InvoiceDate", "min"),
        last_invoice=("InvoiceDate", "max")
    )
    .sort_values("total_delivery_revenue", ascending=False)
)

display(delivery_only_customer_profile.head(20))

# Check whether any non-delivery records occur in these invoices
non_delivery_context = delivery_only_context[
    ~delivery_only_context["StockCode"].astype(str).isin(
        delivery_stockcodes
    )
]

print("\nNon-delivery records in delivery-only candidate invoices:",
      len(non_delivery_context))

print(
    "Invoices containing non-delivery records:",
    non_delivery_context["Invoice"].nunique()
)


Delivery-only invoice charges by StockCode:


,,record_count,unique_invoices,unique_customers,total_quantity,total_revenue,average_line_value,maximum_line_value
StockCode,Description,,,,,,,
POST,POSTAGE,110,110,82,306,21131.07,192.100636,8142.75
C2,CARRIAGE,6,6,4,6,510.00,85.000000,150.00



Corrected customer-level delivery-charge profile:


,invoice_count,delivery_record_count,total_delivery_revenue,first_invoice,last_invoice
Customer ID,,,,,
16029.0,1,1,8142.75,2011-05-03 13:46:00,2011-05-03 13:46:00
12471.0,4,4,924.00,2010-01-26 10:36:00,2011-10-05 12:24:00
12357.0,1,1,850.00,2010-11-16 14:29:00,2010-11-16 14:29:00
12678.0,6,6,818.00,2010-01-08 11:25:00,2011-08-31 10:24:00
12432.0,1,1,700.00,2011-06-22 13:12:00,2011-06-22 13:12:00
12476.0,5,5,612.00,2010-09-14 13:05:00,2010-11-08 09:35:00
17444.0,1,1,550.94,2011-07-15 12:05:00,2011-07-15 12:05:00
12590.0,1,1,523.00,2011-05-12 14:52:00,2011-05-12 14:52:00
14646.0,6,6,450.00,2010-04-22 09:41:00,2011-10-06 10:34:00



Non-delivery records in delivery-only candidate invoices: 0
Invoices containing non-delivery records: 0


### 3.7 Delivery-Related Transaction Classification

Four StockCodes were identified as delivery-related transactions: `POST`, `C2`, `23444`, and `DOT`.

| StockCode | Description | Records | Unique invoices | Recorded revenue |
|---|---|---:|---:|---:|
| POST | POSTAGE | 1,838 | 1,803 | £126,563.04 |
| C2 | CARRIAGE | 253 | 248 | £12,796.00 |
| 23444 | Next Day Carriage | 79 | 79 | £1,200.00 |
| DOT | DOTCOM POSTAGE | 16 | 16 | £11,906.36 |
| **Total** | | **2,186** | — | **£152,465.40** |

The four StockCodes appeared across 2,145 unique invoices. Of these, 2,029 invoices also contained other candidate paid-purchase records, while 116 contained only delivery-related candidate records.

The 116 delivery-only invoices were investigated further:

| StockCode | Records | Unique invoices | Unique customers | Recorded revenue |
|---|---:|---:|---:|---:|
| POST | 110 | 110 | 82 | £21,131.07 |
| C2 | 6 | 6 | 4 | £510.00 |
| **Total** | **116** | **116** | — | **£21,641.07** |

No non-delivery records were found in these 116 invoices.

The 16 `DOT` records were also investigated at invoice level. All belonged to customer `14096` and appeared alongside merchandise records. Their descriptions identified them as `DOTCOM POSTAGE`. Due to their distinct characteristics and relatively high recorded values, they are retained as a separately identifiable delivery-related subtype.

Some standalone delivery charges were unusually high, including a `POST` line worth £8,142.75. These values are preserved and flagged for review rather than automatically treated as errors.

**Treatment decisions:**

- Preserve all delivery-related records.
- Exclude delivery charges from merchandise revenue.
- Retain delivery charges as a separate revenue component.
- Include eligible delivery charges in invoice-level recorded revenue reporting, separately from merchandise revenue.
- Do not allow delivery-only invoices to qualify as merchandise purchases for customer inactivity calculations.
- Retain `DOT` as a distinct delivery-related category rather than merging it indiscriminately with ordinary postage.

# Manual Transaction

In [133]:

# Special financial StockCodes

financial_stockcodes = [
    "M",
    "D",
    "ADJUST",
    "ADJUST2",
    "BANK CHARGES"
]

financial_records = df_transaction_review[
    df_transaction_review["StockCode"].astype(str).isin(
        financial_stockcodes
    )
].copy()

financial_records["line_value"] = (
    financial_records["Quantity"] * financial_records["Price"]
)

# Summarize each StockCode
financial_profile = (
    financial_records
    .groupby(["StockCode", "Description"], dropna=False)
    .agg(
        record_count=("Invoice", "size"),
        unique_invoices=("Invoice", "nunique"),
        unique_customers=("Customer ID", "nunique"),
        total_quantity=("Quantity", "sum"),
        total_revenue=("line_value", "sum"),
        negative_quantity_records=(
            "Quantity",
            lambda x: (x < 0).sum()
        ),
        zero_price_records=(
            "Price",
            lambda x: (x == 0).sum()
        )
    )
    .reset_index()
    .sort_values("record_count", ascending=False)
)

display(financial_profile)


,StockCode,Description,record_count,unique_invoices,unique_customers,total_quantity,total_revenue,negative_quantity_records,zero_price_records
7,M,Manual,1421,1286,582,4607,-82796.320,537,7
6,D,Discount,177,151,55,-2872,-13484.540,172,0
5,BANK CHARGES,Bank Charges,96,89,20,-40,-33493.669,63,0
1,ADJUST,Adjustment by john on 26/01/2010 16,38,38,32,2,1803.330,18,0
2,ADJUST,Adjustment by john on 26/01/2010 17,26,26,19,6,5762.960,10,0
4,BANK CHARGES,Bank Charges,6,6,3,-2,-2068.960,4,0
3,ADJUST2,Adjustment by Peter on Jun 25 2010,3,3,3,3,731.050,0,0
0,ADJUST,Adjustment by Peter on 24/05/2010 1,3,3,3,-3,-731.050,3,0


In [145]:
# Determine whether financial records overlap
# with cancellation invoices and examine quantity direction

financial_records["quantity_category"] = "zero_quantity"

financial_records.loc[
    financial_records["Quantity"] > 0,
    "quantity_category"
] = "positive_quantity"

financial_records.loc[
    financial_records["Quantity"] < 0,
    "quantity_category"
] = "negative_quantity"

financial_invoice_breakdown = (
    financial_records
    .groupby(
        ["StockCode", "is_cancellation", "quantity_category"],
        dropna=False
    )
    .agg(
        record_count=("Invoice", "size"),
        unique_invoices=("Invoice", "nunique"),
        identified_customers=("Customer ID", "nunique"),
        total_line_value=("line_value", "sum")
    )
    .reset_index()
    .sort_values(
        ["StockCode", "is_cancellation", "quantity_category"]
    )
)

display(financial_invoice_breakdown)


,StockCode,is_cancellation,quantity_category,record_count,unique_invoices,identified_customers,total_line_value
0,ADJUST,False,positive_quantity,36,36,25,8897.930
1,ADJUST,True,negative_quantity,31,31,29,-2062.690
2,ADJUST2,False,positive_quantity,3,3,3,731.050
3,BANK CHARGES,False,positive_quantity,35,33,19,534.241
4,BANK CHARGES,True,negative_quantity,67,62,5,-36096.870
5,D,False,positive_quantity,5,5,5,397.890
6,D,True,negative_quantity,172,146,52,-13882.430
7,M,False,positive_quantity,883,790,443,340716.280
8,M,True,negative_quantity,537,495,229,-423886.170
9,M,True,positive_quantity,1,1,0,373.570


In [144]:
# Match financial cancellation invoices
# with their potential original invoices

financial_cancellation_lines = financial_records[
    financial_records["is_cancellation"]
].copy()

financial_cancellation_lines["cancellation_invoice"] = (
    financial_cancellation_lines["Invoice"].astype(str)
)

financial_cancellation_lines["original_invoice_candidate"] = (
    financial_cancellation_lines["cancellation_invoice"]
    .str.replace(r"^C", "", regex=True)
)

# Check whether the corresponding original invoice exists
existing_invoice_ids = set(
    df_transaction_review["Invoice"].astype(str)
)

financial_cancellation_lines["original_invoice_exists"] = (
    financial_cancellation_lines["original_invoice_candidate"]
    .isin(existing_invoice_ids)
)

# Summarize matching availability by StockCode
original_invoice_match_profile = (
    financial_cancellation_lines
    .groupby(
        ["StockCode", "original_invoice_exists"],
        dropna=False
    )
    .agg(
        cancellation_records=("Invoice", "size"),
        unique_cancellation_invoices=("Invoice", "nunique"),
        total_line_value=("line_value", "sum")
    )
    .reset_index()
    .sort_values(["StockCode", "original_invoice_exists"])
)

print("Potential original invoice availability:")
display(original_invoice_match_profile)

# Inspect the positive-quantity cancellation record for M
print("\nPositive-quantity M record on a cancellation invoice:")

display(
    financial_cancellation_lines[
        (financial_cancellation_lines["StockCode"].astype(str) == "M")
        & (financial_cancellation_lines["Quantity"] > 0)
    ][
        [
            "Invoice", "InvoiceDate", "StockCode", "Description",
            "Quantity", "Price", "line_value", "Customer ID"
        ]
    ]
)

Potential original invoice availability:


,StockCode,original_invoice_exists,cancellation_records,unique_cancellation_invoices,total_line_value
0,ADJUST,False,31,31,-2062.69
1,BANK CHARGES,False,67,62,-36096.87
2,D,False,172,146,-13882.43
3,M,False,538,496,-423512.60



Positive-quantity M record on a cancellation invoice:


,Invoice,InvoiceDate,StockCode,Description,Quantity,Price,line_value,Customer ID
76799,C496350,2010-02-01 08:24:00,M,Manual,1,373.57,373.57,NaN


In [143]:
# Examine complete cancellation invoice composition

financial_cancellation_invoice_ids = set(
    financial_cancellation_lines["Invoice"]
)

financial_cancellation_context = df_transaction_review[
    df_transaction_review["Invoice"].isin(
        financial_cancellation_invoice_ids
    )
].copy()

financial_cancellation_context["line_value"] = (
    financial_cancellation_context["Quantity"]
    * financial_cancellation_context["Price"]
)

# Summarize invoice composition by financial StockCode
financial_invoice_ids_by_stockcode = (
    financial_cancellation_lines[
        ["Invoice", "StockCode"]
    ]
    .drop_duplicates()
)

invoice_context_summary = []

for stockcode in financial_stockcodes:
    relevant_invoices = set(
        financial_invoice_ids_by_stockcode.loc[
            financial_invoice_ids_by_stockcode["StockCode"].astype(str)
            == stockcode,
            "Invoice"
        ]
    )

    context = financial_cancellation_context[
        financial_cancellation_context["Invoice"].isin(
            relevant_invoices
        )
    ]

    financial_invoice_ids = set(
        financial_cancellation_lines.loc[
            financial_cancellation_lines["StockCode"].astype(str)
            == stockcode,
            "Invoice"
        ]
    )

    other_records = context[
        ~context["StockCode"].astype(str).isin(financial_stockcodes)
    ]

    invoice_context_summary.append({
        "StockCode": stockcode,
        "cancellation_invoices": len(financial_invoice_ids),
        "total_context_records": len(context),
        "other_stockcode_records": len(other_records),
        "invoices_with_other_stockcodes": other_records[
            "Invoice"
        ].nunique(),
        "unique_other_stockcodes": other_records[
            "StockCode"
        ].nunique(),
        "other_stockcode_value": other_records[
            "line_value"
        ].sum()
    })

invoice_context_summary = pd.DataFrame(
    invoice_context_summary
)

display(invoice_context_summary)

# Inspect the unusual positive-quantity M cancellation invoice
print("\nComplete context for cancellation invoice C496350:")

display(
    financial_cancellation_context[
        financial_cancellation_context["Invoice"].astype(str)
        == "C496350"
    ][
        [
            "Invoice", "InvoiceDate", "StockCode", "Description",
            "Quantity", "Price", "line_value", "Customer ID",
            "Country"
        ]
    ]
)

,StockCode,cancellation_invoices,total_context_records,other_stockcode_records,invoices_with_other_stockcodes,unique_other_stockcodes,other_stockcode_value
0,M,496,919,374,79,305,-7972.38
1,D,146,205,30,5,29,-2776.58
2,ADJUST,31,31,0,0,0,0.00
3,ADJUST2,0,0,0,0,0,0.00
4,BANK CHARGES,62,70,1,1,1,-25.89



Complete context for cancellation invoice C496350:


,Invoice,InvoiceDate,StockCode,Description,Quantity,Price,line_value,Customer ID,Country
76799,C496350,2010-02-01 08:24:00,M,Manual,1,373.57,373.57,NaN,United Kingdom


In [142]:
# Identify other StockCodes in M and D cancellation invoices

target_stockcodes = ["M", "D"]

target_cancellation_invoices = set(
    financial_cancellation_lines.loc[
        financial_cancellation_lines["StockCode"].astype(str).isin(
            target_stockcodes
        ),
        "Invoice"
    ]
)

target_cancellation_context = financial_cancellation_context[
    financial_cancellation_context["Invoice"].isin(
        target_cancellation_invoices
    )
].copy()

# Exclude all five financial StockCodes to isolate accompanying records
other_transaction_records = target_cancellation_context[
    ~target_cancellation_context["StockCode"].astype(str).isin(
        financial_stockcodes
    )
].copy()

# Profile accompanying StockCodes
other_stockcode_profile = (
    other_transaction_records
    .groupby(["StockCode", "Description"], dropna=False)
    .agg(
        record_count=("Invoice", "size"),
        unique_invoices=("Invoice", "nunique"),
        unique_customers=("Customer ID", "nunique"),
        total_quantity=("Quantity", "sum"),
        total_line_value=("line_value", "sum")
    )
    .reset_index()
    .sort_values("record_count", ascending=False)
)

print("Other StockCodes accompanying M and D:")
display(other_stockcode_profile.head(40))

print("\nTotal accompanying records:", len(other_transaction_records))
print(
    "Unique invoices containing accompanying records:",
    other_transaction_records["Invoice"].nunique()
)

# Examine quantity direction
print("\nQuantity direction of accompanying records:")
display(
    other_transaction_records.assign(
        quantity_category=other_transaction_records["Quantity"].apply(
            lambda x: (
                "positive_quantity" if x > 0
                else "negative_quantity" if x < 0
                else "zero_quantity"
            )
        )
    )
    .groupby("quantity_category")
    .agg(
        record_count=("Invoice", "size"),
        unique_invoices=("Invoice", "nunique"),
        total_line_value=("line_value", "sum")
    )
    .reset_index()
)

Other StockCodes accompanying M and D:


,StockCode,Description,record_count,unique_invoices,unique_customers,total_quantity,total_line_value
330,POST,POSTAGE,28,28,0,-28,-793.55
183,22423,REGENCY CAKESTAND 3 TIER,4,3,3,-9,-102.15
101,21843,RED RETROSPOT CAKE STAND,4,3,2,-7,-69.65
266,82482,WOODEN PICTURE FRAME WHITE FINISH,3,3,3,-4,-10.20
155,22197,SMALL POPCORN HOLDER,3,3,3,-19,-16.15
301,75049L,LARGE CIRCULAR MIRROR MOBILE,3,2,2,-102,-86.70
73,21534,DAIRY MAID LARGE MILK JUG,3,3,2,-3,-14.85
199,22624,IVORY KITCHEN SCALES,3,2,1,-12,-91.80
221,22899,CHILDREN'S APRON DOLLY GIRL,3,3,3,-5,-10.25
285,84992,72 SWEETHEART FAIRY CAKE CASES,2,2,2,-40,-17.19



Total accompanying records: 404
Unique invoices containing accompanying records: 84

Quantity direction of accompanying records:


,quantity_category,record_count,unique_invoices,total_line_value
0,negative_quantity,404,84,-10748.96


In [139]:
pd.set_option("display.max_rows", 40)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

display(other_stockcode_profile.head(40))

,StockCode,Description,record_count,unique_invoices,unique_customers,total_quantity,total_line_value
330,POST,POSTAGE,28,28,0,-28,-793.55
183,22423,REGENCY CAKESTAND 3 TIER,4,3,3,-9,-102.15
101,21843,RED RETROSPOT CAKE STAND,4,3,2,-7,-69.65
266,82482,WOODEN PICTURE FRAME WHITE FINISH,3,3,3,-4,-10.20
155,22197,SMALL POPCORN HOLDER,3,3,3,-19,-16.15
301,75049L,LARGE CIRCULAR MIRROR MOBILE,3,2,2,-102,-86.70
73,21534,DAIRY MAID LARGE MILK JUG,3,3,2,-3,-14.85
199,22624,IVORY KITCHEN SCALES,3,2,1,-12,-91.80
221,22899,CHILDREN'S APRON DOLLY GIRL,3,3,3,-5,-10.25
285,84992,72 SWEETHEART FAIRY CAKE CASES,2,2,2,-40,-17.19


In [141]:
# Classify cancellation invoice composition for M and D

target_stockcodes = ["M", "D"]

target_financial_cancellations = financial_cancellation_lines[
    financial_cancellation_lines["StockCode"].astype(str).isin(
        target_stockcodes
    )
].copy()

target_invoice_ids = set(
    target_financial_cancellations["Invoice"]
)

invoice_composition_records = financial_cancellation_context[
    financial_cancellation_context["Invoice"].isin(
        target_invoice_ids
    )
].copy()

# Identify transaction categories within each invoice
invoice_composition_records["stockcode"] = (
    invoice_composition_records["StockCode"].astype(str)
)

delivery_codes = ["POST", "C2", "23444", "DOT"]

financial_codes = financial_stockcodes

invoice_composition_records["is_delivery"] = (
    invoice_composition_records["stockcode"].isin(delivery_codes)
)

invoice_composition_records["is_financial"] = (
    invoice_composition_records["stockcode"].isin(financial_codes)
)

invoice_composition_records["is_other_transaction"] = (
    ~invoice_composition_records["is_delivery"]
    & ~invoice_composition_records["is_financial"]
)

invoice_composition = (
    invoice_composition_records
    .groupby("Invoice")
    .agg(
        has_delivery=("is_delivery", "any"),
        has_other_transactions=("is_other_transaction", "any"),
        has_merchandise=("is_other_transaction", "any")
    )
)

invoice_composition["composition"] = "financial_only"

invoice_composition.loc[
    invoice_composition["has_other_transactions"]
    & ~invoice_composition["has_delivery"],
    "composition"
] = "financial_plus_merchandise"

invoice_composition.loc[
    invoice_composition["has_delivery"]
    & ~invoice_composition["has_other_transactions"],
    "composition"
] = "financial_plus_delivery"

invoice_composition.loc[
    invoice_composition["has_delivery"]
    & invoice_composition["has_other_transactions"],
    "composition"
] = "financial_plus_merchandise_and_delivery"

print("Cancellation invoice composition for M and D:")

display(
    invoice_composition["composition"]
    .value_counts()
    .rename_axis("composition")
    .reset_index(name="invoice_count")
)

# Breakdown by financial StockCode
invoice_stockcode_mapping = (
    target_financial_cancellations[
        ["Invoice", "StockCode"]
    ]
    .drop_duplicates()
)

invoice_composition_by_stockcode = (
    invoice_stockcode_mapping
    .merge(
        invoice_composition.reset_index(),
        on="Invoice",
        how="left"
    )
    .groupby(["StockCode", "composition"])
    .agg(
        invoice_count=("Invoice", "nunique")
    )
    .reset_index()
    .sort_values(["StockCode", "composition"])
)

print("\nComposition breakdown by financial StockCode:")

display(invoice_composition_by_stockcode)


Cancellation invoice composition for M and D:


,composition,invoice_count
0,financial_only,556
1,financial_plus_merchandise,56
2,financial_plus_delivery,27
3,financial_plus_merchandise_and_delivery,1



Composition breakdown by financial StockCode:


,StockCode,composition,invoice_count
0,D,financial_only,141
1,D,financial_plus_merchandise,5
2,M,financial_only,417
3,M,financial_plus_delivery,27
4,M,financial_plus_merchandise,51
5,M,financial_plus_merchandise_and_delivery,1


In [147]:
# Identify cancellation invoices containing both M and D

m_d_invoice_overlap = (
    target_financial_cancellations[
        target_financial_cancellations["StockCode"].astype(str).isin(
            ["M", "D"]
        )
    ]
    .groupby("Invoice")["StockCode"]
    .agg(lambda x: set(x.astype(str)))
)

m_d_invoice_overlap = m_d_invoice_overlap[
    m_d_invoice_overlap.apply(
        lambda x: {"M", "D"}.issubset(x)
    )
]

overlapping_invoice_ids = set(m_d_invoice_overlap.index)

print(
    "Cancellation invoices containing both M and D:",
    len(overlapping_invoice_ids)
)

print("\nOverlapping invoice identifiers:")

display(
    pd.DataFrame({
        "Invoice": sorted(overlapping_invoice_ids)
    })
)

# Inspect complete transaction context
overlapping_invoice_context = financial_cancellation_context[
    financial_cancellation_context["Invoice"].isin(
        overlapping_invoice_ids
    )
].copy()

overlapping_invoice_context["line_value"] = (
    overlapping_invoice_context["Quantity"]
    * overlapping_invoice_context["Price"]
)

display(
    overlapping_invoice_context[
        [
            "Invoice",
            "InvoiceDate",
            "StockCode",
            "Description",
            "Quantity",
            "Price",
            "line_value",
            "Customer ID"
        ]
    ].sort_values(["Invoice", "StockCode"])
)


Cancellation invoices containing both M and D: 2

Overlapping invoice identifiers:


,Invoice
0,C523987
1,C543789


,Invoice,InvoiceDate,StockCode,Description,Quantity,Price,line_value,Customer ID
355993,C523987,2010-09-26 14:12:00,D,Discount,-250,0.10,-25.00,12931.0
355994,C523987,2010-09-26 14:12:00,D,Discount,-156,0.19,-29.64,12931.0
355995,C523987,2010-09-26 14:12:00,D,Discount,-128,0.42,-53.76,12931.0
355996,C523987,2010-09-26 14:12:00,M,Manual,-100,0.41,-41.00,12931.0
613672,C543789,2011-02-11 17:10:00,D,Discount,-1,84.80,-84.80,17450.0
613671,C543789,2011-02-11 17:10:00,M,Manual,-1,856.48,-856.48,17450.0
613673,C543789,2011-02-11 17:10:00,M,Manual,-1,190.80,-190.80,17450.0


In [148]:
# Consolidated financial transaction profile

financial_records["transaction_direction"] = (
    financial_records["is_cancellation"]
    .map({
        True: "cancellation",
        False: "non_cancellation"
    })
)

financial_profile = (
    financial_records
    .groupby(
        ["StockCode", "Description", "transaction_direction"],
        dropna=False
    )
    .agg(
        record_count=("Invoice", "size"),
        unique_invoices=("Invoice", "nunique"),
        identified_customers=("Customer ID", "nunique"),
        positive_quantity_records=(
            "Quantity",
            lambda x: (x > 0).sum()
        ),
        negative_quantity_records=(
            "Quantity",
            lambda x: (x < 0).sum()
        ),
        zero_quantity_records=(
            "Quantity",
            lambda x: (x == 0).sum()
        ),
        total_quantity=("Quantity", "sum"),
        total_line_value=("line_value", "sum")
    )
    .reset_index()
    .sort_values(
        ["StockCode", "transaction_direction"]
    )
)

display(financial_profile)

print("Total financial records:", len(financial_records))
print(
    "Total financial StockCodes:",
    financial_records["StockCode"].nunique()
)


,StockCode,Description,transaction_direction,record_count,unique_invoices,identified_customers,positive_quantity_records,negative_quantity_records,zero_quantity_records,total_quantity,total_line_value
0,ADJUST,Adjustment by Peter on 24/05/2010 1,cancellation,3,3,3,0,3,0,-3,-731.050
1,ADJUST,Adjustment by john on 26/01/2010 16,cancellation,18,18,18,0,18,0,-18,-531.100
3,ADJUST,Adjustment by john on 26/01/2010 17,cancellation,10,10,8,0,10,0,-10,-800.540
2,ADJUST,Adjustment by john on 26/01/2010 16,non_cancellation,20,20,15,20,0,0,20,2334.430
4,ADJUST,Adjustment by john on 26/01/2010 17,non_cancellation,16,16,11,16,0,0,16,6563.500
5,ADJUST2,Adjustment by Peter on Jun 25 2010,non_cancellation,3,3,3,3,0,0,3,731.050
6,BANK CHARGES,Bank Charges,cancellation,4,4,1,0,4,0,-4,-2098.960
8,BANK CHARGES,Bank Charges,cancellation,63,58,4,0,63,0,-73,-33997.910
7,BANK CHARGES,Bank Charges,non_cancellation,2,2,2,2,0,0,2,30.000
9,BANK CHARGES,Bank Charges,non_cancellation,33,31,18,33,0,0,33,504.241


Total financial records: 1770
Total financial StockCodes: 5


In [152]:
df_classified["line_value"] = (
    df_classified["Quantity"] * df_classified["Price"]
)

classification_summary = (
    df_classified
    .groupby("transaction_category", dropna=False)
    .agg(
        record_count=("Invoice", "size"),
        unique_invoices=("Invoice", "nunique"),
        identified_customers=("Customer ID", "nunique"),
        total_line_value=("line_value", "sum")
    )
    .reset_index()
    .sort_values("record_count", ascending=False)
)

display(classification_summary)

print("Original records:", len(df_transaction_review))
print("Classified records:", len(df_classified))
print(
    "Record count preserved:",
    len(df_transaction_review) == len(df_classified)
)

,transaction_category,record_count,unique_invoices,identified_customers,total_line_value
6,merchandise_candidate,802582,36604,5852,1.743349e+07
8,unclassified,252900,10421,2447,1.676077e+06
2,delivery_charge,3931,3863,599,4.495595e+05
7,operational_adjustment_candidate,3457,3393,0,0.000000e+00
9,zero_price_activity,2720,1963,45,0.000000e+00
5,manual_transaction,1426,1290,582,-8.278127e+04
3,discount_related_transaction,177,151,55,-1.348454e+04
1,bank_charge,102,95,22,-3.556263e+04
4,financial_adjustment,70,70,53,7.566290e+03
0,bad_debt_adjustment,6,6,0,-1.476141e+05


Original records: 1067371
Classified records: 1067371
Record count preserved: True


In [153]:
# Profile unclassified transactions

unclassified_records = df_classified[
    df_classified["transaction_category"] == "unclassified"
].copy()

# Quantity classification
unclassified_records["quantity_category"] = "zero_quantity"

unclassified_records.loc[
    unclassified_records["Quantity"] > 0,
    "quantity_category"
] = "positive_quantity"

unclassified_records.loc[
    unclassified_records["Quantity"] < 0,
    "quantity_category"
] = "negative_quantity"

# Price classification
unclassified_records["price_category"] = "zero_price"

unclassified_records.loc[
    unclassified_records["Price"] > 0,
    "price_category"
] = "positive_price"

unclassified_records.loc[
    unclassified_records["Price"] < 0,
    "price_category"
] = "negative_price"

# Customer identification
unclassified_records["customer_status"] = (
    unclassified_records["Customer ID"]
    .notna()
    .map({
        True: "identified_customer",
        False: "missing_customer_id"
    })
)

# Combined profile
unclassified_profile = (
    unclassified_records
    .groupby(
        [
            "is_cancellation",
            "customer_status",
            "quantity_category",
            "price_category"
        ],
        dropna=False
    )
    .agg(
        record_count=("Invoice", "size"),
        unique_invoices=("Invoice", "nunique"),
        identified_customers=("Customer ID", "nunique"),
        total_quantity=("Quantity", "sum"),
        total_line_value=("line_value", "sum")
    )
    .reset_index()
    .sort_values("record_count", ascending=False)
)

display(unclassified_profile)

print("Total unclassified records:", len(unclassified_records))
print(
    "Unique unclassified invoices:",
    unclassified_records["Invoice"].nunique()
)

,is_cancellation,customer_status,quantity_category,price_category,record_count,unique_invoices,identified_customers,total_quantity,total_line_value
0,False,missing_customer_id,positive_quantity,positive_price,234454,2928,0,712311,2711618.42
1,True,identified_customer,negative_quantity,positive_price,17953,7302,2447,-478120,-727630.51
2,True,missing_customer_id,negative_quantity,positive_price,493,191,0,-3932,-307911.33


Total unclassified records: 252900
Unique unclassified invoices: 10421


In [154]:
# Profile StockCodes in the three unclassified groups

# Create the three transaction groups
anonymous_positive_records = unclassified_records[
    ~unclassified_records["is_cancellation"]
    & unclassified_records["Customer ID"].isna()
    & (unclassified_records["Quantity"] > 0)
    & (unclassified_records["Price"] > 0)
].copy()

identified_cancellation_records = unclassified_records[
    unclassified_records["is_cancellation"]
    & unclassified_records["Customer ID"].notna()
    & (unclassified_records["Quantity"] < 0)
    & (unclassified_records["Price"] > 0)
].copy()

anonymous_cancellation_records = unclassified_records[
    unclassified_records["is_cancellation"]
    & unclassified_records["Customer ID"].isna()
    & (unclassified_records["Quantity"] < 0)
    & (unclassified_records["Price"] > 0)
].copy()


def stockcode_profile(data):
    return (
        data
        .groupby(
            ["StockCode", "Description_normalized"],
            dropna=False
        )
        .agg(
            record_count=("Invoice", "size"),
            unique_invoices=("Invoice", "nunique"),
            unique_customers=("Customer ID", "nunique"),
            total_quantity=("Quantity", "sum"),
            total_line_value=("line_value", "sum")
        )
        .reset_index()
        .sort_values("record_count", ascending=False)
    )


anonymous_positive_profile = stockcode_profile(
    anonymous_positive_records
)

identified_cancellation_profile = stockcode_profile(
    identified_cancellation_records
)

anonymous_cancellation_profile = stockcode_profile(
    anonymous_cancellation_records
)

# Display group summaries
print("GROUP A — Anonymous positive transactions")
print("Records:", len(anonymous_positive_records))
print("Unique StockCodes:", anonymous_positive_records["StockCode"].nunique())
display(anonymous_positive_profile.head(40))

print("\nGROUP B — Identified cancellation transactions")
print("Records:", len(identified_cancellation_records))
print("Unique StockCodes:", identified_cancellation_records["StockCode"].nunique())
display(identified_cancellation_profile.head(40))

print("\nGROUP C — Anonymous cancellation transactions")
print("Records:", len(anonymous_cancellation_records))
print("Unique StockCodes:", anonymous_cancellation_records["StockCode"].nunique())
display(anonymous_cancellation_profile.head(40))


GROUP A — Anonymous positive transactions
Records: 234454
Unique StockCodes: 4379


,StockCode,Description_normalized,record_count,unique_invoices,unique_customers,total_quantity,total_line_value
4326,85099B,JUMBO BAG RED RETROSPOT,664,633,0,3101,11843.84
1127,21931,JUMBO STORAGE BAG SUKI,633,615,0,3264,14029.48
1615,22423,REGENCY CAKESTAND 3 TIER,633,601,0,2678,58076.95
2912,47566,PARTY BUNTING,619,597,0,4771,45306.82
1569,22379,RECYCLING BAG RETROSPOT,573,557,0,1568,7031.88
1601,22411,JUMBO SHOPPER VINTAGE RED PAISLEY,527,512,0,3042,12490.82
127,20719,WOODLAND CHARLOTTE BAG,523,511,0,5657,13051.73
120,20712,JUMBO BAG WOODLAND ANIMALS,509,500,0,2028,8525.39
506,21232,STRAWBERRY CERAMIC TRINKET BOX,505,492,0,2884,7951.38
956,21731,RED TOADSTOOL LED NIGHT LIGHT,504,494,0,1848,6625.98



GROUP B — Identified cancellation transactions
Records: 17953
Unique StockCodes: 2843


,StockCode,Description_normalized,record_count,unique_invoices,unique_customers,total_quantity,total_line_value
1247,22423,REGENCY CAKESTAND 3 TIER,347,341,195,-1468,-16749.60
973,22138,BAKING SET 9 PIECE RETROSPOT,210,208,181,-861,-4062.85
345,21232,STRAWBERRY CERAMIC TRINKET BOX,184,178,102,-928,-1077.73
2879,85123A,WHITE HANGING HEART T-LIGHT HOLDER,134,133,99,-3632,-9371.95
2660,79323W,WHITE CHERRY LIGHTS,124,122,109,-1083,-6387.35
765,21843,RED RETROSPOT CAKE STAND,110,105,84,-506,-4394.30
2236,82483,WOOD 2 DRAWER CABINET WHITE FINISH,89,82,57,-287,-1704.65
1699,22960,JAM MAKING SET WITH JARS,89,86,71,-255,-1042.75
2658,79323P,PINK CHERRY LIGHTS,86,86,78,-693,-4021.35
2871,85099B,JUMBO BAG RED RETROSPOT,74,71,42,-1195,-2134.92



GROUP C — Anonymous cancellation transactions
Records: 493
Unique StockCodes: 305


,StockCode,Description_normalized,record_count,unique_invoices,unique_customers,total_quantity,total_line_value
303,S,SAMPLES,101,34,0,-101,-6202.65
302,AMAZONFEE,AMAZON FEE,39,33,0,-39,-294772.71
27,21212,PACK OF 72 RETROSPOT CAKE CASES,3,2,0,-96,-52.80
234,84347,ROTATING SILVER ANGELS T-LIGHT HLDR,3,2,0,-20,-51.00
263,46775D,SUNSET COLOUR CHUNKY KNITTED THROW,3,3,0,-3,-42.35
5,20725,LUNCH BAG RED RETROSPOT,3,3,0,-23,-37.95
31,21333,CLASSIC WHITE FRAME,2,2,0,-6,-23.62
103,22278,OVERNIGHT BAG VINTAGE ROSE PAISLEY,2,2,0,-2,-23.97
126,22429,ENAMEL MEASURING JUG CREAM,2,1,0,-4,-17.00
117,22398,MAGNETS PACK OF 4 SWALLOWS,2,1,0,-6,-7.50


In [155]:
# Identify special transaction candidates

# Previously identified special StockCodes
known_special_stockcodes = {
    "POST", "C2", "23444", "DOT",
    "M", "D", "ADJUST", "ADJUST2",
    "BANK CHARGES", "B", "AMAZONFEE", "S"
}

# Description keywords for discovery only
special_description_pattern = (
    r"AMAZON|FEE|SAMPLE|POSTAGE|POSTAL|CARRIAGE|"
    r"MANUAL|ADJUST|BANK CHARGE|DISCOUNT|"
    r"DELIVERY|COURIER|INSURANCE|TEST|"
    r"DAMAGE|DAMAGED|MISSING|LOST|"
    r"CHARGE|CREDIT|DEBIT|REFUND"
)

def identify_special_candidates(data, group_name):

    profile = (
        data
        .groupby(
            ["StockCode", "Description_normalized"],
            dropna=False
        )
        .agg(
            record_count=("Invoice", "size"),
            unique_invoices=("Invoice", "nunique"),
            total_quantity=("Quantity", "sum"),
            total_line_value=("line_value", "sum")
        )
        .reset_index()
    )

    profile["StockCode_normalized"] = (
        profile["StockCode"]
        .astype("string")
        .str.strip()
        .str.upper()
    )

    known_code_mask = (
        profile["StockCode_normalized"]
        .isin(known_special_stockcodes)
    )

    keyword_mask = (
        profile["Description_normalized"]
        .fillna("")
        .str.contains(
            special_description_pattern,
            case=False,
            regex=True
        )
    )

    candidates = profile[
        known_code_mask | keyword_mask
    ].copy()

    candidates["candidate_reason"] = "description_keyword"

    candidates.loc[
        known_code_mask & keyword_mask,
        "candidate_reason"
    ] = "known_stockcode_and_description_keyword"

    candidates.loc[
        known_code_mask & ~keyword_mask,
        "candidate_reason"
    ] = "known_stockcode"

    candidates = candidates.sort_values(
        "record_count",
        ascending=False
    )

    print(f"\n{group_name} — Special transaction candidates")
    print("Candidate StockCode-description combinations:", len(candidates))

    display(
        candidates[
            [
                "StockCode",
                "Description_normalized",
                "record_count",
                "unique_invoices",
                "total_quantity",
                "total_line_value",
                "candidate_reason"
            ]
        ]
    )

    return profile, candidates


# Group C: display its complete profile
group_c_full_profile = stockcode_profile(
    anonymous_cancellation_records
)

print("GROUP C — Complete StockCode profile")
print("Total StockCode-description combinations:",
      len(group_c_full_profile))

display(group_c_full_profile)


# Search all three groups
group_a_profile, group_a_special_candidates = identify_special_candidates(
    anonymous_positive_records,
    "GROUP A"
)

group_b_profile, group_b_special_candidates = identify_special_candidates(
    identified_cancellation_records,
    "GROUP B"
)

group_c_profile, group_c_special_candidates = identify_special_candidates(
    anonymous_cancellation_records,
    "GROUP C"
)


GROUP C — Complete StockCode profile
Total StockCode-description combinations: 305


,StockCode,Description_normalized,record_count,unique_invoices,unique_customers,total_quantity,total_line_value
303,S,SAMPLES,101,34,0,-101,-6202.65
302,AMAZONFEE,AMAZON FEE,39,33,0,-39,-294772.71
27,21212,PACK OF 72 RETROSPOT CAKE CASES,3,2,0,-96,-52.80
234,84347,ROTATING SILVER ANGELS T-LIGHT HLDR,3,2,0,-20,-51.00
263,46775D,SUNSET COLOUR CHUNKY KNITTED THROW,3,3,0,-3,-42.35
...,...,...,...,...,...,...,...
12,20931,BLUE POT PLANT CANDLE,1,1,0,-1,-3.75
13,20961,STRAWBERRY BATH SPONGE,1,1,0,-21,-26.25
14,20979,36 PENCILS TUBE RED SPOTTY,1,1,0,-16,-20.00
22,21102,"CANDLE IN JAR COLLAGE , VANILLA",1,1,0,-31,-79.05



GROUP A — Special transaction candidates
Candidate StockCode-description combinations: 40


,StockCode,Description_normalized,record_count,unique_invoices,total_quantity,total_line_value,candidate_reason
493,21216,"SET 3 RETROSPOT TEA,COFFEE,SUGAR",222,215,332,3487.42,description_keyword
1471,22301,COFFEE MUG CAT + BIRD DESIGN,185,177,289,1359.27,description_keyword
2867,37370,RETRO COFFEE MUGS ASSORTED,164,148,221,3310.91,description_keyword
1470,22300,COFFEE MUG DOG + BALL DESIGN,120,115,204,910.61,description_keyword
2457,23243,SET OF TEA COFFEE SUGAR TINS PANTRY,85,85,142,1380.34,description_keyword
147,20748,KENSINGTON COFFEE SET,69,69,1241,10895.73,description_keyword
3559,35810A,ENAMEL PINK COFFEE CONTAINER,67,67,88,322.38,description_keyword
492,21216,"SET 3 RED SPOT TIN TEA,COFFEE,SUGAR",65,65,74,798.99,description_keyword
1473,22303,COFFEE MUG APPLES DESIGN,60,60,129,521.68,description_keyword
2860,37342,POLKADOT COFFEE CUP & SAUCER PINK,52,52,146,411.73,description_keyword



GROUP B — Special transaction candidates
Candidate StockCode-description combinations: 30


,StockCode,Description_normalized,record_count,unique_invoices,total_quantity,total_line_value,candidate_reason
1926,23243,SET OF TEA COFFEE SUGAR TINS PANTRY,34,34,-115,-535.65,description_keyword
1130,22301,COFFEE MUG CAT + BIRD DESIGN,27,26,-77,-194.55,description_keyword
1132,22303,COFFEE MUG APPLES DESIGN,27,27,-139,-336.45,description_keyword
1129,22300,COFFEE MUG DOG + BALL DESIGN,20,20,-63,-160.65,description_keyword
2134,37370,RETRO COFFEE MUGS ASSORTED,18,16,-61,-74.92,description_keyword
2896,85159B,"WHITE TEA,COFFEE,SUGAR JARS",14,14,-39,-92.89,description_keyword
338,21216,"SET 3 RETROSPOT TEA,COFFEE,SUGAR",12,12,-18,-87.70,description_keyword
1131,22302,COFFEE MUG PEARS DESIGN,10,10,-37,-94.35,description_keyword
2895,85159A,"BLACK TEA,COFFEE,SUGAR JARS",8,8,-43,-101.45,description_keyword
1710,22971,QUEENS GUARD COFFEE MUG,5,5,-16,-37.65,description_keyword



GROUP C — Special transaction candidates
Candidate StockCode-description combinations: 4


,StockCode,Description_normalized,record_count,unique_invoices,total_quantity,total_line_value,candidate_reason
303,S,SAMPLES,101,34,-101,-6202.65,known_stockcode_and_description_keyword
302,AMAZONFEE,AMAZON FEE,39,33,-39,-294772.71,known_stockcode_and_description_keyword
208,23243,SET OF TEA COFFEE SUGAR TINS PANTRY,1,1,-2,-9.90,description_keyword
11,20847,ZINC HEART LATTICE CHARGER LARGE,1,1,-1,-7.62,description_keyword


In [157]:
df_classified["StockCode_normalized"] = (
    df_classified["StockCode"]
    .astype("string")
    .str.strip()
    .str.upper()
)

In [160]:
# Investigate additional special StockCodes

special_codes_to_review = [
    "AMAZONFEE",
    "S",
    "TEST001",
    "23574"
]

special_records = df_classified[
    df_classified["StockCode_normalized"].isin(
        special_codes_to_review
    )
].copy()

# Profile by StockCode, description, and existing classification
special_stockcode_profile = (
    special_records
    .groupby(
        [
            "StockCode_normalized",
            "Description_normalized",
            "transaction_category",
            "is_cancellation"
        ],
        dropna=False
    )
    .agg(
        record_count=("Invoice", "size"),
        unique_invoices=("Invoice", "nunique"),
        identified_customers=("Customer ID", "nunique"),
        positive_quantity_records=(
            "Quantity",
            lambda x: (x > 0).sum()
        ),
        negative_quantity_records=(
            "Quantity",
            lambda x: (x < 0).sum()
        ),
        zero_price_records=(
            "Price",
            lambda x: (x == 0).sum()
        ),
        total_quantity=("Quantity", "sum"),
        total_line_value=("line_value", "sum")
    )
    .reset_index()
    .sort_values(
        ["StockCode_normalized", "is_cancellation"]
    )
)

print("Special StockCode profile across the entire dataset:")
display(special_stockcode_profile)

print("\nTotal records:", len(special_records))
print(
    "Unique invoices:",
    special_records["Invoice"].nunique()
)

# Inspect individual records
print("\nIndividual transaction records:")

display(
    special_records[
        [
            "Invoice",
            "InvoiceDate",
            "StockCode_normalized",
            "Description",
            "Quantity",
            "Price",
            "line_value",
            "Customer ID",
            "Country",
            "is_cancellation",
            "transaction_category"
        ]
    ].sort_values(
        ["StockCode_normalized", "InvoiceDate"]
    )
)

Special StockCode profile across the entire dataset:


,StockCode_normalized,Description_normalized,transaction_category,is_cancellation,record_count,unique_invoices,identified_customers,positive_quantity_records,negative_quantity_records,zero_price_records,total_quantity,total_line_value
0,23574,PACKING CHARGE,merchandise_candidate,False,14,14,9,14,0,0,14,105.00
2,23574,PACKING CHARGE,zero_price_activity,False,1,1,0,1,0,1,100,0.00
1,23574,PACKING CHARGE,unclassified,True,2,2,2,0,2,0,-2,-15.00
3,AMAZONFEE,AMAZON FEE,unclassified,False,4,3,0,4,0,0,4,34009.13
4,AMAZONFEE,AMAZON FEE,unclassified,True,39,33,0,0,39,0,-39,-294772.71
5,S,SAMPLES,unclassified,False,3,3,0,3,0,0,3,136.85
6,S,SAMPLES,unclassified,True,101,34,0,0,101,0,-101,-6202.65
7,TEST001,THIS IS A TEST PRODUCT.,merchandise_candidate,False,9,9,2,9,0,0,50,225.00
9,TEST001,THIS IS A TEST PRODUCT.,zero_price_activity,False,2,2,2,2,0,2,10,0.00
8,TEST001,THIS IS A TEST PRODUCT.,unclassified,True,4,4,3,0,4,0,-5,-22.50



Total records: 179
Unique invoices: 105

Individual transaction records:


,Invoice,InvoiceDate,StockCode_normalized,Description,Quantity,Price,line_value,Customer ID,Country,is_cancellation,transaction_category
771503,558663,2011-07-01 10:44:00,23574,PACKING CHARGE,100,0.0,0.0,NaN,United Kingdom,False,zero_price_activity
778066,559131,2011-07-06 13:04:00,23574,PACKING CHARGE,1,7.5,7.5,18144.0,United Kingdom,False,merchandise_candidate
830805,563685,2011-08-18 12:26:00,23574,PACKING CHARGE,1,7.5,7.5,14911.0,EIRE,False,merchandise_candidate
834838,C564084,2011-08-23 09:16:00,23574,PACKING CHARGE,-1,7.5,-7.5,14911.0,EIRE,True,unclassified
858600,566178,2011-09-09 12:41:00,23574,PACKING CHARGE,1,7.5,7.5,16353.0,United Kingdom,False,merchandise_candidate
...,...,...,...,...,...,...,...,...,...,...,...
156017,504188,2010-04-12 09:52:00,TEST001,This is a test product.,5,4.5,22.5,16454.0,United Kingdom,False,merchandise_candidate
371777,C525272,2010-10-04 16:29:00,TEST001,This is a test product.,-1,4.5,-4.5,16454.0,United Kingdom,True,unclassified
371778,C525273,2010-10-04 16:30:00,TEST001,This is a test product.,-1,4.5,-4.5,16454.0,United Kingdom,True,unclassified
371779,C525274,2010-10-04 16:33:00,TEST001,This is a test product.,-1,4.5,-4.5,12346.0,United Kingdom,True,unclassified


In [163]:
# Display consolidated special StockCode profile

display(
    special_stockcode_profile.sort_values(
        [
            "StockCode_normalized",
            "is_cancellation"
        ]
    ).reset_index(drop=True)
)

,StockCode_normalized,Description_normalized,transaction_category,is_cancellation,record_count,unique_invoices,identified_customers,positive_quantity_records,negative_quantity_records,zero_price_records,total_quantity,total_line_value
0,23574,PACKING CHARGE,merchandise_candidate,False,14,14,9,14,0,0,14,105.00
1,23574,PACKING CHARGE,zero_price_activity,False,1,1,0,1,0,1,100,0.00
2,23574,PACKING CHARGE,unclassified,True,2,2,2,0,2,0,-2,-15.00
3,AMAZONFEE,AMAZON FEE,unclassified,False,4,3,0,4,0,0,4,34009.13
4,AMAZONFEE,AMAZON FEE,unclassified,True,39,33,0,0,39,0,-39,-294772.71
5,S,SAMPLES,unclassified,False,3,3,0,3,0,0,3,136.85
6,S,SAMPLES,unclassified,True,101,34,0,0,101,0,-101,-6202.65
7,TEST001,THIS IS A TEST PRODUCT.,merchandise_candidate,False,9,9,2,9,0,0,50,225.00
8,TEST001,THIS IS A TEST PRODUCT.,zero_price_activity,False,2,2,2,2,0,2,10,0.00
9,TEST001,THIS IS A TEST PRODUCT.,unclassified,True,4,4,3,0,4,0,-5,-22.50


In [164]:
# Additional special transaction classifications

df_classified.loc[
    df_classified["StockCode_normalized"] == "23574",
    "transaction_category"
] = "packing_charge"

df_classified.loc[
    df_classified["StockCode_normalized"] == "AMAZONFEE",
    "transaction_category"
] = "platform_fee_candidate"

df_classified.loc[
    df_classified["StockCode_normalized"] == "S",
    "transaction_category"
] = "sample_transaction"

df_classified.loc[
    df_classified["StockCode_normalized"] == "TEST001",
    "transaction_category"
] = "test_transaction_candidate"

In [165]:
special_codes_to_review = [
    "23574",
    "AMAZONFEE",
    "S",
    "TEST001"
]

display(
    df_classified[
        df_classified["StockCode_normalized"].isin(
            special_codes_to_review
        )
    ]
    .groupby(
        [
            "StockCode_normalized",
            "transaction_category",
            "is_cancellation"
        ],
        dropna=False
    )
    .agg(
        record_count=("Invoice", "size"),
        unique_invoices=("Invoice", "nunique"),
        total_quantity=("Quantity", "sum"),
        total_line_value=("line_value", "sum")
    )
    .reset_index()
    .sort_values(
        ["StockCode_normalized", "is_cancellation"]
    )
)

,StockCode_normalized,transaction_category,is_cancellation,record_count,unique_invoices,total_quantity,total_line_value
0,23574,packing_charge,False,15,15,114,105.00
1,23574,packing_charge,True,2,2,-2,-15.00
2,AMAZONFEE,platform_fee_candidate,False,4,3,4,34009.13
3,AMAZONFEE,platform_fee_candidate,True,39,33,-39,-294772.71
4,S,sample_transaction,False,3,3,3,136.85
5,S,sample_transaction,True,101,34,-101,-6202.65
6,TEST001,test_transaction_candidate,False,11,11,60,225.00
7,TEST001,test_transaction_candidate,True,4,4,-5,-22.50


In [166]:
# Profile the remaining unclassified records

remaining_unclassified = df_classified[
    df_classified["transaction_category"] == "unclassified"
].copy()

remaining_unclassified["quantity_category"] = "zero_quantity"
remaining_unclassified.loc[
    remaining_unclassified["Quantity"] > 0,
    "quantity_category"
] = "positive_quantity"
remaining_unclassified.loc[
    remaining_unclassified["Quantity"] < 0,
    "quantity_category"
] = "negative_quantity"

remaining_unclassified["price_category"] = "zero_price"
remaining_unclassified.loc[
    remaining_unclassified["Price"] > 0,
    "price_category"
] = "positive_price"
remaining_unclassified.loc[
    remaining_unclassified["Price"] < 0,
    "price_category"
] = "negative_price"

remaining_unclassified["customer_status"] = (
    remaining_unclassified["Customer ID"]
    .notna()
    .map({
        True: "identified_customer",
        False: "missing_customer_id"
    })
)

unclassified_composition = (
    remaining_unclassified
    .groupby(
        [
            "is_cancellation",
            "customer_status",
            "quantity_category",
            "price_category"
        ],
        dropna=False
    )
    .agg(
        record_count=("Invoice", "size"),
        unique_invoices=("Invoice", "nunique"),
        identified_customers=("Customer ID", "nunique"),
        total_quantity=("Quantity", "sum"),
        total_line_value=("line_value", "sum")
    )
    .reset_index()
    .sort_values("record_count", ascending=False)
)

print("Remaining unclassified records:", len(remaining_unclassified))
print(
    "Unique invoices:",
    remaining_unclassified["Invoice"].nunique()
)

display(unclassified_composition)

Remaining unclassified records: 252747
Unique invoices: 10343


,is_cancellation,customer_status,quantity_category,price_category,record_count,unique_invoices,identified_customers,total_quantity,total_line_value
0,False,missing_customer_id,positive_quantity,positive_price,234447,2922,0,712304,2677472.44
1,True,identified_customer,negative_quantity,positive_price,17947,7297,2446,-478113,-727593.01
2,True,missing_customer_id,negative_quantity,positive_price,353,124,0,-3792,-6935.97


In [167]:
# Remaining anonymous cancellation transactions

remaining_group_c = remaining_unclassified[
    remaining_unclassified["is_cancellation"]
    & remaining_unclassified["Customer ID"].isna()
    & (remaining_unclassified["Quantity"] < 0)
    & (remaining_unclassified["Price"] > 0)
].copy()

remaining_group_c_profile = stockcode_profile(
    remaining_group_c
)

print("Remaining Group C records:", len(remaining_group_c))
print(
    "Unique invoices:",
    remaining_group_c["Invoice"].nunique()
)
print(
    "Unique StockCode-description combinations:",
    len(remaining_group_c_profile)
)

display(
    remaining_group_c_profile.sort_values(
        "record_count",
        ascending=False
    )
)

Remaining Group C records: 353
Unique invoices: 124
Unique StockCode-description combinations: 303


,StockCode,Description_normalized,record_count,unique_invoices,unique_customers,total_quantity,total_line_value
263,46775D,SUNSET COLOUR CHUNKY KNITTED THROW,3,3,0,-3,-42.35
27,21212,PACK OF 72 RETROSPOT CAKE CASES,3,2,0,-96,-52.80
234,84347,ROTATING SILVER ANGELS T-LIGHT HLDR,3,2,0,-20,-51.00
5,20725,LUNCH BAG RED RETROSPOT,3,3,0,-23,-37.95
274,79323W,WHITE CHERRY LIGHTS,2,2,0,-5,-33.75
...,...,...,...,...,...,...,...
8,20799,PINK MILKSHAKE GLASS,1,1,0,-6,-25.26
7,20728,LUNCH BAG CARS BLUE,1,1,0,-1,-1.65
6,20727,LUNCH BAG BLACK SKULL.,1,1,0,-10,-16.50
4,20718,RED RETROSPOT SHOPPER BAG,1,1,0,-11,-13.75


In [168]:
# Identified customer cancellation records still unclassified

remaining_group_b = remaining_unclassified[
    remaining_unclassified["is_cancellation"]
    & remaining_unclassified["Customer ID"].notna()
    & (remaining_unclassified["Quantity"] < 0)
    & (remaining_unclassified["Price"] > 0)
].copy()

remaining_group_b_profile = stockcode_profile(
    remaining_group_b
)

print("Remaining Group B records:", len(remaining_group_b))
print(
    "Unique invoices:",
    remaining_group_b["Invoice"].nunique()
)
print(
    "Unique StockCode-description combinations:",
    len(remaining_group_b_profile)
)

display(
    remaining_group_b_profile.head(40)
)

Remaining Group B records: 17947
Unique invoices: 7297
Unique StockCode-description combinations: 3012


,StockCode,Description_normalized,record_count,unique_invoices,unique_customers,total_quantity,total_line_value
1247,22423,REGENCY CAKESTAND 3 TIER,347,341,195,-1468,-16749.60
973,22138,BAKING SET 9 PIECE RETROSPOT,210,208,181,-861,-4062.85
345,21232,STRAWBERRY CERAMIC TRINKET BOX,184,178,102,-928,-1077.73
2878,85123A,WHITE HANGING HEART T-LIGHT HOLDER,134,133,99,-3632,-9371.95
2659,79323W,WHITE CHERRY LIGHTS,124,122,109,-1083,-6387.35
765,21843,RED RETROSPOT CAKE STAND,110,105,84,-506,-4394.30
1699,22960,JAM MAKING SET WITH JARS,89,86,71,-255,-1042.75
2235,82483,WOOD 2 DRAWER CABINET WHITE FINISH,89,82,57,-287,-1704.65
2657,79323P,PINK CHERRY LIGHTS,86,86,78,-693,-4021.35
545,21527,RED RETROSPOT TRADITIONAL TEAPOT,74,69,58,-114,-878.30


In [169]:
group_b_profile, group_b_special_candidates = (
    identify_special_candidates(
        remaining_group_b,
        "REMAINING GROUP B"
    )
)


REMAINING GROUP B — Special transaction candidates
Candidate StockCode-description combinations: 28


,StockCode,Description_normalized,record_count,unique_invoices,total_quantity,total_line_value,candidate_reason
1926,23243,SET OF TEA COFFEE SUGAR TINS PANTRY,34,34,-115,-535.65,description_keyword
1130,22301,COFFEE MUG CAT + BIRD DESIGN,27,26,-77,-194.55,description_keyword
1132,22303,COFFEE MUG APPLES DESIGN,27,27,-139,-336.45,description_keyword
1129,22300,COFFEE MUG DOG + BALL DESIGN,20,20,-63,-160.65,description_keyword
2133,37370,RETRO COFFEE MUGS ASSORTED,18,16,-61,-74.92,description_keyword
2895,85159B,"WHITE TEA,COFFEE,SUGAR JARS",14,14,-39,-92.89,description_keyword
338,21216,"SET 3 RETROSPOT TEA,COFFEE,SUGAR",12,12,-18,-87.70,description_keyword
1131,22302,COFFEE MUG PEARS DESIGN,10,10,-37,-94.35,description_keyword
2894,85159A,"BLACK TEA,COFFEE,SUGAR JARS",8,8,-43,-101.45,description_keyword
337,21216,"SET 3 RED SPOT TIN TEA,COFFEE,SUGAR",5,5,-5,-24.05,description_keyword


In [170]:
# Assign provisional categories to the remaining transaction groups

group_a_mask = (
    ~df_classified["is_cancellation"]
    & df_classified["Customer ID"].isna()
    & (df_classified["Quantity"] > 0)
    & (df_classified["Price"] > 0)
    & df_classified["transaction_category"].eq("unclassified")
)

group_b_mask = (
    df_classified["is_cancellation"]
    & df_classified["Customer ID"].notna()
    & (df_classified["Quantity"] < 0)
    & (df_classified["Price"] > 0)
    & df_classified["transaction_category"].eq("unclassified")
)

group_c_mask = (
    df_classified["is_cancellation"]
    & df_classified["Customer ID"].isna()
    & (df_classified["Quantity"] < 0)
    & (df_classified["Price"] > 0)
    & df_classified["transaction_category"].eq("unclassified")
)

df_classified.loc[
    group_a_mask,
    "transaction_category"
] = "anonymous_merchandise_candidate"

df_classified.loc[
    group_b_mask,
    "transaction_category"
] = "merchandise_cancellation_candidate"

df_classified.loc[
    group_c_mask,
    "transaction_category"
] = "anonymous_merchandise_cancellation_candidate"

print("Group A records classified:", group_a_mask.sum())
print("Group B records classified:", group_b_mask.sum())
print("Group C records classified:", group_c_mask.sum())

print(
    "\nRemaining unclassified records:",
    (df_classified["transaction_category"] == "unclassified").sum()
)

Group A records classified: 234447
Group B records classified: 17947
Group C records classified: 353

Remaining unclassified records: 0


In [171]:
classification_summary = (
    df_classified
    .groupby("transaction_category", dropna=False)
    .agg(
        record_count=("Invoice", "size"),
        unique_invoices=("Invoice", "nunique"),
        identified_customers=("Customer ID", "nunique"),
        total_quantity=("Quantity", "sum"),
        total_line_value=("line_value", "sum")
    )
    .reset_index()
    .sort_values("record_count", ascending=False)
)

display(classification_summary)

print("\nTotal classified records:", len(df_classified))
print(
    "Original records:",
    len(df_combined_raw)
)
print(
    "Record count preserved:",
    len(df_classified) == len(df_combined_raw)
)

,transaction_category,record_count,unique_invoices,identified_customers,total_quantity,total_line_value
9,merchandise_candidate,802559,36595,5852,10690355,1.743316e+07
1,anonymous_merchandise_candidate,234447,2922,0,712304,2.677472e+06
8,merchandise_cancellation_candidate,17947,7297,2446,-478113,-7.275930e+05
4,delivery_charge,3931,3863,599,14040,4.495595e+05
10,operational_adjustment_candidate,3457,3393,0,-573085,0.000000e+00
15,zero_price_activity,2717,1960,43,245037,0.000000e+00
7,manual_transaction,1426,1290,582,4612,-8.278127e+04
0,anonymous_merchandise_cancellation_candidate,353,124,0,-3792,-6.935970e+03
5,discount_related_transaction,177,151,55,-2872,-1.348454e+04
13,sample_transaction,104,37,0,-98,-6.065800e+03



Total classified records: 1067371
Original records: 1067371
Record count preserved: True


In [172]:
# Validate the four main merchandise-related categories

validation_rules = {
    "merchandise_candidate": (
        ~df_classified["is_cancellation"]
        & df_classified["Customer ID"].notna()
        & (df_classified["Quantity"] > 0)
        & (df_classified["Price"] > 0)
    ),

    "anonymous_merchandise_candidate": (
        ~df_classified["is_cancellation"]
        & df_classified["Customer ID"].isna()
        & (df_classified["Quantity"] > 0)
        & (df_classified["Price"] > 0)
    ),

    "merchandise_cancellation_candidate": (
        df_classified["is_cancellation"]
        & df_classified["Customer ID"].notna()
        & (df_classified["Quantity"] < 0)
        & (df_classified["Price"] > 0)
    ),

    "anonymous_merchandise_cancellation_candidate": (
        df_classified["is_cancellation"]
        & df_classified["Customer ID"].isna()
        & (df_classified["Quantity"] < 0)
        & (df_classified["Price"] > 0)
    )
}

validation_results = []

for category, expected_condition in validation_rules.items():

    category_mask = (
        df_classified["transaction_category"] == category
    )

    violations = (
        category_mask & ~expected_condition
    ).sum()

    validation_results.append({
        "transaction_category": category,
        "record_count": category_mask.sum(),
        "rule_violations": violations,
        "validation_passed": violations == 0
    })

validation_results = pd.DataFrame(validation_results)

display(validation_results)

print(
    "Record count preserved:",
    len(df_classified) == len(df_combined_raw)
)

print(
    "Original row index preserved:",
    df_classified.index.equals(df_combined_raw.index)
)

print(
    "Remaining unclassified:",
    (df_classified["transaction_category"] == "unclassified").sum()
)

,transaction_category,record_count,rule_violations,validation_passed
0,merchandise_candidate,802559,0,True
1,anonymous_merchandise_candidate,234447,0,True
2,merchandise_cancellation_candidate,17947,0,True
3,anonymous_merchandise_cancellation_candidate,353,0,True


Record count preserved: True
Original row index preserved: True
Remaining unclassified: 0


In [173]:
# Define the transaction taxonomy

transaction_taxonomy = pd.DataFrame([
    {
        "transaction_category": "merchandise_candidate",
        "merchandise_revenue_treatment": "include_positive",
        "customer_level_eligible": True,
        "qualifying_purchase": True,
        "notes": "Identified-customer merchandise transactions"
    },
    {
        "transaction_category": "anonymous_merchandise_candidate",
        "merchandise_revenue_treatment": "include_positive",
        "customer_level_eligible": False,
        "qualifying_purchase": False,
        "notes": "Merchandise transactions without Customer ID"
    },
    {
        "transaction_category": "merchandise_cancellation_candidate",
        "merchandise_revenue_treatment": "include_negative",
        "customer_level_eligible": True,
        "qualifying_purchase": False,
        "notes": "Identified-customer merchandise cancellations"
    },
    {
        "transaction_category": "anonymous_merchandise_cancellation_candidate",
        "merchandise_revenue_treatment": "include_negative",
        "customer_level_eligible": False,
        "qualifying_purchase": False,
        "notes": "Anonymous merchandise cancellations"
    },
    {
        "transaction_category": "delivery_charge",
        "merchandise_revenue_treatment": "separate",
        "customer_level_eligible": False,
        "qualifying_purchase": False,
        "notes": "Delivery-related charges"
    },
    {
        "transaction_category": "packing_charge",
        "merchandise_revenue_treatment": "separate",
        "customer_level_eligible": False,
        "qualifying_purchase": False,
        "notes": "Packing-related charges"
    },
    {
        "transaction_category": "zero_price_activity",
        "merchandise_revenue_treatment": "exclude",
        "customer_level_eligible": True,
        "qualifying_purchase": False,
        "notes": "Zero-price activity; not a paid purchase"
    },
    {
        "transaction_category": "operational_adjustment_candidate",
        "merchandise_revenue_treatment": "exclude",
        "customer_level_eligible": False,
        "qualifying_purchase": False,
        "notes": "Potential operational or inventory adjustments"
    },
    {
        "transaction_category": "manual_transaction",
        "merchandise_revenue_treatment": "separate",
        "customer_level_eligible": False,
        "qualifying_purchase": False,
        "notes": "Manual transactions with unclear accounting meaning"
    },
    {
        "transaction_category": "discount_related_transaction",
        "merchandise_revenue_treatment": "separate",
        "customer_level_eligible": False,
        "qualifying_purchase": False,
        "notes": "Discount-related records; accounting treatment unresolved"
    },
    {
        "transaction_category": "financial_adjustment",
        "merchandise_revenue_treatment": "separate",
        "customer_level_eligible": False,
        "qualifying_purchase": False,
        "notes": "Financial adjustment records"
    },
    {
        "transaction_category": "bank_charge",
        "merchandise_revenue_treatment": "separate",
        "customer_level_eligible": False,
        "qualifying_purchase": False,
        "notes": "Bank charge records"
    },
    {
        "transaction_category": "platform_fee_candidate",
        "merchandise_revenue_treatment": "separate",
        "customer_level_eligible": False,
        "qualifying_purchase": False,
        "notes": "Potential platform-related fees"
    },
    {
        "transaction_category": "bad_debt_adjustment",
        "merchandise_revenue_treatment": "separate",
        "customer_level_eligible": False,
        "qualifying_purchase": False,
        "notes": "Bad-debt financial adjustments"
    },
    {
        "transaction_category": "sample_transaction",
        "merchandise_revenue_treatment": "exclude",
        "customer_level_eligible": False,
        "qualifying_purchase": False,
        "notes": "Sample-related transactions"
    },
    {
        "transaction_category": "test_transaction_candidate",
        "merchandise_revenue_treatment": "exclude",
        "customer_level_eligible": False,
        "qualifying_purchase": False,
        "notes": "Potential test transactions"
    }
])

display(transaction_taxonomy)

,transaction_category,merchandise_revenue_treatment,customer_level_eligible,qualifying_purchase,notes
0,merchandise_candidate,include_positive,True,True,Identified-customer merchandise transactions
1,anonymous_merchandise_candidate,include_positive,False,False,Merchandise transactions without Customer ID
2,merchandise_cancellation_candidate,include_negative,True,False,Identified-customer merchandise cancellations
3,anonymous_merchandise_cancellation_candidate,include_negative,False,False,Anonymous merchandise cancellations
4,delivery_charge,separate,False,False,Delivery-related charges
5,packing_charge,separate,False,False,Packing-related charges
6,zero_price_activity,exclude,True,False,Zero-price activity; not a paid purchase
7,operational_adjustment_candidate,exclude,False,False,Potential operational or inventory adjustments
8,manual_transaction,separate,False,False,Manual transactions with unclear accounting me...
9,discount_related_transaction,separate,False,False,Discount-related records; accounting treatment...


In [174]:
# Validate transaction taxonomy completeness

data_categories = set(
    df_classified["transaction_category"].dropna().unique()
)

taxonomy_categories = set(
    transaction_taxonomy["transaction_category"].dropna().unique()
)

missing_taxonomy_categories = (
    data_categories - taxonomy_categories
)

unused_taxonomy_categories = (
    taxonomy_categories - data_categories
)

duplicate_taxonomy_categories = (
    transaction_taxonomy[
        transaction_taxonomy["transaction_category"].duplicated(
            keep=False
        )
    ]["transaction_category"]
    .unique()
)

print("Categories in classified data:", len(data_categories))
print("Categories in taxonomy:", len(taxonomy_categories))

print("\nCategories missing from taxonomy:")
print(missing_taxonomy_categories)

print("\nUnused taxonomy categories:")
print(unused_taxonomy_categories)

print("\nDuplicate taxonomy definitions:")
print(duplicate_taxonomy_categories)

taxonomy_validation_passed = (
    len(missing_taxonomy_categories) == 0
    and len(unused_taxonomy_categories) == 0
    and len(duplicate_taxonomy_categories) == 0
)

print("\nTaxonomy validation passed:", taxonomy_validation_passed)

Categories in classified data: 16
Categories in taxonomy: 16

Categories missing from taxonomy:
set()

Unused taxonomy categories:
set()

Duplicate taxonomy definitions:
<ArrowStringArray>
[]
Length: 0, dtype: str

Taxonomy validation passed: True


In [175]:
# Attach analytical treatment rules to each transaction

df_classified = df_classified.merge(
    transaction_taxonomy,
    on="transaction_category",
    how="left",
    validate="many_to_one"
)

print("Updated dataset dimensions:", df_classified.shape)

print(
    "Records without taxonomy treatment:",
    df_classified["merchandise_revenue_treatment"].isna().sum()
)

print(
    "Original record count preserved:",
    len(df_classified) == len(df_combined_raw)
)

Updated dataset dimensions: (1067371, 26)
Records without taxonomy treatment: 0
Original record count preserved: True


In [176]:
# Create analytical eligibility flags

# 1. Merchandise revenue eligibility
df_classified["is_merchandise_revenue_record"] = (
    df_classified["merchandise_revenue_treatment"].isin(
        ["include_positive", "include_negative"]
    )
)

# 2. Customer-level analytical eligibility
df_classified["is_customer_analytics_eligible"] = (
    df_classified["customer_level_eligible"]
    & df_classified["Customer ID"].notna()
)

# 3. Qualifying paid purchase
df_classified["is_qualifying_purchase"] = (
    df_classified["qualifying_purchase"]
    & df_classified["Customer ID"].notna()
    & (df_classified["Quantity"] > 0)
    & (df_classified["Price"] > 0)
)

# 4. Merchandise revenue value
df_classified["merchandise_revenue"] = (
    df_classified["line_value"].where(
        df_classified["is_merchandise_revenue_record"],
        0
    )
)

# 5. Customer-level qualifying purchase value
df_classified["qualifying_purchase_revenue"] = (
    df_classified["line_value"].where(
        df_classified["is_qualifying_purchase"],
        0
    )
)

print("Analytical flags created.")

display(
    df_classified[
        [
            "transaction_category",
            "merchandise_revenue_treatment",
            "is_merchandise_revenue_record",
            "is_customer_analytics_eligible",
            "is_qualifying_purchase",
            "merchandise_revenue",
            "qualifying_purchase_revenue"
        ]
    ].head(10)
)

Analytical flags created.


,transaction_category,merchandise_revenue_treatment,is_merchandise_revenue_record,is_customer_analytics_eligible,is_qualifying_purchase,merchandise_revenue,qualifying_purchase_revenue
0,merchandise_candidate,include_positive,True,True,True,83.4,83.4
1,merchandise_candidate,include_positive,True,True,True,81.0,81.0
2,merchandise_candidate,include_positive,True,True,True,81.0,81.0
3,merchandise_candidate,include_positive,True,True,True,100.8,100.8
4,merchandise_candidate,include_positive,True,True,True,30.0,30.0
5,merchandise_candidate,include_positive,True,True,True,39.6,39.6
6,merchandise_candidate,include_positive,True,True,True,30.0,30.0
7,merchandise_candidate,include_positive,True,True,True,59.5,59.5
8,merchandise_candidate,include_positive,True,True,True,30.6,30.6
9,merchandise_candidate,include_positive,True,True,True,45.0,45.0


In [177]:
# Validate analytical flags across the complete dataset

# Expected merchandise revenue eligibility
expected_merchandise_revenue = (
    df_classified["merchandise_revenue_treatment"].isin(
        ["include_positive", "include_negative"]
    )
)

# Expected customer-level eligibility
expected_customer_eligibility = (
    df_classified["customer_level_eligible"]
    & df_classified["Customer ID"].notna()
)

# Expected qualifying purchase definition
expected_qualifying_purchase = (
    df_classified["qualifying_purchase"]
    & df_classified["Customer ID"].notna()
    & (df_classified["Quantity"] > 0)
    & (df_classified["Price"] > 0)
)

# Validate flag consistency
flag_validation = pd.DataFrame([
    {
        "validation": "Merchandise revenue eligibility",
        "violations": (
            df_classified["is_merchandise_revenue_record"]
            != expected_merchandise_revenue
        ).sum()
    },
    {
        "validation": "Customer analytics eligibility",
        "violations": (
            df_classified["is_customer_analytics_eligible"]
            != expected_customer_eligibility
        ).sum()
    },
    {
        "validation": "Qualifying purchase eligibility",
        "violations": (
            df_classified["is_qualifying_purchase"]
            != expected_qualifying_purchase
        ).sum()
    },
    {
        "validation": "Merchandise revenue calculation",
        "violations": (
            df_classified["merchandise_revenue"]
            != df_classified["line_value"].where(
                expected_merchandise_revenue, 0
            )
        ).sum()
    },
    {
        "validation": "Qualifying purchase revenue calculation",
        "violations": (
            df_classified["qualifying_purchase_revenue"]
            != df_classified["line_value"].where(
                expected_qualifying_purchase, 0
            )
        ).sum()
    }
])

flag_validation["validation_passed"] = (
    flag_validation["violations"] == 0
)

display(flag_validation)

# Summarize analytical eligibility and revenue by category
analytical_summary = (
    df_classified
    .groupby("transaction_category")
    .agg(
        record_count=("Invoice", "size"),
        merchandise_revenue_records=(
            "is_merchandise_revenue_record", "sum"
        ),
        customer_eligible_records=(
            "is_customer_analytics_eligible", "sum"
        ),
        qualifying_purchase_records=(
            "is_qualifying_purchase", "sum"
        ),
        merchandise_revenue=("merchandise_revenue", "sum"),
        qualifying_purchase_revenue=(
            "qualifying_purchase_revenue", "sum"
        )
    )
    .reset_index()
    .sort_values("record_count", ascending=False)
)

display(analytical_summary)

# Overall reconciliation
print(
    "Total merchandise revenue:",
    round(df_classified["merchandise_revenue"].sum(), 2)
)

print(
    "Total qualifying purchase revenue:",
    round(df_classified["qualifying_purchase_revenue"].sum(), 2)
)

print(
    "Qualifying purchase records:",
    df_classified["is_qualifying_purchase"].sum()
)

print(
    "Customer-eligible records:",
    df_classified["is_customer_analytics_eligible"].sum()
)

,validation,violations,validation_passed
0,Merchandise revenue eligibility,0,True
1,Customer analytics eligibility,0,True
2,Qualifying purchase eligibility,0,True
3,Merchandise revenue calculation,0,True
4,Qualifying purchase revenue calculation,0,True


,transaction_category,record_count,merchandise_revenue_records,customer_eligible_records,qualifying_purchase_records,merchandise_revenue,qualifying_purchase_revenue
9,merchandise_candidate,802559,802559,802559,802559,1.743316e+07,1.743316e+07
1,anonymous_merchandise_candidate,234447,234447,0,0,2.677472e+06,0.000000e+00
8,merchandise_cancellation_candidate,17947,17947,17947,0,-7.275930e+05,0.000000e+00
4,delivery_charge,3931,0,0,0,0.000000e+00,0.000000e+00
10,operational_adjustment_candidate,3457,0,0,0,0.000000e+00,0.000000e+00
15,zero_price_activity,2717,0,62,0,0.000000e+00,0.000000e+00
7,manual_transaction,1426,0,0,0,0.000000e+00,0.000000e+00
0,anonymous_merchandise_cancellation_candidate,353,353,0,0,-6.935970e+03,0.000000e+00
5,discount_related_transaction,177,0,0,0,0.000000e+00,0.000000e+00
13,sample_transaction,104,0,0,0,0.000000e+00,0.000000e+00


Total merchandise revenue: 19376104.21
Total qualifying purchase revenue: 17433160.75
Qualifying purchase records: 802559
Customer-eligible records: 820568


In [178]:
# Measure the contribution of exact duplicate records
# using the original, unmodified transaction columns.

duplicate_mask = df_combined_raw.duplicated(keep="first")

print("Excess exact duplicate records:", duplicate_mask.sum())

duplicate_records = df_classified.loc[duplicate_mask].copy()

duplicate_impact = (
    duplicate_records
    .groupby("transaction_category")
    .agg(
        duplicate_records=("Invoice", "size"),
        unique_invoices=("Invoice", "nunique"),
        duplicate_merchandise_revenue=(
            "merchandise_revenue", "sum"
        ),
        duplicate_qualifying_purchase_revenue=(
            "qualifying_purchase_revenue", "sum"
        )
    )
    .reset_index()
    .sort_values("duplicate_records", ascending=False)
)

display(duplicate_impact)

print(
    "\nDuplicate merchandise line value:",
    round(duplicate_records["merchandise_revenue"].sum(), 2)
)

print(
    "Duplicate qualifying purchase value:",
    round(
        duplicate_records["qualifying_purchase_revenue"].sum(),
        2
    )
)

Excess exact duplicate records: 34335


,transaction_category,duplicate_records,unique_invoices,duplicate_merchandise_revenue,duplicate_qualifying_purchase_revenue
7,merchandise_candidate,26055,4963,365882.01,365882.01
1,anonymous_merchandise_candidate,7609,117,100437.37,0.00
6,merchandise_cancellation_candidate,347,160,-9699.55,0.00
11,zero_price_activity,124,65,0.00,0.00
8,operational_adjustment_candidate,64,64,0.00,0.00
3,delivery_charge,62,62,0.00,0.00
5,manual_transaction,34,27,0.00,0.00
0,anonymous_merchandise_cancellation_candidate,24,4,-393.87,0.00
9,platform_fee_candidate,7,7,0.00,0.00
4,discount_related_transaction,4,4,0.00,0.00



Duplicate merchandise line value: 456225.96
Duplicate qualifying purchase value: 365882.01


In [179]:
# Profile excess exact duplicates by month, invoice, and customer

# Reuse the duplicate mask derived from the original eight columns.
df_classified["is_excess_exact_duplicate"] = duplicate_mask

duplicate_purchase_records = df_classified[
    df_classified["is_excess_exact_duplicate"]
    & df_classified["is_qualifying_purchase"]
].copy()

# Monthly duplicate impact
duplicate_purchase_records["invoice_month"] = (
    duplicate_purchase_records["InvoiceDate"]
    .dt.to_period("M")
    .astype(str)
)

monthly_duplicate_impact = (
    duplicate_purchase_records
    .groupby("invoice_month")
    .agg(
        duplicate_records=("Invoice", "size"),
        affected_invoices=("Invoice", "nunique"),
        affected_customers=("Customer ID", "nunique"),
        duplicate_purchase_value=(
            "qualifying_purchase_revenue", "sum"
        )
    )
    .reset_index()
    .sort_values("invoice_month")
)

print("Monthly duplicate impact:")
display(monthly_duplicate_impact)

# Invoice-level concentration
duplicate_invoice_impact = (
    duplicate_purchase_records
    .groupby("Invoice")
    .agg(
        duplicate_records=("Invoice", "size"),
        duplicate_purchase_value=(
            "qualifying_purchase_revenue", "sum"
        ),
        customer_id=("Customer ID", "first"),
        invoice_date=("InvoiceDate", "first")
    )
    .reset_index()
    .sort_values(
        "duplicate_purchase_value",
        ascending=False
    )
)

print("\nTop 20 invoices by duplicate purchase value:")
display(duplicate_invoice_impact.head(20))

# Customer-level concentration
duplicate_customer_impact = (
    duplicate_purchase_records
    .groupby("Customer ID")
    .agg(
        duplicate_records=("Invoice", "size"),
        affected_invoices=("Invoice", "nunique"),
        duplicate_purchase_value=(
            "qualifying_purchase_revenue", "sum"
        )
    )
    .reset_index()
    .sort_values(
        "duplicate_purchase_value",
        ascending=False
    )
)

print("\nTop 20 customers by duplicate purchase value:")
display(duplicate_customer_impact.head(20))

Monthly duplicate impact:


,invoice_month,duplicate_records,affected_invoices,affected_customers,duplicate_purchase_value
0,2009-12,482,199,167,3150.15
1,2010-01,317,122,113,1516.39
2,2010-02,327,135,121,1812.11
3,2010-03,516,203,178,2603.52
4,2010-04,403,166,147,2626.34
5,2010-05,411,167,152,2152.41
6,2010-06,500,189,170,2695.45
7,2010-07,389,163,142,1900.57
8,2010-08,363,158,143,2018.05
9,2010-09,470,173,162,2600.25



Top 20 invoices by duplicate purchase value:


,Invoice,duplicate_records,duplicate_purchase_value,customer_id,invoice_date
2952,537659,9,15885.49,18102.0,2010-12-07 16:43:00
2950,537657,5,9639.12,18102.0,2010-12-07 16:42:00
2597,536783,38,4076.48,15061.0,2010-12-02 15:19:00
3085,537899,1,3794.40,12755.0,2010-12-09 10:44:00
2409,536387,5,3193.92,16029.0,2010-12-01 09:58:00
3009,537793,29,2746.75,15032.0,2010-12-08 12:57:00
2599,536785,6,2730.96,15061.0,2010-12-02 15:22:00
3053,537852,86,2671.27,17340.0,2010-12-08 15:56:00
2796,537235,5,2567.64,17381.0,2010-12-06 09:45:00
2491,536576,16,2558.42,13777.0,2010-12-01 16:11:00



Top 20 customers by duplicate purchase value:


,Customer ID,duplicate_records,affected_invoices,duplicate_purchase_value
1844,18102.0,18,4,27834.61
742,15061.0,89,7,11429.50
332,13777.0,35,9,6840.16
1709,17850.0,297,34,5391.21
1079,16210.0,25,2,4738.54
1033,16029.0,13,4,4445.52
674,14911.0,165,15,4401.82
59,12755.0,1,1,3794.40
11,12433.0,147,2,3787.12
1539,17381.0,11,3,3603.72


In [180]:
# Investigate exact duplicate qualifying purchase groups
# specifically in December 2010.

december_2010_purchases = df_classified[
    (df_classified["InvoiceDate"].dt.year == 2010)
    & (df_classified["InvoiceDate"].dt.month == 12)
    & df_classified["is_qualifying_purchase"]
].copy()

# Group by all original transaction columns.
# This identifies groups of completely identical transaction lines.

december_duplicate_groups = (
    december_2010_purchases
    .groupby(source_columns, dropna=False)
    .size()
    .reset_index(name="occurrences")
)

# Retain only groups with repeated identical records.
december_duplicate_groups = december_duplicate_groups[
    december_duplicate_groups["occurrences"] > 1
].copy()

# Calculate excess occurrences and their financial impact.
december_duplicate_groups["excess_occurrences"] = (
    december_duplicate_groups["occurrences"] - 1
)

december_duplicate_groups["line_value"] = (
    december_duplicate_groups["Quantity"]
    * december_duplicate_groups["Price"]
)

december_duplicate_groups["duplicate_purchase_value"] = (
    december_duplicate_groups["excess_occurrences"]
    * december_duplicate_groups["line_value"]
)

december_duplicate_groups = december_duplicate_groups.sort_values(
    "duplicate_purchase_value",
    ascending=False
)

print(
    "December qualifying purchase records:",
    len(december_2010_purchases)
)

print(
    "Exact duplicate groups:",
    len(december_duplicate_groups)
)

print(
    "Excess duplicate occurrences:",
    december_duplicate_groups["excess_occurrences"].sum()
)

print(
    "Duplicate purchase value:",
    round(
        december_duplicate_groups["duplicate_purchase_value"].sum(),
        2
    )
)

print("\nTop 30 duplicate transaction groups:")

display(december_duplicate_groups.head(30))

December qualifying purchase records: 40501
Exact duplicate groups: 14272
Excess duplicate occurrences: 14932
Duplicate purchase value: 311814.91

Top 30 duplicate transaction groups:


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,occurrences,excess_occurrences,line_value,duplicate_purchase_value
10479,537659,21623,VINTAGE UNION JACK MEMOBOARD,600,2010-12-07 16:43:00,6.38,18102.0,United Kingdom,2,1,3828.00,3828.00
12631,537899,22328,ROUND SNACK BOXES SET OF 4 FRUITS,1488,2010-12-09 10:44:00,2.55,12755.0,Japan,2,1,3794.40,3794.40
10486,537659,82484,WOOD BLACK BOARD ANT WHITE FINISH,600,2010-12-07 16:43:00,4.78,18102.0,United Kingdom,2,1,2868.00,2868.00
10471,537657,21623,VINTAGE UNION JACK MEMOBOARD,408,2010-12-07 16:42:00,6.38,18102.0,United Kingdom,2,1,2603.04,2603.04
10485,537659,22833,HALL CABINET WITH 3 DRAWERS,72,2010-12-07 16:43:00,32.69,18102.0,United Kingdom,2,1,2353.68,2353.68
10481,537659,22188,BLACK HEART CARD HOLDER,1008,2010-12-07 16:43:00,2.31,18102.0,United Kingdom,2,1,2328.48,2328.48
10482,537659,22189,CREAM HEART CARD HOLDER,1008,2010-12-07 16:43:00,2.31,18102.0,United Kingdom,2,1,2328.48,2328.48
10472,537657,22188,BLACK HEART CARD HOLDER,972,2010-12-07 16:42:00,2.31,18102.0,United Kingdom,2,1,2245.32,2245.32
10473,537657,22189,CREAM HEART CARD HOLDER,972,2010-12-07 16:42:00,2.31,18102.0,United Kingdom,2,1,2245.32,2245.32
10475,537657,82484,WOOD BLACK BOARD ANT WHITE FINISH,408,2010-12-07 16:42:00,4.78,18102.0,United Kingdom,2,1,1950.24,1950.24


In [181]:
# Examine the structure of December 2010 duplicate groups

# 1. Distribution of identical transaction occurrences
occurrence_distribution = (
    december_duplicate_groups
    .groupby("occurrences")
    .agg(
        duplicate_groups=("Invoice", "size"),
        excess_occurrences=("excess_occurrences", "sum"),
        duplicate_purchase_value=(
            "duplicate_purchase_value", "sum"
        )
    )
    .reset_index()
    .sort_values("occurrences")
)

print("Distribution of duplicate group occurrences:")
display(occurrence_distribution)


# 2. Concentration by invoice
invoice_duplicate_profile = (
    december_duplicate_groups
    .groupby("Invoice")
    .agg(
        duplicate_groups=("StockCode", "size"),
        excess_duplicate_records=("excess_occurrences", "sum"),
        duplicate_purchase_value=(
            "duplicate_purchase_value", "sum"
        )
    )
    .reset_index()
    .sort_values(
        "duplicate_purchase_value",
        ascending=False
    )
)

print("\nTop 20 invoices by duplicate group count:")
display(
    invoice_duplicate_profile
    .sort_values("duplicate_groups", ascending=False)
    .head(20)
)


# 3. Inspect all original transaction lines in two highly affected invoices
invoices_to_inspect = ["537659", "537657"]

invoice_details = december_2010_purchases[
    december_2010_purchases["Invoice"].astype(str).isin(
        invoices_to_inspect
    )
].copy()

invoice_line_occurrences = (
    invoice_details
    .groupby(source_columns, dropna=False)
    .size()
    .reset_index(name="occurrences")
)

print("\nInvoice-level details:")
print("Invoices inspected:", invoices_to_inspect)
print("Total transaction records:", len(invoice_details))

display(
    invoice_line_occurrences
    .sort_values(
        ["Invoice", "StockCode"]
    )
)

Distribution of duplicate group occurrences:


,occurrences,duplicate_groups,excess_occurrences,duplicate_purchase_value
0,2,13962,13962,308021.36
1,3,13,26,152.20
2,4,272,816,3265.05
3,5,1,4,15.00
4,6,23,115,346.45
5,10,1,9,14.85



Top 20 invoices by duplicate group count:


,Invoice,duplicate_groups,excess_duplicate_records,duplicate_purchase_value
399,537224,152,186,1462.88
616,537781,136,186,610.59
350,537135,117,117,722.36
342,537126,117,119,479.61
221,536796,105,137,402.26
358,537144,96,114,357.53
664,537852,86,86,2671.27
450,537374,82,92,395.33
52,536464,81,89,289.60
373,537196,76,96,440.27



Invoice-level details:
Invoices inspected: ['537659', '537657']
Total transaction records: 28


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,occurrences
0,537657,21623,VINTAGE UNION JACK MEMOBOARD,408,2010-12-07 16:42:00,6.38,18102.0,United Kingdom,2
1,537657,22188,BLACK HEART CARD HOLDER,972,2010-12-07 16:42:00,2.31,18102.0,United Kingdom,2
2,537657,22189,CREAM HEART CARD HOLDER,972,2010-12-07 16:42:00,2.31,18102.0,United Kingdom,2
3,537657,22830,UTILTY CABINET WITH HOOKS,30,2010-12-07 16:42:00,19.84,18102.0,United Kingdom,2
4,537657,82484,WOOD BLACK BOARD ANT WHITE FINISH,408,2010-12-07 16:42:00,4.78,18102.0,United Kingdom,2
5,537659,21623,VINTAGE UNION JACK MEMOBOARD,600,2010-12-07 16:43:00,6.38,18102.0,United Kingdom,2
6,537659,21769,VINTAGE POST OFFICE CABINET,10,2010-12-07 16:43:00,52.77,18102.0,United Kingdom,2
7,537659,22188,BLACK HEART CARD HOLDER,1008,2010-12-07 16:43:00,2.31,18102.0,United Kingdom,2
8,537659,22189,CREAM HEART CARD HOLDER,1008,2010-12-07 16:43:00,2.31,18102.0,United Kingdom,2
9,537659,22507,MEMO BOARD RETROSPOT DESIGN,208,2010-12-07 16:43:00,3.86,18102.0,United Kingdom,2


In [182]:
# Inspect original row positions of the two invoices

invoices_to_inspect = ["537657", "537659"]

invoice_original_rows = df_combined_raw[
    df_combined_raw["Invoice"].astype(str).isin(
        invoices_to_inspect
    )
].copy()

invoice_original_rows["original_row_index"] = (
    invoice_original_rows.index
)

invoice_original_rows = invoice_original_rows.sort_values(
    ["Invoice", "original_row_index"]
)

display(
    invoice_original_rows[
        [
            "original_row_index",
            "Invoice",
            "StockCode",
            "Description",
            "Quantity",
            "InvoiceDate",
            "Price",
            "Customer ID"
        ]
    ]
)

,original_row_index,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID
519363,519363,537657,82484,WOOD BLACK BOARD ANT WHITE FINISH,408,2010-12-07 16:42:00,4.78,18102.0
519364,519364,537657,22830,UTILTY CABINET WITH HOOKS,30,2010-12-07 16:42:00,19.84,18102.0
519365,519365,537657,22189,CREAM HEART CARD HOLDER,972,2010-12-07 16:42:00,2.31,18102.0
519366,519366,537657,22188,BLACK HEART CARD HOLDER,972,2010-12-07 16:42:00,2.31,18102.0
519367,519367,537657,21623,VINTAGE UNION JACK MEMOBOARD,408,2010-12-07 16:42:00,6.38,18102.0
541886,541886,537657,82484,WOOD BLACK BOARD ANT WHITE FINISH,408,2010-12-07 16:42:00,4.78,18102.0
541887,541887,537657,22830,UTILTY CABINET WITH HOOKS,30,2010-12-07 16:42:00,19.84,18102.0
541888,541888,537657,22189,CREAM HEART CARD HOLDER,972,2010-12-07 16:42:00,2.31,18102.0
541889,541889,537657,22188,BLACK HEART CARD HOLDER,972,2010-12-07 16:42:00,2.31,18102.0
541890,541890,537657,21623,VINTAGE UNION JACK MEMOBOARD,408,2010-12-07 16:42:00,6.38,18102.0


In [183]:
# Calculate the original row-position gaps between duplicate occurrences

invoice_position_gaps = []

for invoice, group in invoice_original_rows.groupby("Invoice"):

    for _, duplicate_group in group.groupby(
        source_columns,
        dropna=False
    ):

        positions = sorted(duplicate_group.index.tolist())

        if len(positions) > 1:
            invoice_position_gaps.append({
                "Invoice": invoice,
                "StockCode": duplicate_group["StockCode"].iloc[0],
                "original_positions": positions,
                "position_gaps": [
                    positions[i + 1] - positions[i]
                    for i in range(len(positions) - 1)
                ]
            })

invoice_position_gaps = pd.DataFrame(invoice_position_gaps)

display(invoice_position_gaps)

,Invoice,StockCode,original_positions,position_gaps
0,537657,21623,"[519367, 541890]",[22523]
1,537657,22188,"[519366, 541889]",[22523]
2,537657,22189,"[519365, 541888]",[22523]
3,537657,22830,"[519364, 541887]",[22523]
4,537657,82484,"[519363, 541886]",[22523]
5,537659,21623,"[519376, 541899]",[22523]
6,537659,21769,"[519375, 541898]",[22523]
7,537659,22188,"[519374, 541897]",[22523]
8,537659,22189,"[519373, 541896]",[22523]
9,537659,22507,"[519369, 541895]",[22526]


In [184]:
# Analyze original row-position gaps for December 2010
# exact duplicate qualifying purchase groups.

position_gap_records = []

for _, group in december_2010_purchases.groupby(
    source_columns,
    dropna=False,
    sort=False
):
    if len(group) > 1:

        positions = sorted(group.index.tolist())

        for i in range(len(positions) - 1):

            position_gap_records.append({
                "position_gap": positions[i + 1] - positions[i],
                "first_position": positions[i],
                "second_position": positions[i + 1]
            })

position_gap_profile = (
    pd.DataFrame(position_gap_records)
    .groupby("position_gap")
    .agg(
        matching_pairs=("position_gap", "size")
    )
    .reset_index()
    .sort_values("matching_pairs", ascending=False)
)

print("Total duplicate position pairs:", len(position_gap_records))

print("\nMost frequent original row-position gaps:")
display(position_gap_profile.head(20))

# Measure the contribution of the previously identified offset
target_offset = 22523

matching_offset_count = sum(
    record["position_gap"] == target_offset
    for record in position_gap_records
)

print(
    f"\nPairs with offset {target_offset:,}:",
    matching_offset_count
)

print(
    "Share of duplicate position pairs:",
    round(
        matching_offset_count / len(position_gap_records) * 100,
        2
    )
)

Total duplicate position pairs: 14932

Most frequent original row-position gaps:


,position_gap,matching_pairs
168,22523,11947
166,22521,193
167,22522,192
172,22527,98
1,2,85
162,22517,82
121,22474,78
0,1,78
135,22490,72
153,22508,68



Pairs with offset 22,523: 11947
Share of duplicate position pairs: 80.01


In [185]:
# Examine duplicate position gaps around the dominant offset

target_offset = 22523
offset_tolerance = 150

position_gap_analysis = pd.DataFrame(position_gap_records)

position_gap_analysis["gap_difference"] = (
    position_gap_analysis["position_gap"] - target_offset
)

position_gap_analysis["within_offset_window"] = (
    position_gap_analysis["gap_difference"].abs()
    <= offset_tolerance
)

offset_window_summary = (
    position_gap_analysis
    .groupby("within_offset_window")
    .agg(
        matching_pairs=("position_gap", "size"),
        minimum_gap=("position_gap", "min"),
        maximum_gap=("position_gap", "max")
    )
    .reset_index()
)

display(offset_window_summary)

within_window = position_gap_analysis[
    position_gap_analysis["within_offset_window"]
]

print(
    "Pairs within ±150 rows of the dominant offset:",
    len(within_window)
)

print(
    "Percentage of all duplicate position pairs:",
    round(
        len(within_window) / len(position_gap_analysis) * 100,
        2
    )
)

print(
    "\nDistinct position gaps within the window:",
    within_window["position_gap"].nunique()
)

,within_offset_window,matching_pairs,minimum_gap,maximum_gap
0,False,808,1,22368
1,True,14124,22383,22641


Pairs within ±150 rows of the dominant offset: 14124
Percentage of all duplicate position pairs: 94.59

Distinct position gaps within the window: 120


In [186]:
# Attach qualifying purchase values to each duplicate position pair

position_value_records = []

for _, group in december_2010_purchases.groupby(
    source_columns,
    dropna=False,
    sort=False
):
    if len(group) > 1:

        ordered_group = group.sort_index()
        positions = ordered_group.index.tolist()
        line_value = (
            ordered_group["Quantity"].iloc[0]
            * ordered_group["Price"].iloc[0]
        )

        for i in range(len(positions) - 1):
            position_value_records.append({
                "position_gap": positions[i + 1] - positions[i],
                "pair_value": line_value
            })

position_value_analysis = pd.DataFrame(position_value_records)

position_value_analysis["gap_difference"] = (
    position_value_analysis["position_gap"] - 22523
)

position_value_analysis["within_offset_window"] = (
    position_value_analysis["gap_difference"].abs() <= 150
)

offset_value_summary = (
    position_value_analysis
    .groupby("within_offset_window")
    .agg(
        matching_pairs=("pair_value", "size"),
        associated_value=("pair_value", "sum")
    )
    .reset_index()
)

offset_value_summary["value_share_pct"] = (
    offset_value_summary["associated_value"]
    / position_value_analysis["pair_value"].sum()
    * 100
)

display(offset_value_summary)

print(
    "Total pair-associated value:",
    round(position_value_analysis["pair_value"].sum(), 2)
)

,within_offset_window,matching_pairs,associated_value,value_share_pct
0,False,808,3514.04,1.126963
1,True,14124,308300.87,98.873037


Total pair-associated value: 311814.91


In [187]:
# Combine position-gap information with associated transaction values

position_excess_analysis = position_gap_analysis[
    ["first_position", "second_position", "position_gap"]
].copy()

position_excess_analysis["pair_value"] = (
    position_value_analysis["pair_value"].values
)

position_excess_analysis["within_offset_window"] = (
    position_excess_analysis["position_gap"].sub(22523).abs()
    <= 150
)

# Check whether each excess row appears only once
print(
    "Total position pairs:",
    len(position_excess_analysis)
)

print(
    "Unique later-side positions:",
    position_excess_analysis["second_position"].nunique()
)

print(
    "Repeated later-side positions:",
    position_excess_analysis["second_position"].duplicated().sum()
)

# Summarize unique excess rows and their associated value
unique_excess_summary = (
    position_excess_analysis
    .groupby("within_offset_window")
    .agg(
        excess_records=("second_position", "nunique"),
        associated_value=("pair_value", "sum")
    )
    .reset_index()
)

unique_excess_summary["record_share_pct"] = (
    unique_excess_summary["excess_records"]
    / unique_excess_summary["excess_records"].sum()
    * 100
)

unique_excess_summary["value_share_pct"] = (
    unique_excess_summary["associated_value"]
    / unique_excess_summary["associated_value"].sum()
    * 100
)

display(unique_excess_summary)

print(
    "\nTotal unique excess records:",
    position_excess_analysis["second_position"].nunique()
)

print(
    "Total associated value:",
    round(position_excess_analysis["pair_value"].sum(), 2)
)

Total position pairs: 14932
Unique later-side positions: 14932
Repeated later-side positions: 0


,within_offset_window,excess_records,associated_value,record_share_pct,value_share_pct
0,False,808,3514.04,5.411197,1.126963
1,True,14124,308300.87,94.588803,98.873037



Total unique excess records: 14932
Total associated value: 311814.91


In [188]:
# Identify later-side positions associated with the dominant offset window

repeated_block_positions = (
    position_excess_analysis.loc[
        position_excess_analysis["within_offset_window"],
        "second_position"
    ]
    .sort_values()
    .reset_index(drop=True)
)

# Identify contiguous sequences of original row positions
position_differences = repeated_block_positions.diff()

block_starts = (
    position_differences.ne(1)
    .cumsum()
)

repeated_block_summary = (
    pd.DataFrame({
        "position": repeated_block_positions,
        "block_id": block_starts
    })
    .groupby("block_id")
    .agg(
        start_position=("position", "min"),
        end_position=("position", "max"),
        excess_records=("position", "size")
    )
    .reset_index(drop=True)
)

repeated_block_summary["position_span"] = (
    repeated_block_summary["end_position"]
    - repeated_block_summary["start_position"]
    + 1
)

repeated_block_summary = repeated_block_summary.sort_values(
    "excess_records",
    ascending=False
)

print(
    "Number of contiguous repeated-position blocks:",
    len(repeated_block_summary)
)

print("\nLargest repeated-position blocks:")
display(repeated_block_summary.head(20))

print(
    "\nTotal excess records across identified blocks:",
    repeated_block_summary["excess_records"].sum()
)

Number of contiguous repeated-position blocks: 469

Largest repeated-position blocks:


,start_position,end_position,excess_records,position_span
66,529155,529357,203,203
38,526684,526883,200,200
80,529872,530058,187,187
415,544623,544808,186,186
338,539999,540176,178,178
56,528632,528798,167,167
139,532435,532599,165,165
420,545093,545252,160,160
52,527712,527866,155,155
166,533783,533933,151,151



Total excess records across identified blocks: 14124


In [189]:
# Map each later-side duplicate position to its earlier counterpart

source_position_analysis = position_excess_analysis[
    position_excess_analysis["within_offset_window"]
].copy()

source_position_analysis["earlier_position"] = (
    source_position_analysis["second_position"]
    - source_position_analysis["position_gap"]
)

# Check uniqueness of the mapped earlier positions
print(
    "Later-side positions:",
    source_position_analysis["second_position"].nunique()
)

print(
    "Mapped earlier positions:",
    source_position_analysis["earlier_position"].nunique()
)

print(
    "Repeated earlier positions:",
    source_position_analysis["earlier_position"].duplicated().sum()
)

# Identify contiguous sequences among the earlier counterpart positions
earlier_positions = (
    source_position_analysis["earlier_position"]
    .sort_values()
    .reset_index(drop=True)
)

earlier_block_ids = earlier_positions.diff().ne(1).cumsum()

earlier_block_summary = (
    pd.DataFrame({
        "position": earlier_positions,
        "block_id": earlier_block_ids
    })
    .groupby("block_id")
    .agg(
        start_position=("position", "min"),
        end_position=("position", "max"),
        source_records=("position", "size")
    )
    .reset_index(drop=True)
)

earlier_block_summary["position_span"] = (
    earlier_block_summary["end_position"]
    - earlier_block_summary["start_position"]
    + 1
)

earlier_block_summary = earlier_block_summary.sort_values(
    "source_records",
    ascending=False
)

print(
    "\nNumber of contiguous earlier-source blocks:",
    len(earlier_block_summary)
)

print("\nLargest earlier-source blocks:")
display(earlier_block_summary.head(20))

print(
    "\nTotal mapped earlier-source records:",
    earlier_block_summary["source_records"].sum()
)

Later-side positions: 14124
Mapped earlier positions: 14124
Repeated earlier positions: 0

Number of contiguous earlier-source blocks: 466

Largest earlier-source blocks:


,start_position,end_position,source_records,position_span
66,506632,506848,217,217
39,504156,504360,205,205
410,522096,522285,190,190
334,517476,517653,178,178
57,506109,506275,167,167
141,509912,510076,165,165
155,510488,510649,162,162
415,522570,522729,160,160
81,507349,507507,159,159
164,511003,511160,158,158



Total mapped earlier-source records: 14124


In [190]:
# Assign block IDs to later-side positions
later_sorted = source_position_analysis.sort_values(
    "second_position"
).copy()

later_sorted["later_block_id"] = (
    later_sorted["second_position"].diff().ne(1).cumsum()
)

# Assign block IDs to earlier-source positions
earlier_sorted = source_position_analysis.sort_values(
    "earlier_position"
).copy()

earlier_sorted["earlier_block_id"] = (
    earlier_sorted["earlier_position"].diff().ne(1).cumsum()
)

# Merge block IDs back to each position mapping
block_mapping = later_sorted[
    ["second_position", "earlier_position", "later_block_id"]
].merge(
    earlier_sorted[
        ["second_position", "earlier_block_id"]
    ],
    on="second_position",
    how="left",
    validate="one_to_one"
)

# Summarize how many earlier blocks each later block maps to
later_to_earlier_summary = (
    block_mapping
    .groupby("later_block_id")
    .agg(
        later_records=("second_position", "size"),
        earlier_blocks=("earlier_block_id", "nunique"),
        mapped_earlier_start=("earlier_position", "min"),
        mapped_earlier_end=("earlier_position", "max")
    )
    .reset_index()
)

later_to_earlier_summary["single_source_block"] = (
    later_to_earlier_summary["earlier_blocks"] == 1
)

print("Later-side blocks:", len(later_to_earlier_summary))

print(
    "Later blocks mapping to exactly one earlier block:",
    later_to_earlier_summary["single_source_block"].sum()
)

print(
    "Later blocks mapping to multiple earlier blocks:",
    (~later_to_earlier_summary["single_source_block"]).sum()
)

print("\nLargest later blocks and their source-block mapping:")
display(
    later_to_earlier_summary
    .sort_values("later_records", ascending=False)
    .head(20)
)

print("\nDistribution of earlier blocks per later block:")
display(
    later_to_earlier_summary["earlier_blocks"]
    .value_counts()
    .sort_index()
    .rename_axis("earlier_blocks_per_later_block")
    .reset_index(name="later_block_count")
)

Later-side blocks: 469
Later blocks mapping to exactly one earlier block: 309
Later blocks mapping to multiple earlier blocks: 160

Largest later blocks and their source-block mapping:


,later_block_id,later_records,earlier_blocks,mapped_earlier_start,mapped_earlier_end,single_source_block
66,67,203,1,506632,506834,True
38,39,200,1,504161,504360,True
80,81,187,12,507349,507625,False
415,416,186,1,522100,522285,True
338,339,178,1,517476,517653,True
56,57,167,1,506109,506275,True
139,140,165,1,509912,510076,True
420,421,160,1,522570,522729,True
52,53,155,2,505165,505343,False
166,167,151,3,511260,511419,False



Distribution of earlier blocks per later block:


,earlier_blocks_per_later_block,later_block_count
0,1,309
1,2,96
2,3,23
3,4,23
4,5,6
5,6,7
6,7,2
7,8,1
8,12,1
9,13,1


In [191]:
# Quantify records by source-block mapping consistency

mapping_consistency_summary = (
    later_to_earlier_summary
    .groupby("single_source_block")
    .agg(
        later_blocks=("later_block_id", "size"),
        excess_records=("later_records", "sum")
    )
    .reset_index()
)

mapping_consistency_summary["block_share_pct"] = (
    mapping_consistency_summary["later_blocks"]
    / mapping_consistency_summary["later_blocks"].sum()
    * 100
)

mapping_consistency_summary["record_share_pct"] = (
    mapping_consistency_summary["excess_records"]
    / mapping_consistency_summary["excess_records"].sum()
    * 100
)

display(mapping_consistency_summary)

print(
    "Total excess records:",
    mapping_consistency_summary["excess_records"].sum()
)

,single_source_block,later_blocks,excess_records,block_share_pct,record_share_pct
0,False,160,6134,34.115139,43.429623
1,True,309,7990,65.884861,56.570377


Total excess records: 14124


In [192]:
# Validate mapped source and later records against original data

source_positions = (
    source_position_analysis["earlier_position"].astype(int).tolist()
)

later_positions = (
    source_position_analysis["second_position"].astype(int).tolist()
)

source_records = df_combined_raw.loc[
    source_positions, source_columns
].reset_index(drop=True)

later_records = df_combined_raw.loc[
    later_positions, source_columns
].reset_index(drop=True)

# Compare each mapped pair, including missing values
pair_matches = (
    source_records.eq(later_records)
    | (source_records.isna() & later_records.isna())
).all(axis=1)

print("Mapped record pairs:", len(pair_matches))
print("Exact matches across all original columns:", pair_matches.sum())
print("Non-matching pairs:", (~pair_matches).sum())
print(
    "Exact match percentage:",
    round(pair_matches.mean() * 100, 2)
)

Mapped record pairs: 14124
Exact matches across all original columns: 14124
Non-matching pairs: 0
Exact match percentage: 100.0


In [193]:
# Profile exact duplicate excess records by month across the full dataset

all_duplicate_records = df_classified.loc[
    df_classified["is_excess_exact_duplicate"]
].copy()

all_duplicate_records["duplicate_month"] = (
    all_duplicate_records["InvoiceDate"].dt.to_period("M")
)

duplicate_month_profile = (
    all_duplicate_records
    .groupby("duplicate_month")
    .agg(
        excess_duplicate_records=("Invoice", "size"),
        affected_invoices=("Invoice", "nunique"),
        affected_customers=("Customer ID", "nunique"),
        merchandise_revenue_impact=("merchandise_revenue", "sum"),
        qualifying_purchase_impact=(
            "qualifying_purchase_revenue", "sum"
        )
    )
    .reset_index()
)

duplicate_month_profile["record_share_pct"] = (
    duplicate_month_profile["excess_duplicate_records"]
    / duplicate_month_profile["excess_duplicate_records"].sum()
    * 100
)

display(
    duplicate_month_profile.sort_values(
        "excess_duplicate_records",
        ascending=False
    )
)

print(
    "Total excess duplicate records:",
    duplicate_month_profile["excess_duplicate_records"].sum()
)

print(
    "Total duplicate qualifying purchase impact:",
    round(
        duplicate_month_profile["qualifying_purchase_impact"].sum(),
        2
    )
)

,duplicate_month,excess_duplicate_records,affected_invoices,affected_customers,merchandise_revenue_impact,qualifying_purchase_impact,record_share_pct
12,2010-12,23023,1156,662,404596.92,311814.91,67.054027
11,2010-11,1431,429,341,5957.44,5876.02,4.167759
23,2011-11,1368,421,326,5610.45,5611.77,3.984273
10,2010-10,849,302,260,3482.74,3567.99,2.472696
22,2011-10,773,256,224,976.44,3316.34,2.251347
3,2010-03,526,209,180,2574.52,2603.52,1.531964
6,2010-06,514,197,172,2725.68,2695.45,1.497015
0,2009-12,506,208,167,3198.61,3150.15,1.473715
9,2010-09,484,182,163,2543.67,2600.25,1.409640
5,2010-05,415,171,154,2052.11,2152.41,1.208679


Total excess duplicate records: 34335
Total duplicate qualifying purchase impact: 365882.01


In [194]:
# Compare monthly duplicate rates against transaction volume

df_monthly_duplicate_rates = df_classified.copy()

df_monthly_duplicate_rates["month"] = (
    df_monthly_duplicate_rates["InvoiceDate"].dt.to_period("M")
)

monthly_duplicate_rates = (
    df_monthly_duplicate_rates
    .groupby("month")
    .agg(
        total_records=("Invoice", "size"),
        excess_duplicate_records=("is_excess_exact_duplicate", "sum"),
        qualifying_purchase_records=("is_qualifying_purchase", "sum"),
        duplicate_qualifying_records=(
            "is_excess_exact_duplicate",
            lambda x: (
                x
                & df_monthly_duplicate_rates.loc[
                    x.index, "is_qualifying_purchase"
                ]
            ).sum()
        )
    )
    .reset_index()
)

monthly_duplicate_rates["overall_duplicate_rate_pct"] = (
    monthly_duplicate_rates["excess_duplicate_records"]
    / monthly_duplicate_rates["total_records"]
    * 100
)

monthly_duplicate_rates["qualifying_purchase_duplicate_rate_pct"] = (
    monthly_duplicate_rates["duplicate_qualifying_records"]
    / monthly_duplicate_rates["qualifying_purchase_records"]
    * 100
)

display(
    monthly_duplicate_rates.sort_values(
        "overall_duplicate_rate_pct",
        ascending=False
    )
)

,month,total_records,excess_duplicate_records,qualifying_purchase_records,duplicate_qualifying_records,overall_duplicate_rate_pct,qualifying_purchase_duplicate_rate_pct
12,2010-12,65004,23023,40501,14932,35.417820,36.868225
11,2010-11,78015,1431,60126,1407,1.834263,2.340086
23,2011-11,84711,1368,64271,1363,1.614902,2.120708
10,2010-10,59098,849,49413,838,1.436597,1.695910
6,2010-06,39983,514,31095,500,1.285546,1.607976
22,2011-10,60742,773,49358,756,1.272596,1.531667
3,2010-03,41511,526,32164,516,1.267134,1.604278
4,2010-04,34057,410,27124,403,1.203864,1.485769
5,2010-05,35323,415,28563,411,1.174872,1.438924
7,2010-07,33383,391,26936,389,1.171255,1.444164


In [195]:
# Investigate positional gaps across all December 2010 exact duplicates

december_2010_all = df_classified[
    (df_classified["InvoiceDate"].dt.year == 2010)
    & (df_classified["InvoiceDate"].dt.month == 12)
].copy()

all_december_position_pairs = []

for _, group in december_2010_all.groupby(
    source_columns,
    dropna=False,
    sort=False
):
    if len(group) > 1:
        positions = sorted(group.index.tolist())

        for i in range(len(positions) - 1):
            all_december_position_pairs.append({
                "first_position": positions[i],
                "second_position": positions[i + 1],
                "position_gap": positions[i + 1] - positions[i]
            })

all_december_position_analysis = pd.DataFrame(
    all_december_position_pairs
)

# Profile position gaps
all_december_gap_profile = (
    all_december_position_analysis
    .groupby("position_gap")
    .size()
    .reset_index(name="matching_pairs")
    .sort_values("matching_pairs", ascending=False)
)

print(
    "December excess duplicate pairs:",
    len(all_december_position_analysis)
)

print("\nMost frequent position gaps:")
display(all_december_gap_profile.head(20))

# Measure alignment with the previously identified offset
target_offset = 22523
offset_tolerance = 150

all_december_position_analysis["within_offset_window"] = (
    all_december_position_analysis["position_gap"]
    .sub(target_offset)
    .abs()
    .le(offset_tolerance)
)

all_december_offset_summary = (
    all_december_position_analysis
    .groupby("within_offset_window")
    .agg(
        matching_pairs=("position_gap", "size"),
        minimum_gap=("position_gap", "min"),
        maximum_gap=("position_gap", "max")
    )
    .reset_index()
)

all_december_offset_summary["pair_share_pct"] = (
    all_december_offset_summary["matching_pairs"]
    / len(all_december_position_analysis)
    * 100
)

display(all_december_offset_summary)

December excess duplicate pairs: 23023

Most frequent position gaps:


,position_gap,matching_pairs
171,22523,18474
172,22524,777
212,22682,401
170,22522,208
169,22521,200
83,22122,159
175,22527,98
164,22516,93
1,2,87
0,1,84


,within_offset_window,matching_pairs,minimum_gap,maximum_gap,pair_share_pct
0,False,1384,1,22682,6.01138
1,True,21639,22383,22641,93.98862


In [196]:
# Map the later-side positions to their transaction classifications

all_december_mapped_records = (
    all_december_position_analysis
    .merge(
        df_classified[
            [
                "transaction_category",
                "merchandise_revenue",
                "qualifying_purchase_revenue"
            ]
        ],
        left_on="second_position",
        right_index=True,
        how="left",
        validate="many_to_one"
    )
)

# Confirm unique later-side positions
print(
    "Unique later-side positions:",
    all_december_mapped_records["second_position"].nunique()
)

print(
    "Total position pairs:",
    len(all_december_mapped_records)
)

print(
    "Missing transaction classifications:",
    all_december_mapped_records["transaction_category"].isna().sum()
)

# Summarize by transaction category and offset-window status
category_offset_summary = (
    all_december_mapped_records
    .groupby(
        ["transaction_category", "within_offset_window"],
        dropna=False
    )
    .agg(
        excess_records=("second_position", "nunique"),
        merchandise_revenue_impact=("merchandise_revenue", "sum"),
        qualifying_purchase_impact=(
            "qualifying_purchase_revenue", "sum"
        )
    )
    .reset_index()
)

display(
    category_offset_summary.sort_values(
        ["transaction_category", "within_offset_window"]
    )
)

Unique later-side positions: 23023
Total position pairs: 23023
Missing transaction classifications: 0


,transaction_category,within_offset_window,excess_records,merchandise_revenue_impact,qualifying_purchase_impact
0,anonymous_merchandise_cancellation_candidate,True,20,-219.72,0.00
1,anonymous_merchandise_candidate,False,561,6645.13,0.00
2,anonymous_merchandise_candidate,True,6929,93059.93,0.00
3,bank_charge,True,2,0.00,0.00
4,delivery_charge,False,1,0.00,0.00
5,delivery_charge,True,61,0.00,0.00
6,discount_related_transaction,True,4,0.00,0.00
7,manual_transaction,True,23,0.00,0.00
8,merchandise_cancellation_candidate,False,9,-60.00,0.00
9,merchandise_cancellation_candidate,True,292,-6643.33,0.00


In [197]:
# Investigate December duplicate records outside the dominant offset window

outside_offset_records = all_december_position_analysis[
    ~all_december_position_analysis["within_offset_window"]
].copy()

outside_offset_profile = (
    outside_offset_records
    .groupby("position_gap")
    .size()
    .reset_index(name="matching_pairs")
    .sort_values("matching_pairs", ascending=False)
)

print("Outside-window duplicate pairs:", len(outside_offset_records))

print("\nMost frequent outside-window gaps:")
display(outside_offset_profile.head(25))

# Add transaction categories to the outside-window records
outside_offset_classified = outside_offset_records.merge(
    df_classified[
        [
            "transaction_category",
            "merchandise_revenue",
            "qualifying_purchase_revenue"
        ]
    ],
    left_on="second_position",
    right_index=True,
    how="left",
    validate="many_to_one"
)

outside_offset_category_profile = (
    outside_offset_classified
    .groupby("transaction_category")
    .agg(
        excess_records=("second_position", "nunique"),
        merchandise_revenue_impact=("merchandise_revenue", "sum"),
        qualifying_purchase_impact=(
            "qualifying_purchase_revenue", "sum"
        )
    )
    .reset_index()
    .sort_values("excess_records", ascending=False)
)

print("\nOutside-window records by transaction category:")
display(outside_offset_category_profile)

Outside-window duplicate pairs: 1384

Most frequent outside-window gaps:


,position_gap,matching_pairs
86,22682,401
83,22122,159
1,2,87
0,1,84
5,6,50
4,5,38
2,3,37
3,4,35
6,7,34
11,12,22



Outside-window records by transaction category:


,transaction_category,excess_records,merchandise_revenue_impact,qualifying_purchase_impact
3,merchandise_candidate,808,3514.04,3514.04
0,anonymous_merchandise_candidate,561,6645.13,0.00
2,merchandise_cancellation_candidate,9,-60.00,0.00
5,zero_price_activity,4,0.00,0.00
1,delivery_charge,1,0.00,0.00
4,platform_fee_candidate,1,0.00,0.00


In [198]:
# Investigate the two additional long-range offsets

additional_offsets = [22682, 22122]

additional_offset_records = outside_offset_records[
    outside_offset_records["position_gap"].isin(additional_offsets)
].copy()

additional_offset_records["earlier_position"] = (
    additional_offset_records["second_position"]
    - additional_offset_records["position_gap"]
)

for offset in additional_offsets:
    subset = additional_offset_records[
        additional_offset_records["position_gap"] == offset
    ].copy()

    earlier_positions = (
        subset["earlier_position"]
        .sort_values()
        .reset_index(drop=True)
    )

    later_positions = (
        subset["second_position"]
        .sort_values()
        .reset_index(drop=True)
    )

    earlier_blocks = earlier_positions.diff().ne(1).cumsum()
    later_blocks = later_positions.diff().ne(1).cumsum()

    print(f"\nOffset: {offset:,}")
    print("Matching pairs:", len(subset))
    print("Unique earlier positions:", earlier_positions.nunique())
    print("Unique later positions:", later_positions.nunique())
    print("Earlier contiguous blocks:", earlier_blocks.nunique())
    print("Later contiguous blocks:", later_blocks.nunique())

    print(
        "Repeated earlier positions:",
        earlier_positions.duplicated().sum()
    )

    print(
        "Repeated later positions:",
        later_positions.duplicated().sum()
    )

# Summarize the remaining short-range gaps
short_gap_records = outside_offset_records[
    ~outside_offset_records["position_gap"].isin(additional_offsets)
]

print("\nRemaining short-range duplicate pairs:", len(short_gap_records))
print(
    "Distinct short-range gaps:",
    short_gap_records["position_gap"].nunique()
)


Offset: 22,682
Matching pairs: 401
Unique earlier positions: 401
Unique later positions: 401
Earlier contiguous blocks: 1
Later contiguous blocks: 1
Repeated earlier positions: 0
Repeated later positions: 0

Offset: 22,122
Matching pairs: 159
Unique earlier positions: 159
Unique later positions: 159
Earlier contiguous blocks: 1
Later contiguous blocks: 1
Repeated earlier positions: 0
Repeated later positions: 0

Remaining short-range duplicate pairs: 824
Distinct short-range gaps: 85


In [199]:
# Summarize the boundaries of the three long-range alignments

long_range_alignment = outside_offset_records[
    outside_offset_records["position_gap"].isin([22682, 22122])
].copy()

long_range_alignment["earlier_position"] = (
    long_range_alignment["second_position"]
    - long_range_alignment["position_gap"]
)

alignment_boundaries = (
    long_range_alignment
    .groupby("position_gap")
    .agg(
        matching_pairs=("second_position", "size"),
        earlier_start=("earlier_position", "min"),
        earlier_end=("earlier_position", "max"),
        later_start=("second_position", "min"),
        later_end=("second_position", "max")
    )
    .reset_index()
)

alignment_boundaries["earlier_span"] = (
    alignment_boundaries["earlier_end"]
    - alignment_boundaries["earlier_start"] + 1
)

alignment_boundaries["later_span"] = (
    alignment_boundaries["later_end"]
    - alignment_boundaries["later_start"] + 1
)

display(alignment_boundaries)

,position_gap,matching_pairs,earlier_start,earlier_end,later_start,later_end,earlier_span,later_span
0,22122,159,514192,514350,536314,536472,159,159
1,22682,401,513791,514191,536473,536873,401,401


In [200]:
# Validate the two adjacent blocks as one reordered section

ordered_alignment = long_range_alignment.sort_values(
    "earlier_position"
).copy()

source_positions = ordered_alignment[
    "earlier_position"
].astype(int).tolist()

later_positions = ordered_alignment[
    "second_position"
].astype(int).tolist()

source_records = df_combined_raw.loc[
    source_positions, source_columns
].reset_index(drop=True)

later_records = df_combined_raw.loc[
    later_positions, source_columns
].reset_index(drop=True)

exact_matches = (
    source_records.eq(later_records)
    | (source_records.isna() & later_records.isna())
).all(axis=1)

print("Mapped records:", len(ordered_alignment))
print("Exact matches:", exact_matches.sum())
print("Non-matching records:", (~exact_matches).sum())

print(
    "Earlier source positions:",
    min(source_positions),
    "to",
    max(source_positions)
)

print(
    "Later positions:",
    min(later_positions),
    "to",
    max(later_positions)
)

# Verify the two blocks are adjacent and reordered
print(
    "\nEarlier blocks are adjacent:",
    514192 - 514191 == 1
)

print(
    "Later blocks are adjacent:",
    536473 - 536472 == 1
)

Mapped records: 560
Exact matches: 560
Non-matching records: 0
Earlier source positions: 513791 to 514350
Later positions: 536314 to 536873

Earlier blocks are adjacent: True
Later blocks are adjacent: True


In [201]:
# Profile the remaining short-range duplicate records

short_gap_classified = short_gap_records.merge(
    df_classified[
        [
            "transaction_category",
            "Invoice",
            "Customer ID",
            "merchandise_revenue",
            "qualifying_purchase_revenue"
        ]
    ],
    left_on="second_position",
    right_index=True,
    how="left",
    validate="many_to_one"
)

print("Short-range duplicate pairs:", len(short_gap_classified))

print(
    "Unique later-side positions:",
    short_gap_classified["second_position"].nunique()
)

print(
    "Unique earlier-side positions:",
    short_gap_classified["first_position"].nunique()
)

# Distribution of short-range gaps
short_gap_distribution = (
    short_gap_classified
    .groupby("position_gap")
    .agg(
        matching_pairs=("second_position", "size"),
        affected_invoices=("Invoice", "nunique"),
        affected_customers=("Customer ID", "nunique")
    )
    .reset_index()
    .sort_values("matching_pairs", ascending=False)
)

print("\nMost frequent short-range gaps:")
display(short_gap_distribution.head(25))

# Breakdown by transaction category
short_gap_category_summary = (
    short_gap_classified
    .groupby("transaction_category")
    .agg(
        excess_records=("second_position", "nunique"),
        affected_invoices=("Invoice", "nunique"),
        affected_customers=("Customer ID", "nunique"),
        merchandise_revenue_impact=("merchandise_revenue", "sum"),
        qualifying_purchase_impact=(
            "qualifying_purchase_revenue", "sum"
        )
    )
    .reset_index()
    .sort_values("excess_records", ascending=False)
)

print("\nShort-range duplicates by transaction category:")
display(short_gap_category_summary)

print(
    "\nTotal merchandise revenue impact:",
    round(
        short_gap_classified["merchandise_revenue"].sum(),
        2
    )
)

Short-range duplicate pairs: 824
Unique later-side positions: 824
Unique earlier-side positions: 824

Most frequent short-range gaps:


,position_gap,matching_pairs,affected_invoices,affected_customers
1,2,87,43,39
0,1,84,42,41
5,6,50,28,27
4,5,38,19,19
2,3,37,23,22
3,4,35,20,20
6,7,34,17,16
11,12,22,11,10
23,24,18,10,10
19,20,17,6,5



Short-range duplicates by transaction category:


,transaction_category,excess_records,affected_invoices,affected_customers,merchandise_revenue_impact,qualifying_purchase_impact
2,merchandise_candidate,808,147,131,3514.04,3514.04
1,merchandise_cancellation_candidate,9,1,1,-60.00,0.00
4,zero_price_activity,4,3,0,0.00,0.00
0,anonymous_merchandise_candidate,2,2,0,2.25,0.00
3,platform_fee_candidate,1,1,0,0.00,0.00



Total merchandise revenue impact: 3456.29


In [202]:
# Investigate short-range duplicate merchandise at invoice level

short_range_merchandise = short_gap_classified[
    short_gap_classified["transaction_category"].eq(
        "merchandise_candidate"
    )
].copy()

short_range_invoice_profile = (
    short_range_merchandise
    .groupby("Invoice")
    .agg(
        duplicate_records=("second_position", "size"),
        unique_stockcodes=("second_position", lambda x: (
            df_classified.loc[x, "StockCode_normalized"].nunique()
        )),
        affected_customers=("Customer ID", "nunique"),
        duplicate_value=("qualifying_purchase_revenue", "sum")
    )
    .reset_index()
    .sort_values("duplicate_records", ascending=False)
)

print(
    "Affected merchandise invoices:",
    short_range_invoice_profile["Invoice"].nunique()
)

print("\nDuplicate records per affected invoice:")
display(
    short_range_invoice_profile["duplicate_records"]
    .value_counts()
    .sort_index()
    .rename_axis("duplicate_records_per_invoice")
    .reset_index(name="invoice_count")
)

print("\nInvoices with the most short-range duplicate records:")
display(short_range_invoice_profile.head(20))

Affected merchandise invoices: 147

Duplicate records per affected invoice:


,duplicate_records_per_invoice,invoice_count
0,1,32
1,2,41
2,3,5
3,4,15
4,5,4
5,6,13
6,7,1
7,8,12
8,9,2
9,10,4



Invoices with the most short-range duplicate records:


,Invoice,duplicate_records,unique_stockcodes,affected_customers,duplicate_value
73,537781,51,22,1,195.23
49,537224,34,13,1,93.82
16,536796,32,15,1,53.22
1,536412,26,11,1,121.90
53,537265,20,8,1,113.28
42,537196,20,9,1,57.70
33,537144,18,7,1,28.50
68,537646,18,7,1,44.60
35,537154,18,8,1,136.80
6,536528,18,9,1,78.00


In [204]:
# Inspect complete transaction details for two affected invoices

invoices_to_review = ["537781", "537224"]

invoice_detail = df_classified[
    df_classified["Invoice"].astype(str).isin(invoices_to_review)
].copy()

invoice_detail["original_position"] = invoice_detail.index

invoice_duplicate_profile = (
    invoice_detail
    .groupby(source_columns, dropna=False)
    .agg(
        occurrences=("Invoice", "size"),
        original_positions=(
            "Invoice",
            lambda x: list(x.index)
        )
    )
    .reset_index()
)

invoice_duplicate_profile["excess_occurrences"] = (
    invoice_duplicate_profile["occurrences"] - 1
).clip(lower=0)

invoice_duplicate_profile["line_value"] = (
    invoice_duplicate_profile["Quantity"]
    * invoice_duplicate_profile["Price"]
)

for invoice in invoices_to_review:
    print(f"\n{'=' * 60}")
    print(f"INVOICE {invoice}")
    print(f"{'=' * 60}")

    invoice_rows = invoice_detail[
        invoice_detail["Invoice"].astype(str) == invoice
    ].sort_index()

    print("Total transaction records:", len(invoice_rows))
    print(
        "Distinct StockCodes:",
        invoice_rows["StockCode_normalized"].nunique()
    )

    print("\nTransaction lines and their original positions:")

    display(
        invoice_duplicate_profile[
            invoice_duplicate_profile["Invoice"].astype(str) == invoice
        ][
            [
                "StockCode",
                "Description",
                "Quantity",
                "Price",
                "occurrences",
                "excess_occurrences",
                "original_positions",
                "line_value"
            ]
        ].sort_values(
            ["excess_occurrences", "StockCode"],
            ascending=[False, True]
        )
    )


INVOICE 537781
Total transaction records: 322
Distinct StockCodes: 127

Transaction lines and their original positions:


,StockCode,Description,Quantity,Price,occurrences,excess_occurrences,original_positions,line_value
163,21098,CHRISTMAS TOILET ROLL,1,1.25,6,5,"[520754, 520863, 520909, 543277, 543386, 543432]",1.25
248,22745,POPPY'S PLAYHOUSE BEDROOM,1,2.10,6,5,"[520770, 520845, 520879, 543293, 543368, 543402]",2.10
261,22898,CHILDRENS APRON APPLES DESIGN,1,1.95,6,5,"[520847, 520870, 520891, 543370, 543393, 543414]",1.95
155,20751,FUNKY WASHING UP GLOVES ASSORTED,1,2.10,4,3,"[520759, 520895, 543282, 543418]",2.10
164,21136,PAINTED METAL PEARS ASSORTED,8,1.69,4,3,"[520809, 520871, 543332, 543394]",13.52
...,...,...,...,...,...,...,...,...
283,84559B,3D SHEET OF CAT STICKERS,1,0.85,2,1,"[520841, 543364]",0.85
284,85099B,JUMBO BAG RED RETROSPOT,1,1.95,2,1,"[520843, 543366]",1.95
285,85099C,JUMBO BAG BAROQUE BLACK WHITE,1,1.95,2,1,"[520878, 543401]",1.95
286,85099F,JUMBO BAG STRAWBERRY,1,1.95,2,1,"[520908, 543431]",1.95



INVOICE 537224
Total transaction records: 338
Distinct StockCodes: 122

Transaction lines and their original positions:


,StockCode,Description,Quantity,Price,occurrences,excess_occurrences,original_positions,line_value
122,70007,HI TEC ALPINE HAND WARMER,1,1.65,10,9,"[512850, 512852, 512875, 512887, 512906, 53537...",1.65
50,22086,PAPER CHAIN KIT 50'S CHRISTMAS,1,2.95,6,5,"[512856, 512876, 512974, 535379, 535399, 535497]",2.95
6,21187,WHITE BELL HONEYCOMB PAPER GARLAND,2,1.65,4,3,"[512898, 512939, 535421, 535462]",3.30
11,21202,DOLLY HONEYCOMB GARLAND,1,1.65,4,3,"[512857, 512869, 535380, 535392]",1.65
29,21802,CHRISTMAS TREE HEART DECORATION,3,0.42,4,3,"[512947, 512996, 535470, 535519]",1.26
...,...,...,...,...,...,...,...,...
147,84970L,SINGLE HEART ZINC T-LIGHT HOLDER,12,0.95,2,1,"[513003, 535526]",11.40
148,84970S,HANGING HEART ZINC T-LIGHT HOLDER,12,0.85,2,1,"[513004, 535527]",10.20
149,85025C,PINK HEART SHAPE PHOTO FRAME,1,1.65,2,1,"[512854, 535377]",1.65
150,85025C,PINK HEART SHAPE PHOTO FRAME,2,1.65,2,1,"[512961, 535484]",3.30


In [205]:
# Inspect original row-position sequences for the two invoices

for invoice in ["537781", "537224"]:

    invoice_rows = df_classified[
        df_classified["Invoice"].astype(str) == invoice
    ].sort_index().copy()

    positions = invoice_rows.index.to_numpy()

    print(f"\n{'=' * 60}")
    print(f"INVOICE {invoice}")
    print(f"{'=' * 60}")

    print("Total records:", len(invoice_rows))
    print("First original position:", positions.min())
    print("Last original position:", positions.max())

    # Separate positions into potential earlier and later sections
    earlier_positions = positions[positions < 530000]
    later_positions = positions[positions >= 530000]

    print("\nEarlier-section records:", len(earlier_positions))
    print("Later-section records:", len(later_positions))

    if len(earlier_positions) > 0 and len(later_positions) > 0:

        print(
            "Earlier position range:",
            earlier_positions.min(),
            "to",
            earlier_positions.max()
        )

        print(
            "Later position range:",
            later_positions.min(),
            "to",
            later_positions.max()
        )

        print(
            "Exact 22,523 offset matches:",
            sum(
                (position + 22523) in set(later_positions)
                for position in earlier_positions
            )
        )

    # Show the distribution of row-position gaps
    print("\nMost frequent gaps between identical transaction records:")

    invoice_duplicate_groups = (
        invoice_rows
        .groupby(source_columns, dropna=False)
    )

    invoice_gaps = []

    for _, group in invoice_duplicate_groups:
        group_positions = sorted(group.index.tolist())

        for i in range(len(group_positions) - 1):
            invoice_gaps.append(
                group_positions[i + 1] - group_positions[i]
            )

    display(
        pd.Series(invoice_gaps)
        .value_counts()
        .head(15)
        .rename_axis("position_gap")
        .reset_index(name="matching_pairs")
    )


INVOICE 537781
Total records: 322
First original position: 520749
Last original position: 543432

Earlier-section records: 161
Later-section records: 161
Earlier position range: 520749 to 520909
Later position range: 543272 to 543432
Exact 22,523 offset matches: 161

Most frequent gaps between identical transaction records:


,position_gap,matching_pairs
0,22523,114
1,109,4
2,4,4
3,126,4
4,136,2
5,46,2
6,62,2
7,22519,2
8,72,2
9,94,2



INVOICE 537224
Total records: 338
First original position: 512849
Last original position: 535540

Earlier-section records: 169
Later-section records: 169
Earlier position range: 512849 to 513017
Later position range: 535372 to 535540
Exact 22,523 offset matches: 169

Most frequent gaps between identical transaction records:


,position_gap,matching_pairs
0,22523,139
1,12,4
2,20,4
3,11,4
4,41,2
5,49,2
6,140,2
7,98,2
8,133,2
9,22512,2


In [206]:
# Test all short-range affected invoices for a complete
# 22,523-position repeated invoice block.

affected_invoice_ids = (
    short_range_merchandise["Invoice"]
    .astype(str)
    .unique()
)

invoice_block_results = []

for invoice in affected_invoice_ids:

    invoice_rows = df_classified[
        df_classified["Invoice"].astype(str) == invoice
    ].sort_index()

    positions = set(invoice_rows.index.tolist())

    earlier_positions = sorted(
        position for position in positions
        if position < 530000
    )

    later_positions = sorted(
        position for position in positions
        if position >= 530000
    )

    matching_positions = sum(
        position + 22523 in positions
        for position in earlier_positions
    )

    invoice_block_results.append({
        "Invoice": invoice,
        "total_records": len(invoice_rows),
        "earlier_records": len(earlier_positions),
        "later_records": len(later_positions),
        "offset_matches": matching_positions,
        "earlier_positions": len(earlier_positions),
        "later_positions": len(later_positions),
        "complete_offset_match": (
            len(earlier_positions) > 0
            and len(earlier_positions) == len(later_positions)
            and matching_positions == len(earlier_positions)
        )
    })

invoice_block_results = pd.DataFrame(invoice_block_results)

print(
    "Affected invoices examined:",
    len(invoice_block_results)
)

print("\nComplete 22,523-offset invoice blocks:")
print(
    invoice_block_results["complete_offset_match"].sum()
)

print("\nDistribution of offset-match coverage:")
display(
    invoice_block_results["complete_offset_match"]
    .value_counts()
    .rename_axis("complete_offset_match")
    .reset_index(name="invoice_count")
)

print("\nInvoices with complete offset matches:")
display(
    invoice_block_results[
        invoice_block_results["complete_offset_match"]
    ].sort_values("total_records", ascending=False).head(20)
)

Affected invoices examined: 147

Complete 22,523-offset invoice blocks:
54

Distribution of offset-match coverage:


,complete_offset_match,invoice_count
0,False,93
1,True,54



Invoices with complete offset matches:


,Invoice,total_records,earlier_records,later_records,offset_matches,earlier_positions,later_positions,complete_offset_match
49,537224,338,169,169,169,169,169,True
73,537781,322,161,161,161,161,161,True
16,536796,242,121,121,121,121,121,True
33,537144,210,105,105,105,105,105,True
68,537646,160,80,80,80,80,80,True
18,536846,152,76,76,76,76,76,True
78,538035,148,74,74,74,74,74,True
20,536874,138,69,69,69,69,69,True
72,537773,138,69,69,69,69,69,True
71,537772,132,66,66,66,66,66,True


In [207]:
# Validate 22,523-offset correspondence for all 147 affected invoices

invoice_offset_validation = []

for invoice in affected_invoice_ids:

    invoice_rows = df_classified[
        df_classified["Invoice"].astype(str) == invoice
    ].sort_index()

    earlier_positions = [
        pos for pos in invoice_rows.index
        if pos < 530000
    ]

    later_positions = set(
        pos for pos in invoice_rows.index
        if pos >= 530000
    )

    exact_matches = 0
    positional_matches = 0

    for earlier_pos in earlier_positions:

        later_pos = earlier_pos + 22523

        if later_pos in later_positions:

            positional_matches += 1

            source_row = df_combined_raw.loc[
                earlier_pos, source_columns
            ]

            later_row = df_combined_raw.loc[
                later_pos, source_columns
            ]

            rows_match = (
                source_row.eq(later_row)
                | (source_row.isna() & later_row.isna())
            ).all()

            if rows_match:
                exact_matches += 1

    invoice_offset_validation.append({
        "Invoice": invoice,
        "earlier_records": len(earlier_positions),
        "later_records": len(later_positions),
        "positional_matches": positional_matches,
        "exact_matches": exact_matches,
        "earlier_match_rate_pct": (
            exact_matches / len(earlier_positions) * 100
            if earlier_positions else 0
        )
    })

invoice_offset_validation = pd.DataFrame(
    invoice_offset_validation
)

invoice_offset_validation["match_class"] = pd.cut(
    invoice_offset_validation["earlier_match_rate_pct"],
    bins=[-1, 0, 99.999, 100],
    labels=[
        "no_exact_matches",
        "partial_exact_matches",
        "complete_exact_matches"
    ]
)

print("Invoice-level validation:")
display(
    invoice_offset_validation["match_class"]
    .value_counts()
    .rename_axis("match_class")
    .reset_index(name="invoice_count")
)

print("\nLowest matching rates:")
display(
    invoice_offset_validation.sort_values(
        "earlier_match_rate_pct"
    ).head(20)
)

print("\nOverall validation:")
print(
    "Earlier records:",
    invoice_offset_validation["earlier_records"].sum()
)

print(
    "Positional matches:",
    invoice_offset_validation["positional_matches"].sum()
)

print(
    "Exact record matches:",
    invoice_offset_validation["exact_matches"].sum()
)

Invoice-level validation:


,match_class,invoice_count
0,no_exact_matches,90
1,complete_exact_matches,54
2,partial_exact_matches,3



Lowest matching rates:


,Invoice,earlier_records,later_records,positional_matches,exact_matches,earlier_match_rate_pct,match_class
0,536409,116,0,0,0,0.0,no_exact_matches
1,536412,162,0,0,0,0.0,no_exact_matches
2,536446,64,0,0,0,0.0,no_exact_matches
3,536464,170,0,0,0,0.0,no_exact_matches
4,536488,70,0,0,0,0.0,no_exact_matches
5,536522,108,0,0,0,0.0,no_exact_matches
6,536528,114,0,0,0,0.0,no_exact_matches
7,536530,46,0,0,0,0.0,no_exact_matches
8,536531,46,0,0,0,0.0,no_exact_matches
9,536551,70,0,0,0,0.0,no_exact_matches



Overall validation:
Earlier records: 5064
Positional matches: 3134
Exact record matches: 2705


In [208]:
# Identify records at the expected offset that are not exact matches

mismatch_records = []

for invoice in affected_invoice_ids:

    invoice_rows = df_classified[
        df_classified["Invoice"].astype(str) == invoice
    ]

    earlier_positions = [
        pos for pos in invoice_rows.index
        if pos < 530000
    ]

    later_positions = set(
        pos for pos in invoice_rows.index
        if pos >= 530000
    )

    for earlier_pos in earlier_positions:

        later_pos = earlier_pos + 22523

        if later_pos not in later_positions:
            continue

        source_row = df_combined_raw.loc[
            earlier_pos, source_columns
        ]

        later_row = df_combined_raw.loc[
            later_pos, source_columns
        ]

        equal_columns = (
            source_row.eq(later_row)
            | (source_row.isna() & later_row.isna())
        )

        if not equal_columns.all():

            differing_columns = equal_columns[
                ~equal_columns
            ].index.tolist()

            mismatch_records.append({
                "Invoice": invoice,
                "earlier_position": earlier_pos,
                "later_position": later_pos,
                "differing_columns": differing_columns
            })

mismatch_details = pd.DataFrame(mismatch_records)

print("Total positional mismatches:", len(mismatch_details))

print("\nColumns responsible for mismatches:")

column_mismatch_counts = (
    mismatch_details["differing_columns"]
    .explode()
    .value_counts()
    .rename_axis("column")
    .reset_index(name="mismatch_count")
)

display(column_mismatch_counts)

print("\nMost frequently affected invoices:")
display(
    mismatch_details["Invoice"]
    .value_counts()
    .head(20)
    .rename_axis("Invoice")
    .reset_index(name="mismatch_count")
)

print("\nSample mismatched record pairs:")
display(mismatch_details.head(15))

Total positional mismatches: 429

Columns responsible for mismatches:


,column,mismatch_count
0,StockCode,429
1,Description,429
2,Price,382
3,Quantity,248



Most frequently affected invoices:


,Invoice,mismatch_count
0,537126,116
1,537154,57
2,537196,53
3,537133,48
4,537208,39
5,537374,38
6,537382,34
7,537155,26
8,536981,12
9,537262,6



Sample mismatched record pairs:


,Invoice,earlier_position,later_position,differing_columns
0,536981,509734,532257,"[StockCode, Description, Quantity, Price]"
1,536981,509735,532258,"[StockCode, Description, Quantity, Price]"
2,536981,509736,532259,"[StockCode, Description, Quantity, Price]"
3,536981,509737,532260,"[StockCode, Description, Quantity, Price]"
4,536981,509738,532261,"[StockCode, Description, Price]"
5,536981,509739,532262,"[StockCode, Description, Price]"
6,536981,509740,532263,"[StockCode, Description, Quantity, Price]"
7,536981,509741,532264,"[StockCode, Description, Quantity, Price]"
8,536981,509742,532265,"[StockCode, Description, Quantity, Price]"
9,536981,509743,532266,"[StockCode, Description, Price]"


### Final Assessment and Duplicate-Handling Decision

The duplicate-record investigation identified **34,335 excess exact duplicate records**, representing approximately 3.22% of the original dataset.

The investigation revealed a substantial concentration in December 2010, which accounted for 23,023 excess duplicate records. This represented approximately 67.05% of all identified excess exact duplicates.

Further positional analysis identified a dominant offset pattern of 22,523 rows. However, positional alignment alone was not sufficient to establish exact duplication. Among the 429 positionally aligned mismatches examined, all differed in `StockCode` and `Description`, while 382 also differed in `Price` and 248 in `Quantity`.

These findings suggest substantial repeated data patterns in the December 2010 records, but the precise mechanism responsible for the anomaly remains unconfirmed.

**Duplicate-handling decision:**

1. Preserve the original dataset unchanged.
2. Identify exact duplicates using all eight original source columns.
3. Retain the first occurrence of each exact record.
4. Exclude subsequent exact occurrences from the primary analytical dataset.
5. Preserve duplicate indicators and original record positions for traceability.
6. Do not remove records solely because they follow a positional offset pattern.
7. Document the December 2010 anomaly as a data-quality limitation.

This approach establishes a reproducible baseline while avoiding unsupported assumptions about the cause of the duplication. Subsequent analyses should compare relevant metrics before and after duplicate exclusion to assess sensitivity to the handling decision.

In [209]:
# Create an audit dataset containing excess exact duplicates
df_duplicate_audit = df_classified.loc[duplicate_mask].copy()

df_duplicate_audit["original_row_position"] = df_duplicate_audit.index
df_duplicate_audit["is_excess_exact_duplicate"] = True

# Create the primary analytical dataset
df_clean = df_classified.loc[~duplicate_mask].copy()

df_clean["original_row_position"] = df_clean.index

# Reset indexes while preserving original row positions
df_duplicate_audit = df_duplicate_audit.reset_index(drop=True)
df_clean = df_clean.reset_index(drop=True)

# Validate the results
print("Original records:", len(df_classified))
print("Excluded exact duplicates:", len(df_duplicate_audit))
print("Cleaned records:", len(df_clean))

print(
    "\nRemaining exact duplicates:",
    df_clean.duplicated(subset=source_columns).sum()
)

print(
    "Original record count reconciled:",
    len(df_clean) + len(df_duplicate_audit) == len(df_classified)
)

print(
    "Original row positions preserved:",
    df_clean["original_row_position"].notna().all()
    and df_duplicate_audit["original_row_position"].notna().all()
)

print("\nCleaned dataset dimensions:", df_clean.shape)
print("Duplicate audit dimensions:", df_duplicate_audit.shape)

Original records: 1067371
Excluded exact duplicates: 34335
Cleaned records: 1033036

Remaining exact duplicates: 0
Original record count reconciled: True
Original row positions preserved: True

Cleaned dataset dimensions: (1033036, 33)
Duplicate audit dimensions: (34335, 33)


In [210]:
# 1. Align cleaned records with their original source records
raw_aligned = (
    df_combined_raw
    .loc[df_clean["original_row_position"].to_numpy(), source_columns]
    .reset_index(drop=True)
)

clean_aligned = df_clean.reset_index(drop=True)

# 2. Validate original source fields, excluding Description
# Description may have been legitimately imputed earlier.
source_validation = {}

for column in source_columns:
    if column == "Description":
        continue

    equal_values = (
        raw_aligned[column].eq(clean_aligned[column])
        | (
            raw_aligned[column].isna()
            & clean_aligned[column].isna()
        )
    )

    source_validation[column] = equal_values.all()

# 3. Validate Description preservation and imputation tracking
original_description_missing = raw_aligned["Description"].isna()

description_flag_matches = (
    original_description_missing
    == clean_aligned["Description_was_missing"]
).all()

non_missing_descriptions_preserved = (
    raw_aligned.loc[~original_description_missing, "Description"]
    .reset_index(drop=True)
    .equals(
        clean_aligned.loc[~original_description_missing, "Description"]
        .reset_index(drop=True)
    )
)

# 4. Validate transaction classification
classification_missing = (
    df_clean["transaction_category"].isna().sum()
)

qualifying_purchase_violations = (
    df_clean["is_qualifying_purchase"]
    & (
        ~df_clean["qualifying_purchase"]
        | df_clean["Customer ID"].isna()
        | (df_clean["Quantity"] <= 0)
        | (df_clean["Price"] <= 0)
    )
).sum()

# 5. Validate analytical revenue fields
expected_merchandise_revenue = df_clean["line_value"].where(
    df_clean["is_merchandise_revenue_record"], 0
)

expected_qualifying_revenue = df_clean["line_value"].where(
    df_clean["is_qualifying_purchase"], 0
)

merchandise_revenue_valid = (
    df_clean["merchandise_revenue"]
    .equals(expected_merchandise_revenue)
)

qualifying_revenue_valid = (
    df_clean["qualifying_purchase_revenue"]
    .equals(expected_qualifying_revenue)
)

# 6. Display validation results
print("SOURCE FIELD VALIDATION")
for column, result in source_validation.items():
    print(f"{column}: {result}")

print("\nDESCRIPTION VALIDATION")
print("Missing-value indicator preserved:", description_flag_matches)
print("Originally available descriptions preserved:",
      non_missing_descriptions_preserved)

print("\nCLASSIFICATION VALIDATION")
print("Missing transaction categories:", classification_missing)
print("Qualifying purchase violations:", qualifying_purchase_violations)

print("\nREVENUE VALIDATION")
print("Merchandise revenue valid:", merchandise_revenue_valid)
print("Qualifying purchase revenue valid:", qualifying_revenue_valid)

SOURCE FIELD VALIDATION
Invoice: True
StockCode: True
Quantity: True
InvoiceDate: True
Price: True
Customer ID: True
Country: True

DESCRIPTION VALIDATION
Missing-value indicator preserved: True
Originally available descriptions preserved: True

CLASSIFICATION VALIDATION
Missing transaction categories: 0
Qualifying purchase violations: 0

REVENUE VALIDATION
Merchandise revenue valid: True
Qualifying purchase revenue valid: True


In [211]:
# 1. Inspect data types and missing values
final_structure = pd.DataFrame({
    "data_type": df_clean.dtypes.astype(str),
    "missing_count": df_clean.isna().sum(),
    "missing_percentage": (
        df_clean.isna().mean() * 100
    ).round(2),
    "unique_values": df_clean.nunique(dropna=True)
})

display(final_structure)

# 2. Validate original row-position integrity
clean_positions = set(df_clean["original_row_position"])
audit_positions = set(df_duplicate_audit["original_row_position"])

print("\nROW-POSITION VALIDATION")
print("Unique cleaned positions:", len(clean_positions))
print("Unique audit positions:", len(audit_positions))
print("Overlap between datasets:", len(clean_positions & audit_positions))
print(
    "All original positions accounted for:",
    len(clean_positions | audit_positions) == len(df_classified)
)

# 3. Validate duplicate audit records
print("\nDUPLICATE AUDIT VALIDATION")
print(
    "All audit records marked as excess duplicates:",
    df_duplicate_audit["is_excess_exact_duplicate"].all()
)

print(
    "All audit records match an earlier retained record:",
    df_duplicate_audit["original_row_position"].notna().all()
)

# 4. Check numerical fields for infinite values
numeric_columns = df_clean.select_dtypes(include="number").columns

infinite_counts = (
    df_clean[numeric_columns]
    .isin([float("inf"), float("-inf")])
    .sum()
)

print("\nINFINITE VALUE CHECK")
display(infinite_counts[infinite_counts > 0])

# 5. Review transaction-category distribution
print("\nTRANSACTION CATEGORY DISTRIBUTION")
display(
    df_clean["transaction_category"]
    .value_counts(dropna=False)
    .rename_axis("transaction_category")
    .reset_index(name="record_count")
)

,data_type,missing_count,missing_percentage,unique_values
Invoice,object,0,0.00,53628
StockCode,object,0,0.00,5305
Description,object,1622,0.16,5698
Quantity,int64,0,0.00,1057
InvoiceDate,datetime64[us],0,0.00,47635
Price,float64,0,0.00,2807
Customer ID,float64,235151,22.76,5942
Country,str,0,0.00,43
YearMonth,period[M],0,0.00,25
Description_was_missing,bool,0,0.00,2



ROW-POSITION VALIDATION
Unique cleaned positions: 1033036
Unique audit positions: 34335
Overlap between datasets: 0
All original positions accounted for: True

DUPLICATE AUDIT VALIDATION
All audit records marked as excess duplicates: True
All audit records match an earlier retained record: True

INFINITE VALUE CHECK


Series([], dtype: int64)


TRANSACTION CATEGORY DISTRIBUTION


,transaction_category,record_count
0,merchandise_candidate,776504
1,anonymous_merchandise_candidate,226838
2,merchandise_cancellation_candidate,17600
3,delivery_charge,3869
4,operational_adjustment_candidate,3393
5,zero_price_activity,2593
6,manual_transaction,1392
7,anonymous_merchandise_cancellation_candidate,329
8,discount_related_transaction,173
9,sample_transaction,101


In [215]:
from pathlib import Path
import importlib.util
import pandas as pd

# 1. Create processed-data directory
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# 2. Export primary cleaned dataset as CSV
clean_csv_path = PROCESSED_DIR / "online_retail_clean.csv"

df_clean.to_csv(
    clean_csv_path,
    index=False,
    encoding="utf-8"
)

# 3. Export duplicate audit dataset as CSV
audit_csv_path = PROCESSED_DIR / "duplicate_audit.csv"

df_duplicate_audit.to_csv(
    audit_csv_path,
    index=False,
    encoding="utf-8"
)

print("CSV EXPORT")
print("Clean dataset:", clean_csv_path.resolve())
print("Audit dataset:", audit_csv_path.resolve())

# 4. Check Parquet engine availability
parquet_available = (
    importlib.util.find_spec("pyarrow") is not None
    or importlib.util.find_spec("fastparquet") is not None
)

# 5. Export primary cleaned dataset as Parquet
if parquet_available:

    # Work on a separate copy
    df_clean_export = df_clean.copy()

    # Normalize all object columns to consistent strings
    object_columns = df_clean_export.select_dtypes(
        include=["object"]
    ).columns

    def normalize_text_value(value):
        if pd.isna(value):
            return pd.NA

        if isinstance(value, bytes):
            return value.decode("utf-8", errors="replace")

        return str(value)

    for column in object_columns:
        df_clean_export[column] = (
            df_clean_export[column]
            .map(normalize_text_value)
            .astype("string")
        )

    clean_parquet_path = (
        PROCESSED_DIR / "online_retail_clean.parquet"
    )

    df_clean_export.to_parquet(
        clean_parquet_path,
        index=False,
        engine="pyarrow"
    )

    print("\nPARQUET EXPORT")
    print("Export successful:", clean_parquet_path.resolve())
    print("Normalized object columns:", list(object_columns))

else:
    clean_parquet_path = None
    print("\nParquet export skipped: no Parquet engine installed.")

# 6. Verify exported files
print("\nFILE VERIFICATION")

exported_files = [
    clean_csv_path,
    audit_csv_path,
    clean_parquet_path
]

for path in exported_files:
    if path is not None:
        print(
            f"{path.name}: "
            f"exists={path.exists()}, "
            f"size={path.stat().st_size / (1024 ** 2):.2f} MB"
        )

# 7. Verify Parquet can be read back
if clean_parquet_path is not None:
    df_parquet_check = pd.read_parquet(clean_parquet_path)

    print("\nPARQUET READ-BACK VALIDATION")
    print("Original cleaned dimensions:", df_clean.shape)
    print("Parquet dimensions:", df_parquet_check.shape)
    print(
        "Record count preserved:",
        len(df_parquet_check) == len(df_clean)
    )
    print(
        "Column names preserved:",
        df_parquet_check.columns.tolist() == df_clean.columns.tolist()
    )

CSV EXPORT
Clean dataset: C:\Users\Kia\Downloads\DATA-SCIENCE-PORTFOLIO\customer-intelligence-revenue-optimization\data\processed\online_retail_clean.csv
Audit dataset: C:\Users\Kia\Downloads\DATA-SCIENCE-PORTFOLIO\customer-intelligence-revenue-optimization\data\processed\duplicate_audit.csv


C:\Users\Kia\AppData\Local\Temp\ipykernel_6224\4105798108.py:44: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_columns = df_clean_export.select_dtypes(



PARQUET EXPORT
Export successful: C:\Users\Kia\Downloads\DATA-SCIENCE-PORTFOLIO\customer-intelligence-revenue-optimization\data\processed\online_retail_clean.parquet
Normalized object columns: ['Invoice', 'StockCode', 'Description', 'Country', 'zero_price_category', 'transaction_category', 'merchandise_revenue_treatment', 'notes']

FILE VERIFICATION
online_retail_clean.csv: exists=True, size=331.41 MB
duplicate_audit.csv: exists=True, size=10.94 MB
online_retail_clean.parquet: exists=True, size=19.02 MB

PARQUET READ-BACK VALIDATION
Original cleaned dimensions: (1033036, 33)
Parquet dimensions: (1033036, 33)
Record count preserved: True
Column names preserved: True


## Final Data Quality Assessment and Cleaning Outcome

### 1. Duplicate Handling

The investigation identified 34,335 excess exact duplicate records out of 1,067,371 original records.

A substantial concentration was observed in December 2010, accounting for 23,023 excess duplicate records. Positional analysis revealed repeated data patterns, although positional alignment alone was insufficient to establish exact duplication.

The implemented handling policy was:

- Preserve the original dataset unchanged.
- Identify exact duplicates using all eight original source columns.
- Retain the first occurrence of each exact record.
- Exclude subsequent exact occurrences from the primary analytical dataset.
- Preserve excluded records in a separate duplicate audit dataset.
- Retain original row positions for traceability.

### 2. Missing-Value Handling

**Customer ID**
- No customer identifiers were imputed.
- Anonymous transactions were retained for appropriate transaction-level analysis.
- Records without reliable customer identification were excluded from customer-level analysis requiring customer identity.

**Description**
- A controlled mapping based on StockCode was used to recover descriptions where sufficient supporting evidence existed.
- 2,740 missing descriptions were recovered.
- 1,642 missing descriptions remained unresolved.
- Original descriptions were preserved, and imputed records were identified using `Description_was_missing`.

### 3. Transaction Classification

Transactions were classified into 16 categories using a project-specific taxonomy.

The classification distinguishes merchandise purchases, cancellations, anonymous merchandise, delivery charges, financial adjustments, operational activities, and other special transactions.

Analytical flags and revenue fields were derived from the approved classification rules.

### 4. Anomaly Handling

Negative quantities, zero-price transactions, negative prices, extreme quantities, and special transaction codes were investigated.

Records were not automatically removed merely because their values appeared unusual. Their treatment depended on transaction context and customer-level eligibility.

### 5. Final Dataset

| Metric | Result |
|---|---:|
| Original records | 1,067,371 |
| Excluded exact duplicates | 34,335 |
| Primary cleaned records | 1,033,036 |
| Original columns | 8 |
| Cleaned dataset columns | 33 |
| Remaining exact duplicates | 0 |
| Missing transaction categories | 0 |
| Qualifying purchase rule violations | 0 |

### 6. Validation and Export

The cleaned dataset passed the following checks:

- Original source-field preservation.
- Description imputation tracking.
- Transaction classification completeness.
- Analytical revenue calculation consistency.
- Duplicate exclusion reconciliation.
- Original row-position traceability.
- Absence of infinite numerical values.
- Successful CSV and Parquet exports.
- Successful Parquet read-back with dimensions and column names preserved.

### 7. Output Files

The following files were generated in `data/processed/`:

- `online_retail_clean.csv`
- `online_retail_clean.parquet`
- `duplicate_audit.csv`

The original raw dataset remains unchanged.

### 8. Remaining Limitations

The cleaning process establishes a reproducible analytical baseline, but it does not prove that every source transaction is economically correct.

The December 2010 duplication anomaly remains an unresolved source-data limitation. Unresolved descriptions, anonymous transactions, and ambiguous operational records also remain documented.

Subsequent customer analytics and predictive modeling must respect the transaction classification rules and the distinction between recorded purchase activity and other transaction activity.

In [216]:
from pathlib import Path
import sys

# Add project root to Python's import path
PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.extract import extract_online_retail

# Extract original dataset
df_extracted = extract_online_retail()

# Validate extraction
print("Extracted dimensions:", df_extracted.shape)
print("Expected dimensions:", df_combined_raw.shape)

print(
    "Dimensions match:",
    df_extracted.shape == df_combined_raw.shape
)

print(
    "Column order matches:",
    df_extracted.columns.tolist() == source_columns
)

print(
    "Original records preserved:",
    df_extracted.equals(
        df_combined_raw.reset_index(drop=True)
    )
)

Extracted dimensions: (1067371, 8)
Expected dimensions: (1067371, 8)
Dimensions match: True
Column order matches: True
Original records preserved: True


In [217]:
print("DESCRIPTION MAPPING")
print("Dimensions:", description_mapping.shape)
print("Columns:", description_mapping.columns.tolist())
display(description_mapping.head())

print("\nTRANSACTION TAXONOMY")
print("Dimensions:", transaction_taxonomy.shape)
print("Columns:", transaction_taxonomy.columns.tolist())
display(transaction_taxonomy)

DESCRIPTION MAPPING
Dimensions: (1387, 2)
Columns: ['imputed_description', 'evidence_count']


,imputed_description,evidence_count
StockCode,,
79321,CHILLI LIGHTS,1211
22084,PAPER CHAIN KIT EMPIRE,843
37461,FUNKY MONKEY MUG,98
37509,NEW ENGLAND MUG W GIFT BOX,18
84977,WIRE FLOWER T-LIGHT HOLDER,45



TRANSACTION TAXONOMY
Dimensions: (16, 5)
Columns: ['transaction_category', 'merchandise_revenue_treatment', 'customer_level_eligible', 'qualifying_purchase', 'notes']


,transaction_category,merchandise_revenue_treatment,customer_level_eligible,qualifying_purchase,notes
0,merchandise_candidate,include_positive,True,True,Identified-customer merchandise transactions
1,anonymous_merchandise_candidate,include_positive,False,False,Merchandise transactions without Customer ID
2,merchandise_cancellation_candidate,include_negative,True,False,Identified-customer merchandise cancellations
3,anonymous_merchandise_cancellation_candidate,include_negative,False,False,Anonymous merchandise cancellations
4,delivery_charge,separate,False,False,Delivery-related charges
5,packing_charge,separate,False,False,Packing-related charges
6,zero_price_activity,exclude,True,False,Zero-price activity; not a paid purchase
7,operational_adjustment_candidate,exclude,False,False,Potential operational or inventory adjustments
8,manual_transaction,separate,False,False,Manual transactions with unclear accounting me...
9,discount_related_transaction,separate,False,False,Discount-related records; accounting treatment...


In [218]:
# Create reference-data directory
REFERENCE_DIR = Path("../data/reference")
REFERENCE_DIR.mkdir(parents=True, exist_ok=True)

# 1. Export description mapping
description_mapping_export = (
    description_mapping
    .rename_axis("StockCode")
    .reset_index()
)

description_mapping_path = (
    REFERENCE_DIR / "description_mapping.csv"
)

description_mapping_export.to_csv(
    description_mapping_path,
    index=False,
    encoding="utf-8"
)

# 2. Export transaction taxonomy
transaction_taxonomy_path = (
    REFERENCE_DIR / "transaction_taxonomy.csv"
)

transaction_taxonomy.to_csv(
    transaction_taxonomy_path,
    index=False,
    encoding="utf-8"
)

# 3. Verify both reference files
description_mapping_check = pd.read_csv(
    description_mapping_path,
    dtype={"StockCode": "string"}
)

transaction_taxonomy_check = pd.read_csv(
    transaction_taxonomy_path
)

print("DESCRIPTION MAPPING")
print("Original dimensions:", description_mapping_export.shape)
print("Exported dimensions:", description_mapping_check.shape)
print(
    "StockCode mapping preserved:",
    description_mapping_check["StockCode"].nunique()
    == len(description_mapping)
)

print("\nTRANSACTION TAXONOMY")
print("Original dimensions:", transaction_taxonomy.shape)
print("Exported dimensions:", transaction_taxonomy_check.shape)
print(
    "All categories preserved:",
    set(transaction_taxonomy["transaction_category"])
    == set(transaction_taxonomy_check["transaction_category"])
)

print("\nSAVED FILES")
print(description_mapping_path.resolve())
print(transaction_taxonomy_path.resolve())

DESCRIPTION MAPPING
Original dimensions: (1387, 3)
Exported dimensions: (1387, 3)
StockCode mapping preserved: True

TRANSACTION TAXONOMY
Original dimensions: (16, 5)
Exported dimensions: (16, 5)
All categories preserved: True

SAVED FILES
C:\Users\Kia\Downloads\DATA-SCIENCE-PORTFOLIO\customer-intelligence-revenue-optimization\data\reference\description_mapping.csv
C:\Users\Kia\Downloads\DATA-SCIENCE-PORTFOLIO\customer-intelligence-revenue-optimization\data\reference\transaction_taxonomy.csv


In [219]:
# Inspect existing notebook objects related to transformation rules

keywords = [
    "mapping",
    "category",
    "classification",
    "stockcode",
    "description",
    "taxonomy"
]

print("RELEVANT DATAFRAME OBJECTS")

for name, obj in globals().copy().items():
    if any(keyword in name.lower() for keyword in keywords):
        if isinstance(obj, pd.DataFrame):
            print(
                f"{name}: shape={obj.shape}, "
                f"columns={obj.columns.tolist()}"
            )

print("\nRELEVANT DICTIONARY OBJECTS")

for name, obj in globals().copy().items():
    if any(keyword in name.lower() for keyword in keywords):
        if isinstance(obj, dict):
            print(f"{name}: {len(obj)} entries")
            print("Sample:", list(obj.items())[:5])

RELEVANT DATAFRAME OBJECTS
missing_description_df: shape=(4382, 9), columns=['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country', 'YearMonth']
stockcode_description_profile: shape=(5305, 3), columns=['total_records', 'known_description_records', 'unique_descriptions']
missing_description_by_stockcode: shape=(2451, 1), columns=['missing_description_records']
missing_description_analysis: shape=(2451, 5), columns=['missing_description_records', 'total_records', 'known_description_records', 'unique_descriptions', 'recovery_category']
description_consistency: shape=(1616, 4), columns=['unique_descriptions', 'first_seen', 'last_seen', 'total_records']
candidate_descriptions: shape=(1616, 5), columns=['known_description', 'known_description_count', 'total_records', 'missing_description_records', 'evidence_category']
description_mapping: shape=(1387, 2), columns=['imputed_description', 'evidence_count']
df_description_cleaning: shape=(1067371, 1

In [220]:
display(
    special_stockcode_profile.sort_values(
        ["transaction_category", "StockCode_normalized"]
    )
)

,StockCode_normalized,Description_normalized,transaction_category,is_cancellation,record_count,unique_invoices,identified_customers,positive_quantity_records,negative_quantity_records,zero_price_records,total_quantity,total_line_value
0,23574,PACKING CHARGE,merchandise_candidate,False,14,14,9,14,0,0,14,105.00
7,TEST001,THIS IS A TEST PRODUCT.,merchandise_candidate,False,9,9,2,9,0,0,50,225.00
1,23574,PACKING CHARGE,unclassified,True,2,2,2,0,2,0,-2,-15.00
3,AMAZONFEE,AMAZON FEE,unclassified,False,4,3,0,4,0,0,4,34009.13
4,AMAZONFEE,AMAZON FEE,unclassified,True,39,33,0,0,39,0,-39,-294772.71
5,S,SAMPLES,unclassified,False,3,3,0,3,0,0,3,136.85
6,S,SAMPLES,unclassified,True,101,34,0,0,101,0,-101,-6202.65
8,TEST001,THIS IS A TEST PRODUCT.,unclassified,True,4,4,3,0,4,0,-5,-22.50
2,23574,PACKING CHARGE,zero_price_activity,False,1,1,0,1,0,1,100,0.00
9,TEST001,THIS IS A TEST PRODUCT.,zero_price_activity,False,2,2,2,2,0,2,10,0.00


In [221]:
keywords = [
    "transaction_category",
    "transaction_taxonomy",
    "StockCode_normalized",
    "Description_normalized"
]

print("CLASSIFICATION-RELATED CODE CELLS\n")

for cell_number, code in enumerate(In):
    if (
        cell_number > 0
        and any(keyword in code for keyword in keywords)
        and (
            "transaction_category" in code
            or "transaction_taxonomy" in code
        )
    ):
        print(f"\n{'=' * 70}")
        print(f"EXECUTION CELL: {cell_number}")
        print(f"{'=' * 70}")
        print(code)

CLASSIFICATION-RELATED CODE CELLS


EXECUTION CELL: 149
# Step 9 — Apply provisional transaction classification

df_classified = df_transaction_review.copy()

df_classified["StockCode_normalized"] = (
    df_classified["StockCode"].astype("string").str.strip().str.upper()
)

delivery_stockcodes = ["POST", "C2", "23444", "DOT"]

# Default classification
df_classified["transaction_category"] = "unclassified"

# 1. Delivery charges
df_classified.loc[
    df_classified["StockCode_normalized"].isin(delivery_stockcodes),
    "transaction_category"
] = "delivery_charge"

# 2. Manual transactions
df_classified.loc[
    df_classified["StockCode_normalized"] == "M",
    "transaction_category"
] = "manual_transaction"

# 3. Discount-related financial transactions
df_classified.loc[
    df_classified["StockCode_normalized"] == "D",
    "transaction_category"
] = "discount_related_transaction"

# 4. Financial adjustments
df_classified.loc[
    df_classified["StockCode_normalized"].isin(["ADJUST", "

In [222]:
import re

patterns = {
    "description cleaning": r"df_description_cleaning\s*=|Description_was_missing",
    "description mapping": r"description_mapping|imputed_description",
    "transaction review": r"df_transaction_review\s*=",
    "cancellation identification": r"is_cancellation\s*=",
    "purchase flags": r"candidate_paid_purchase\s*=|has_positive_line_value",
    "description normalization": r"Description_normalized\s*=",
    "duplicate handling": r"duplicate_mask\s*=|df_clean\s*=",
}

for section, pattern in patterns.items():
    print(f"\n{'=' * 75}")
    print(section.upper())
    print("=" * 75)

    found = False

    for cell_number, code in enumerate(In):
        if re.search(pattern, code):
            found = True
            print(f"\n--- Execution cell {cell_number} ---")
            print(code)

    if not found:
        print("No matching execution cells found.")


DESCRIPTION CLEANING

--- Execution cell 70 ---
# Create a separate dataset for the missing-description transformation

df_description_cleaning = df_work.copy(deep=True)

# Track which descriptions were originally missing
df_description_cleaning["Description_was_missing"] = (
    df_description_cleaning["Description"].isna()
)

# Map approved descriptions using StockCode
mapped_descriptions = (
    df_description_cleaning["StockCode"]
    .map(description_mapping["imputed_description"])
)

# Fill only missing descriptions with approved mappings
imputation_mask = (
    df_description_cleaning["Description"].isna()
    & mapped_descriptions.notna()
)

df_description_cleaning.loc[
    imputation_mask, "Description"
] = mapped_descriptions[imputation_mask]

# Report results
print("Original missing descriptions:", missing_description.sum())
print("Descriptions imputed:", imputation_mask.sum())
print(
    "Remaining missing descriptions:",
    df_description_cleaning["Description"].isna().s

In [223]:
# Retrieve all notebook cells related to description normalization

keywords = [
    "Description_normalized",
    "original_description",
    "normalized_description",
    "description_normalization_review"
]

for cell_number, code in enumerate(In):
    if any(keyword in code for keyword in keywords):
        print(f"\n{'=' * 70}")
        print(f"EXECUTION CELL: {cell_number}")
        print("=" * 70)
        print(code)


EXECUTION CELL: 122
import re

# Work on a separate copy
description_normalization_review = description_variation_records.copy()

# Preserve the original description
description_normalization_review["original_description"] = (
    description_normalization_review["Description"]
)

# Apply conservative text normalization
description_normalization_review["normalized_description"] = (
    description_normalization_review["Description"]
    .astype("string")
    .str.upper()
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

# Compare original and normalized description counts

normalization_comparison = (
    description_normalization_review
    .groupby("StockCode")
    .agg(
        original_description_count=(
            "original_description",
            "nunique"
        ),
        normalized_description_count=(
            "normalized_description",
            "nunique"
        ),
        record_count=("Invoice", "size")
    )
)

normalization_comparison["descriptions_

In [224]:
print("DF_CLEAN")
print("Shape:", df_clean.shape)
print("Columns:")
for i, column in enumerate(df_clean.columns, start=1):
    print(f"{i:02d}. {column} ({df_clean[column].dtype})")

print("\nDF_CLASSIFIED")
print("Shape:", df_classified.shape)
print("Columns:")
for i, column in enumerate(df_classified.columns, start=1):
    print(f"{i:02d}. {column} ({df_classified[column].dtype})")

print("\nTRANSACTION TAXONOMY")
print(transaction_taxonomy.columns.tolist())
display(transaction_taxonomy)

DF_CLEAN
Shape: (1033036, 33)
Columns:
01. Invoice (object)
02. StockCode (object)
03. Description (object)
04. Quantity (int64)
05. InvoiceDate (datetime64[us])
06. Price (float64)
07. Customer ID (float64)
08. Country (str)
09. YearMonth (period[M])
10. Description_was_missing (bool)
11. is_cancellation (bool)
12. has_customer_id (bool)
13. has_positive_quantity (bool)
14. has_positive_price (bool)
15. has_positive_line_value (bool)
16. candidate_paid_purchase (bool)
17. Description_normalized (string)
18. is_zero_price (bool)
19. zero_price_category (str)
20. StockCode_normalized (string)
21. transaction_category (str)
22. line_value (float64)
23. merchandise_revenue_treatment (str)
24. customer_level_eligible (bool)
25. qualifying_purchase (bool)
26. notes (str)
27. is_merchandise_revenue_record (bool)
28. is_customer_analytics_eligible (bool)
29. is_qualifying_purchase (bool)
30. merchandise_revenue (float64)
31. qualifying_purchase_revenue (float64)
32. is_excess_exact_duplicate 

,transaction_category,merchandise_revenue_treatment,customer_level_eligible,qualifying_purchase,notes
0,merchandise_candidate,include_positive,True,True,Identified-customer merchandise transactions
1,anonymous_merchandise_candidate,include_positive,False,False,Merchandise transactions without Customer ID
2,merchandise_cancellation_candidate,include_negative,True,False,Identified-customer merchandise cancellations
3,anonymous_merchandise_cancellation_candidate,include_negative,False,False,Anonymous merchandise cancellations
4,delivery_charge,separate,False,False,Delivery-related charges
5,packing_charge,separate,False,False,Packing-related charges
6,zero_price_activity,exclude,True,False,Zero-price activity; not a paid purchase
7,operational_adjustment_candidate,exclude,False,False,Potential operational or inventory adjustments
8,manual_transaction,separate,False,False,Manual transactions with unclear accounting me...
9,discount_related_transaction,separate,False,False,Discount-related records; accounting treatment...


In [225]:
keywords = [
    "is_zero_price",
    "zero_price_category",
    "YearMonth"
]

for keyword in keywords:
    print(f"\n{'=' * 70}")
    print(f"SEARCHING: {keyword}")
    print("=" * 70)

    found = False

    for cell_number, code in enumerate(In):
        if keyword in code:
            found = True
            print(f"\n--- Execution cell {cell_number} ---")
            print(code)

    if not found:
        print("No matching cells found.")


SEARCHING: is_zero_price

--- Execution cell 125 ---
# Create zero-price classification flags

df_transaction_review["is_zero_price"] = (
    df_transaction_review["Price"] == 0
)

df_transaction_review["zero_price_category"] = "not_zero_price"

zero_price_mask = df_transaction_review["is_zero_price"]

df_transaction_review.loc[
    zero_price_mask & (df_transaction_review["Quantity"] > 0)
    & df_transaction_review["Customer ID"].notna(),
    "zero_price_category"
] = "positive_zero_price_identified_customer"

df_transaction_review.loc[
    zero_price_mask & (df_transaction_review["Quantity"] > 0)
    & df_transaction_review["Customer ID"].isna(),
    "zero_price_category"
] = "positive_zero_price_anonymous"

df_transaction_review.loc[
    zero_price_mask & (df_transaction_review["Quantity"] < 0),
    "zero_price_category"
] = "negative_zero_price"

df_transaction_review.loc[
    zero_price_mask & (df_transaction_review["Quantity"] == 0),
    "zero_price_category"
] = "zero_quantity

In [226]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.transform import transform_online_retail

print("Transformation module imported successfully.")

Transformation module imported successfully.


In [227]:
df_classified_rebuilt, df_clean_rebuilt, df_duplicate_audit_rebuilt = (
    transform_online_retail(df_combined_raw)
)

print("TRANSFORMATION RESULTS")
print("Classified:", df_classified_rebuilt.shape)
print("Cleaned:", df_clean_rebuilt.shape)
print("Duplicate audit:", df_duplicate_audit_rebuilt.shape)

TRANSFORMATION RESULTS
Classified: (1067371, 33)
Cleaned: (1033036, 33)
Duplicate audit: (34335, 33)


In [229]:
import importlib
import src.data.transform as transform_module

# Reload the updated Python module
importlib.reload(transform_module)

# Rebind the function
transform_online_retail = transform_module.transform_online_retail

print("Updated transformation module reloaded.")

Updated transformation module reloaded.


In [230]:
df_classified_rebuilt, df_clean_rebuilt, df_duplicate_audit_rebuilt = (
    transform_online_retail(df_combined_raw)
)

print("Classified:", df_classified_rebuilt.shape)
print("Cleaned:", df_clean_rebuilt.shape)
print("Duplicate audit:", df_duplicate_audit_rebuilt.shape)

print("\nClassified columns:", df_classified_rebuilt.columns.tolist())

Classified: (1067371, 32)
Cleaned: (1033036, 33)
Duplicate audit: (34335, 33)

Classified columns: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country', 'YearMonth', 'Description_was_missing', 'is_cancellation', 'has_customer_id', 'has_positive_quantity', 'has_positive_price', 'has_positive_line_value', 'candidate_paid_purchase', 'Description_normalized', 'is_zero_price', 'zero_price_category', 'StockCode_normalized', 'transaction_category', 'line_value', 'merchandise_revenue_treatment', 'customer_level_eligible', 'qualifying_purchase', 'notes', 'is_merchandise_revenue_record', 'is_customer_analytics_eligible', 'is_qualifying_purchase', 'merchandise_revenue', 'qualifying_purchase_revenue', 'is_excess_exact_duplicate']


In [231]:
# Compare original and rebuilt datasets column by column

datasets_to_compare = {
    "Classified": (df_classified, df_classified_rebuilt),
    "Cleaned": (df_clean, df_clean_rebuilt),
    "Duplicate audit": (
        df_duplicate_audit,
        df_duplicate_audit_rebuilt
    ),
}

for dataset_name, (original, rebuilt) in datasets_to_compare.items():

    print(f"\n{'=' * 65}")
    print(dataset_name.upper())
    print("=" * 65)

    print("Same shape:", original.shape == rebuilt.shape)
    print("Same column order:",
          original.columns.tolist() == rebuilt.columns.tolist())

    if original.columns.tolist() != rebuilt.columns.tolist():
        print("Column mismatch detected.")
        continue

    comparison = pd.DataFrame({
        "column": original.columns,
        "values_match": [
            original[col].reset_index(drop=True).equals(
                rebuilt[col].reset_index(drop=True)
            )
            for col in original.columns
        ],
    })

    print("Matching columns:",
          comparison["values_match"].sum(),
          "/",
          len(comparison))

    print("All values match:",
          comparison["values_match"].all())

    if not comparison["values_match"].all():
        display(comparison[~comparison["values_match"]])


CLASSIFIED
Same shape: True
Same column order: True
Matching columns: 32 / 32
All values match: True

CLEANED
Same shape: True
Same column order: True
Matching columns: 33 / 33
All values match: True

DUPLICATE AUDIT
Same shape: True
Same column order: True
Matching columns: 33 / 33
All values match: True


In [232]:
import importlib
import src.data.validate as validate_module

importlib.reload(validate_module)

validate_online_retail = (
    validate_module.validate_online_retail
)

print("Validation module imported successfully.")

Validation module imported successfully.


In [233]:
validation_report = validate_online_retail(
    raw_df=df_combined_raw,
    df_classified=df_classified_rebuilt,
    df_clean=df_clean_rebuilt,
    df_duplicate_audit=df_duplicate_audit_rebuilt,
)

display(validation_report)

,check,passed,details
0,Required source columns,True,All eight original dataset columns must be pre...
1,Classified schema,True,Expected 32 columns; found 32.
2,Cleaned schema,True,Expected 33 columns; found 33.
3,Duplicate audit schema,True,Expected 33 columns; found 33.
4,Classified record count,True,Source: 1067371; classified: 1067371.
5,Cleaned and audit reconciliation,True,Cleaned: 1033036; audit: 34335; source: 1067371.
6,Duplicate identification,True,Expected excess duplicates: 34335.
7,Unique original row positions,True,Positions must be unique within each output da...
8,Disjoint original row positions,True,Cleaned and audit datasets must not share posi...
9,Complete original row positions,True,Every source row must appear in exactly one ou...


In [234]:
total_checks = len(validation_report)
passed_checks = validation_report["passed"].sum()
failed_checks = total_checks - passed_checks

print("VALIDATION SUMMARY")
print("Total checks:", total_checks)
print("Passed:", passed_checks)
print("Failed:", failed_checks)

if failed_checks == 0:
    print("\nAll validation checks passed.")
else:
    print("\nFailed checks:")
    display(
        validation_report.loc[
            ~validation_report["passed"]
        ]
    )

VALIDATION SUMMARY
Total checks: 29
Passed: 29
Failed: 0

All validation checks passed.


In [235]:
import importlib.util

packages = [
    "sqlalchemy",
    "psycopg",
    "psycopg2",
]

print("DATABASE DEPENDENCY CHECK")

for package in packages:
    available = importlib.util.find_spec(package) is not None
    print(f"{package}: {'Available' if available else 'Not installed'}")

DATABASE DEPENDENCY CHECK
sqlalchemy: Available
psycopg: Not installed
psycopg2: Not installed
